# BE prediction: focused model comparison with full per-row evaluation

V15 retains the same feature combinations, cohort eligibility, five patient-grouped folds,
sampling contrasts (including eICU prevalence matching), historical/current laboratory
representations and separate CNN comparisons. Every evaluated model/configuration retains
an inner MIMIC fit for threshold estimation and an outer full-training-partition refit.
Thresholds are transferred unchanged to held-out MIMIC patients and external eICU patients.
No eICU outcomes select thresholds or feature sets. eICU outcomes inform the explicitly
labelled prevalence-matching arm; that arm uses aggregate target-cohort label prevalence.

## Model schedule and rationale

RF, XGBoost and L2 logistic regression (`SummaryLR` in cache/output identifiers) run on
every grid configuration. This linear comparator uses all requested predictor blocks,
not only labs. DecisionTree and L1 logistic regression are not part of the main sweep.
RF+L2 SoftVote and Stack run on four prespecified primary anchors: current+historical labs
in all_labs and all blocks in icd_history, at both landmarks, using weighted natural
training prevalence (`none`). Ensembles reuse the already-computed base predictions;
Stack adds a small meta-model. Sensitivities retain fixed feature anchors and the same
three primary models, without ensembles or CNN retraining.

`V15_L1_ANCHORS=False` is the default. Enabling it adds L1 only at these same four primary
anchors (48 additional inner/outer fits with five folds and external refit). Its original
objective, convergence requirement and bounded continuation remain unchanged.

L1 is not a prerequisite for the paper's question: whether feature representations and
sampling choices transport across cohorts. It provides sparse coefficient selection,
which is a different objective from accurate prediction. A clinical simulation study
found ridge and elastic net competitive with highly correlated predictors
([2020 study](https://pubmed.ncbi.nlm.nih.gov/32227652/)); another study of low-dimensional,
correlated clinical predictors found instability in data-driven selection, including
Lasso ([2022 study](https://pmc.ncbi.nlm.nih.gov/articles/PMC8751246/)). These settings are
not identical to this sparse lab/ICD design and do not establish that ridge is superior here.
An IE mortality study compared Lasso with RF/SVM/KNN and reported RF as the strongest
discriminator ([2025 study](https://link.springer.com/article/10.1186/s12911-025-03025-4));
its prognosis task differs from the diagnostic-risk task here. The SABIER study developed
an IE risk score from random forests in patients with S. aureus bacteremia
([2024 study](https://academic.oup.com/jid/article/230/3/606/7616139)); that selected
bacteremia population also differs from the present admission-level population. These examples support a
justified comparator choice, not omission of validation. TRIPOD+AI asks for transparent
model specification/evaluation, not a required L1 algorithm
([reporting guidance](https://pmc.ncbi.nlm.nih.gov/articles/PMC11019967/)).

## Metrics and selection

Every complete MIMIC/eICU row contains AUROC, AP, PR_AUC, recall, specificity, PPV, F1,
Brier score, alert rate and both lift measures, with patient-cluster 95% bootstrap CIs.
AP and PR_AUC are identical average-precision values, not two different integrals.
PR_AUC_lift = AP / observed evaluation-cohort prevalence (random-ranking baseline 1).
AUROC_lift = AUROC / 0.5 (random-ranking baseline 1; perfect discrimination 2).
The latter is an explicitly defined descriptive ratio, not a standard clinical utility
measure, and contains no ranking information beyond AUROC. F1 = 2TP/(2TP+FP+FN).
PPV is undefined when there are no positive predictions; F1 is zero when positives
exist but none are predicted. Bootstrap lift recomputes prevalence in each patient-cluster
resample. Missing/failed folds produce incomplete rows with missing metrics, never
apparently valid partial-fold performance. The default operating point remains the
training-only Youden threshold; training-only recall-target thresholds remain available.

Use MIMIC development results for selection and prespecify a recall/specificity/PPV
trade-off rather than maximizing recall alone. Comparisons selected after inspecting
eICU performance are exploratory, not untouched external validation. CIs are per-model
conditional performance intervals, not correction for selecting a winner from many rows.
Only the top 50 complete rows sorted by eICU recall are printed; the entire grid is saved.

## Execution and reuse

V15 writes separate outputs and never modifies V14 checkpoints. Exact-fingerprint,
individually accepted RF/L2/XGBoost stages can be copied from V14; whole-fold/ensemble
outputs are not reused because the model schedule changed. Raw-feature caches remain
available. Reporting is separately fingerprinted, so adding these metrics does not
require retraining accepted models. Real optimizer failures remain explicit and isolated.
This design reduces the default tabular fit count from 6,624 to 4,320, principally by
removing 1,440 L1 fits. It retains 72 separate CNN training stages. No under-24-hour
runtime is claimed without a representative timing benchmark.


In [ ]:
from pathlib import Path
import sys,json,hashlib,marshal,re,time
import numpy as np
import pandas as pd
import sklearn,torch
import matplotlib.pyplot as plt
from IPython.display import display
DATA_ROOT=Path(r'C:\Users\local-researcher\Downloads\Bacterial Endocarditis Prediction Pipeline')
OUTPUT_ROOT=DATA_ROOT/'BE_integrated_grid_v15_outputs'
V15_MAIN_MODELS=('RandomForest','SummaryLR','XGBoost')  # Active V15 grid and sensitivity models.
V15_L1_ANCHORS=False  # Optional four prespecified L1 comparisons; never the full grid.
# The inherited CFG/model declarations below reproduce V14 cache signatures.
# The V15 runtime installs V15_MAIN_MODELS and anchor-only ensembles before any fit.
USE_FEATURE_CACHE=True
RUN_GRID=True
DEFAULT_CONFIG = dict(
    seed=42, chunksize=1_000_000, first_draw_max_hours=24., second_draw_max_hours=48.,
    set_width_hours=1., repeat_min_gap_hours=6., release_lag_hours=4.,
    min_core_analytes=2, min_slope_span_hours=1., history_slope_span_hours=6.,
    cnn_bin_hours=3., cnn_window_hours=54.,
    outer_folds=5, inner_holdout=.2, max_epochs=40, patience=8, batch_size=256,
    width=16, dropout=.1, weight_decay=.001, lr=.001, torch_threads=4,
    n_boot=500, forest_trees=300,
)
CFG=dict(DEFAULT_CONFIG)
CFG.update(
    age_policy='adults',  # Primary: age >=18, known age. Use 'all_ages' for sensitivity.
    grid_profile='compact',  # 'full' enables all 127 non-empty combinations of seven blocks
    seed=42,outer_folds=5,inner_holdout=.2,max_epochs=40,patience=8,n_boot=500,
    first_draw_max_hours=24.,second_draw_max_hours=48.,release_lag_hours=4.,
    include_demographics=True,icd_temporal_budget=25,icd_ever_budget=25,icd_min_patients=2,
    selection_trees=300,forest_trees=300,xgb_trees=500,
    base_models=('RandomForest','Lasso','SummaryLR','XGBoost','DecisionTree'),
    include_ensembles=True,include_cnn=True,
    oversample_ratios=(.01,.1,.2,.3),undersample_ratios=(.05,.1,.2,.3),
    prevalence_targets=('eicu','midpoint'),
)
CFG.update(lab_panel='expanded',include_observation_controls=True,
    recall_targets=(.80,.90,.95),  # Exploratory inner-MIMIC recall targets, not clinical recommendations.
    run_sensitivities=True,sensitivity_lags=(8.,12.),
    all_ages_sensitivity=True,core_panel_sensitivity=True)
RUN_CRP_AUDIT=True  # Source-unit review; never guesses conversions.
OUTPUT_DIR=OUTPUT_ROOT/'primary'
DISPLAY_SCENARIO='primary'  # The displayed table is only this scenario's top 50 by eICU recall.
print({'Python':sys.version.split()[0],'pandas':pd.__version__,'sklearn':sklearn.__version__,'torch':torch.__version__})
print('Research experiment: timing uses an explicit result-availability assumption.')
FEATURE_BATCH_SIZE=250  # Source admissions per atomic checkpoint; does not change model features.
LEGACY_CACHE_DIRS=[OUTPUT_ROOT/'primary',DATA_ROOT/'BE_integrated_grid_v14_outputs'/'primary',
    DATA_ROOT/'BE_integrated_grid_v13_outputs'/'primary',
    DATA_ROOT/'BE_integrated_grid_v12_outputs'/'primary',
    DATA_ROOT/'BE_integrated_grid_v11_outputs'/'primary',
    DATA_ROOT/'BE_integrated_grid_v10_outputs'/'primary',
    DATA_ROOT/'BE_integrated_grid_v9_outputs'/'adults',
    DATA_ROOT/'BE_integrated_grid_v8_outputs'/'adults']

# Sampling-aware CNN; one-hour draw clusters and three-hour tensor bins are distinct.
CFG.update(cnn_dilations=(1,2,4,8),cnn_value_transform='log1p',
    cnn_carry_forward=True,cnn_add_current_summaries=True,cnn_grad_clip=5.)
# Optional finer-bin experiment: set cnn_bin_hours=1. and cnn_dilations=(1,2,4,8,16).
# It requires new raw tensors and CNN fits; it is NOT run automatically.

# Both lab-only CNN comparisons retain the usual age/sex inputs.
CFG.update(cnn_include_demographics=True,summary_semantics="observed_support_v12")

# Focused primary design; full combinatorial profile remains available explicitly.
CFG.update(grid_profile='representative',sensitivity_profile='anchors',
    sensitivity_models=('RandomForest','Lasso','SummaryLR'),transform_cache_entries=16)

# Separate CNN scheduling. Tabular grid and all sensitivities never fit CNNs.
CFG['include_cnn']=False
RUN_CNN=True


## Grid definition and sampling policy

Seven independently selectable blocks: temporal ICD one-hot (`onehot`), category rates (`rates`), code-burden shares (`shares`), category presence (`ever`), non-temporal individual ICD codes (`icd_ever`), current-lab summaries (`lab_current`) and historical-lab summaries (`lab_history`). Current-minus-historical-median features are included when both lab summary blocks are present. Demographics are consistently included by default. Turning them off also removes reference-derived lab flags whose definitions or availability encode age/sex; raw lab slopes and changes remain. This does not remove all biological demographic proxies.

The default representative profile evaluates all combinations of three groups (all ICD blocks, current labs, historical labs), supplemented by single ICD-block baselines and leave-one-ICD-block-out comparisons. These supplementary contrasts use balanced-natural training only. This is not exhaustive testing of every ICD sub-block interaction. The optional compact profile retains its eight prespecified combinations. It uses unweighted, weighted-natural and control-duplication prevalence-match training. The full profile evaluates all 127 combinations and 12 sampling settings in the ICD-history subgroup, plus three lab-only combinations on the broader lab-eligible population. The primary grid uses every configured base model and RF/Lasso ensembles. Sensitivities use six fixed configurations each, with RF, Lasso, SummaryLR and ensembles; no CNNs. The separate all-labs CNN experiment retains age/sex in both models: sequences alone and sequences plus current/historical summaries. Neither receives ICD. Historical information remains summaries, not a historical sequence.

Sampling settings:

- `unweighted`: unchanged training rows, ordinary loss.
- `none`: unchanged training rows, balanced loss.
- `oversample r`: duplicate cases toward case:control ratio r; target prevalence r/(1+r).
- `undersample r`: remove controls toward the same ratio; test rows are unchanged.
- `prevalence_match eicu`: represent duplicated controls toward this population and landmark's observed eICU prevalence.
- `prevalence_match midpoint`: duplicate controls toward the arithmetic midpoint of training-MIMIC and this population and landmark's eICU prevalence.

Resampling follows patient splitting and preprocessing; duplicated rows cannot enter a held-out fold. Resampled modes do not also apply balanced loss. A target that does not require the specified operation leaves rows unchanged; achieved prevalence and duplicate counts are exported. **Prevalence matching intentionally uses an aggregate eICU label statistic and is marked as target-domain-informed sensitivity analysis**, not untouched external validation. Other settings do not use eICU labels for fitting, feature selection or thresholds.

Full-grid runtime can be much longer than an overnight run: adding labs, CNNs, repeated fits and CIs multiplies the work. The plan below prints the exact row count before data processing. Completed folds are checkpointed. Identical fitting inputs, schemas, code and configuration reuse them. CIs for every row run in a separate cached phase: changing n_boot recomputes uncertainty without retraining models. Full-grid work may still take days; the plan reports model-fit counts before execution.
For Lasso and SummaryLR only, repeated training rows are represented by exact integer multiplicity weights, preserving the same logistic objective. Trees and CNNs retain literal sampling. Five-fold evaluation, independent inner calibration and confidence intervals for every row remain enabled. Detailed fit counts and runtime provisions appear in the focused-design section below.


V14 scheduling: the tabular feature/sampling grid excludes CNNs. The separate CNN comparison below uses only all-labs patients; it does not require ICD history. CNN policy sensitivities are not estimated by the tabular-only sensitivity analyses.


## Clinical lab definitions and reference features

The 28-analyte expanded panel, canonical units and source-linked reference intervals are retained from the lab-landmark notebook. Reference intervals are research comparators, not BE-specific cutoffs, treatment targets or verified local laboratory limits. Raw values remain available where an age/sex-specific range is unsupported. Pediatric values are not assigned adult ranges; unknown or top-coded demographics are handled explicitly. The primary analysis uses `age_policy='adults'`: index admissions must have known age >=18 in both datasets and both populations. Under-18 and unknown-age exclusions are counted explicitly. For the all-ages sensitivity analysis, set `age_policy='all_ages'` in the configuration cell and rerun top to bottom. This retains all ages and unknown ages; outputs and checkpoints are kept in separate age-policy folders. Historical data are not age-filtered. Eligibility counts are exported by cohort, landmark and outcome class.

Current summaries include mean, median, extrema, SD/IQR, first/last values, counts, actual-time slopes and changes. Rising/falling, toward/away-range, nearest-boundary and crossing indicators distinguish numerical direction from abnormality. Missing/undefined trends are not encoded as clinical improvement. Historical summaries include observation- and stay-level summaries plus valid within-stay slopes; no continuous eICU cross-hospital timeline is invented. Imputation medians are training-set quantities, distinct from a patient's historical median or a published reference interval.

In [ ]:
"""Prespecified reference-feature metadata, checked 2026-10-10.

Reference intervals are assay/site dependent; these are research comparators,
not local laboratory limits, therapeutic targets or BE diagnostic thresholds.
Adult ranges only unless explicitly stated. Unknown demographics return missing.
Raw values remain available when a reference cannot be assigned.
"""
import numpy as np
import pandas as pd

SOURCES = {
    'CBC': 'https://www.mayocliniclabs.com/test-catalog/overview/9109',
    'CMP': 'https://www.mayocliniclabs.com/api/sitecore/TestCatalog/DownloadTestCatalog?testId=113631',
    'CRP': 'https://www.mayocliniclabs.com/api/sitecore/TestCatalog/DownloadTestCatalog?testId=800164',
    'ESR': 'https://bronsonlab.testcatalog.org/show/ESR-or-LAB322',
    'MG': 'https://www.mayocliniclabs.com/api/sitecore/TestCatalog/DownloadTestCatalog?testId=8448',
    'PHOS': 'https://www.mayocliniclabs.com/api/sitecore/TestCatalog/DownloadTestCatalog?testId=8408',
    'LDH': 'https://cphlab.testcatalog.org/show/LDH',
    'PTT': 'https://www.mayocliniclabs.com/api/sitecore/TestCatalog/DownloadTestCatalog?testId=40935',
    'INR': 'https://marshfieldlabs.testcatalog.org/show/PT',
    'LACTATE': 'https://prd1.mayocliniclabs.com/test-catalog/overview/601685',
}

# analyte: (MIMIC item IDs, canonical unit, broad plausibility min/max, eICU name)
LABS = {
 'hemoglobin': ((51222,), 'g/dl', 0, 50, 'Hgb'),
 'wbc': ((51301,51300), 'k/ul', 0, 1000, 'WBC x 1000'),
 'neutrophils_pct': ((51256,), '%', 0, 100, '-polys'),
 'lymphocytes_pct': ((51244,51245), '%', 0, 100, '-lymphs'),
 'platelet': ((51265,), 'k/ul', 0, 10000, 'platelets x 1000'),
 'creatinine': ((50912,), 'mg/dl', 0, 150, 'creatinine'),
 'bun': ((51006,), 'mg/dl', 0, 300, 'BUN'),
 'albumin': ((50862,), 'g/dl', 0, 10, 'albumin'),
 'crp': ((50889,), 'mg/dl', 0, 100, 'CRP'),
 'bilirubin': ((50885,), 'mg/dl', 0, 150, 'total bilirubin'),
 'lactate': ((50813,), 'mmol/l', 0, 50, 'lactate'),
 'esr': ((51288,), 'mm/hr', 0, 200, 'ESR'),
 'rdw': ((51277,), '%', 0, 100, 'RDW'),
 'mcv': ((51250,), 'fl', 0, 250, 'MCV'),
 'sodium': ((50983,), 'mmol/l', 50, 250, 'sodium'),
 'potassium': ((50971,), 'mmol/l', 0, 20, 'potassium'),
 'chloride': ((50902,), 'mmol/l', 30, 200, 'chloride'),
 'bicarbonate': ((50882,), 'mmol/l', 0, 80, 'bicarbonate'),
 'glucose': ((50931,), 'mg/dl', 0, 2500, 'glucose'),
 'calcium_total': ((50893,), 'mg/dl', 0, 30, 'calcium'),
 'magnesium': ((50960,), 'mg/dl', 0, 20, 'magnesium'),
 'phosphate': ((50970,), 'mg/dl', 0, 30, 'phosphate'),
 'alt': ((50861,), 'u/l', 0, 30000, 'ALT (SGPT)'),
 'ast': ((50878,), 'u/l', 0, 30000, 'AST (SGOT)'),
 'alkaline_phosphatase': ((50863,), 'u/l', 0, 10000, 'alkaline phos.'),
 'ldh': ((50954,), 'u/l', 0, 30000, 'LDH'),
 'inr': ((51237,), 'ratio', 0, 30, 'PT - INR'),
 'ptt': ((51275,), 'sec', 0, 500, 'PTT'),
}

# (male lower/upper, female lower/upper, source). No race correction.
ADULT_RANGES = {
 'hemoglobin': (13.2,16.6,11.6,15.0,'CBC'),
 'wbc': (3.4,9.6,3.4,9.6,'CBC'),
 'platelet': (135,317,157,371,'CBC'),
 'mcv': (78.2,97.9,78.2,97.93,'CBC'),
 'rdw': (11.8,14.5,12.2,16.1,'CBC'),
 'creatinine': (.74,1.35,.59,1.04,'CMP'),
 'bun': (8,24,6,21,'CMP'),
 'albumin': (3.5,5,3.5,5,'CMP'),
 'sodium': (135,145,135,145,'CMP'),
 'potassium': (3.6,5.2,3.6,5.2,'CMP'),
 'chloride': (98,107,98,107,'CMP'),
 'bicarbonate': (22,29,22,29,'CMP'),
 'glucose': (70,140,70,140,'CMP'),  # random, not fasting
 'bilirubin': (0,1.2,0,1.2,'CMP'),
 'alt': (7,55,7,45,'CMP'),
 'ast': (8,48,8,43,'CMP'),
 'alkaline_phosphatase': (40,129,35,104,'CMP'),
 'magnesium': (1.7,2.3,1.7,2.3,'MG'),
 'phosphate': (2.5,4.5,2.5,4.5,'PHOS'),
 'ldh': (122,222,122,222,'LDH'),
 'lactate': (.5,2,.5,2,'LACTATE'),
 'inr': (.9,1.1,.9,1.1,'INR'),  # NOT an anticoagulation treatment target
 'ptt': (25,37,25,37,'PTT'),
}

def reference_interval(analyte, age, sex, age_topcoded=False):
    """Return lower/upper; no fabricated pediatric/unknown-sex intervals."""
    if analyte == 'crp':
        return 0., .5  # <5 mg/L, expressed in pipeline mg/dL; all ages
    if not np.isfinite(age) or age < 18:
        return np.nan, np.nan
    if analyte == 'esr':
        if sex not in ('M','F'): return np.nan,np.nan
        return 0., (15 if age < 50 else 20) if sex == 'M' else (20 if age < 50 else 30)
    if analyte == 'calcium_total':
        if age_topcoded: return np.nan,np.nan  # cannot distinguish age 90 vs >90
        return (8.6,10.) if age < 60 else (8.8,10.2) if age <= 90 else (8.2,9.6)
    r = ADULT_RANGES.get(analyte)
    if r is None: return np.nan,np.nan  # percentage differential: no transplanted absolute-count ranges
    if analyte == 'alkaline_phosphatase' and age < 19 and sex == 'M': return 55.,149.
    if analyte in ('bilirubin','ldh') and age == 18: return np.nan,np.nan  # source age-18 boundary unclear
    if sex == 'M': return r[0],r[1]
    if sex == 'F': return r[2],r[3]
    return (r[0],r[1]) if r[:2] == r[2:4] else (np.nan,np.nan)

def boundary_features(first, last, hours, low, high, upper_exclusive=False):
    """Separate raw direction, boundary approach and abnormal-distance change.

    'toward_boundary' is approaching the closest boundary FROM INSIDE normal.
    'toward_range' is decreasing distance FROM OUTSIDE the interval.
    Two binary indicators + explicit validity/stability prevent undefined=improving.
    No clinical improvement claim (e.g. anticoagulated INR or treatment response).
    """
    names = ('reference_known','below','above','distance_outside','toward_boundary',
             'away_boundary','toward_range','away_range','stable_distance','trend_valid',
             'rising','falling','crossed_boundary')
    out = dict.fromkeys(names, np.nan)
    out['reference_known'] = float(np.isfinite(low) and np.isfinite(high) and high > low)
    out['trend_valid'] = 0.
    if not out['reference_known'] or not np.isfinite(last): return out
    width=high-low
    dist=lambda x:max(low-x,0.,x-high)/width
    out.update(below=float(last < low),above=float(last >= high if upper_exclusive else last > high),distance_outside=dist(last))
    if not np.isfinite(first) or not np.isfinite(hours) or hours <= 0: return out
    delta=dist(last)-dist(first)
    out.update(toward_range=float(delta < -1e-8),away_range=float(delta > 1e-8),
               stable_distance=float(abs(delta)<=1e-8),trend_valid=1.,
               rising=float(last>first),falling=float(last<first),
               crossed_boundary=float((first<low)!=(last<low) or
                   ((first>=high)!=(last>=high) if upper_exclusive else (first>high)!=(last>high))))
    if low <= first and (first < high if upper_exclusive else first <= high):
        # Distance to the nearest boundary at baseline; crossing is also reported.
        boundary=low if first-low < high-first else high
        out['toward_boundary']=float(abs(last-boundary)<abs(first-boundary)-1e-8)
        out['away_boundary']=float(abs(last-boundary)>abs(first-boundary)+1e-8)
    return out

def reference_table():
    rows=[]
    for a in LABS:
        for age in (18,40,65,91):
            for sex in ('M','F','U'):
                lo,hi=reference_interval(a,age,sex)
                src=ADULT_RANGES.get(a, (None,None,None,None,
                      'ESR' if a=='esr' else 'CMP' if a=='calcium_total' else 'CRP' if a=='crp' else None))[-1]
                rows.append(dict(analyte=a,age=age,sex=sex,unit=LABS[a][1],lower=lo,upper=hi,
                                 source=SOURCES.get(src,''),checked='2026-10-10'))
    return pd.DataFrame(rows)

## Cohort and time alignment

Cases are hospital-level ICD-9 421.0/421.1/421.9, with corresponding eICU I33-family entries also recognized. Code-negative ENDOCARD-text admissions are excluded as uncertain controls. This harmonizes the intent of case definitions but does not establish identical coding sensitivity between databases or identify infection onset.

First/second sets are one-hour draw clusters containing at least two core results available by the scoring cutoff, starting within 24/48 hours; the repeat cluster is at least six hours after the first. The configurable four-hour release lag is an assumption because MIMIC does not supply a reliable result-release timestamp. eICU additionally respects revision-entry time. Late results and future admissions cannot enter an earlier prediction. eICU's first recorded hospital-relative set may not be the actual first draw at that hospital.

MIMIC history requires completion before current admission; timestamp-valid outpatient labs can contribute. eICU histories require strictly earlier discharge years. Separate same-year hospitalizations are not ordered using their IDs. For ICD temporal features, tied historical order causes the temporal block to remain absent, while valid non-temporal/category information is retained and the condition is audited. No control pseudo-index/history matching is applied to this actual-admission task. Lab-only analyses retain patients without history; ICD-containing analyses require a usable non-BE diagnosis in an eligible prior hospitalization, before feature ranking.

Within each population and landmark, all combinations use identical evaluation patients. Lab-only comparators are refitted in the ICD-history subgroup. Index-admission eICU lab sets are allowed; separate same-year hospitalizations are not historical data. Insufficient-class populations/landmarks are explicitly skipped and reported, never replaced by a broader cohort. First and second landmarks have distinct eligible populations; paired update metrics additionally restrict evaluation to identical repeat-tested admissions.

In [ ]:
"""Admission-level BE diagnostic-risk landmarks; no current discharge predictors.

All eligible admissions, including patients with no history, are retained.
Patients (not admissions) are separated during evaluation. eICU histories use
strictly earlier discharge years; no inferred order of distinct same-year stays.
"""
from pathlib import Path
from collections import Counter
import json
import re
import numpy as np
import pandas as pd

# Defaults configured near the top.
CORE = ('hemoglobin','wbc','platelet','sodium','potassium','creatinine','bun','glucose')
BE9={'4210','4211','4219'}

def locate(root, *candidates):
    for candidate in candidates:
        p=Path(root)/candidate
        if p.exists(): return p
    raise FileNotFoundError(f'Missing input: one of {candidates} under {root}')

def load_admissions(root):
    """Outcome is a code-defined BE hospitalization, not proven incident onset."""
    root=Path(root)
    ma=pd.read_csv(root/'ADMISSIONS.csv',usecols=['SUBJECT_ID','HADM_ID','ADMITTIME','DISCHTIME','DIAGNOSIS'])
    mp=pd.read_csv(root/'PATIENTS.csv',usecols=['SUBJECT_ID','GENDER','DOB'])
    dx=pd.read_csv(root/'DIAGNOSES_ICD.csv',usecols=['HADM_ID','ICD9_CODE'],dtype={'ICD9_CODE':str})
    be=set(dx.loc[dx.ICD9_CODE.str.replace('.','',regex=False).isin(BE9),'HADM_ID'])
    ma=ma.merge(mp,on='SUBJECT_ID',validate='many_to_one')
    for c in ('ADMITTIME','DISCHTIME','DOB'): ma[c]=pd.to_datetime(ma[c],errors='coerce')
    # datetime subtraction can overflow for MIMIC's deidentified age >300: use year arithmetic.
    age=(ma.ADMITTIME.dt.year-ma.DOB.dt.year)-((ma.ADMITTIME.dt.month*100+ma.ADMITTIME.dt.day)<(ma.DOB.dt.month*100+ma.DOB.dt.day))
    ma['age_topcoded']=age.gt(89)
    ma['age']=age.mask(age.gt(89),90.).mask(age.lt(0))
    ma['label']=ma.HADM_ID.isin(be).astype(int)
    ma['valid_time']=ma.ADMITTIME.notna()&ma.DISCHTIME.gt(ma.ADMITTIME)
    ma['eligible']=ma.valid_time.copy()
    ma['eligible'] &= ~(ma.label.eq(0)&ma.DIAGNOSIS.fillna('').str.contains('ENDOCARD',case=False))
    m=pd.DataFrame(dict(patient_id=ma.SUBJECT_ID.astype(str),hospital_id=ma.HADM_ID.astype(int),
        start=ma.ADMITTIME,end=ma.DISCHTIME,year=ma.DISCHTIME.dt.year,age=ma.age,
        age_topcoded=ma.age_topcoded,sex=ma.GENDER.where(ma.GENDER.isin(['M','F']),'U'),
        label=ma.label,eligible=ma.eligible,valid_time=ma.valid_time))
    ep=pd.read_csv(locate(root,'eicu/patients.csv','eicu/patient.csv','patients.csv','patient.csv'),dtype={'uniquepid':str})
    ed=pd.read_csv(locate(root,'eicu/diagnoses.csv','eicu/diagnosis.csv','diagnoses.csv','diagnosis.csv'),
                   usecols=['patientunitstayid','icd9code','diagnosisstring'],dtype={'icd9code':str})
    unit=ep.set_index('patientunitstayid')
    def is_be(s):
        return any(t in BE9 or t.startswith('I33') for t in re.split(r'[,;\s]+',str(s).upper().replace('.','')))
    ed['hid']=ed.patientunitstayid.map(unit.patienthealthsystemstayid)
    eb=set(ed.loc[ed.icd9code.map(is_be),'hid'].dropna().astype(int))
    text=set(ed.loc[ed.diagnosisstring.fillna('').str.contains('ENDOCARD',case=False),'hid'].dropna().astype(int))
    for c in ('hospitaladmitoffset','hospitaldischargeoffset','hospitaldischargeyear'):
        ep[c]=pd.to_numeric(ep[c],errors='coerce')
    ep['duration']=(ep.hospitaldischargeoffset-ep.hospitaladmitoffset)/60
    ep['age_num']=pd.to_numeric(ep.age.astype(str).str.replace('>','',regex=False),errors='coerce')
    ep['sex_code']=ep.gender.str.upper().map({'MALE':'M','FEMALE':'F'})
    ep['age_topcoded']=ep.age.astype(str).str.contains('>')
    g=ep.groupby('patienthealthsystemstayid',sort=False).agg(
        patient_id=('uniquepid','first'),patient_count=('uniquepid','nunique'),
        duration=('duration','min'),duration_max=('duration','max'),
        year=('hospitaldischargeyear','first'),year_count=('hospitaldischargeyear','nunique'),
        age=('age_num','max'),age_topcoded=('age_topcoded','max'),
        sex=('sex_code','first'),sex_count=('sex_code','nunique')).reset_index()
    if g.patient_count.ne(1).any():raise ValueError('Hospital assigned to missing/multiple eICU patients')
    g=g.rename(columns={'patienthealthsystemstayid':'hospital_id'})
    g['patient_id']=g.patient_id.astype(str);g['age']=g.age.mask(g.age_topcoded,90.)
    g['sex']=g.sex.where(g.sex_count.eq(1),'U');g['label']=g.hospital_id.isin(eb).astype(int)
    g['valid_time']=g.duration.gt(0)&(g.duration_max-g.duration).le(1/60+1e-8)&g.year_count.eq(1)
    g['eligible']=g.valid_time&~(g.hospital_id.isin(text)&g.label.eq(0))
    e=g[['patient_id','hospital_id','duration','year','age','age_topcoded','sex','label','eligible','valid_time']].copy()
    if m.hospital_id.duplicated().any() or e.hospital_id.duplicated().any(): raise ValueError('Repeated hospital key')
    return m,e,ep

def clean_labs(z,audit):
    z=z.copy()
    u=z.unit.astype('string').str.lower().str.replace(r'\s+','',regex=True)
    u=u.replace({'k/mcl':'k/ul','k/mm3':'k/ul','k/mmcu':'k/ul','10^3/ul':'k/ul'})
    mono=z.analyte.isin(['sodium','potassium','chloride','bicarbonate'])
    u=u.mask(mono & u.eq('meq/l').fillna(False),'mmol/l')
    enzymes=z.analyte.isin(['alt','ast','alkaline_phosphatase','ldh'])
    u=u.mask(enzymes & u.isin(['iu/l','units/l']),'u/l')
    if 'ITEMID' in z:
        u=u.mask(z.ITEMID.eq(51237)&(u.isna()|u.eq('')),'ratio')
    x=pd.to_numeric(z.value,errors='coerce')
    crp=z.analyte.eq('crp')&u.eq('mg/l').fillna(False)
    x=x.mask(crp,x/10);u=u.mask(crp,'mg/dl')
    expected=z.analyte.map({k:v[1] for k,v in LABS.items()})
    good=u.eq(expected).fillna(False)&np.isfinite(x)
    plausible=x.ge(z.analyte.map({k:v[2] for k,v in LABS.items()}))&x.le(z.analyte.map({k:v[3] for k,v in LABS.items()}))
    audit['unit_or_nonnumeric_excluded']+=int((~good).sum())
    audit['implausible_excluded']+=int((good&~plausible).sum())
    z['value']=x
    return z[good&plausible].copy()

def extract_labs(root,m,e,ep,cfg):
    """Chunked reads. No cohort freezes or previously selected feature CSVs."""
    audit=Counter();parts=[]; mp=set(m.patient_id)
    itemmap={i:a for a,s in LABS.items() for i in s[0]}
    # Retain full histories only where a later hospitalization can use them.
    # Single/latest admissions need only the prespecified current draw window.
    latest=m.groupby('patient_id').start.max()
    old_m=set(m.loc[m.end.lt(m.patient_id.map(latest))&m.valid_time,'hospital_id'])
    starts=m.set_index('hospital_id').start
    draw_limit=cfg['second_draw_max_hours']+cfg['set_width_hours']
    for chunk in pd.read_csv(Path(root)/'LABEVENTS.csv',chunksize=cfg['chunksize'],
          usecols=['SUBJECT_ID','HADM_ID','ITEMID','CHARTTIME','VALUENUM','VALUEUOM']):
        z=chunk[chunk.ITEMID.isin(itemmap)].copy()
        z['patient_id']=z.SUBJECT_ID.astype(str);z=z[z.patient_id.isin(mp)]
        if z.empty: continue
        z['analyte']=z.ITEMID.map(itemmap);z['time']=pd.to_datetime(z.CHARTTIME,errors='coerce')
        start=z.HADM_ID.map(starts)
        retain=z.HADM_ID.isin(old_m)|(z.time.ge(start)&z.time.lt(start+pd.to_timedelta(draw_limit,unit='h')))
        retain|=z.HADM_ID.isna()&z.time.lt(z.patient_id.map(latest))
        z=z[retain]
        z=z.rename(columns={'HADM_ID':'hospital_id','VALUENUM':'value','VALUEUOM':'unit'})
        z=clean_labs(z,audit)
        z['available']=z.time+pd.to_timedelta(cfg['release_lag_hours'],unit='h')
        # NULL HADM outpatient labs are retained for timestamp-valid prior history.
        parts.append(z[['patient_id','hospital_id','analyte','time','available','value']].dropna(subset=['time']))
    ml=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(columns=['patient_id','hospital_id','analyte','time','available','value'])
    ml=resolve_revisions(ml,['patient_id','analyte','time'],audit)
    ma=dict(audit);audit=Counter();parts=[]
    ep=ep[ep.uniquepid.astype(str).isin(set(e.patient_id))].drop_duplicates('patientunitstayid').set_index('patientunitstayid')
    latest_e=e.groupby('patient_id').year.max()
    old_e=set(e.loc[e.year.lt(e.patient_id.map(latest_e))&e.valid_time,'hospital_id'])
    names={s[4]:a for a,s in LABS.items()}
    for chunk in pd.read_csv(locate(root,'eicu/lab.csv','lab.csv'),chunksize=cfg['chunksize'],
            usecols=['patientunitstayid','labname','labresultoffset','labresultrevisedoffset','labresult','labmeasurenamesystem','labmeasurenameinterface']):
        z=chunk[chunk.patientunitstayid.isin(ep.index)&chunk.labname.isin(names)].copy()
        if z.empty:continue
        z['patient_id']=z.patientunitstayid.map(ep.uniquepid).astype(str)
        z['hospital_id']=z.patientunitstayid.map(ep.patienthealthsystemstayid).astype(int)
        off=z.patientunitstayid.map(ep.hospitaladmitoffset)
        z['time']=(pd.to_numeric(z.labresultoffset,errors='coerce')-off)/60
        revised=(pd.to_numeric(z.labresultrevisedoffset,errors='coerce')-off)/60
        # Same prespecified minimum lag as MIMIC + conservative revision availability.
        z['available']=np.maximum(z.time+cfg['release_lag_hours'],revised)
        z['year']=z.patientunitstayid.map(ep.hospitaldischargeyear)
        z=z[z.hospital_id.isin(old_e)|z.time.lt(draw_limit)].copy()
        z['analyte']=z.labname.map(names)
        z=z.rename(columns={'labresult':'value','labmeasurenamesystem':'unit'})
        interface=z.labmeasurenameinterface.astype('string').str.lower().str.replace(r'\s+','',regex=True)
        sys=z.unit.astype('string').str.lower().str.replace(r'\s+','',regex=True)
        conflict=(z.analyte.eq('crp')&sys.isin(['mg/l','mg/dl'])&interface.isin(['mg/l','mg/dl'])&sys.ne(interface)).fillna(False)
        conflict |= ((z.analyte.isin(['magnesium','calcium_total'])&interface.isin(['meq/l','mmol/l']))|
          (z.analyte.eq('rdw')&interface.isin(['fl','femtoliter','femtolitre']))|
          (z.analyte.isin(['neutrophils_pct','lymphocytes_pct'])&interface.isin(['1000/ul','1000/µl','x10']))|
          (z.analyte.isin(['sodium','potassium','chloride','bicarbonate'])&interface.isin(['meq/ml','mmol/ml']))|
          (z.analyte.isin(['alt','ast','alkaline_phosphatase','ldh'])&interface.isin(['unit/ml','units/ml','u/ml','iu/ml']))|
          (z.analyte.eq('ptt')&interface.str.fullmatch(r'\d+(\.\d+)?').fillna(False))).fillna(False)
        audit['conflicting_units_excluded']+=int(conflict.sum())
        z=clean_labs(z[~conflict],audit)
        z=z[z.time.ge(0)&np.isfinite(z.available)&z.available.ge(z.time)]
        # Keep versions until the particular prediction cutoff is known.
        parts.append(z[['patient_id','hospital_id','analyte','time','available','year','value']])
    el=pd.concat(parts,ignore_index=True).drop_duplicates() if parts else pd.DataFrame(columns=['patient_id','hospital_id','analyte','time','available','year','value'])
    return ml,el,{'MIMIC':ma,'eICU':dict(audit)}

def resolve_revisions(z,keys,audit=None):
    if z.empty:return z.copy()
    z=z[z.available.eq(z.groupby(keys,dropna=False).available.transform('max'))].copy()
    conflict=z.groupby(keys,dropna=False).value.transform('nunique').gt(1)
    if audit is not None:audit['conflicting_latest_values']+=int(conflict.sum())
    return z[~conflict].drop_duplicates(keys).copy()

def draw_landmarks(z,cfg):
    """Fixed-width draw clusters anchored to first recorded core-panel time.

    Not a claim of accession identity. Second set is a new qualifying cluster,
    at least repeat_min_gap_hours after first, not a revision or adjacent analyte.
    Trigger eligibility depends on recorded test times, not values or labels.
    """
    if z.empty:return []
    times=np.sort(z.loc[z.analyte.isin(CORE),'hour'].dropna().unique())
    sets=[];i=0
    while i<len(times):
        t=float(times[i]);end=t+cfg['set_width_hours']
        g=z[z.hour.ge(t)&z.hour.lt(end)&z.analyte.isin(CORE)]
        if 'release_hour' in g:
            g=g[g.release_hour.le(end+cfg['release_lag_hours'])]
            g=resolve_revisions(g,['analyte','hour'])
        if g.analyte.nunique()>=cfg['min_core_analytes']:
            if not sets and t<=cfg['first_draw_max_hours']:sets.append((t,end))
            elif sets and t>=sets[0][0]+cfg['repeat_min_gap_hours'] and t<=cfg['second_draw_max_hours']:
                sets.append((t,end));break
        i=int(np.searchsorted(times,end,side='left'))
    return sets

def series_stats(z,min_span):
    """Observation-weighted means/medians; actual-time slopes, never interpolated."""
    d={k:np.nan for k in ('mean','median','min','max','sd','iqr','first','last','slope_per_day','change','span_hours')}
    d.update(count=0.,slope_valid=0.)
    if z.empty:return d
    z=z.groupby('hour',as_index=False).value.mean().sort_values('hour')
    x=z.value.to_numpy(float);t=z.hour.to_numpy(float);span=t[-1]-t[0]
    d.update(mean=x.mean(),median=np.median(x),min=x.min(),max=x.max(),sd=x.std(),
             iqr=np.quantile(x,.75)-np.quantile(x,.25),first=x[0],last=x[-1],
             count=float(len(x)),span_hours=float(span))
    if len(x)>=2 and span>=min_span:
        centered=t-t.mean();d['slope_per_day']=float(np.dot(centered,x-x.mean())/np.dot(centered,centered)*24)
        d['change']=float(x[-1]-x[0]);d['slope_valid']=1.
    return d

def build_features(admissions,long,cohort,cfg):
    """History summaries do not invent between-stay time gaps in eICU."""
    current={int(h):g for h,g in long.dropna(subset=['hospital_id']).groupby('hospital_id',sort=False)}
    patients={str(p):g for p,g in long.groupby('patient_id',sort=False)}
    stays={str(p):g for p,g in admissions.groupby('patient_id',sort=False)}
    names=list(LABS);n_bins=int(np.ceil(cfg['cnn_window_hours']/cfg['cnn_bin_hours']))
    rows=[];metadata_rows=[];feature_names=None;matrices=[];valids=[];event_times=[];cutoffs=[];flow=Counter()
    for r in admissions.itertuples(index=False):
        flow['source_admissions']+=1
        if not r.eligible:flow['ineligible_admissions']+=1;continue
        z=current.get(int(r.hospital_id),pd.DataFrame())
        if z.empty:flow['no_current_labs']+=1;continue
        z=z.copy()
        z['hour']=(z.time-r.start).dt.total_seconds()/3600 if cohort=='MIMIC' else z.time
        z['release_hour']=(z.available-r.start).dt.total_seconds()/3600 if cohort=='MIMIC' else z.available
        duration=(r.end-r.start).total_seconds()/3600 if cohort=='MIMIC' else r.duration
        z=z[z.hour.ge(0)&z.hour.le(duration)]
        sets=draw_landmarks(z,cfg)
        if not sets:flow['no_qualifying_first_set']+=1;continue
        ph=patients[str(r.patient_id)]
        if cohort=='MIMIC':
            earlier=stays[str(r.patient_id)]
            earlier_ids=earlier.loc[earlier.end.lt(r.start)&earlier.start.lt(r.start)&earlier.valid_time,'hospital_id']
            hist=ph[(ph.hospital_id.isin(earlier_ids)|ph.hospital_id.isna())&ph.available.lt(r.start)&ph.time.lt(r.start)].copy()
            hist['hour']=(hist.time-pd.Timestamp('1970-01-01')).dt.total_seconds()/3600
            hist['episode']=hist.hospital_id.astype('string').fillna('outpatient_'+hist.time.dt.strftime('%Y-%m-%d'))
        else:
            earlier=stays[str(r.patient_id)]
            earlier_ids=earlier.loc[earlier.year.lt(r.year)&earlier.valid_time,'hospital_id']
            hist=ph[ph.hospital_id.isin(earlier_ids)].copy()
            ends=earlier.set_index('hospital_id').duration
            hist=hist[hist.available.le(hist.hospital_id.map(ends))].copy()
            hist=resolve_revisions(hist,['hospital_id','analyte','time'])
            hist['hour']=hist.time;hist['episode']=hist.hospital_id.astype(str)
        hd={};hs={}; history_groups=dict(tuple(hist.groupby('analyte')))
        for a in names:
            az=history_groups.get(a,hist.iloc[:0])
            # No pooled first/last/slope across uncertain historical stay order.
            vals=az.value.to_numpy(float)
            h={k:np.nan for k in ('mean','median','min','max','iqr','stay_mean','within_stay_slope_mean','within_stay_slope_median')}
            h.update(count=float(len(vals)),stays=float(az.episode.nunique()))
            if len(vals):
                h.update(mean=vals.mean(),median=np.median(vals),min=vals.min(),max=vals.max(),iqr=np.quantile(vals,.75)-np.quantile(vals,.25))
                ss=[series_stats(g,cfg['history_slope_span_hours']) for _,g in az.groupby('episode')]
                h['stay_mean']=float(np.mean([s['mean'] for s in ss]))
                slopes=[s['slope_per_day'] for s in ss if s['slope_valid']]
                if slopes:h.update(within_stay_slope_mean=float(np.mean(slopes)),within_stay_slope_median=float(np.median(slopes)))
            hd.update({f'history__{a}__{k}':v for k,v in h.items()});hs[a]=h
        for landmark,(draw_start,draw_end) in enumerate(sets,1):
            score_time=draw_end+cfg['release_lag_hours']
            if score_time>duration:flow[f'landmark_{landmark}_after_discharge']+=1;continue
            visible=z[z.hour.lt(draw_end)&z.release_hour.le(score_time)].copy()
            visible=resolve_revisions(visible,['hospital_id','analyte','hour'])
            features=dict(hd)
            features.update(age=r.age,age_missing=float(not np.isfinite(r.age)),sex_male=float(r.sex=='M'),
                            sex_unknown=float(r.sex not in ('M','F')),age_topcoded=float(r.age_topcoded),
                            current__elapsed_hours=score_time,history__available=float(len(hist)>0))
            seq=np.full((n_bins,len(names)),np.nan,np.float32)
            seq_time=np.full_like(seq,np.nan)
            valid=(np.arange(n_bins)*cfg['cnn_bin_hours']<draw_end).astype(np.float32)
            visible_groups=dict(tuple(visible.groupby('analyte')))
            for j,a in enumerate(names):
                az=visible_groups.get(a,visible.iloc[:0]);s=series_stats(az,cfg['min_slope_span_hours'])
                features.update({f'current__{a}__{k}':v for k,v in s.items()})
                lo,hi=reference_interval(a,r.age,r.sex,r.age_topcoded)
                bf=boundary_features(s['first'],s['last'],s['span_hours'] if s['slope_valid'] else np.nan,lo,hi, a in ('crp','esr'))
                features.update({f'current__{a}__{k}':v for k,v in bf.items()})
                features[f'combined__{a}__delta_history_median']=s['last']-hs[a]['median']
                # Baseline-to-current is a difference, NOT a rate without a known time gap.
                for q in az.sort_values('hour').itertuples():
                    b=int(q.hour//cfg['cnn_bin_hours'])
                    if 0<=b<n_bins:seq[b,j]=q.value;seq_time[b,j]=q.hour
            # Avoid retaining a thousand freshly allocated string keys per admission.
            # Fixed-schema float arrays markedly reduce full-cohort memory use.
            if feature_names is None:feature_names=list(features)
            if list(features)!=feature_names:raise AssertionError('Inconsistent feature schema')
            rows.append(np.fromiter(features.values(),dtype=np.float32,count=len(features)))
            metadata_rows.append(dict(patient_id=str(r.patient_id),hospital_id=int(r.hospital_id),label=int(r.label),
                                      landmark=landmark,score_hour=score_time))
            matrices.append(seq);valids.append(valid);event_times.append(seq_time);cutoffs.append(draw_end)
            flow[f'landmark_{landmark}_rows']+=1
    if not rows:raise ValueError(f'{cohort}: no eligible landmark rows')
    frame=pd.concat([pd.DataFrame(metadata_rows),pd.DataFrame(np.stack(rows),columns=feature_names)],axis=1)
    return frame,dict(values=np.stack(matrices),valid=np.stack(valids),
                      hours=np.stack(event_times),cutoff=np.array(cutoffs)),dict(flow)

## ICD vocabulary and crosswalk

The fixed category definitions and mixed-code resolver below are carried from v5. The CMS diagnosis dictionary and `icdmappings` package are existing dependencies. Numeric ICD-9 tokens are validated against the dictionary; mixed ICD-10 codes are mapped before aggregation. Ambiguous undotted E/V codes are excluded and counted. GEM is a single-equivalent crosswalk here, not a lossless/combination-aware clinical ontology.

Diagnosis codes are deduplicated per historical hospitalization and BE outcome codes are scrubbed. Rates divide by eligible historical hospitalizations; shares divide by total coded burden. These summaries are not guaranteed invariant to different coding practices. The temporal cap remains five. Individual-code presence has its own training-only support filter and ranking, independent of temporal selection.

In [ ]:
CARDIAC_PREFIXES = ["39891", "40201", "40211", "40291", "40401", "40403", "40411", "40413", "40491", "40493", "4254", "4255", "4257", "4258", "4259", "428", "4280", "4281", "4282", "42820", "42821", "42822", "42823", "42830", "42831", "42832", "42833", "42840", "42841", "42842", "42843", "4289", "4260", "42610", "42611", "42612", "42613", "4267", "42650", "42651", "42652", "42653", "42654", "4269", "4270", "4271", "4272", "42731", "42732", "4273", "42741", "42742", "4275", "4276", "42760", "42761", "42769", "42781", "42789", "4279", "7850", "7851"]

VASCULAR_PREFIXES = ["0930", "4373", "440", "4400", "4401", "44020", "44021", "44022", "44023", "44024", "44029", "44030", "44031", "44032", "4408", "4409", "441", "4410", "4411", "4412", "4413", "4414", "4415", "4416", "4417", "4419", "4421", "4423", "44281", "44282", "44283", "44284", "44289", "4429", "4431", "4432", "4433", "44381", "44389", "4439", "4471", "5571", "5579", "V434", "4150", "41511", "41512", "41513", "41519", "416", "4160", "4161", "4162", "4168", "4169", "4170", "4178", "4179"]

DIABETES_COMP_PREFIXES = ["2504", "25040", "25041", "25042", "25043", "2505", "25050", "25051", "25052", "25053", "2506", "25060", "25061", "25062", "25063", "2507", "25070", "25071", "25072", "25073", "2508", "25080", "25081", "25082", "25083", "2509", "25090", "25091", "25092", "25093", "3572", "36201", "36202", "36203", "36204", "36205", "36206", "36641"]

LIVER_PREFIXES = ["07022", "07023", "07032", "07033", "07044", "07054", "0706", "0709", "4560", "4561", "45620", "45621", "570", "571", "5710", "5711", "5712", "5713", "5714", "57140", "57141", "57142", "57149", "5715", "5716", "5718", "5719", "5723", "5724", "5728", "V427"]

VALVULAR_PREFIXES = ["394", "395", "396", "397", "4240", "4241", "4242", "4243", "42490", "42491", "42499"]

CONGENITAL_PREFIXES = ["745", "746", "747"]

RENAL_ACCESS_PREFIXES = ["585", "5851", "5852", "5853", "5854", "5855", "5856", "5859", "586", "V451", "V4511", "V4512", "V56", "V560", "V561", "V562", "V568", "40301", "40311", "40391", "40402", "40403", "40412", "40413", "40492", "40493", "99673", "99674", "99656"]

DEVICE_PREFIXES = ["V450", "V4500", "V4501", "V4502", "V4509", "V533", "V5331", "V5332", "V5339", "99601", "99602", "99604", "9961", "99661", "99662", "99671", "99672", "99931", "99932", "99933"]

PROSTHETIC_VALVE_DX_PREFIXES = ["V422", "V433", "99602", "99661", "99671"]

IVDU_PREFIXES = ["3040", "30400", "30401", "30402", "30403", "3041", "3042", "3044", "3045", "3047", "3048", "3049", "3055", "30550", "30551", "30552", "30553", "3056", "30560", "3057", "3058", "3059", "6483", "6484", "65550", "V6542"]

In [ ]:
# Mixed eICU diagnosis tokens retain decimal placement until the coding system is resolved.
# ICD-9 E834.2 and ICD-10 E83.42 have different meanings despite identical compact strings.
import re as _xw_re
from functools import lru_cache as _xw_lru
from icdmappings import Mapper as _XW_Mapper
_XW_MAPPER = _XW_Mapper()
_XW_OK = True
_XW_KNOWN_ICD9 = None

@_xw_lru(maxsize=None)
def _xw_icd10_to_icd9(code):
    result = _XW_MAPPER.map(code, source="icd10", target="icd9")
    return str(result or "").replace(".", "").upper()

def classify_mixed_icd(raw):
    """Return (compact ICD-9 or empty, provenance); never guess a known E/V collision.

    Dotted E/V tokens identify the coding system by decimal position. Undotted E/V
    tokens valid in both dictionaries are excluded and counted as ambiguous. GEM is
    a single-equivalent mapping here, not a lossless or combination-aware crosswalk.
    """
    global _XW_KNOWN_ICD9
    token = str(raw or "").strip().upper()
    c = token.replace(".", "")
    if not c or c in {"NAN", "NONE"}:
        return "", "missing"
    if _XW_KNOWN_ICD9 is None:
        descriptions = globals().get("mimic_tables", {}).get("ICD_9_DX_DESCRIPTIONS")
        if descriptions is None:
            raise RuntimeError("Load the CMS ICD-9 diagnosis dictionary before crosswalking")
        _XW_KNOWN_ICD9 = set(descriptions["DIAGNOSIS CODE"].dropna().astype(str)
                             .str.strip().str.replace(".", "", regex=False).str.upper())
        # The CMS billing dictionary lists leaf codes; eICU also records legitimate
        # ICD-9 categories/subcategories (e.g. 780.6). Retain these parent concepts.
        _XW_KNOWN_ICD9 |= {c[:k] for c in list(_XW_KNOWN_ICD9)
                            for k in range(4 if c.startswith("E") else 3, len(c))}
    known9 = c in _XW_KNOWN_ICD9
    if c[0].isdigit():
        return (c, "direct") if known9 else ("", "unknown_icd9")
    if c.startswith(("E", "V")):
        if "." in token:
            before = len(token.split(".", 1)[0])
            icd9_position = 4 if c.startswith("E") else 3
            # ICD-10 always has three characters before the decimal; ICD-9 V also
            # has three, but Vxx.xx ICD-10 external-cause forms are incomplete
            # without their encounter suffix. Dictionary checks resolve these.
            if before == icd9_position and known9:
                if c.startswith("E") or not _xw_icd10_to_icd9(c):
                    return c, "direct"
            if before != 3:
                return "", "invalid_decimal"
        mapped = _xw_icd10_to_icd9(c)
        if "." not in token and known9 and mapped and mapped != c:
            return "", "ambiguous_undotted"
        if known9 and not mapped:
            return c, "direct"
        if c.startswith("V") and known9 and mapped and mapped != c:
            return "", "ambiguous_v"
    else:
        mapped = _xw_icd10_to_icd9(c)
    if mapped and mapped in _XW_KNOWN_ICD9:
        return mapped, "gem_exact"
    return "", "unmapped_icd10"

def map_code_to_icd9(raw):
    return classify_mixed_icd(raw)[0]

def resolve_code_to_vocab(raw, vocab):
    """Exact ICD-9 identity on both cohorts; no external-only prefix expansion."""
    code, tag = classify_mixed_icd(raw)
    return (code, tag) if code and code in vocab else (None, "miss")


import re

# ===========================================================================
#  eICU EXTERNAL VALIDATION
# ===========================================================================

In [ ]:
CATEGORY_RATE_SPECS = {
    "cardiac": CARDIAC_PREFIXES, "vascular": VASCULAR_PREFIXES,
    "diabetes": DIABETES_COMP_PREFIXES, "liver": LIVER_PREFIXES,
    "valvular": VALVULAR_PREFIXES, "congenital": CONGENITAL_PREFIXES,
    "renal_access": RENAL_ACCESS_PREFIXES, "device": DEVICE_PREFIXES,
    "prosthetic_valve": PROSTHETIC_VALVE_DX_PREFIXES, "ivdu": IVDU_PREFIXES,
}

## Shared nested fitting and uncertainty

Outer folds group all hospitalizations from a patient together. All models in a feature/sampling cell use the same inner patient holdout. Each inner and outer training partition independently fits ICD support/ranking and lab preprocessing. Rankings are cached only for identical training partitions, not learned once on the full cohort. Fold-local feature names and actual training prevalence are exported.

CNN stopping uses the inner patient holdout; refitting uses its selected duration. RF/Lasso soft voting averages probabilities. Stacking fits a logistic meta-model to RF/Lasso predictions on the disjoint inner holdout, then applies it to refitted base models. This is holdout stacking within outer training—not the old v5 inner-K-fold stacking implementation. Operating thresholds use that MIMIC training-side holdout only. Test and external labels never choose thresholds. Threshold transport after base-model refitting is part of the evaluated method.

Every summary/CNN imputer and normalizer is training-only. Inputs absent throughout training remain masked. Channels constant during training, including missingness flags and demographics, stay at their training value externally. Resampled CNN examples use repeated indices without materializing duplicated tensor banks. Current discharge codes, raw identities and discharge timing are excluded from predictors.

Every grid row's AP, AUROC, Brier score, recall, specificity, PPV and alert rate receive patient-cluster bootstrap 95% intervals. Score sorting is reused across patient-multiplicity resamples. Valid bootstrap counts are exported, especially important for the small ICD-history eICU subgroup. These are conditional uncertainty estimates for the evaluated predictions, not complete retraining uncertainty or multiple-comparison correction. Average precision is the PR summary, not trapezoidal PR area. Repeated examination of eICU limits claims that it remains a fresh confirmatory test set.

In [ ]:
"""Patient-grouped development, MIMIC-only preprocessing and external scoring."""
import random
import copy
import hashlib
import time
import sklearn
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, roc_curve,
                             precision_recall_curve, brier_score_loss)
import torch
from torch import nn
from torch.utils.data import DataLoader, TensorDataset, Subset

META={'patient_id','hospital_id','label','landmark','score_hour'}
DEMOGRAPHICS=['age','age_missing','sex_male','sex_unknown','age_topcoded']


class TrainOnlyTransform:
    """Fixed output schema; all-empty training columns use 0 and missing flag."""
    def fit(self,x):
        x=np.asarray(x,float);self.median=np.zeros(x.shape[1]);self.scale=np.ones(x.shape[1])
        self.supported=np.isfinite(x).any(axis=0)
        for j in range(x.shape[1]):
            a=x[np.isfinite(x[:,j]),j]
            if len(a):
                self.median[j]=np.median(a);s=np.quantile(a,.75)-np.quantile(a,.25)
                self.scale[j]=s if s>1e-8 else max(float(a.std()),1.)
        self.active=None
        fitted=self.transform(x)
        self.active=np.ptp(fitted,axis=0)>0
        self.baseline=fitted[0].copy()
        return self
    def transform(self,x):
        x=np.asarray(x,float);mask=np.isfinite(x)&self.supported
        z=(np.where(mask,x,self.median)-self.median)/self.scale
        # Prespecified numeric guard, not learned from validation or eICU.
        out=np.concatenate([np.clip(z,-50,50),~mask],axis=1).astype(np.float32)
        # A channel constant during training cannot acquire an externally activated,
        # untrained coefficient (including a previously always-zero missing flag).
        if self.active is not None:out[:,~self.active]=self.baseline[~self.active]
        return out

class SequenceTransform:
    def __init__(self,cfg):self.cfg=cfg
    def fit(self,seq):
        x=seq['values'];valid=seq['valid'].astype(bool)
        if not valid.any():raise ValueError('No valid sequence bins in training')
        self.norm=TrainOnlyTransform().fit(x[valid]);self.active=None
        fitted=self.transform(seq).transpose(0,2,1)[valid]
        self.active=np.ptp(fitted,axis=0)>0;self.baseline=fitted[0].copy()
        return self
    def transform(self,seq):
        x=seq['values'];observed=np.isfinite(x)&self.norm.supported[None,None,:];valid=seq['valid'];n,t,k=x.shape
        values=(np.where(observed,x,self.norm.median)-self.norm.median)/self.norm.scale
        # No backward fill, interpolation, cross-admission concatenation or future values.
        # Actual last-observation time retained inside each bin, not just a bin index.
        last=np.full((n,k),np.nan);gap=np.ones_like(x)
        for b in range(t):
            last=np.where(observed[:,b],seq['hours'][:,b],last)
            clock=np.minimum((b+1)*self.cfg['cnn_bin_hours'],seq['cutoff'])[:,None]
            gap[:,b]=np.where(np.isfinite(last),np.maximum(clock-last,0)/self.cfg['cnn_window_hours'],1.)
        clock=np.minimum((np.arange(t)+1)*self.cfg['cnn_bin_hours'],seq['cutoff'][:,None])/self.cfg['cnn_window_hours']
        channels=np.concatenate([np.clip(values,-10,10),observed.astype(float),gap,
                                 clock[:,:,None],valid[:,:,None]],axis=2)
        if self.active is not None:channels[:,:,~self.active]=self.baseline[~self.active]
        channels*=valid[:,:,None]
        assert np.isfinite(channels).all()
        return channels.transpose(0,2,1).astype(np.float32)

def take_sequence(seq,idx):return {k:v[idx] for k,v in seq.items()}

def patient_folds(frame,n_folds,seed):
    patients=frame.groupby('patient_id',sort=True).label.max()
    if patients.value_counts().min()<n_folds:raise ValueError('Too few positive/negative patients for requested folds')
    out={}
    for f,(_,te) in enumerate(StratifiedKFold(n_folds,shuffle=True,random_state=seed).split(patients.index,patients)):
        out.update({p:f for p in patients.index[te]})
    return out

def inner_split(frame,seed,fraction):
    p=frame.groupby('patient_id',sort=True).label.max()
    if len(p.unique())<2 or p.value_counts().min()<2:raise ValueError('Too few patients for two-class inner validation')
    va_count=max(2,int(np.ceil(len(p)*fraction)))
    a,b=train_test_split(p.index.to_numpy(),test_size=va_count,random_state=seed,stratify=p.to_numpy())
    tr=np.flatnonzero(frame.patient_id.isin(a));va=np.flatnonzero(frame.patient_id.isin(b))
    assert not set(frame.iloc[tr].patient_id)&set(frame.iloc[va].patient_id)
    if frame.iloc[tr].label.nunique()!=2 or frame.iloc[va].label.nunique()!=2:
        raise ValueError('Inner split lacks a class; increase the prespecified holdout or cohort size')
    return tr,va


class HybridLabCNN(nn.Module):
    def __init__(self,channels,tab_dim,cfg):
        super().__init__();w=cfg['width']
        self.c1=nn.Conv1d(channels,w,3,padding=1);self.c2=nn.Conv1d(w,w,3,padding=1)
        self.tab=nn.Sequential(nn.Linear(tab_dim,w),nn.ReLU(),nn.Dropout(cfg['dropout']))
        self.out=nn.Sequential(nn.Linear(2*w,w),nn.ReLU(),nn.Dropout(cfg['dropout']),nn.Linear(w,1))
    def forward(self,s,x,mask):
        mask=mask[:,None,:]
        h=torch.relu(self.c1(s))*mask;h=torch.relu(self.c2(h))*mask
        pooled=h.sum(2)/mask.sum(2).clamp_min(1)
        return self.out(torch.cat([pooled,self.tab(x)],1)).squeeze(1)

def seed_everything(seed,cfg):
    random.seed(seed);np.random.seed(seed);torch.manual_seed(seed);torch.set_num_threads(cfg['torch_threads'])
    torch.use_deterministic_algorithms(True)

def predict_cnn(model,s,x,v,batch=1024):
    model.eval();out=[]
    with torch.no_grad():
        for i in range(0,len(x),batch):
            out.append(torch.sigmoid(model(torch.from_numpy(s[i:i+batch]),torch.from_numpy(x[i:i+batch]),
                                          torch.from_numpy(v[i:i+batch]))).numpy())
    return np.concatenate(out)

def train_cnn(s,x,v,y,mode,cfg,seed,epochs,validation=None,row_indices=None):
    seed_everything(seed,cfg);model=HybridLabCNN(s.shape[1],x.shape[1],cfg)
    ix=np.arange(len(y)) if row_indices is None else np.asarray(row_indices,int)
    yr=np.asarray(y)[ix]
    weight=float((yr==0).sum()/max((yr==1).sum(),1)) if mode=='balanced' else 1.
    loss_fn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight,dtype=torch.float32))
    opt=torch.optim.AdamW(model.parameters(),lr=cfg['lr'],weight_decay=cfg['weight_decay'])
    # Repeated indices are literal repeated training examples, without copying
    # the entire tensor bank for control oversampling.
    bank=TensorDataset(torch.from_numpy(s),torch.from_numpy(x),torch.from_numpy(v),torch.tensor(np.asarray(y),dtype=torch.float32))
    data=Subset(bank,ix.tolist())
    loader=DataLoader(data,batch_size=cfg['batch_size'],shuffle=True,generator=torch.Generator().manual_seed(seed))
    best=-np.inf;best_epoch=1;best_state=None;logs=[]
    for epoch in range(1,epochs+1):
        model.train();total=0.
        for sb,xb,vb,yb in loader:
            opt.zero_grad();loss=loss_fn(model(sb,xb,vb),yb);loss.backward();opt.step();total+=loss.item()*len(yb)
        ap=np.nan
        if validation is not None:
            vs,vx,vv,vy=validation;ap=average_precision_score(vy,predict_cnn(model,vs,vx,vv))
            if ap>best+1e-6:best=ap;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict())
        logs.append(dict(epoch=epoch,loss=total/len(data),inner_AP=ap))
        if validation is not None and epoch-best_epoch>=cfg['patience']:break
    if validation is not None:model.load_state_dict(best_state)
    else:best_epoch=epochs
    return model,best_epoch,logs

def youden_threshold(y,p):
    f,t,thr=roc_curve(y,p);ok=np.isfinite(thr)
    if not ok.any():return .5
    return float(thr[ok][np.argmax((t-f)[ok])])


def metric_values(y,p,decision):
    y=np.asarray(y);decision=np.asarray(decision,bool);positive=y==1;negative=~positive
    return dict(AUROC=roc_auc_score(y,p) if len(np.unique(y))==2 else np.nan,
                AP=average_precision_score(y,p) if positive.any() else np.nan,
                Brier=brier_score_loss(y,p),Recall=float(decision[positive].mean()) if positive.any() else np.nan,
                Specificity=float((~decision[negative]).mean()) if negative.any() else np.nan,
                PPV=float(y[decision].mean()) if decision.any() else np.nan,Alert_rate=float(decision.mean()))

In [ ]:
"""Patient-cluster percentile CIs, sorting scores once rather than per draw."""
class WeightedPredictionMetrics:
    def __init__(self,y,p,decision):
        self.y=np.asarray(y,int);self.p=np.asarray(p,float);self.decision=np.asarray(decision,bool)
        self.order=np.argsort(-self.p,kind='stable')
        ordered=self.p[self.order]
        self.ends=np.r_[np.flatnonzero(np.diff(ordered)!=0),len(ordered)-1]
    def __call__(self,weights):
        w=np.asarray(weights,float);y=self.y;d=self.decision
        pos=float(w@y);neg=float(w@(1-y));total=pos+neg
        if not pos or not neg:return None
        wo=w[self.order];yo=y[self.order]
        tp=np.cumsum(wo*yo)[self.ends];fp=np.cumsum(wo*(1-yo))[self.ends]
        # Removing zero-weight thresholds matches explicit row duplication.
        present=np.diff(np.r_[0,tp+fp])>0;tp=tp[present];fp=fp[present]
        ap=np.sum(np.diff(np.r_[0,tp])/pos*tp/(tp+fp))
        tpr=np.r_[0,tp/pos];fpr=np.r_[0,fp/neg]
        auc=float(np.sum(np.diff(fpr)*(tpr[1:]+tpr[:-1])/2))
        alerts=float(w@d)
        return dict(AUROC=auc,AP=float(ap),Brier=float(w@((y-self.p)**2)/total),
            Recall=float(w@(d*y)/pos),Specificity=float(w@((~d)*(1-y))/neg),
            PPV=float(w@(d*y)/alerts) if alerts else np.nan,Alert_rate=alerts/total)

def cluster_draws(ids,n_boot,seed):
    codes,unique=pd.factorize(np.asarray(ids),sort=True)
    rng=np.random.default_rng(seed);n=len(unique)
    for _ in range(n_boot):
        counts=np.bincount(rng.integers(0,n,n),minlength=n)
        yield counts[codes]

def patient_bootstrap(d,n_boot,seed):
    calc=WeightedPredictionMetrics(d.label,d.probability,d.probability>=d.threshold)
    rows=[]
    for weights in cluster_draws(d.patient_id,n_boot,seed):
        row=calc(weights)
        if row is not None:rows.append(row)
    return pd.DataFrame(rows)

def summarize_predictions(pred,cfg):
    rows=[];keys=['population','cohort','landmark','scope','model','sampling']
    for key,g in pred.groupby(keys,sort=False):
        met=metric_values(g.label,g.probability,g.probability>=g.threshold)
        boot=patient_bootstrap(g,cfg['n_boot'],cfg['seed']);row=dict(zip(keys,key));row.update(met)
        row.update(admissions=len(g),patients=g.patient_id.nunique(),cases=int(g.label.sum()),
            prevalence=float(g.label.mean()),bootstrap_requested=cfg['n_boot'],bootstrap_valid=len(boot))
        for name in met:
            row[name+'_lo']=boot[name].quantile(.025) if name in boot else np.nan
            row[name+'_hi']=boot[name].quantile(.975) if name in boot else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

def paired_landmarks(pred,cfg):
    rows=[];keys=['population','cohort','scope','model','sampling']
    for key,g in pred.groupby(keys,sort=False):
        p=g[g.landmark.eq(1)].merge(g[g.landmark.eq(2)],on=['patient_id','hospital_id'],
                                   suffixes=('_first','_second'),validate='one_to_one')
        if p.empty or p.label_first.nunique()<2:continue
        assert (p.label_first==p.label_second).all()
        first=WeightedPredictionMetrics(p.label_first,p.probability_first,p.probability_first>=p.threshold_first)
        second=WeightedPredictionMetrics(p.label_second,p.probability_second,p.probability_second>=p.threshold_second)
        a=first(np.ones(len(p)));b=second(np.ones(len(p)));deltas=[]
        for weights in cluster_draws(p.patient_id,cfg['n_boot'],cfg['seed']):
            x=first(weights);z=second(weights)
            if x is not None:deltas.append([z['AP']-x['AP'],z['AUROC']-x['AUROC']])
        row=dict(zip(keys,key),admissions=len(p),patients=p.patient_id.nunique(),cases=int(p.label_first.sum()),
                 bootstrap_requested=cfg['n_boot'],bootstrap_valid=len(deltas))
        for j,metric in enumerate(('AP','AUROC')):
            row[metric+'_first']=a[metric];row[metric+'_second']=b[metric]
            row[metric+'_delta']=b[metric]-a[metric]
            lo,hi=np.quantile(np.asarray(deltas)[:,j],[.025,.975]) if deltas else (np.nan,np.nan)
            row[metric+'_delta_lo']=lo;row[metric+'_delta_hi']=hi
        rows.append(row)
    return pd.DataFrame(rows)

In [ ]:
"""Local outputs and compact diagnostics; no external uploads."""
import marshal
import matplotlib.pyplot as plt

def validate_config(cfg):
    if cfg.get('age_policy') not in ('adults','all_ages'):raise ValueError('Choose age_policy adults or all_ages')
    retired={'scopes','sampling_modes','control_target_prevalence'}&set(cfg)
    if retired:raise ValueError(f'Retired standalone-runner settings: {sorted(retired)}; use grid_profile and prevalence_targets')
    if cfg['include_ensembles'] and not {'RandomForest','Lasso'}<=set(cfg['base_models']):
        raise ValueError('RF and Lasso are required for the configured ensembles')
    if not cfg['base_models'] or not set(cfg['base_models'])<=set(BASE_MODELS):raise ValueError('Invalid base_models')
    assert cfg['release_lag_hours']>=0 and cfg['set_width_hours']>0
    assert cfg['repeat_min_gap_hours']>=cfg['set_width_hours']
    assert cfg['cnn_window_hours']>=cfg['second_draw_max_hours']+cfg['set_width_hours']
    assert cfg['cnn_bin_hours']>0 and cfg['max_epochs']>=1 and cfg['n_boot']>=1
    assert 0<cfg['inner_holdout']<.5 and cfg['outer_folds']>=2

def input_manifest(root,cfg):
    paths=[Path(root)/n for n in ['ADMISSIONS.csv','PATIENTS.csv','DIAGNOSES_ICD.csv','LABEVENTS.csv']]
    paths += [locate(root,'eicu/patients.csv','eicu/patient.csv','patients.csv','patient.csv'),
              locate(root,'eicu/diagnoses.csv','eicu/diagnosis.csv','diagnoses.csv','diagnosis.csv'),
              locate(root,'eicu/lab.csv','lab.csv')]
    files=[dict(path=str(p.resolve()),bytes=p.stat().st_size,modified_ns=p.stat().st_mtime_ns) for p in paths]
    functions=[load_admissions,extract_labs,clean_labs,resolve_revisions,draw_landmarks,series_stats,
               build_features,reference_interval,boundary_features]
    code=hashlib.sha256(b''.join(stable_code_bytes(f.__code__) for f in functions)).hexdigest()
    manifest=dict(feature_cache_schema=2,files=files,config=cfg,labs=LABS,ranges=ADULT_RANGES,core=CORE,be_codes=sorted(BE9),feature_code_sha256=code)
    feature_keys=['first_draw_max_hours','second_draw_max_hours','set_width_hours','repeat_min_gap_hours',
                  'release_lag_hours','min_core_analytes','min_slope_span_hours','history_slope_span_hours',
                  'cnn_bin_hours','cnn_window_hours']
    feature_manifest=dict(manifest,config={k:cfg[k] for k in feature_keys})
    fingerprint=hashlib.sha256(json.dumps(feature_manifest,sort_keys=True,default=str).encode()).hexdigest()
    return manifest,fingerprint

def prepare_experiment(root,out,cfg,use_cache=True):
    validate_config(cfg);out=Path(out);out.mkdir(parents=True,exist_ok=True)
    manifest,fingerprint=input_manifest(root,cfg);cache=out/'feature_cache'/fingerprint[:16]
    required=['MIMIC.csv.gz','eICU.csv.gz','MIMIC.npz','eICU.npz','complete.json']
    if use_cache and all((cache/n).exists() for n in required):
        saved=json.loads((cache/'complete.json').read_text())
        if saved['fingerprint']!=fingerprint:raise ValueError('Cache fingerprint mismatch')
        m=pd.read_csv(cache/'MIMIC.csv.gz',dtype=saved['dtypes']['MIMIC'],float_precision='round_trip');e=pd.read_csv(cache/'eICU.csv.gz',dtype=saved['dtypes']['eICU'],float_precision='round_trip')
        with np.load(cache/'MIMIC.npz',allow_pickle=False) as z:ms={k:z[k] for k in z.files}
        with np.load(cache/'eICU.npz',allow_pickle=False) as z:es={k:z[k] for k in z.files}
        print('Using matching local feature cache; no fitted model is cached.')
        return m,ms,e,es,saved['audit']
    print('Loading admission-level cohort. Prior hospitalization/labs are optional.',flush=True)
    ma,ea,ep=load_admissions(root)
    source=[]
    for name,d in [('MIMIC',ma),('eICU',ea)]:
        source.append(dict(cohort=name,source_admissions=len(d),eligible_admissions=int(d.eligible.sum()),
                           eligible_cases=int(d.loc[d.eligible,'label'].sum()),eligible_patients=d.loc[d.eligible,'patient_id'].nunique()))
    print('Scanning laboratory CSVs. Full histories retained where usable; current data stop at the maximum draw window.',flush=True)
    ml,el,unit_audit=extract_labs(root,ma,ea,ep,cfg)
    print('Building MIMIC summaries and sequences',flush=True);m,ms,mflow=build_features(ma,ml,'MIMIC',cfg)
    del ml
    print('Building eICU summaries and sequences',flush=True);e,es,eflow=build_features(ea,el,'eICU',cfg)
    del el
    audit=dict(source=source,flow={'MIMIC':mflow,'eICU':eflow},units=unit_audit)
    cache.mkdir(parents=True,exist_ok=True)
    m.to_csv(cache/'MIMIC.csv.gz',index=False);e.to_csv(cache/'eICU.csv.gz',index=False)
    np.savez_compressed(cache/'MIMIC.npz',**ms);np.savez_compressed(cache/'eICU.npz',**es)
    (cache/'complete.json').write_text(json.dumps(dict(fingerprint=fingerprint,manifest=manifest,audit=audit,dtypes={c:{k:str(v) for k,v in d.dtypes.items()} for c,d in [('MIMIC',m),('eICU',e)]}),indent=2,default=str))
    return m,ms,e,es,audit

def cohort_summary(m,e):
    rows=[]
    for name,d in [('MIMIC',m),('eICU',e)]:
        for landmark,g in d.groupby('landmark'):
            rows.append(dict(cohort=name,landmark=landmark,admissions=len(g),patients=g.patient_id.nunique(),
                cases=int(g.label.sum()),prevalence=g.label.mean(),no_history_fraction=g.history__available.eq(0).mean(),
                median_age=g.age.median(),under18_fraction=g.age.lt(18).mean(),unknown_age_fraction=g.age.isna().mean(),
                median_score_hour=g.score_hour.median()))
    return pd.DataFrame(rows)

def coverage_report(m,e,out):
    rows=[]
    for name,d in [('MIMIC',m),('eICU',e)]:
        for landmark,g in d.groupby('landmark'):
            for a in LABS:
                for source,stat in [('current','last'),('history','mean')]:
                    rows.append(dict(cohort=name,landmark=landmark,analyte=a,source=source,
                                     fraction_observed=g[f'{source}__{a}__{stat}'].notna().mean()))
    coverage=pd.DataFrame(rows);coverage.to_csv(Path(out)/'lab_coverage.csv',index=False)
    fig,axes=plt.subplots(1,2,figsize=(12,9),sharey=True)
    for ax,source in zip(axes,['current','history']):
        p=coverage[coverage.source.eq(source)].pivot(index='analyte',columns=['cohort','landmark'],values='fraction_observed').reindex(list(LABS))
        im=ax.imshow(p.to_numpy(),vmin=0,vmax=1,cmap='viridis',aspect='auto')
        ax.set_xticks(range(len(p.columns)),[f'{c}\nset {l}' for c,l in p.columns]);ax.set_yticks(range(len(p)),p.index)
        ax.set_title(f'{source.title()} lab availability')
    fig.colorbar(im,ax=axes.ravel().tolist(),label='Fraction with observed value',shrink=.8)
    fig.savefig(Path(out)/'lab_coverage.png',dpi=180,bbox_inches='tight');plt.show()
    return coverage

def plot_results(pred,metrics,logs,cfg,out):
    out=Path(out)
    population=cfg.get('population_label','')
    # Separate cohorts and landmark cutoffs. Curves use natural test prevalence.
    for landmark in (1,2):
        fig,axes=plt.subplots(2,2,figsize=(13,9))
        for i,cohort in enumerate(['MIMIC_OOF','eICU_external']):
            d=pred[pred.landmark.eq(landmark)&pred.cohort.eq(cohort)&pred.scope.eq('current_plus_history')]
            for (model,sampling),g in d.groupby(['model','sampling']):
                fpr,tpr,_=roc_curve(g.label,g.probability);pr,re,_=precision_recall_curve(g.label,g.probability)
                label=f'{model} / {sampling}'
                axes[i,0].plot(fpr,tpr,label=f'{label} ({roc_auc_score(g.label,g.probability):.3f})')
                axes[i,1].plot(re,pr,label=f'{label} ({average_precision_score(g.label,g.probability):.3f})')
            base=d.drop_duplicates('hospital_id').label.mean()
            axes[i,0].plot([0,1],[0,1],'k--',lw=1)
            axes[i,1].axhline(base,color='gray',ls='--',label=f'Prevalence {base:.3%}')
            for j,title in enumerate(['ROC','Precision–recall']):
                ax=axes[i,j];ax.set_title(f'{cohort} | set {landmark} | {title}');ax.set_xlim(0,1);ax.set_ylim(0,1)
                ax.set_xlabel('False positive rate' if j==0 else 'Recall');ax.set_ylabel('Recall' if j==0 else 'Precision')
                ax.legend(fontsize=7);ax.grid(alpha=.2)
        fig.suptitle(population);fig.tight_layout();fig.savefig(out/f'landmark_{landmark}_curves.png',dpi=180);plt.show()
    # Fixed model/scopes, not external-winner selection. Error bars are patient-cluster CIs.
    largest=metrics.groupby(['cohort','landmark']).size().max()
    fig,axes=plt.subplots(2,2,figsize=(17,max(10,largest*.58)))
    for i,cohort in enumerate(['MIMIC_OOF','eICU_external']):
        for j,landmark in enumerate([1,2]):
            d=metrics[metrics.cohort.eq(cohort)&metrics.landmark.eq(landmark)].sort_values(['sampling','model','scope'])
            x=np.arange(len(d));ax=axes[i,j]
            ax.barh(x,d.AP,height=.6,color=['tab:blue' if s=='current_plus_history' else 'tab:orange' if s=='current_only' else 'tab:gray' for s in d.scope])
            # Percentile intervals need not contain the point estimate; draw interval directly.
            ax.hlines(x,d.AP_lo,d.AP_hi,color='black');ax.scatter(d.AP_lo,x,marker='|',c='black');ax.scatter(d.AP_hi,x,marker='|',c='black')
            short={'current_plus_history':'current + history','current_only':'current','history_only':'history',
                   'SummaryLR':'LR','SummaryRF':'RF','RandomForest':'RF','LabCNN':'CNN'}
            ax.set_yticks(x,[f'{short[r.model]} | {short[r.scope]} | {r.sampling}' for r in d.itertuples()],fontsize=8)
            ax.invert_yaxis();ax.axvline(d.prevalence.iloc[0],color='gray',ls='--');ax.set_xlabel('Average precision (95% CI)')
            ax.set_title(f'{cohort} | set {landmark}');ax.grid(axis='x',alpha=.2)
    fig.suptitle(population);fig.tight_layout();fig.savefig(out/'scope_comparison_AP.png',dpi=180);plt.show()
    if not logs.empty:
        mode='balanced' if 'balanced' in cfg['sampling_modes'] else cfg['sampling_modes'][0]
        d=logs[logs.scope.eq('current_plus_history')&logs.sampling.eq(mode)&logs.fold.ge(0)]
        fig,axes=plt.subplots(1,2,figsize=(12,4))
        for (landmark,fold),g in d.groupby(['landmark','fold']):
            label=f'set {landmark}, fold {fold+1}'
            axes[0].plot(g.epoch,g.loss,label=label);axes[1].plot(g.epoch,g.inner_AP,label=label)
        axes[0].set_title('CNN inner-training loss');axes[1].set_title('CNN inner-validation AP')
        axes[0].set_ylabel('Training loss');axes[1].set_ylabel('Average precision')
        for ax in axes:ax.set_xlabel('Epoch');ax.grid(alpha=.2)
        axes[1].legend(fontsize=7);fig.suptitle(population);fig.tight_layout();fig.savefig(out/'cnn_learning_curves.png',dpi=180);plt.show()

In [ ]:
"""Unified ICD + laboratory experiment grid. Raw features only are shared across folds."""
from itertools import combinations
from collections import defaultdict
from sklearn.feature_extraction import DictVectorizer
from scipy import sparse
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from functools import lru_cache
import gc
from types import CodeType

def stable_code_bytes(code):
    """Executable fingerprint independent of CPython marshal/interning state."""
    def encode(value):
        if isinstance(value,CodeType):
            return dict(bytecode=value.co_code.hex(),constants=[encode(c) for c in value.co_consts],
                names=value.co_names,varnames=value.co_varnames,freevars=value.co_freevars,
                cellvars=value.co_cellvars,argcount=value.co_argcount,posonly=value.co_posonlyargcount,
                kwonly=value.co_kwonlyargcount,flags=value.co_flags,
                exceptiontable=getattr(value,'co_exceptiontable',b'').hex())
        if isinstance(value,tuple):return ['tuple',[encode(v) for v in value]]
        if isinstance(value,frozenset):return ['frozenset',sorted([encode(v) for v in value],key=repr)]
        return [type(value).__name__,repr(value)]
    return json.dumps(encode(code),sort_keys=True,separators=(',',':')).encode()

ICD_BLOCKS=('onehot','rates','shares','ever','icd_ever')
ALL_GRID_BLOCKS=ICD_BLOCKS+('lab_current','lab_history')
BASE_MODELS=('RandomForest','Lasso','SummaryLR','XGBoost','DecisionTree')
ENSEMBLES=('SoftVote(RF+Lasso)','Stack(RF+Lasso)')
ICD_RAW={};ICD_SELECTION_CACHE={}

def icd_block(name):
    if name.startswith('T-'):return 'onehot'
    if name.startswith('ICD_') and name.endswith('_ever'):return 'icd_ever'
    if name.endswith('_rate_per_adm'):return 'rates'
    if name.endswith('_share'):return 'shares'
    if name.endswith('_ever_hx'):return 'ever'
    return 'other'

def history_icd_features(earlier,codes,cohort,cap=5):
    """No current-admission diagnoses; one occurrence per normalized code/stay."""
    rows=[];features={};total=0;counts=Counter()
    # MIMIC admission-time ties and eICU same-year ties have no defensible order.
    order_col='start' if cohort=='MIMIC' else 'year'
    ordered=earlier.sort_values(order_col,ascending=False)
    ambiguous=bool(ordered[order_col].duplicated().any())
    for rank,r in enumerate(ordered.itertuples(),1):
        clean={c for c in codes.get(int(r.hospital_id),set()) if c and c not in BE9}
        total+=len(clean)
        for c in clean:
            features[f'ICD_{c}_ever']=1.
            if not ambiguous:features[f'T-{min(rank,cap)} {c}']=1.
            for cat,pfx in CATEGORY_RATE_SPECS.items():
                if c.startswith(tuple(pfx)):counts[cat]+=1
    n=len(earlier)
    for cat in CATEGORY_RATE_SPECS:
        features[f'{cat}_rate_per_adm']=counts[cat]/max(n,1)
        features[f'{cat}_share']=counts[cat]/max(total,1)
        features[f'{cat}_ever_hx']=float(counts[cat]>0)
    features['total_icd_rate_per_adm']=total/max(n,1)
    features['total_icd_ever_hx']=float(total>0)
    return features,dict(prior_hospitalizations=n,prior_unique_code_mentions=total,
                         temporal_order_ambiguous=ambiguous)

def build_icd_history(root,m_frame,e_frame,out):
    """Rebuild historical ICD by hospital ID at the new lab landmarks, never join by position."""
    global mimic_tables,_XW_KNOWN_ICD9,ICD_RAW,ICD_SELECTION_CACHE
    ma,ea,ep=load_admissions(root)
    dictionary=pd.read_excel(Path(root)/'CMS32_DESC_LONG_SHORT_DX.xlsx',dtype=str)
    mimic_tables={'ICD_9_DX_DESCRIPTIONS':dictionary};_XW_KNOWN_ICD9=None
    md=pd.read_csv(Path(root)/'DIAGNOSES_ICD.csv',usecols=['HADM_ID','ICD9_CODE'],dtype={'ICD9_CODE':str})
    md['code']=md.ICD9_CODE.fillna('').str.strip().str.replace('.','',regex=False).str.upper()
    mc=md.groupby('HADM_ID').code.agg(set).to_dict()
    ed=pd.read_csv(locate(root,'eicu/diagnoses.csv','eicu/diagnosis.csv','diagnoses.csv','diagnosis.csv'),
                   usecols=['patientunitstayid','icd9code'],dtype={'icd9code':str})
    unit=ep.set_index('patientunitstayid').patienthealthsystemstayid
    ed['hospital_id']=ed.patientunitstayid.map(unit)
    raw=ed.icd9code.fillna('').str.split(r'[,;\s]+',regex=True)
    ed=ed.assign(token=raw).explode('token');tokens=ed.token.dropna().unique()
    mapping={str(t):classify_mixed_icd(str(t)) for t in tokens}
    audit=pd.DataFrame([dict(token=t,ICD9=c,provenance=p) for t,(c,p) in mapping.items()])
    audit['occurrences']=audit.token.map(ed.token.value_counts()).fillna(0).astype(int)
    ed['code']=ed.token.map({k:v[0] for k,v in mapping.items()})
    ec=ed[ed.code.fillna('').ne('')].groupby('hospital_id').code.agg(set).to_dict()
    ICD_RAW={};history_audit=[]
    for cohort,admissions,frame,codes in [('MIMIC',ma,m_frame,mc),('eICU',ea,e_frame,ec)]:
        patients={str(p):g for p,g in admissions.groupby('patient_id',sort=False)}
        index=admissions.set_index('hospital_id');lookup={}
        for hid in frame.hospital_id.unique():
            r=index.loc[int(hid)];history=patients[str(r.patient_id)]
            if cohort=='MIMIC':
                history=history[history.valid_time&history.end.lt(r.start)&history.start.lt(r.start)]
            else:history=history[history.valid_time&history.year.lt(r.year)]
            assert int(hid) not in set(history.hospital_id)
            feats,meta=history_icd_features(history,codes,cohort)
            lookup[int(hid)]=feats
            history_audit.append(dict(cohort=cohort,hospital_id=int(hid),**meta))
        ICD_RAW[cohort]=lookup
        assert set(frame.hospital_id).issubset(lookup)
        mapped=index.loc[frame.hospital_id]
        assert np.array_equal(mapped.patient_id.astype(str).to_numpy(),frame.patient_id.astype(str).to_numpy())
        assert np.array_equal(mapped.label.to_numpy(),frame.label.to_numpy())
    ICD_SELECTION_CACHE={}
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    audit.to_csv(out/'icd_crosswalk_audit.csv',index=False)
    pd.DataFrame(history_audit).to_csv(out/'icd_history_alignment_audit.csv',index=False)
    # Raw dictionary exports are optional caches/audits, not globally fitted predictors.
    for cohort,rows in ICD_RAW.items():
        with (out/f'{cohort}_raw_icd_history.json').open('w',encoding='utf-8') as f:json.dump(rows,f)
    return audit,pd.DataFrame(history_audit)

def selected_icd_names(train,cfg,seed):
    key=(tuple(train.hospital_id.astype(int)),tuple(train.label.astype(int)),seed,
         tuple(train.patient_id.astype(str)),
         cfg['icd_min_patients'],cfg['icd_temporal_budget'],cfg['icd_ever_budget'],cfg['selection_trees'])
    if key in ICD_SELECTION_CACHE:return ICD_SELECTION_CACHE[key]
    records=[ICD_RAW['MIMIC'][int(h)] for h in train.hospital_id]
    vec=DictVectorizer(sparse=True);x=vec.fit_transform(records);names=vec.get_feature_names_out()
    pid,unique=pd.factorize(train.patient_id)
    patients=sparse.csr_matrix((np.ones(len(train)),(pid,np.arange(len(train)))),shape=(len(unique),len(train)))
    support=np.asarray(((patients@(x!=0))>0).sum(axis=0)).ravel()
    selected=[]
    for block,budget in [('onehot',cfg['icd_temporal_budget']),('icd_ever',cfg['icd_ever_budget'])]:
        candidates=np.array([i for i,n in enumerate(names) if icd_block(n)==block and support[i]>=cfg['icd_min_patients']],int)
        if len(candidates):
            rf=RandomForestClassifier(n_estimators=cfg['selection_trees'],max_depth=8,min_samples_leaf=5,
                                       class_weight='balanced',n_jobs=cfg['torch_threads'],random_state=seed)
            rf.fit(x[:,candidates],train.label)
            rank=np.argsort(-rf.feature_importances_,kind='stable')[:budget]
            selected.extend(names[candidates[rank]])
    selected.extend(n for n in names if icd_block(n) in ('rates','shares','ever'))
    ICD_SELECTION_CACHE[key]=tuple(selected)
    return tuple(selected)

class GridTransform:
    def fit(self,train,combo,cfg,seed,cnn=False):
        self.combo=tuple(combo);self.cnn=cnn
        has_icd=bool(set(combo)&set(ICD_BLOCKS))
        self.icd_names=[n for n in selected_icd_names(train,cfg,seed) if icd_block(n) in combo] if has_icd else []
        self.icd_presence=has_icd
        prefixes=[]
        if 'lab_history' in combo:prefixes.append('history__')
        if 'lab_current' in combo and not cnn:prefixes.append('current__')
        if {'lab_current','lab_history'}<=set(combo) and not cnn:prefixes.append('combined__')
        self.lab_names=[n for n in train if n.startswith(tuple(prefixes))]
        if not cfg['include_demographics']:
            reference_stats={'reference_known','below','above','distance_outside','toward_boundary',
                'away_boundary','toward_range','away_range','stable_distance','trend_valid',
                'rising','falling','crossed_boundary'}
            # These directions are computed only with a supported reference interval;
            # even their missingness can reveal age/sex. Raw slope/change stay available.
            self.lab_names=[n for n in self.lab_names if n.split('__')[-1] not in reference_stats]
        if cfg['include_demographics']:self.lab_names+=DEMOGRAPHICS
        self.norm=TrainOnlyTransform().fit(self.raw(train,'MIMIC'))
        return self
    def raw(self,frame,cohort):
        pieces=[]
        if self.icd_presence:
            rows=[ICD_RAW[cohort][int(h)] for h in frame.hospital_id]
            a=np.array([[r.get(n,0.) for n in self.icd_names] for r in rows],np.float32)
            missing=np.array([not r.get('total_icd_ever_hx',0) for r in rows],float)[:,None]
            pieces.extend([a,missing])
        if self.lab_names:pieces.append(frame[self.lab_names].to_numpy(float))
        # Intercept-only tabular branch when current-only CNN excludes demographics.
        if not pieces:pieces=[np.zeros((len(frame),1))]
        return np.concatenate(pieces,axis=1)
    def transform(self,frame,cohort):return self.norm.transform(self.raw(frame,cohort))

def sampling_plan(cfg):
    if cfg['grid_profile']=='compact':
        return [('unweighted',None),('none',None),('prevalence_match','eicu')]
    return [('unweighted',None),('none',None)]+[('oversample',r) for r in cfg['oversample_ratios']]+[
        ('undersample',r) for r in cfg['undersample_ratios']]+[('prevalence_match',t) for t in cfg['prevalence_targets']]

def grid_combinations(cfg):
    if cfg['grid_profile']=='full':
        return [tuple(c) for k in range(1,len(ALL_GRID_BLOCKS)+1) for c in combinations(ALL_GRID_BLOCKS,k)]
    if cfg['grid_profile']!='compact':raise ValueError('grid_profile must be compact or full')
    # Small prespecified comparison, not selected using any performance results.
    return [('rates','shares'),tuple(ICD_BLOCKS),('lab_history',),('lab_current',),('lab_current','lab_history'),
            ('rates','shares','lab_history'),('rates','shares','lab_current','lab_history'),tuple(ALL_GRID_BLOCKS)]

def resolved_sampling(mode,value,y_train,y_external):
    if mode!='prevalence_match':return dict(mode=mode,value=value,external_prevalence_used=False)
    pe=float(np.mean(y_external));pm=float(np.mean(y_train))
    target=pe if value=='eicu' else (pm+pe)/2 if value=='midpoint' else float(value)
    return dict(mode=mode,value=target,external_prevalence_used=value in ('eicu','midpoint'))

def sample_indices(y,setting,seed):
    rng=np.random.default_rng(seed);y=np.asarray(y);pos=np.flatnonzero(y==1);neg=np.flatnonzero(y==0)
    if not len(pos) or not len(neg):raise ValueError('Both classes required in training')
    mode=setting['mode'];v=setting['value'];ix=np.arange(len(y))
    if mode in ('none','unweighted'):return ix
    if mode=='oversample':
        if not 0<v<=1:raise ValueError('Case:control ratio must be in (0,1]')
        need=int(np.ceil(v*len(neg)))
        if need>len(pos):ix=np.r_[ix,rng.choice(pos,need-len(pos),replace=True)]
    elif mode=='undersample':
        if not 0<v<=1:raise ValueError('Case:control ratio must be in (0,1]')
        need=int(np.floor(len(pos)/v))
        if need<len(neg):ix=np.r_[pos,rng.choice(neg,max(1,need),replace=False)]
    elif mode=='prevalence_match':
        if not 0<v<1:raise ValueError('Target prevalence must be in (0,1)')
        need=int(round(len(pos)*(1-v)/v))
        if need>len(neg):ix=np.r_[ix,rng.choice(neg,need-len(neg),replace=True)]
    else:raise ValueError(mode)
    return ix

def grid_estimator(name,y,setting,cfg,seed):
    balanced=setting['mode']=='none';cw='balanced' if balanced else None
    if name=='RandomForest':return RandomForestClassifier(n_estimators=cfg['forest_trees'],max_depth=8,
        min_samples_leaf=5,class_weight=cw,random_state=seed,n_jobs=cfg['torch_threads'])
    if name=='DecisionTree':return DecisionTreeClassifier(max_depth=5,min_samples_leaf=5,class_weight=cw,random_state=seed)
    if name=='Lasso':
        modern=tuple(int(x) for x in re.findall(r'\d+',sklearn.__version__)[:2])>=(1,8)
        kw=dict(l1_ratio=1.) if modern else dict(penalty='l1')
        return LogisticRegression(solver='liblinear',C=1.,class_weight=cw,max_iter=3000,random_state=seed,**kw)
    if name=='SummaryLR':return LogisticRegression(C=1.,max_iter=3000,class_weight=cw,random_state=seed)
    if name=='XGBoost':return XGBClassifier(n_estimators=cfg['xgb_trees'],max_depth=3,learning_rate=.05,
        scale_pos_weight=float((y==0).sum()/max((y==1).sum(),1)) if balanced else 1.,
        eval_metric='logloss',random_state=seed,n_jobs=cfg['torch_threads'])
    raise ValueError(name)

def fit_grid_partition(train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,checkpoint_dir=None):
    """All selection/scaling nested. Same inner patient holdout for all model families."""
    seq=panel_sequence(seq,cfg);target_seq=panel_sequence(target_seq,cfg)
    ti,vi=inner_split(train,seed,cfg['inner_holdout']);y=train.label.to_numpy(int)
    inner=GridTransform().fit(train.iloc[ti],combo,cfg,seed)
    xt=inner.transform(train.iloc[ti],'MIMIC');xv=inner.transform(train.iloc[vi],'MIMIC')
    outer=GridTransform().fit(train,combo,cfg,seed)
    xf=outer.transform(train,'MIMIC');xe=outer.transform(target,target_cohort)
    ix=sample_indices(y[ti],setting,seed);fx=sample_indices(y,setting,seed)
    preds={};cal={};logs=[]
    fit_records=[]
    for name in cfg['base_models']:
        cal[name],inner_record=_fit_model_checkpoint(name,xt[ix],y[ti][ix],xv,setting,cfg,seed,'inner',checkpoint_dir)
        preds[name],outer_record=_fit_model_checkpoint(name,xf[fx],y[fx],xe,setting,cfg,seed,'outer',checkpoint_dir)
        fit_records.extend([inner_record,outer_record])
    if cfg['include_ensembles']:
        if not {'RandomForest','Lasso'}<=set(preds):raise ValueError('RF and Lasso are required for ensembles')
        ip=np.column_stack([cal['RandomForest'],cal['Lasso']]);ep=np.column_stack([preds['RandomForest'],preds['Lasso']])
        meta=LogisticRegression(max_iter=1000).fit(ip,y[vi])
        cal[ENSEMBLES[0]]=ip.mean(1);preds[ENSEMBLES[0]]=ep.mean(1)
        cal[ENSEMBLES[1]]=meta.predict_proba(ip)[:,1];preds[ENSEMBLES[1]]=meta.predict_proba(ep)[:,1]
    if 'lab_current' in combo and cfg['include_cnn']:
        ci=GridTransform().fit(train.iloc[ti],combo,cfg,seed,cnn=True)
        co=GridTransform().fit(train,combo,cfg,seed,cnn=True)
        sn=SequenceTransform(cfg).fit(take_sequence(seq,ti));st=sn.transform(take_sequence(seq,ti));sv=sn.transform(take_sequence(seq,vi))
        # Duplication occurs only here, after all patient splits and transforms were fitted.
        mode='balanced' if setting['mode']=='none' else 'unweighted'
        c,epochs,logs=train_cnn(st,ci.transform(train.iloc[ti],'MIMIC'),seq['valid'][ti],y[ti],mode,cfg,seed,
            cfg['max_epochs'],(sv,ci.transform(train.iloc[vi],'MIMIC'),seq['valid'][vi],y[vi]),row_indices=ix)
        name='ICD_LabCNN_Fusion' if set(combo)&set(ICD_BLOCKS) else 'LabCNN'
        cal[name]=predict_cnn(c,sv,ci.transform(train.iloc[vi],'MIMIC'),seq['valid'][vi])
        sn=SequenceTransform(cfg).fit(seq);sf=sn.transform(seq)
        c,_,_=train_cnn(sf,co.transform(train,'MIMIC'),seq['valid'],y,mode,cfg,seed,epochs,row_indices=fx)
        preds[name]=predict_cnn(c,sn.transform(target_seq),co.transform(target,target_cohort),target_seq['valid'])
    thresholds={name:youden_threshold(y[vi],p) for name,p in cal.items()}
    operating={model:{operating_key(t):operating_threshold(y[vi],p,t) for t in cfg['recall_targets']} for model,p in cal.items()}
    audit=dict(operating_thresholds_json=json.dumps(operating),train_admissions=len(train),train_patients=train.patient_id.nunique(),
        inner_validation_patients=train.iloc[vi].patient_id.nunique(),raw_predictors=outer.raw(train,'MIMIC').shape[1],
        natural_prevalence=float(y.mean()),fit_prevalence=float(y[fx].mean()),fit_rows=len(fx),
        duplicated_rows=len(fx)-len(np.unique(fx)),external_prevalence_used=setting['external_prevalence_used'],
        selected_ICD_features='|'.join(outer.icd_names),threshold_source='MIMIC inner patient holdout',model_timing_json=json.dumps(fit_records))
    return preds,thresholds,logs,audit



def wide_grid_results(metrics):
    keys=['population','landmark','scope','sampling','model','external_prevalence_used']
    keep=[c for c in metrics if c not in keys+['cohort']]
    m=metrics[metrics.cohort.eq('MIMIC_OOF')][keys+keep].rename(columns={c:'MIMIC_'+c for c in keep})
    e=metrics[metrics.cohort.eq('eICU_external')][keys+keep].rename(columns={c:'eICU_'+c for c in keep})
    w=m.merge(e,on=keys,validate='one_to_one').rename(columns={'scope':'Feature_combo','sampling':'Sampling','model':'Model'})
    w=w.rename(columns={p+'AP'+suffix:p+'PR_AUC'+suffix for p in ('MIMIC_','eICU_') for suffix in ('','_lo','_hi')})
    w['Experiment']=w.landmark.map({1:'first_lab_set',2:'second_lab_set'})
    w['Prediction_question']='current_hospitalization_BE'
    w['Lab_input']=w.Feature_combo.map(lambda c:'current+history' if 'lab_current' in c and 'lab_history' in c else 'current' if 'lab_current' in c else 'history' if 'lab_history' in c else 'none')
    w['ICD_input']=w.Feature_combo.map(lambda c:'+'.join(b for b in c.split('+') if b in ICD_BLOCKS) or 'none')
    return w

def integrated_plots_one_population(grid,paired,learning,out,population=''):
    out=Path(out)
    for experiment,d in grid.groupby('Experiment'):
        models=list(d.Model.unique());fig,axes=plt.subplots(2,int(np.ceil(len(models)/2)),figsize=(16,8),squeeze=False)
        for ax,name in zip(axes.flat,models):
            g=d[d.Model.eq(name)];ax.scatter(g.MIMIC_PR_AUC,g.eICU_PR_AUC,s=18,alpha=.65)
            ax.set_title(name);ax.set_xlabel('MIMIC AP');ax.set_ylabel('eICU AP');ax.grid(alpha=.2)
        for ax in list(axes.flat)[len(models):]:ax.axis('off')
        fig.suptitle(f'{population}: {experiment}');fig.tight_layout();fig.savefig(out/f'{experiment}_scatter_by_model.png',dpi=160);plt.show()
        # Same internally ranked representative in both panels; no eICU winner selection.
        selected=d.sort_values(['MIMIC_PR_AUC','Model'],ascending=[False,True]).drop_duplicates(['Feature_combo','Sampling'])
        selected.to_csv(out/f'{experiment}_heatmap_representatives.csv',index=False)
        fig,axes=plt.subplots(1,2,figsize=(15,max(5,.23*selected.Feature_combo.nunique())))
        order=selected.groupby('Feature_combo').MIMIC_PR_AUC.mean().sort_values(ascending=False).index
        for ax,metric,title in zip(axes,['MIMIC_PR_AUC','eICU_PR_AUC'],['MIMIC AP','eICU AP']):
            p=selected.pivot(index='Feature_combo',columns='Sampling',values=metric).reindex(order)
            im=ax.imshow(p.to_numpy(),aspect='auto',cmap='viridis');ax.set_xticks(range(len(p.columns)),p.columns,rotation=90,fontsize=7)
            ax.set_yticks(range(len(p)),p.index,fontsize=6);ax.set_title(title);fig.colorbar(im,ax=ax,fraction=.04)
        fig.suptitle(f'{population}: {experiment} — same MIMIC-selected model');fig.tight_layout();fig.savefig(out/f'{experiment}_heatmap.png',dpi=160);plt.show()
    # Per-model mean across combinations is descriptive, not an independent replication CI.
    fig,axes=plt.subplots(1,2,figsize=(15,5))
    for ax,(experiment,d) in zip(axes,grid.groupby('Experiment')):
        means=d.groupby(['Model','Sampling']).eICU_PR_AUC.mean().unstack('Sampling')
        for model,row in means.iterrows():ax.plot(row.index,row.to_numpy(),marker='o',label=model)
        ax.set_title(experiment);ax.set_ylabel('Mean eICU AP across model-eligible combinations');ax.tick_params(axis='x',rotation=60);ax.grid(alpha=.2)
    fig.suptitle(population);axes[-1].legend(fontsize=7);fig.tight_layout();fig.savefig(out/'sampling_means_by_model.png',dpi=160);plt.show()

In [ ]:
"""Population-specific grid, fit checkpoints and independently cached all-row CIs."""
POPULATIONS=('all_labs','icd_history')

def population_combinations(population,cfg):
    combos=grid_combinations(cfg)
    return combos if population=='icd_history' else [c for c in combos if not set(c)&set(ICD_BLOCKS)]

def population_indices(frame,cohort,population,cfg):
    age=cfg['age_policy']
    if age not in ('adults','all_ages'):raise ValueError('age_policy must be adults or all_ages')
    keep=np.ones(len(frame),bool) if age=='all_ages' else frame.age.ge(18).fillna(False).to_numpy(dtype=bool,copy=True)
    if population=='icd_history':
        keep &= np.array([ICD_RAW[cohort][int(h)].get('total_icd_ever_hx',0)>0 for h in frame.hospital_id])
    elif population!='all_labs':raise ValueError(population)
    return np.flatnonzero(keep)

def population_audit(m,e,cfg):
    rows=[]
    for cohort,frame in [('MIMIC',m),('eICU',e)]:
        for landmark,g in frame.groupby('landmark'):
            for label,d in g.groupby('label'):
                ageok=np.ones(len(d),bool) if cfg['age_policy']=='all_ages' else d.age.ge(18).fillna(False).to_numpy(bool)
                hx=np.array([ICD_RAW[cohort][int(h)].get('total_icd_ever_hx',0)>0 for h in d.hospital_id])
                rows.append(dict(cohort=cohort,landmark=landmark,label=label,lab_eligible=len(d),
                    under18=int(d.age.lt(18).sum()),unknown_age=int(d.age.isna().sum()),
                    excluded_age=int((~ageok).sum()),all_labs=int(ageok.sum()),
                    icd_history=int((ageok&hx).sum()),excluded_no_usable_icd_history=int((ageok&~hx).sum())))
    return pd.DataFrame(rows)

def grid_plan(cfg):
    ordinary=len(cfg['base_models'])+len(ENSEMBLES)*cfg['include_ensembles']
    sampling=len(sampling_plan(cfg));rows=0;cells=0;cnn=0
    for population in POPULATIONS:
        combos=population_combinations(population,cfg);cells+=2*sampling*len(combos)
        rows+=2*sampling*sum(ordinary+int('lab_current' in c and cfg['include_cnn']) for c in combos)
        cnn+=2*sampling*sum('lab_current' in c and cfg['include_cnn'] for c in combos)
    return dict(profile=cfg['grid_profile'],age_policy=cfg['age_policy'],populations=POPULATIONS,
        grid_cells=cells,expected_wide_result_rows=rows,cohort_metric_rows=rows*2,
        base_estimator_fits=cells*(cfg['outer_folds']+1)*2*len(cfg['base_models']),
        cnn_training_runs=cnn*(cfg['outer_folds']+1)*2,bootstrap_draws_per_row=cfg['n_boot'],
        note='Counts assume both populations and both landmarks pass preflight. CIs cover every row.')

def frame_digest(frame):
    h=hashlib.sha256(json.dumps(dict(columns=list(frame.columns),dtypes=[str(t) for t in frame.dtypes],
        index_dtype=str(frame.index.dtype)),sort_keys=True,default=str).encode())
    h.update(pd.util.hash_pandas_object(frame,index=True).values.tobytes());return h.hexdigest()

def fitting_fingerprint(m,ms,e,es,cfg):
    fitcfg={k:v for k,v in cfg.items() if k not in {'n_boot'}}
    h=hashlib.sha256(json.dumps(dict(schema=8,config=fitcfg,frames=[frame_digest(m),frame_digest(e)],
        demographics=DEMOGRAPHICS,lab_schema=LABS,blocks=ALL_GRID_BLOCKS,ensembles=ENSEMBLES,categories=CATEGORY_RATE_SPECS,
        BE=sorted(BE9),raw_icd=ICD_RAW,
        versions=[sklearn.__version__,torch.__version__,np.__version__,pd.__version__,
                  __import__('xgboost').__version__]),sort_keys=True,default=str).encode())
    for label,seq in [('MIMIC',ms),('eICU',es)]:
        for k in sorted(seq):
            v=np.ascontiguousarray(seq[k])
            h.update(json.dumps([label,k,v.shape,str(v.dtype)]).encode());h.update(v.view(np.uint8))
    functions=[fit_grid_partition,grid_estimator,sample_indices,GridTransform.fit,GridTransform.raw,
        selected_icd_names,train_cnn,TrainOnlyTransform.fit,TrainOnlyTransform.transform,
        SequenceTransform.fit,SequenceTransform.transform,HybridLabCNN.__init__,HybridLabCNN.forward,
        GridTransform.transform,inner_split,predict_cnn,seed_everything,youden_threshold,
        patient_folds,resolved_sampling,grid_combinations,sampling_plan,icd_block,take_sequence,
        SequenceTransform.__init__,population_indices,population_combinations,run_unified_grid]
    for f in functions:
        h.update(stable_code_bytes(f.__code__))
        h.update(repr((f.__defaults__,f.__kwdefaults__)).encode())
    return h.hexdigest()[:24]

def report_fingerprint(cfg):
    h=hashlib.sha256(json.dumps(dict(n_boot=cfg['n_boot'],seed=cfg['seed'],confidence=.95)).encode())
    for f in [metric_values,patient_bootstrap,cluster_draws,WeightedPredictionMetrics.__init__,
              WeightedPredictionMetrics.__call__,summarize_predictions,paired_landmarks]:
        h.update(stable_code_bytes(f.__code__))
    return h.hexdigest()[:24]

def prediction_token(population,landmark,combo,sampling):
    return hashlib.sha256(f'{population}|{landmark}|{combo}|{sampling}'.encode()).hexdigest()[:20]

def read_predictions(path):
    return pd.read_csv(path,dtype={'patient_id':str},float_precision='round_trip')

def run_unified_grid(m,ms,e,es,cfg,out):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    ICD_SELECTION_CACHE.clear()
    population_audit(m,e,cfg).to_csv(out/'population_eligibility.csv',index=False)
    checkpoint=out/'grid_checkpoints'/fitting_fingerprint(m,ms,e,es,cfg)
    checkpoint.mkdir(parents=True,exist_ok=True)
    tasks=[];status=[]
    # Validate all patient splits before fitting; a sparse subgroup never falls
    # back to the broader population or a globally fitted feature selection.
    for population in POPULATIONS:
        mi=population_indices(m,'MIMIC',population,cfg);ei=population_indices(e,'eICU',population,cfg)
        pm=m.iloc[mi].reset_index(drop=True);pe=e.iloc[ei].reset_index(drop=True)
        ps=take_sequence(ms,mi);pes=take_sequence(es,ei)
        try:fold_map=patient_folds(pm,cfg['outer_folds'],cfg['seed'])
        except ValueError as err:
            status.append(dict(population=population,landmark='both',status='skipped',reason=str(err)));continue
        for landmark in (1,2):
            mi=np.flatnonzero(pm.landmark.eq(landmark));ei=np.flatnonzero(pe.landmark.eq(landmark))
            mm=pm.iloc[mi].reset_index(drop=True);ee=pe.iloc[ei].reset_index(drop=True)
            ss=take_sequence(ps,mi);se=take_sequence(pes,ei);folds=mm.patient_id.map(fold_map).to_numpy()
            try:
                if mm.label.nunique()!=2 or ee.label.nunique()!=2:raise ValueError('Both cohorts need two outcome classes at this landmark')
                inner_split(mm,cfg['seed']+1000,cfg['inner_holdout'])
                for fold in range(cfg['outer_folds']):
                    tr=np.flatnonzero(folds!=fold);te=np.flatnonzero(folds==fold)
                    if not len(te) or mm.iloc[te].label.nunique()!=2:raise ValueError('Outer test fold empty or single-class; revise fold count explicitly')
                    inner_split(mm.iloc[tr],cfg['seed']+fold,cfg['inner_holdout'])
            except ValueError as err:
                status.append(dict(population=population,landmark=landmark,status='skipped',reason=str(err)));continue
            status.append(dict(population=population,landmark=landmark,status='ready',reason='',
                MIMIC_admissions=len(mm),MIMIC_cases=int(mm.label.sum()),eICU_admissions=len(ee),eICU_cases=int(ee.label.sum())))
            for combo in population_combinations(population,cfg):
                for mode,value in sampling_plan(cfg):tasks.append((population,landmark,mm,ee,ss,se,folds,combo,mode,value))
    pd.DataFrame(status).to_csv(out/'population_run_status.csv',index=False)
    for s in status:print('[population]',s,flush=True)
    if not tasks:raise ValueError('No population/landmark passed preflight; inspect population_run_status.csv')
    completed=[];audits=[];curves=[]
    for number,(population,landmark,mm,ee,ss,se,folds,combo,mode,value) in enumerate(tasks,1):
        label='+'.join(combo);sl=mode if value is None else f'{mode} {value}'
        token=prediction_token(population,landmark,label,sl);tags=dict(population=population,landmark=landmark,scope=label,sampling=sl)
        pfile=checkpoint/f'{token}.csv.gz';afile=checkpoint/f'{token}.audit.csv';lfile=checkpoint/f'{token}.learning.csv'
        ready=checkpoint/f'{token}.complete'
        print(f'[fits] {number}/{len(tasks)} {population} set {landmark} {label} {sl}',flush=True)
        if not (ready.exists() and pfile.exists() and afile.exists()):
            cp=[];ca=[];cl=[]
            for fold in list(range(cfg['outer_folds']))+[-1]:
                part=checkpoint/f'{token}.fold{fold}.pkl';mark=checkpoint/f'{token}.fold{fold}.complete'
                # Only locally generated checkpoints in this exact fingerprint namespace.
                if part.exists() and mark.exists():q,a,ll=pd.read_pickle(part)
                else:
                    tr=np.arange(len(mm)) if fold==-1 else np.flatnonzero(folds!=fold)
                    te=np.flatnonzero(folds==fold) if fold!=-1 else np.arange(len(ee))
                    train=mm.iloc[tr];target=ee if fold==-1 else mm.iloc[te]
                    t_seq=se if fold==-1 else take_sequence(ss,te)
                    if fold!=-1:assert not set(train.patient_id)&set(target.patient_id)
                    setting=resolved_sampling(mode,value,train.label,ee.label)
                    pp,tt,ll,aa=fit_grid_partition(train,take_sequence(ss,tr),target,t_seq,
                        'eICU' if fold==-1 else 'MIMIC',combo,setting,cfg,cfg['seed']+(1000 if fold==-1 else fold),
                        checkpoint_dir=checkpoint/f'{token}.fold{fold}.models')
                    q=pd.concat([target[['patient_id','hospital_id','label']].copy().assign(**tags,
                        cohort='eICU_external' if fold==-1 else 'MIMIC_OOF',model=model,fold=fold,
                        probability=prob,threshold=tt[model]) for model,prob in pp.items()],ignore_index=True)
                    op=json.loads(aa['operating_thresholds_json'])
                    for target_recall in cfg['recall_targets']:
                        key=operating_key(target_recall)
                        q[key]=q.model.map({model:values[key] for model,values in op.items()})
                    if not np.isfinite(q.probability).all():raise ValueError('Nonfinite predictions')
                    a=dict(**tags,fold=fold,**aa);ll=[dict(**tags,fold=fold,**l) for l in ll]
                    pd.to_pickle((q,a,ll),part);mark.write_text('complete')
                cp.append(q);ca.append(a);cl.extend(ll)
            p=pd.concat(cp,ignore_index=True);a=pd.DataFrame(ca);ll=pd.DataFrame(cl)
            assert not p.duplicated(['cohort','hospital_id','model']).any()
            p.to_csv(pfile,index=False);a.to_csv(afile,index=False)
            if not ll.empty:ll.to_csv(lfile,index=False)
            ready.write_text('complete')
        audits.append(pd.read_csv(afile))
        if lfile.exists():curves.append(pd.read_csv(lfile))
        completed.append(dict(**tags,token=token,external_prevalence_used=mode=='prevalence_match' and value in ('eicu','midpoint')))
    # A separate phase: changing n_boot never retrains models. Every row is covered.
    reports=checkpoint/'reports'/report_fingerprint(cfg);reports.mkdir(parents=True,exist_ok=True)
    rows=[]
    for number,task in enumerate(completed,1):
        token=task['token'];file=reports/f'{token}.metrics.csv';mark=reports/f'{token}.complete'
        print(f'[all-row CIs] {number}/{len(completed)}',flush=True)
        if file.exists() and mark.exists():met=pd.read_csv(file,float_precision='round_trip')
        else:
            met=summarize_predictions(read_predictions(checkpoint/f'{token}.csv.gz'),cfg)
            met['external_prevalence_used']=task['external_prevalence_used']
            met.to_csv(file,index=False);mark.write_text('complete')
        rows.append(met)
    metrics=pd.concat(rows,ignore_index=True);audit=pd.concat(audits,ignore_index=True)
    learning=pd.concat(curves,ignore_index=True) if curves else pd.DataFrame()
    wide=wide_grid_results(metrics);wide.to_csv(out/'feature_sampling_grid.csv',index=False)
    metrics.to_csv(out/'unified_grid_metrics_long.csv',index=False);audit.to_csv(out/'unified_grid_fit_audit.csv',index=False)
    learning.to_csv(out/'unified_grid_learning_curves.csv',index=False)
    paired=[];tasks_frame=pd.DataFrame(completed)
    for _,g in tasks_frame.groupby(['population','scope','sampling'],sort=False):
        if set(g.landmark)!={1,2}:continue
        file=reports/f'{g.iloc[0].token}.paired.pkl';mark=file.with_suffix('.complete')
        if file.exists() and mark.exists():q=pd.read_pickle(file)
        else:
            p=pd.concat([read_predictions(checkpoint/f'{t}.csv.gz') for t in g.token],ignore_index=True)
            q=paired_landmarks(p,cfg);q.to_pickle(file);mark.write_text('complete')
        paired.append(q)
    paired=pd.concat(paired,ignore_index=True) if paired else pd.DataFrame()
    paired.to_csv(out/'paired_landmark_grid.csv',index=False)
    pd.DataFrame(completed).to_csv(checkpoint/'prediction_index.csv',index=False)
    return wide,metrics,paired,audit,learning,checkpoint

In [ ]:
def integrated_plots(grid,paired,learning,out):
    for population,g in grid.groupby('population'):
        destination=Path(out)/population;destination.mkdir(parents=True,exist_ok=True)
        print('Plots for population:',population)
        integrated_plots_one_population(g,paired,learning,destination,population=population)

## Checkpoints and execution

Compatible completed raw-feature caches are read from v9/v8 or the v10 primary run. Changed
release delays get different feature fingerprints and rebuild extraction and summaries.
Every scenario writes to its own V14 directory; preprocessing, sampling and model checkpoints
are specific to its fingerprint. Model-fit checkpoints, full-fold checkpoints and all-row CI
checkpoints remain separate. A restart can resume completed work in the same configuration.
The raw CSV extraction scan itself is not resumable mid-scan.

One patient-level outer-fold map is defined from eligible source MIMIC admissions before lab
eligibility, then reused across age/panel/lag policies. All learned preprocessing and feature
selection still fit within each training partition. Sparse subgroups fail preflight rather
than silently changing cohorts. Inner splits remain training-only and can differ when training
membership changes; cross-policy results are policy comparisons, not isolated causal effects.

V14 additionally checks the V13 primary raw-feature cache and exact-fingerprint per-model predictions. Tabular and CNN model outputs occupy separate namespaces. Existing V13 outputs remain intact.


In [ ]:
def _build_features_batch(admissions,long,cohort,cfg,context):
    """History summaries do not invent between-stay time gaps in eICU."""
    current,patients,stays=context
    names=list(LABS);n_bins=int(np.ceil(cfg['cnn_window_hours']/cfg['cnn_bin_hours']))
    rows=[];metadata_rows=[];feature_names=None;matrices=[];valids=[];event_times=[];cutoffs=[];flow=Counter()
    for r in admissions.itertuples(index=False):
        flow['source_admissions']+=1
        if not r.eligible:flow['ineligible_admissions']+=1;continue
        z=current.get(int(r.hospital_id),pd.DataFrame())
        if z.empty:flow['no_current_labs']+=1;continue
        z=z.copy()
        z['hour']=(z.time-r.start).dt.total_seconds()/3600 if cohort=='MIMIC' else z.time
        z['release_hour']=(z.available-r.start).dt.total_seconds()/3600 if cohort=='MIMIC' else z.available
        duration=(r.end-r.start).total_seconds()/3600 if cohort=='MIMIC' else r.duration
        z=z[z.hour.ge(0)&z.hour.le(duration)]
        sets=draw_landmarks(z,cfg)
        if not sets:flow['no_qualifying_first_set']+=1;continue
        ph=patients[str(r.patient_id)]
        if cohort=='MIMIC':
            earlier=stays[str(r.patient_id)]
            earlier_ids=earlier.loc[earlier.end.lt(r.start)&earlier.start.lt(r.start)&earlier.valid_time,'hospital_id']
            hist=ph[(ph.hospital_id.isin(earlier_ids)|ph.hospital_id.isna())&ph.available.lt(r.start)&ph.time.lt(r.start)].copy()
            hist['hour']=(hist.time-pd.Timestamp('1970-01-01')).dt.total_seconds()/3600
            hist['episode']=hist.hospital_id.astype('string').fillna('outpatient_'+hist.time.dt.strftime('%Y-%m-%d'))
        else:
            earlier=stays[str(r.patient_id)]
            earlier_ids=earlier.loc[earlier.year.lt(r.year)&earlier.valid_time,'hospital_id']
            hist=ph[ph.hospital_id.isin(earlier_ids)].copy()
            ends=earlier.set_index('hospital_id').duration
            hist=hist[hist.available.le(hist.hospital_id.map(ends))].copy()
            hist=resolve_revisions(hist,['hospital_id','analyte','time'])
            hist['hour']=hist.time;hist['episode']=hist.hospital_id.astype(str)
        hd={};hs={}; history_groups=dict(tuple(hist.groupby('analyte')))
        for a in names:
            az=history_groups.get(a,hist.iloc[:0])
            # No pooled first/last/slope across uncertain historical stay order.
            vals=az.value.to_numpy(float)
            h={k:np.nan for k in ('mean','median','min','max','iqr','stay_mean','within_stay_slope_mean','within_stay_slope_median')}
            h.update(count=float(len(vals)),stays=float(az.episode.nunique()))
            if len(vals):
                h.update(mean=vals.mean(),median=np.median(vals),min=vals.min(),max=vals.max(),iqr=np.quantile(vals,.75)-np.quantile(vals,.25))
                ss=[series_stats(g,cfg['history_slope_span_hours']) for _,g in az.groupby('episode')]
                h['stay_mean']=float(np.mean([s['mean'] for s in ss]))
                slopes=[s['slope_per_day'] for s in ss if s['slope_valid']]
                if slopes:h.update(within_stay_slope_mean=float(np.mean(slopes)),within_stay_slope_median=float(np.median(slopes)))
            hd.update({f'history__{a}__{k}':v for k,v in h.items()});hs[a]=h
        for landmark,(draw_start,draw_end) in enumerate(sets,1):
            score_time=draw_end+cfg['release_lag_hours']
            if score_time>duration:flow[f'landmark_{landmark}_after_discharge']+=1;continue
            visible=z[z.hour.lt(draw_end)&z.release_hour.le(score_time)].copy()
            visible=resolve_revisions(visible,['hospital_id','analyte','hour'])
            features=dict(hd)
            features.update(age=r.age,age_missing=float(not np.isfinite(r.age)),sex_male=float(r.sex=='M'),
                            sex_unknown=float(r.sex not in ('M','F')),age_topcoded=float(r.age_topcoded),
                            current__elapsed_hours=score_time,history__available=float(len(hist)>0))
            seq=np.full((n_bins,len(names)),np.nan,np.float32)
            seq_time=np.full_like(seq,np.nan)
            valid=(np.arange(n_bins)*cfg['cnn_bin_hours']<draw_end).astype(np.float32)
            visible_groups=dict(tuple(visible.groupby('analyte')))
            for j,a in enumerate(names):
                az=visible_groups.get(a,visible.iloc[:0]);s=series_stats(az,cfg['min_slope_span_hours'])
                features.update({f'current__{a}__{k}':v for k,v in s.items()})
                lo,hi=reference_interval(a,r.age,r.sex,r.age_topcoded)
                bf=boundary_features(s['first'],s['last'],s['span_hours'] if s['slope_valid'] else np.nan,lo,hi, a in ('crp','esr'))
                features.update({f'current__{a}__{k}':v for k,v in bf.items()})
                features[f'combined__{a}__delta_history_median']=s['last']-hs[a]['median']
                # Baseline-to-current is a difference, NOT a rate without a known time gap.
                for q in az.sort_values('hour').itertuples():
                    b=int(q.hour//cfg['cnn_bin_hours'])
                    if 0<=b<n_bins:seq[b,j]=q.value;seq_time[b,j]=q.hour
            # Avoid retaining a thousand freshly allocated string keys per admission.
            # Fixed-schema float arrays markedly reduce full-cohort memory use.
            if feature_names is None:feature_names=list(features)
            if list(features)!=feature_names:raise AssertionError('Inconsistent feature schema')
            rows.append(np.fromiter(features.values(),dtype=np.float32,count=len(features)))
            metadata_rows.append(dict(patient_id=str(r.patient_id),hospital_id=int(r.hospital_id),label=int(r.label),
                                      landmark=landmark,score_hour=score_time))
            matrices.append(seq);valids.append(valid);event_times.append(seq_time);cutoffs.append(draw_end)
            flow[f'landmark_{landmark}_rows']+=1
    if not rows:return None,None,dict(flow)
    frame=pd.concat([pd.DataFrame(metadata_rows),pd.DataFrame(np.stack(rows),columns=feature_names)],axis=1)
    return frame,dict(values=np.stack(matrices),valid=np.stack(valids),
                      hours=np.stack(event_times),cutoff=np.array(cutoffs)),dict(flow)

"""Local resumable raw features; never store fitted preprocessing in this cache."""
import os
import uuid
import shutil


def _v9_atomic(path, writer):
    path=Path(path);path.parent.mkdir(parents=True,exist_ok=True)
    temp=path.with_name('.'+uuid.uuid4().hex+'.'+path.name)
    try:
        writer(temp)
        os.replace(temp,path)
    finally:
        if temp.exists():temp.unlink()


def _v9_pickle(path,value):
    compression={'method':'gzip','compresslevel':1} if str(path).endswith('.gz') else None
    _v9_atomic(path,lambda p:pd.to_pickle(value,p,compression=compression))


def _v9_read_pickle(path):
    # Trusted, locally produced checkpoints ONLY; never load downloaded pickles.
    return pd.read_pickle(path,compression='gzip' if str(path).endswith('.gz') else None)


def _v9_load_final(cache,fingerprint):
    required=['MIMIC.csv.gz','eICU.csv.gz','MIMIC.npz','eICU.npz','complete.json']
    if not all((cache/n).is_file() for n in required):return None
    saved=json.loads((cache/'complete.json').read_text())
    if saved['fingerprint']!=fingerprint:raise ValueError('Feature-cache fingerprint mismatch')
    frames=[];sequences=[]
    for cohort in ('MIMIC','eICU'):
        frames.append(pd.read_csv(cache/f'{cohort}.csv.gz',dtype=saved['dtypes'][cohort],float_precision='round_trip'))
        with np.load(cache/f'{cohort}.npz',allow_pickle=False) as z:sequences.append({k:z[k] for k in z.files})
    print(f'[features] Reusing completed cache: {cache}',flush=True)
    return frames[0],sequences[0],frames[1],sequences[1],saved['audit']


def build_features_checkpointed(admissions,long,cohort,cfg,folder,batch_size=250):
    """Batch only prediction admissions, retaining FULL patient history context."""
    if not isinstance(batch_size,int) or isinstance(batch_size,bool) or batch_size<1:
        raise ValueError('FEATURE_BATCH_SIZE must be a positive integer')
    folder=Path(folder)/f'batch_{batch_size}';folder.mkdir(parents=True,exist_ok=True)
    # The parent folder binds inputs, configuration, code and runtime versions.
    # Each atomic shard contains both results and metadata; no partial shard is accepted.
    total=len(admissions);context=None;parts=[];seqparts=[];flow=Counter()
    for start in range(0,total,batch_size):
        stop=min(start+batch_size,total);path=folder/f'{start:09d}_{stop:09d}.pkl'
        if path.exists():
            saved=_v9_read_pickle(path)
            if saved['range']!=(start,stop) or saved['cohort']!=cohort:raise ValueError('Batch metadata mismatch')
            frame,seq,counts=saved['result'];action='resumed'
        else:
            if context is None:
                print(f'[features {cohort}] Preparing full-history lookup tables...',flush=True)
                context=(
                    {int(h):g for h,g in long.dropna(subset=['hospital_id']).groupby('hospital_id',sort=False)},
                    {str(p):g for p,g in long.groupby('patient_id',sort=False)},
                    {str(p):g for p,g in admissions.groupby('patient_id',sort=False)})
            frame,seq,counts=_build_features_batch(admissions.iloc[start:stop],long,cohort,cfg,context)
            _v9_pickle(path,dict(range=(start,stop),cohort=cohort,result=(frame,seq,counts)))
            action='saved'
        flow.update(counts)
        if frame is not None:parts.append(frame);seqparts.append(seq)
        print(f'[features {cohort}] {stop:,}/{total:,} admissions ({action}); '
              f"landmark rows={flow['landmark_1_rows']+flow['landmark_2_rows']:,}",flush=True)
    if not parts:raise ValueError(f'{cohort}: no eligible landmark rows')
    return pd.concat(parts,ignore_index=True),{k:np.concatenate([s[k] for s in seqparts]) for k in seqparts[0]},dict(flow)


def _v9_publish_final(cache,fingerprint,manifest,m,ms,e,es,audit):
    for cohort,frame,seq in [('MIMIC',m,ms),('eICU',e,es)]:
        _v9_atomic(cache/f'{cohort}.csv.gz',lambda p:frame.to_csv(p,index=False,compression='gzip'))
        _v9_atomic(cache/f'{cohort}.npz',lambda p:np.savez_compressed(p,**seq))
    saved=dict(fingerprint=fingerprint,manifest=manifest,audit=audit,
               dtypes={c:{k:str(v) for k,v in d.dtypes.items()} for c,d in [('MIMIC',m),('eICU',e)]})
    _v9_atomic(cache/'complete.json',lambda p:p.write_text(json.dumps(saved,indent=2,default=str)))


def v9_input_manifest(root,cfg):
    manifest,fingerprint=input_manifest(root,cfg)
    actual=hashlib.sha256(b''.join(stable_code_bytes(f.__code__) for f in
        [_build_features_batch,build_features_checkpointed])).hexdigest()
    # Only the tested, unchanged batch implementation can share v8's final key.
    # Manual changes to batch calculations automatically use a new cache namespace.
    if actual!=V9_COMPATIBLE_BUILD_CODE_SHA256:
        manifest=dict(manifest,feature_execution_code_sha256=actual)
        fingerprint=hashlib.sha256((fingerprint+actual).encode()).hexdigest()
    return manifest,fingerprint


def prepare_experiment_v9(root,out,cfg,use_cache=True):
    validate_config(cfg);out=Path(out)
    out.mkdir(parents=True,exist_ok=True)
    manifest,fingerprint=v9_input_manifest(root,cfg)
    cache=out/'feature_cache'/fingerprint[:16]
    if use_cache:
        for candidate in [cache]+[Path(p)/'feature_cache'/fingerprint[:16] for p in LEGACY_CACHE_DIRS]:
            restored=_v9_load_final(candidate,fingerprint)
            if restored is not None:return restored
    # Preserve the v8 final-feature key because feature values are unchanged.
    # Independently version the NEW execution checkpoints to invalidate changed helpers.
    runtime=[np.__version__,pd.__version__]
    code=b''.join(stable_code_bytes(f.__code__) for f in
        [_build_features_batch,build_features_checkpointed,prepare_experiment_v9])
    key=hashlib.sha256(fingerprint.encode()+repr(runtime).encode()+code).hexdigest()[:24]
    stage=out/'feature_build_checkpoints'/key
    if not use_cache:
        # Explicit cache-off never reads or replaces prior partial checkpoints.
        stage=stage/('fresh_'+uuid.uuid4().hex)
    stage.mkdir(parents=True,exist_ok=True)
    def obtain(name,create):
        path=stage/(name+'.pkl.gz')
        if path.exists():
            print(f'[features] Resuming {name}',flush=True);return _v9_read_pickle(path)
        print(f'[features] Building {name}',flush=True)
        result=create();_v9_pickle(path,result)
        print(f'[features] Saved {name}',flush=True);return result
    # If BOTH cohorts completed, avoid reloading the much larger raw lab checkpoint.
    done={c:stage/(c+'_features.pkl.gz') for c in ('MIMIC','eICU')}
    meta=stage/'audit_base.pkl.gz'
    if all(p.exists() for p in done.values()) and meta.exists():
        source,unit_audit=_v9_read_pickle(meta)
        m,ms,mflow=_v9_read_pickle(done['MIMIC']);e,es,eflow=_v9_read_pickle(done['eICU'])
        print('[features] Both completed cohort checkpoints restored',flush=True)
    else:
        ma,ea,ep=obtain('admissions',lambda:load_admissions(root))
        source=[dict(cohort=c,source_admissions=len(d),eligible_admissions=int(d.eligible.sum()),
                     eligible_cases=int(d.loc[d.eligible,'label'].sum()),
                     eligible_patients=d.loc[d.eligible,'patient_id'].nunique()) for c,d in [('MIMIC',ma),('eICU',ea)]]
        ml,el,unit_audit=obtain('extracted_labs',lambda:extract_labs(root,ma,ea,ep,cfg))
        _v9_pickle(meta,(source,unit_audit))
        m,ms,mflow=obtain('MIMIC_features',lambda:build_features_checkpointed(ma,ml,'MIMIC',cfg,stage/'MIMIC',FEATURE_BATCH_SIZE))
        del ml
        e,es,eflow=obtain('eICU_features',lambda:build_features_checkpointed(ea,el,'eICU',cfg,stage/'eICU',FEATURE_BATCH_SIZE))
        del el
    audit=dict(source=source,flow={'MIMIC':mflow,'eICU':eflow},units=unit_audit)
    _v9_publish_final(cache,fingerprint,manifest,m,ms,e,es,audit)
    print(f'[features] Complete cache saved: {cache}',flush=True)
    return m,ms,e,es,audit


def import_v8_grid_checkpoints(m,ms,e,es,cfg,out):
    """Read-only import of completed, exactly compatible v8 training units."""
    fingerprint=fitting_fingerprint(m,ms,e,es,cfg)
    target=Path(out)/'grid_checkpoints'/fingerprint
    copied=0
    for legacy in LEGACY_CACHE_DIRS:
        source=Path(legacy)/'grid_checkpoints'/fingerprint
        if not source.is_dir() or source.resolve()==target.resolve():continue
        for mark in source.rglob('*.complete'):
            rel=mark.relative_to(source);dest=target/rel
            if dest.exists():continue
            stem=mark.name[:-len('.complete')]
            if mark.parent==source:
                required=[mark.with_name(stem+'.pkl')] if '.fold' in stem else [mark.with_name(stem+'.csv.gz'),mark.with_name(stem+'.audit.csv')]
                optional=[mark.with_name(stem+'.learning.csv')] if '.fold' not in stem else []
            elif '.paired' in stem:required=[mark.with_name(stem+'.pkl')];optional=[]
            else:required=[mark.with_name(stem+'.metrics.csv')];optional=[]
            if not all(p.is_file() for p in required):continue
            for p in required+[p for p in optional if p.is_file()]:
                _v9_atomic(target/p.relative_to(source),lambda t,p=p:shutil.copyfile(p,t))
            _v9_atomic(dest,lambda t:t.write_text('complete'))
            copied+=1
    print(f'[resume] Imported {copied} completed compatible v8 grid checkpoints',flush=True)

V9_COMPATIBLE_BUILD_CODE_SHA256='5dd9f88e70edaa43323d3f30d8b9281a7bb1d5e48585eb000a89c4b763d47c8b'


## Build once: laboratory features, tensors and aligned historical ICD

The same-run feature bank is joined by cohort, hospital ID and patient ID—not row position. Label equality is checked when attaching ICD history to lab landmarks. CSVs contain summaries and results; multidimensional CNN arrays use NPZ with masks/timing information. Neither cache stores a global imputer, scaler or feature selection. `model_cohort_ids.csv` is not required.

Feature caches detect changes to their input files, feature configuration and feature-building code. Grid checkpoints also fingerprint feature/tensor contents, raw ICD inputs, model configuration and fitting code. All output paths are separate from the earlier notebooks' output directories.

## Optional L1 fitting contract

The weighted logistic-loss plus L1 penalty, C=1, and penalized unit-intercept convention are retained.
Fits are accepted only at the configured relative primal-dual gap (0.0001 by default).
The active resumable solver is defined below; its runtime controls are `LASSO_RUNTIME`,
not the compatibility-era CFG time entries. Each base model's inner and outer predictions
is checkpointed separately. An uncertified fit is never saved as successful.

V15 runs this solver only if `V15_L1_ANCHORS=True`, at four primary anchors. It is disabled by default.


In [ ]:
"""Same weighted L1 logistic objective, with feature-space Newton subproblems."""
from sklearn.base import BaseEstimator,ClassifierMixin
from sklearn.linear_model import lasso_path
from sklearn.exceptions import ConvergenceWarning
from scipy.special import expit,xlogy
from threadpoolctl import threadpool_limits
import warnings


class WeightedL1LogisticGram(ClassifierMixin,BaseEstimator):
    """Binary L1 logistic regression with the LIBLINEAR C/intercept convention.

    Minimize sum_i weight_i * logistic_loss_i + ||[coef, intercept]||_1 / C.
    Solve weighted quadratic Newton subproblems using sklearn's Gram-coordinate
    descent; backtrack on the ORIGINAL logistic objective. Accept a fit only
    after an independently computed feasible dual bound closes the primal-dual
    gap. No outcome-based feature removal or changed regularization strength.
    """
    def __init__(self,C=1.,class_weight=None,max_iter=60,max_seconds=240.,gap_tol=1e-4,threads=2):
        self.C=C;self.class_weight=class_weight;self.max_iter=max_iter
        self.max_seconds=max_seconds;self.gap_tol=gap_tol;self.threads=threads

    @staticmethod
    def _objective(x,y,w,beta,penalty):
        z=x@beta
        return float(np.sum(w*(np.logaddexp(0,z)-y*z))+penalty*np.abs(beta).sum())

    @staticmethod
    def _certificate(x,y,w,beta,penalty):
        residual=w*(expit(x@beta)-y)
        largest=float(np.max(np.abs(x.T@residual)))
        alpha=min(1.,penalty/max(largest,1e-300))
        q=np.clip(y+alpha*residual/w,0.,1.)
        dual=float(-np.sum(w*(xlogy(q,q)+xlogy(1-q,1-q))))
        primal=WeightedL1LogisticGram._objective(x,y,w,beta,penalty)
        return primal,dual,max(0.,primal-dual)/max(1.,abs(primal))

    def fit(self,X,y,sample_weight=None):
        started=time.perf_counter();last_report=started
        X=np.asarray(X,dtype=np.float64);y=np.asarray(y,dtype=np.float64)
        if X.ndim!=2 or y.shape!=(len(X),) or not np.isfinite(X).all():raise ValueError('Invalid finite training matrix')
        if set(np.unique(y))!={0.,1.}:raise ValueError('Both binary classes required')
        if self.C<=0 or self.max_seconds<=0 or self.max_iter<1 or self.gap_tol<=0:raise ValueError('Invalid L1 settings')
        w=np.ones(len(y)) if sample_weight is None else np.asarray(sample_weight,dtype=float).copy()
        if w.shape!=y.shape or not np.isfinite(w).all() or np.any(w<=0):raise ValueError('Weights must be finite and positive')
        if self.class_weight=='balanced':
            total=w.sum()
            factors={label:total/(2*w[y==label].sum()) for label in (0.,1.)}
            for label in (0.,1.):w[y==label]*=factors[label]
        elif self.class_weight is not None:raise ValueError('Use balanced or None')
        self.n_features_in_=X.shape[1];self.classes_=np.array([0,1]);self.converged_=False
        x=np.asfortranarray(np.column_stack([X,np.ones(len(X))]));penalty=1./self.C
        beta=np.zeros(x.shape[1]);self.optimization_trace_=[]
        def check_time():
            if time.perf_counter()-started>self.max_seconds:
                raise TimeoutError(f'Lasso exceeded {self.max_seconds:g}s; fit not certified or saved. Completed model checkpoints remain reusable.')
        with threadpool_limits(limits=self.threads):
            for iteration in range(1,self.max_iter+1):
                check_time();z=x@beta;prob=expit(z)
                curvature=np.maximum(prob*(1-prob),1e-8);h=w*curvature
                target=z+(y-prob)/curvature
                # Form the small feature-space system in blocks, avoiding a second
                # full dense weighted design matrix in this memory-limited setting.
                gram=np.zeros((x.shape[1],x.shape[1]));xy=np.zeros(x.shape[1])
                for start in range(0,len(x),2048):
                    check_time();end=min(start+2048,len(x));a=x[start:end]
                    gram+=a.T@(h[start:end,None]*a)
                    xy+=a.T@(h[start:end]*target[start:end])
                gram=np.ascontiguousarray(gram);xy=np.ascontiguousarray(xy)
                weighted_target=np.ascontiguousarray(np.sqrt(h)*target)
                candidate=beta.copy();inner_iterations=0
                for chunk in range(10):
                    check_time()
                    # X is used for dimensions; the validated weighted Gram and Xy
                    # supply the quadratic system. No centering/intercept is added.
                    with warnings.catch_warnings(record=True) as caught:
                        warnings.simplefilter('always',ConvergenceWarning)
                        _,coef,_,nit=lasso_path(x,weighted_target,alphas=[penalty/len(y)],
                            precompute=gram,Xy=xy,coef_init=candidate,copy_X=False,
                            max_iter=10000,tol=min(1e-7,self.gap_tol*1e-3),check_input=False,return_n_iter=True)
                    candidate=coef[:,0].copy();inner_iterations+=int(nit[0])
                    other=[q for q in caught if not issubclass(q.category,ConvergenceWarning)]
                    for q in other:warnings.warn(str(q.message),q.category)
                    if not any(issubclass(q.category,ConvergenceWarning) for q in caught):break
                direction=candidate-beta;before=self._objective(x,y,w,beta,penalty);step=1.
                while step>=2**-20:
                    check_time()
                    if self._objective(x,y,w,beta+step*direction,penalty)<=before:break
                    step*=.5
                if step<2**-20:raise RuntimeError('Lasso line search stalled; uncertified fit not saved')
                beta+=step*direction
                primal,dual,gap=self._certificate(x,y,w,beta,penalty)
                elapsed=time.perf_counter()-started
                self.optimization_trace_.append(dict(iteration=iteration,seconds=elapsed,
                    objective=primal,dual_bound=dual,relative_dual_gap=gap,inner_iterations=inner_iterations))
                if time.perf_counter()-last_report>=10 or gap<=self.gap_tol:
                    print(f'[Lasso] step {iteration}, {elapsed:.1f}s, certified objective gap {gap:.3g}',flush=True)
                    last_report=time.perf_counter()
                if gap<=self.gap_tol:
                    self.converged_=True;break
        if not self.converged_:raise RuntimeError(f'Lasso did not reach the required objective gap: {gap:g}; fit not saved')
        self.coef_=beta[:-1][None,:];self.intercept_=np.array([beta[-1]])
        self.n_iter_=np.array([iteration]);self.fit_seconds_=time.perf_counter()-started
        self.relative_dual_gap_=gap;self.objective_=primal
        return self

    def decision_function(self,X):
        if not getattr(self,'converged_',False):raise RuntimeError('Lasso is not certified fitted')
        X=np.asarray(X,dtype=float)
        if X.ndim!=2 or X.shape[1]!=self.n_features_in_:raise ValueError('Feature mismatch')
        return X@self.coef_[0]+self.intercept_[0]

    def predict_proba(self,X):
        p=expit(self.decision_function(X));return np.column_stack([1-p,p])

    def predict(self,X):return (self.decision_function(X)>0).astype(int)


# Reexecuting this cell must not wrap an already wrapped function recursively.
if '_GRID_ESTIMATOR_BEFORE_LASSO_FIX' not in globals():
    _GRID_ESTIMATOR_BEFORE_LASSO_FIX=grid_estimator
    _FINGERPRINT_BEFORE_LASSO_FIX=fitting_fingerprint

def grid_estimator(name,y,setting,cfg,seed):
    if name=='Lasso':
        return WeightedL1LogisticGram(C=1.,class_weight='balanced' if setting['mode']=='none' else None,
            max_iter=cfg['lasso_newton_steps'],max_seconds=cfg['lasso_max_seconds'],
            gap_tol=cfg['lasso_objective_gap'],threads=min(2,cfg['torch_threads']))
    return _GRID_ESTIMATOR_BEFORE_LASSO_FIX(name,y,setting,cfg,seed)


def _fit_model_checkpoint(name,x,y,target,setting,cfg,seed,stage,folder):
    path=None if folder is None else Path(folder)/f'{name}_{stage}.pkl'
    if path is not None and path.is_file():
        saved=_v9_read_pickle(path)
        if saved['rows']!=len(target):raise ValueError('Model-checkpoint target length mismatch')
        print(f'[model] {name} {stage}: resumed',flush=True)
        return saved['probs'],saved['audit']
    print(f'[model] {name} {stage}: fitting {len(y):,} rows x {x.shape[1]} columns',flush=True)
    started=time.perf_counter();model=grid_estimator(name,y,setting,cfg,seed).fit(x,y)
    probs=model.predict_proba(target)[:,1]
    if not np.isfinite(probs).all():raise ValueError('Nonfinite model probabilities')
    audit=dict(model=name,stage=stage,seconds=time.perf_counter()-started,
               iterations=np.asarray(getattr(model,'n_iter_',[])).tolist(),
               lasso_relative_dual_gap=getattr(model,'relative_dual_gap_',None))
    if path is not None:_v9_pickle(path,dict(rows=len(target),probs=probs,audit=audit))
    print(f"[model] {name} {stage}: saved after {audit['seconds']:.1f}s",flush=True)
    return probs,audit


def fitting_fingerprint(m,ms,e,es,cfg):
    base=_FINGERPRINT_BEFORE_LASSO_FIX(m,ms,e,es,cfg)
    functions=[WeightedL1LogisticGram.__init__,WeightedL1LogisticGram.fit,
        WeightedL1LogisticGram._objective,WeightedL1LogisticGram._certificate,
        WeightedL1LogisticGram.decision_function,WeightedL1LogisticGram.predict_proba,
        _fit_model_checkpoint]
    digest=hashlib.sha256(base.encode()+b''.join(stable_code_bytes(f.__code__) for f in functions))
    return digest.hexdigest()[:24]


CFG.update(lasso_newton_steps=60,lasso_max_seconds=240.,lasso_objective_gap=1e-4)
print('[solver patch] Certified feature-space L1 solver enabled; completed feature caches remain compatible.',flush=True)


## V10 comparison and reporting functions

The diagnostic controls contain no lab values. Scenario selection is prespecified and not chosen using external metrics.

In [ ]:
"""V10 prespecified comparisons. No live-kernel mutation or external-data tuning."""
from scipy.optimize import minimize
from scipy.special import expit, logit

if '_V9_GRID_TRANSFORM' not in globals():
    _V9_GRID_TRANSFORM=GridTransform
    _V9_GRID_COMBINATIONS=grid_combinations
    _V9_PATIENT_FOLDS=patient_folds
    _V9_FITTING_FINGERPRINT=fitting_fingerprint
    _V9_REPORT_FINGERPRINT=report_fingerprint
    _V9_WIDE_RESULTS=wide_grid_results
    V10_PATIENT_FOLD_MAP={}
    V10_FOLD_DIGEST=None
    V10_FOLD_SPEC=None
OBS_BLOCKS=('observation_current','observation_history')

def initialize_study_folds(root,cfg,out):
    """One patient assignment from eligible source admissions, before lab policies."""
    global V10_PATIENT_FOLD_MAP,V10_FOLD_DIGEST,V10_FOLD_SPEC
    ma,_,_=load_admissions(root)
    frame=ma.loc[ma.eligible,['patient_id','label']]
    V10_PATIENT_FOLD_MAP=_V9_PATIENT_FOLDS(frame,cfg['outer_folds'],cfg['seed'])
    V10_FOLD_DIGEST=hashlib.sha256(json.dumps(sorted(V10_PATIENT_FOLD_MAP.items())).encode()).hexdigest()
    V10_FOLD_SPEC=(cfg['outer_folds'],cfg['seed'])
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    pd.DataFrame(sorted(V10_PATIENT_FOLD_MAP.items()),columns=['patient_id','fold']).to_csv(out/'patient_fold_assignment.csv',index=False)
    return V10_FOLD_DIGEST

def patient_folds(frame,n_folds,seed):
    if not V10_PATIENT_FOLD_MAP:raise RuntimeError('Run initialize_study_folds before the grid.')
    if V10_FOLD_SPEC!=(n_folds,seed):raise ValueError('Seed/fold count changed; rerun study setup')
    absent=set(frame.patient_id.astype(str))-set(V10_PATIENT_FOLD_MAP)
    if absent:raise ValueError('Patients absent from the fixed study fold map; rebuild the study setup.')
    if any(not 0<=v<n_folds for v in V10_PATIENT_FOLD_MAP.values()):raise ValueError('Fold-count mismatch')
    return {str(p):V10_PATIENT_FOLD_MAP[str(p)] for p in frame.patient_id.unique()}

def panel_sequence(seq,cfg):
    panel=cfg.get('lab_panel','expanded')
    if panel=='expanded':return seq
    if panel!='core':raise ValueError('lab_panel must be expanded or core')
    columns=[list(LABS).index(a) for a in CORE]
    if seq['values'].shape[2]!=len(LABS):raise ValueError('Expected the full cached lab tensor before panel selection')
    return {k:(v[:,:,columns] if k in ('values','hours') else v) for k,v in seq.items()}

class GridTransform(_V9_GRID_TRANSFORM):
    def fit(self,train,combo,cfg,seed,cnn=False):
        self.observation_only=bool(set(combo)&set(OBS_BLOCKS))
        self.panel=cfg.get('lab_panel','expanded')
        self.requested_combo=tuple(combo)
        if self.observation_only and not set(combo)<=set(OBS_BLOCKS):raise ValueError('Observation controls cannot mix with value blocks')
        if 'demographics' in combo and tuple(combo)!=('demographics',):raise ValueError('Standalone demographics control only')
        if (self.observation_only or 'demographics' in combo) and not cfg['include_demographics']:
            raise ValueError('Diagnostic controls require include_demographics=True')
        mapped=tuple({'observation_current':'lab_current','observation_history':'lab_history'}.get(b,b) for b in combo)
        return super().fit(train,mapped,cfg,seed,cnn=cnn)
    def raw(self,frame,cohort):
        pieces=[]
        if self.icd_presence:
            rows=[ICD_RAW[cohort][int(h)] for h in frame.hospital_id]
            pieces.append(np.array([[r.get(n,0.) for n in self.icd_names] for r in rows],np.float32))
            pieces.append(np.array([not r.get('total_icd_ever_hx',0) for r in rows],float)[:,None])
        for name in self.lab_names:
            tokens=name.split('__')
            if self.panel=='core' and len(tokens)>2 and tokens[1] in LABS and tokens[1] not in CORE:continue
            if self.panel=='core' and name=='history__available':
                a=frame[[f'history__{lab}__count' for lab in CORE]].gt(0).any(axis=1).to_numpy(float)
            else:a=frame[name].to_numpy(float)
            if not self.observation_only or name in DEMOGRAPHICS:
                pieces.append(a[:,None]);continue
            if name in ('history__available','current__elapsed_hours') or tokens[-1] in ('count','stays','span_hours','slope_valid'):
                pieces.append(a[:,None])
            elif (tokens[0]=='current' and tokens[-1]=='last') or (tokens[0]=='history' and tokens[-1]=='mean'):
                pieces.append(np.isfinite(a).astype(float)[:,None])
            # No raw values, changes, signed slopes, thresholds, or abnormality flags.
        return np.concatenate(pieces,axis=1) if pieces else np.zeros((len(frame),1))

def grid_combinations(cfg):
    combos=list(_V9_GRID_COMBINATIONS(cfg))
    if cfg.get('include_observation_controls',True):
        combos.extend([('demographics',),('observation_current',),('observation_history',),OBS_BLOCKS])
    return list(dict.fromkeys(combos))

def operating_threshold(y,p,target):
    """Highest MIMIC-inner threshold reaching a prespecified empirical recall."""
    if not 0<target<=1:raise ValueError('Recall targets must be in (0,1]')
    _,recall,threshold=roc_curve(y,p,drop_intermediate=False)
    valid=np.isfinite(threshold)&(recall>=target)
    if not valid.any():raise ValueError('Recall target unavailable in the inner holdout')
    return float(threshold[valid].max())

def operating_key(target):return 'threshold_recall_'+format(float(target),'.6g').replace('.','p')

def fitting_fingerprint(m,ms,e,es,cfg):
    if not V10_FOLD_DIGEST:raise RuntimeError('Initialize fixed patient folds first')
    fit_cfg={k:v for k,v in cfg.items() if k not in ('run_sensitivities','sensitivity_lags','all_ages_sensitivity','core_panel_sensitivity')}
    base=_V9_FITTING_FINGERPRINT(m,ms,e,es,fit_cfg)
    funcs=[GridTransform.fit,GridTransform.raw,_V9_GRID_TRANSFORM.fit,_V9_GRID_TRANSFORM.raw,
           panel_sequence,operating_threshold,operating_key,patient_folds,initialize_study_folds,
           grid_combinations,_V9_GRID_COMBINATIONS]
    return hashlib.sha256(base.encode()+V10_FOLD_DIGEST.encode()+b''.join(stable_code_bytes(f.__code__) for f in funcs)).hexdigest()[:24]

def report_fingerprint(cfg):
    return hashlib.sha256((_V9_REPORT_FINGERPRINT(cfg)+'v10-report').encode()).hexdigest()[:24]

def wide_grid_results(metrics):
    w=_V9_WIDE_RESULTS(metrics)
    w['Input_kind']=w.Feature_combo.map(lambda c:'demographics_only' if c=='demographics' else 'observation_patterns' if 'observation_' in c else 'values_and_observation_patterns')
    return w

def analyzed_cohort_reports(m,e,cfg,out):
    """Every denominator matches the actual age/history eligibility policy."""
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    summaries=[];coverage=[];ages=[]
    for cohort,frame in [('MIMIC',m),('eICU',e)]:
        for population in POPULATIONS:
            d=frame.iloc[population_indices(frame,cohort,population,cfg)]
            for landmark,g in d.groupby('landmark'):
                summaries.append(dict(cohort=cohort,population=population,landmark=landmark,
                    admissions=len(g),patients=g.patient_id.nunique(),cases=int(g.label.sum()),
                    case_patients=g.loc[g.label.eq(1),'patient_id'].nunique(),prevalence=g.label.mean(),
                    median_age=g.age.median(),under18=int(g.age.lt(18).sum()),unknown_age=int(g.age.isna().sum()),
                    no_lab_history_fraction=(~g[[f'history__{a}__count' for a in CORE]].gt(0).any(axis=1)).mean() if cfg.get('lab_panel')=='core' else g.history__available.eq(0).mean(),
                    score_hour_q10=g.score_hour.quantile(.1),score_hour_median=g.score_hour.median(),
                    score_hour_q90=g.score_hour.quantile(.9),score_after24_fraction=g.score_hour.gt(24).mean()))
                for label,sub in g.groupby('label'):
                    bins=pd.cut(sub.age,[-np.inf,18,40,65,80,90,np.inf],right=False,
                        labels=['under18','18-39','40-64','65-79','80-89','90_topcoded']).astype('string').fillna('unknown')
                    for band,n in bins.value_counts().items():
                        ages.append(dict(cohort=cohort,population=population,landmark=landmark,label=label,age_band=band,admissions=n,denominator=len(sub)))
                    for a in LABS:
                        coverage.append(dict(cohort=cohort,population=population,landmark=landmark,label=label,
                            analyte=a,denominator=len(sub),current_observed=sub[f'current__{a}__last'].notna().mean(),
                            history_observed=sub[f'history__{a}__mean'].notna().mean(),
                            valid_current_slope=sub[f'current__{a}__slope_valid'].eq(1).mean(),
                            included_in_panel=cfg.get('lab_panel','expanded')=='expanded' or a in CORE))
    summary=pd.DataFrame(summaries);cov=pd.DataFrame(coverage)
    summary.to_csv(out/'cohort_summary.csv',index=False);cov.to_csv(out/'lab_coverage.csv',index=False)
    pd.DataFrame(ages).to_csv(out/'age_composition.csv',index=False)
    population_audit(m,e,cfg).to_csv(out/'population_eligibility.csv',index=False)
    return summary,cov

def scenario_plan(cfg):
    """One-factor-at-a-time, not a combinatorial multiplication of assumptions."""
    if cfg.get('lab_panel') not in ('core','expanded'):raise ValueError('Invalid lab panel')
    if not cfg['recall_targets'] or any(not 0<t<=1 for t in cfg['recall_targets']):raise ValueError('Invalid recall targets')
    if len(set(map(operating_key,cfg['recall_targets'])))!=len(cfg['recall_targets']):raise ValueError('Duplicate recall targets')
    if any(not np.isfinite(t) or t<0 for t in cfg['sensitivity_lags']):raise ValueError('Invalid release lag')
    base=dict(cfg)
    rows=[('primary',base)]
    if cfg.get('run_sensitivities',True):
        rows.extend((f'lag_{lag:g}h',dict(base,release_lag_hours=float(lag))) for lag in cfg['sensitivity_lags'] if lag!=cfg['release_lag_hours'])
        if cfg.get('all_ages_sensitivity',True):rows.append(('all_ages',dict(base,age_policy='all_ages')))
        if cfg.get('core_panel_sensitivity',True):rows.append(('core_panel',dict(base,lab_panel='core')))
    if len({n for n,_ in rows})!=len(rows):raise ValueError('Duplicate sensitivity scenario names')
    return rows

def prepare_v10_scenario(root,out,cfg):
    validate_config(cfg)
    m,ms,e,es,audit=prepare_experiment_v9(root,out,cfg,USE_FEATURE_CACHE)
    cohort_summary(m,e).to_csv(Path(out)/'pre_age_filter_cohort_summary.csv',index=False)
    pd.DataFrame(audit['source']).to_csv(Path(out)/'source_admissions.csv',index=False)
    pd.DataFrame(audit['flow']).fillna(0).to_csv(Path(out)/'lab_eligibility_flow.csv')
    pd.DataFrame(audit['units']).fillna(0).to_csv(Path(out)/'unit_exclusion_counts.csv')
    build_icd_history(root,m,e,out)
    summary,coverage=analyzed_cohort_reports(m,e,cfg,out)
    reference_table().to_csv(Path(out)/'reference_intervals_with_sources.csv',index=False)
    display(summary)
    return m,ms,e,es,audit

def calibration_parameters(y,p):
    """Descriptive diagnostic fit only; never modifies deployed probabilities."""
    y=np.asarray(y,float);p=np.clip(np.asarray(p,float),1e-6,1-1e-6);x=logit(p)
    if np.unique(y).size<2 or np.ptp(x)<1e-10:return dict(intercept=np.nan,slope=np.nan,status='not_estimable')
    design=np.c_[np.ones(len(x)),x]
    def objective(b):
        z=design@b
        return float(np.mean(np.logaddexp(0,z)-y*z)),design.T@(expit(z)-y)/len(y)
    opt=minimize(objective,[0.,1.],jac=True,method='BFGS',options={'maxiter':1000,'gtol':1e-7})
    if not opt.success or not np.isfinite(opt.x).all():return dict(intercept=np.nan,slope=np.nan,status='optimizer_failed')
    return dict(intercept=float(opt.x[0]),slope=float(opt.x[1]),status='ok')

def finalist_diagnostics(wide,checkpoint,cfg,out):
    """Choose per-model configurations using MIMIC AP only; never eICU ranking."""
    out=Path(out)/'finalist_diagnostics';out.mkdir(parents=True,exist_ok=True)
    digest=hashlib.sha256((frame_digest(wide)+report_fingerprint(cfg)).encode()+b''.join(stable_code_bytes(f.__code__) for f in
        [finalist_diagnostics,calibration_parameters,operating_key])).hexdigest()[:16]
    folder=out/digest;folder.mkdir(exist_ok=True)
    if (folder/'complete.json').exists():return pd.read_csv(folder/'calibration.csv'),pd.read_csv(folder/'operating_points.csv')
    # Keep data-adapted prevalence runs separate from target-independent evaluations.
    finalists=wide[wide.Input_kind.eq('values_and_observation_patterns')].sort_values(
        ['MIMIC_PR_AUC','Feature_combo','Sampling'],ascending=[False,True,True]).drop_duplicates(
        ['population','landmark','Model','external_prevalence_used'])
    finalists.to_csv(folder/'mimic_selected_configurations.csv',index=False)
    calrows=[];oprows=[]
    for (pop,landmark,adapted),group in finalists.groupby(['population','landmark','external_prevalence_used']):
        fig,axes=plt.subplots(1,2,figsize=(12,5))
        for r in group.itertuples():
            token=prediction_token(pop,landmark,r.Feature_combo,r.Sampling)
            pred=read_predictions(Path(checkpoint)/f'{token}.csv.gz')
            pred=pred[pred.model.eq(r.Model)]
            for ax,(cohort,g) in zip(axes,[(c,pred[pred.cohort.eq(c)]) for c in ('MIMIC_OOF','eICU_external')]):
                tags=dict(population=pop,landmark=landmark,model=r.Model,scope=r.Feature_combo,
                    sampling=r.Sampling,cohort=cohort,external_prevalence_used=bool(adapted))
                calrows.append(dict(**tags,**calibration_parameters(g.label,g.probability),admissions=len(g),cases=int(g.label.sum())))
                cuts=np.unique(np.quantile(g.probability,np.linspace(0,1,11)))
                if len(cuts)>1:
                    bins=pd.cut(g.probability,cuts,include_lowest=True,duplicates='drop')
                    grouped=g.groupby(bins,observed=True).agg(predicted=('probability','mean'),observed=('label','mean'))
                    ax.plot(grouped.predicted,grouped.observed,'o-',label=r.Model)
                for target in cfg['recall_targets']:
                    key=operating_key(target)
                    if key not in g:raise ValueError('Missing training-derived operating thresholds; incompatible prediction cache')
                    q=g.copy();q['threshold']=q[key]
                    met=metric_values(q.label,q.probability,q.probability>=q.threshold)
                    boots=patient_bootstrap(q,cfg['n_boot'],cfg['seed'])
                    row=dict(**tags,target_inner_recall=target,threshold_source='MIMIC inner patient holdout',
                        clinical_status='exploratory; not a clinical recommendation',**met)
                    for metric in met:
                        row[metric+'_lo']=boots[metric].quantile(.025) if metric in boots else np.nan
                        row[metric+'_hi']=boots[metric].quantile(.975) if metric in boots else np.nan
                    row['bootstrap_valid']=len(boots);oprows.append(row)
        for ax,cohort in zip(axes,['MIMIC OOF (post-selection descriptive)','eICU external']):
            ax.plot([0,1],[0,1],'--',color='gray');ax.set(xlabel='Mean predicted probability',ylabel='Observed BE fraction',title=cohort)
            ax.legend(fontsize=7);ax.grid(alpha=.2)
        fig.suptitle(f'{pop}, set {landmark}; external-prevalence adaptation={adapted}')
        fig.tight_layout();fig.savefig(folder/f'{pop}_set{landmark}_adapted{adapted}_calibration.png',dpi=160);plt.close(fig)
    cal=pd.DataFrame(calrows);ops=pd.DataFrame(oprows)
    cal.to_csv(folder/'calibration.csv',index=False);ops.to_csv(folder/'operating_points.csv',index=False)
    (folder/'complete.json').write_text(json.dumps(dict(status='complete',selection='MIMIC AP, per model and adaptation stratum',
        note='Calibration fits are diagnostic only. No external recalibration. MIMIC finalist estimates are post-selection.')))
    return cal,ops

def compare_scenario_overlap(primary_wide,primary_checkpoint,other_checkpoint,cfg,out):
    """Paired held-out policy comparison on common patients/admissions, no refit."""
    out=Path(out)/'paired_policy_comparison';out.mkdir(exist_ok=True)
    digest=hashlib.sha256((str(primary_checkpoint)+str(other_checkpoint)+frame_digest(primary_wide)+report_fingerprint(cfg)).encode()+
        stable_code_bytes(compare_scenario_overlap.__code__)).hexdigest()[:16]
    path=out/f'{digest}.csv';mark=out/f'{digest}.complete'
    if path.exists() and mark.exists():return pd.read_csv(path)
    winners=primary_wide[primary_wide.Input_kind.eq('values_and_observation_patterns')].sort_values(
        ['MIMIC_PR_AUC','Feature_combo','Sampling'],ascending=[False,True,True]).drop_duplicates(
        ['population','landmark','Model','external_prevalence_used'])
    records=[]
    for r in winners.itertuples():
        token=prediction_token(r.population,r.landmark,r.Feature_combo,r.Sampling)
        first=Path(primary_checkpoint)/f'{token}.csv.gz';second=Path(other_checkpoint)/f'{token}.csv.gz'
        if not first.exists() or not second.exists():
            records.append(dict(population=r.population,landmark=r.landmark,model=r.Model,status='configuration_unavailable'));continue
        a=read_predictions(first);b=read_predictions(second)
        a=a[a.model.eq(r.Model)];b=b[b.model.eq(r.Model)]
        p=a.merge(b,on=['cohort','patient_id','hospital_id','model'],suffixes=('_primary','_sensitivity'),validate='one_to_one')
        assert p.label_primary.eq(p.label_sensitivity).all()
        internal=p.cohort.eq('MIMIC_OOF')
        assert p.loc[internal,'fold_primary'].eq(p.loc[internal,'fold_sensitivity']).all(), 'Cross-policy fold mismatch'
        for cohort,g in p.groupby('cohort'):
            tags=dict(population=r.population,landmark=r.landmark,model=r.Model,scope=r.Feature_combo,sampling=r.Sampling,
                external_prevalence_used=bool(r.external_prevalence_used),cohort=cohort,
                common_admissions=len(g),common_patients=g.patient_id.nunique(),common_cases=int(g.label_primary.sum()))
            if g.label_primary.nunique()<2:records.append(dict(**tags,status='single_class_overlap'));continue
            f=WeightedPredictionMetrics(g.label_primary,g.probability_primary,g.probability_primary>=g.threshold_primary)
            s=WeightedPredictionMetrics(g.label_sensitivity,g.probability_sensitivity,g.probability_sensitivity>=g.threshold_sensitivity)
            x=f(np.ones(len(g)));z=s(np.ones(len(g)));delta=[]
            for weights in cluster_draws(g.patient_id,cfg['n_boot'],cfg['seed']):
                xx=f(weights);zz=s(weights)
                if xx is not None:delta.append({k:zz[k]-xx[k] for k in ('AP','AUROC','Brier')})
            boot=pd.DataFrame(delta);row=dict(**tags,status='ok',bootstrap_valid=len(boot))
            for metric in ('AP','AUROC','Brier'):
                row[metric+'_primary']=x[metric];row[metric+'_sensitivity']=z[metric];row[metric+'_delta']=z[metric]-x[metric]
                row[metric+'_delta_lo']=boot[metric].quantile(.025) if metric in boot else np.nan
                row[metric+'_delta_hi']=boot[metric].quantile(.975) if metric in boot else np.nan
            records.append(row)
    frame=pd.DataFrame(records);frame.to_csv(path,index=False);mark.write_text('complete')
    return frame

def run_v10_sensitivities(root,study_root,cfg,primary_wide,primary_checkpoint):
    """Load, rebuild/refit and release one scenario at a time. Never mutate CFG."""
    tables=[]
    for name,setting in scenario_plan(cfg)[1:]:
        out=Path(study_root)/name
        print(f'[sensitivity] {name}: fresh policy-specific feature/fit fingerprints',flush=True)
        m,ms,e,es,audit=prepare_v10_scenario(root,out,setting)
        result=run_unified_grid(m,ms,e,es,setting,out)
        wide,metrics,paired,fit_audit,learning,checkpoint=result
        finalist_diagnostics(wide,checkpoint,setting,out)
        compare_scenario_overlap(primary_wide,primary_checkpoint,checkpoint,setting,out)
        manifest,fingerprint=input_manifest(root,setting)
        (out/'run_manifest.json').write_text(json.dumps(dict(scenario=name,inputs=manifest,feature_fingerprint=fingerprint,
            patient_fold_digest=V10_FOLD_DIGEST,checkpoint_directory=str(checkpoint)),indent=2,default=str))
        tables.append(wide.assign(Scenario=name))
        del m,ms,e,es,audit,result,metrics,paired,fit_audit,learning
        ICD_RAW.clear();ICD_SELECTION_CACHE.clear();gc.collect()
    return pd.concat(tables,ignore_index=True) if tables else pd.DataFrame()

def audit_crp_source_units(root,out,cfg):
    """Source-unit review only. No inferred or automatic conversion overrides."""
    ma,ea,ep=load_admissions(root);del ma
    ep=ep.drop_duplicates('patientunitstayid').set_index('patientunitstayid')
    latest=ea.groupby('patient_id').year.max()
    old=set(ea.loc[ea.year.lt(ea.patient_id.map(latest))&ea.valid_time,'hospital_id'])
    count=Counter();chunks=0
    for z in pd.read_csv(locate(root,'eicu/lab.csv','lab.csv'),chunksize=100_000,
        usecols=['patientunitstayid','labname','labresultoffset','labresult','labmeasurenamesystem','labmeasurenameinterface']):
        z=z[z.labname.eq(LABS['crp'][4])&z.patientunitstayid.isin(ep.index)].copy()
        if z.empty:continue
        hid=z.patientunitstayid.map(ep.patienthealthsystemstayid)
        hour=(pd.to_numeric(z.labresultoffset,errors='coerce')-z.patientunitstayid.map(ep.hospitaladmitoffset))/60
        z=z[hid.isin(old)|hour.lt(cfg['second_draw_max_hours']+cfg['set_width_hours'])].copy()
        z['site']=z.patientunitstayid.map(ep.hospitalid) if 'hospitalid' in ep else 'unknown'
        z['system']=z.labmeasurenamesystem.astype('string').str.lower().str.replace(r'\s+','',regex=True).fillna('<missing>')
        z['interface']=z.labmeasurenameinterface.astype('string').str.lower().str.replace(r'\s+','',regex=True).fillna('<missing>')
        conflict=z.system.isin(['mg/l','mg/dl'])&z.interface.isin(['mg/l','mg/dl'])&z.system.ne(z.interface)
        z['analyte']='crp';z['value']=z.labresult;z['unit']=z.labmeasurenamesystem
        cleaned=clean_labs(z.loc[~conflict],Counter())
        z['decision']=np.where(conflict,'excluded_conflicting_units',np.where(z.index.isin(cleaned.index),'accepted','excluded_unit_or_value'))
        for key,n in z.groupby(['site','system','interface','decision'],dropna=False).size().items():count[key]+=int(n)
        chunks+=1
        if chunks%20==0:print('[CRP unit review] matching chunks:',chunks,flush=True)
    rows=pd.DataFrame([dict(site=k[0],system_unit=k[1],interface_unit=k[2],decision=k[3],lab_rows=n) for k,n in count.items()],columns=['site','system_unit','interface_unit','decision','lab_rows'])
    out=Path(out);out.mkdir(parents=True,exist_ok=True);rows.to_csv(out/'crp_source_unit_review.csv',index=False)
    print('CRP review saved. Ambiguous conversions remain excluded; no rules were relaxed.',flush=True)
    return rows


## Optional Lasso execution and resume

The weighted L1 logistic objective, regularization and convergence criterion are unchanged.
Zero training columns are omitted from solver algebra and restored with coefficient zero.
Each Newton update uses a coordinate working set; all features remain subject to the full
gradient and primal/dual convergence test. This is numerical acceleration, not feature selection.

`LASSO_RUNTIME` controls the per-invocation time and iteration allowances (900 seconds and
180 iterations). These operational settings do not invalidate completed fits. Accepted iterates
are checkpointed; if interrupted, rerunning the grid continues from the last saved iterate.
Predictions are saved only after the original objective-gap requirement is satisfied.

Compatible completed v10 outputs are copied into the repaired checkpoint namespace after
reproducing the exact previous input/configuration fingerprint and verifying stored Lasso
certificates. The original files remain untouched. A migration manifest records provenance.
Do not change `CFG['lasso_objective_gap']` to bypass convergence. The historical timeout/step
entries in CFG are retained to identify old checkpoints; runtime controls are now separate.

V15 runs this solver only if `V15_L1_ANCHORS=True`, at four primary anchors. It is disabled by default.


In [ ]:
"""Exact weighted L1 objective; working-set Newton and resumable iterates.

No patient, measured feature, loss weight, regularization or acceptance threshold changes.
All-column primal/dual certification remains mandatory before prediction checkpoints.
"""
class ResumableWeightedL1Logistic(WeightedL1LogisticGram):
    def fit(self,X,y,sample_weight=None):
        started=time.perf_counter();last_report=started
        X=np.asarray(X,dtype=np.float64);y=np.asarray(y,dtype=np.float64)
        if X.ndim!=2 or y.shape!=(len(X),) or not np.isfinite(X).all():raise ValueError('Invalid finite training matrix')
        if set(np.unique(y))!={0.,1.}:raise ValueError('Both binary classes required')
        if self.C<=0 or self.max_seconds<=0 or self.max_iter<1 or self.gap_tol<=0:raise ValueError('Invalid L1 settings')
        w=np.ones(len(y)) if sample_weight is None else np.asarray(sample_weight,dtype=float).copy()
        if w.shape!=y.shape or not np.isfinite(w).all() or np.any(w<=0):raise ValueError('Weights must be finite and positive')
        if self.class_weight=='balanced':
            total=w.sum()
            for label in (0.,1.):
                mask=y==label;w[mask]*=total/(2*w[mask].sum())
        elif self.class_weight is not None:raise ValueError('Use balanced or None')
        self.n_features_in_=X.shape[1];self.classes_=np.array([0,1]);self.converged_=False
        # Zero training columns have identically zero gradient and L1-optimal coefficient 0.
        # Nonzero duplicate columns are NOT collapsed: they may differ in held-out patients.
        keep=np.flatnonzero(np.any(X!=0,axis=0))
        x=np.asfortranarray(np.column_stack([X[:,keep],np.ones(len(X))]));penalty=1./self.C
        self.zero_columns_removed_=X.shape[1]-len(keep)
        state_path=getattr(self,'checkpoint_path',None)
        h=hashlib.sha256(json.dumps(dict(schema='working-set-v1',shape=X.shape,keep=keep.tolist(),C=self.C,
            gap_tol=self.gap_tol,threads=self.threads),sort_keys=True).encode())
        for a in (x,y,w):h.update(np.ascontiguousarray(a).view(np.uint8))
        for f in (ResumableWeightedL1Logistic.fit,self._objective,self._certificate):h.update(stable_code_bytes(f.__code__))
        signature=h.hexdigest();beta=np.zeros(x.shape[1]);self.optimization_trace_=[];previous_seconds=0.
        if state_path is not None and Path(state_path).exists():
            saved=_v9_read_pickle(state_path)
            if saved['signature']!=signature:raise ValueError('Lasso iterate does not match this training matrix/objective/solver')
            beta=np.asarray(saved['beta'],float)
            if beta.shape!=(x.shape[1],) or not np.isfinite(beta).all():raise ValueError('Invalid saved Lasso coefficients')
            self.optimization_trace_=list(saved['trace']);previous_seconds=float(saved['seconds'])
            print(f'[Lasso] resumed accepted iterate {len(self.optimization_trace_)}',flush=True)
        def save():
            if state_path is not None:
                _v9_pickle(state_path,dict(signature=signature,beta=beta,trace=self.optimization_trace_,
                    seconds=previous_seconds+time.perf_counter()-started,certified=False))
        def check_time():
            if time.perf_counter()-started>self.max_seconds:
                save()
                raise TimeoutError(f'Lasso reached its {self.max_seconds:g}s per-invocation budget. Accepted iterates and completed models are saved; rerun the grid cell to continue. No uncertified predictions were saved.')
        with threadpool_limits(limits=self.threads):
            primal,dual,gap=self._certificate(x,y,w,beta,penalty)
            try:
                for iteration in range(1,self.max_iter+1):
                    if gap<=self.gap_tol:self.converged_=True;break
                    check_time();z=x@beta;prob=expit(z);gradient=x.T@(w*(prob-y))
                    # Coordinate working set, not feature selection. Full gradients and the
                    # global certificate test every column again at each Newton iteration.
                    active=np.flatnonzero((beta!=0)|(np.abs(gradient)>penalty))
                    if not len(active):raise RuntimeError('Empty Lasso working set before certification')
                    curvature=np.maximum(prob*(1-prob),1e-8);hess=w*curvature
                    target=z+(y-prob)/curvature
                    work=np.asfortranarray(x[:,active]);gram=np.zeros((len(active),len(active)));xy=np.zeros(len(active))
                    gram_start=time.perf_counter()
                    for start in range(0,len(x),2048):
                        check_time();end=min(start+2048,len(x));a=work[start:end]
                        gram+=a.T@(hess[start:end,None]*a);xy+=a.T@(hess[start:end]*target[start:end])
                    gram_seconds=time.perf_counter()-gram_start
                    gram=np.ascontiguousarray(gram);xy=np.ascontiguousarray(xy)
                    weighted_target=np.ascontiguousarray(np.sqrt(hess)*target)
                    candidate=beta[active].copy();inner_iterations=0;solve_start=time.perf_counter()
                    for chunk in range(10):
                        check_time()
                        with warnings.catch_warnings(record=True) as caught:
                            warnings.simplefilter('always',ConvergenceWarning)
                            _,coef,_,nit=lasso_path(work,weighted_target,alphas=[penalty/len(y)],precompute=gram,
                                Xy=xy,coef_init=candidate,copy_X=False,max_iter=10000,
                                tol=min(1e-7,self.gap_tol*1e-3),check_input=False,return_n_iter=True)
                        candidate=coef[:,0].copy();inner_iterations+=int(nit[0])
                        for q in caught:
                            if not issubclass(q.category,ConvergenceWarning):warnings.warn(str(q.message),q.category)
                        if not any(issubclass(q.category,ConvergenceWarning) for q in caught):break
                    solve_seconds=time.perf_counter()-solve_start
                    direction=np.zeros_like(beta);direction[active]=candidate-beta[active]
                    before=primal;step=1.
                    while step>=2**-20:
                        check_time()
                        if self._objective(x,y,w,beta+step*direction,penalty)<=before:break
                        step*=.5
                    if step<2**-20:raise RuntimeError('Lasso line search stalled; last accepted iterate saved, no uncertified predictions')
                    beta+=step*direction;primal,dual,gap=self._certificate(x,y,w,beta,penalty)
                    elapsed=time.perf_counter()-started
                    self.optimization_trace_.append(dict(iteration=len(self.optimization_trace_)+1,
                        seconds=previous_seconds+elapsed,objective=primal,dual_bound=dual,relative_dual_gap=gap,
                        working_columns=len(active),inner_iterations=inner_iterations,gram_seconds=gram_seconds,solve_seconds=solve_seconds))
                    save()
                    if time.perf_counter()-last_report>=10 or gap<=self.gap_tol:
                        print(f'[Lasso] step {len(self.optimization_trace_)}, {elapsed:.1f}s this invocation, gap {gap:.3g}, working columns {len(active)}/{x.shape[1]}',flush=True)
                        last_report=time.perf_counter()
                    if gap<=self.gap_tol:self.converged_=True;break
            except BaseException:
                save();raise
        if not self.converged_:
            save();raise RuntimeError(f'Lasso gap {gap:g} exceeds {self.gap_tol:g}; accepted iterates saved. Rerun to continue; no uncertified predictions saved.')
        self.coef_=np.zeros((1,self.n_features_in_));self.coef_[0,keep]=beta[:-1];self.intercept_=np.array([beta[-1]])
        self.n_iter_=np.array([len(self.optimization_trace_)]);self.fit_seconds_=previous_seconds+time.perf_counter()-started
        self.relative_dual_gap_=gap;self.objective_=primal
        return self


"""Lasso runtime repair; existing certified v10 fits can be migrated explicitly."""
import shutil
if '_L10_BEFORE_REPAIR' not in globals():
    _L10_BEFORE_REPAIR={name:globals()[name] for name in
        ('grid_estimator','_fit_model_checkpoint','fitting_fingerprint','run_unified_grid')}
LASSO_RUNTIME=globals().get('LASSO_RUNTIME',dict(max_seconds=900.,newton_steps=180))

def grid_estimator(name,y,setting,cfg,seed):
    if name=='Lasso':
        return ResumableWeightedL1Logistic(C=1.,class_weight='balanced' if setting['mode']=='none' else None,
            max_iter=LASSO_RUNTIME['newton_steps'],max_seconds=LASSO_RUNTIME['max_seconds'],
            gap_tol=cfg['lasso_objective_gap'],threads=min(2,cfg['torch_threads']))
    return _L10_BEFORE_REPAIR['grid_estimator'](name,y,setting,cfg,seed)

def _fit_model_checkpoint(name,x,y,target,setting,cfg,seed,stage,folder):
    if name!='Lasso':return _L10_BEFORE_REPAIR['_fit_model_checkpoint'](name,x,y,target,setting,cfg,seed,stage,folder)
    path=None if folder is None else Path(folder)/f'{name}_{stage}.pkl'
    if path is not None and path.is_file():
        saved=_v9_read_pickle(path)
        gap=saved['audit'].get('lasso_relative_dual_gap')
        if gap is None or not np.isfinite(gap) or gap>cfg['lasso_objective_gap']:raise ValueError('Cached Lasso fit fails requested certificate')
        if saved['rows']!=len(target) or not np.isfinite(saved['probs']).all():raise ValueError('Invalid cached Lasso predictions')
        print(f'[model] {name} {stage}: resumed certified predictions',flush=True)
        return saved['probs'],saved['audit']
    print(f'[model] {name} {stage}: fitting {len(y):,} rows x {x.shape[1]} columns',flush=True)
    model=grid_estimator(name,y,setting,cfg,seed)
    if path is not None:model.checkpoint_path=path.with_name(path.stem+'.iterate.pkl')
    started=time.perf_counter();model.fit(x,y);probs=model.predict_proba(target)[:,1]
    if not np.isfinite(probs).all():raise ValueError('Nonfinite model probabilities')
    audit=dict(model=name,stage=stage,seconds=time.perf_counter()-started,
        iterations=model.n_iter_.tolist(),lasso_relative_dual_gap=model.relative_dual_gap_,
        solver_revision='v10-working-set-resume-1',zero_columns_removed=model.zero_columns_removed_,
        cumulative_solver_seconds=model.fit_seconds_)
    if path is not None:_v9_pickle(path,dict(rows=len(target),probs=probs,audit=audit))
    print(f"[model] {name} {stage}: certified and saved after {audit['seconds']:.1f}s",flush=True)
    return probs,audit

def _lasso_legacy_fingerprint(m,ms,e,es,cfg):
    """Reproduce the exact published-v10 key; never guess a checkpoint directory."""
    names=('grid_estimator','_fit_model_checkpoint','run_unified_grid')
    current={name:globals()[name] for name in names}
    try:
        for name in names:globals()[name]=_L10_BEFORE_REPAIR[name]
        return _L10_BEFORE_REPAIR['fitting_fingerprint'](m,ms,e,es,cfg)
    finally:globals().update(current)

def fitting_fingerprint(m,ms,e,es,cfg):
    # Runtime allowances do not change the model, accepted certificate or patient split.
    science={k:v for k,v in cfg.items() if k not in ('lasso_max_seconds','lasso_newton_steps')}
    base=_L10_BEFORE_REPAIR['fitting_fingerprint'](m,ms,e,es,science)
    return hashlib.sha256(base.encode()+stable_code_bytes(ResumableWeightedL1Logistic.fit.__code__)).hexdigest()[:24]

def _migrate_certified_lasso_checkpoints(source,target,cfg):
    source=Path(source);target=Path(target)
    if not source.is_dir() or source.resolve()==target.resolve():return
    manifest=target/'solver_migration.json'
    if manifest.exists():return
    # Exact old key already covers inputs, labels, ordering, folds, hyperparameters and code.
    # Explicitly confirm its Lasso outputs met the SAME acceptance requirement.
    models=list(source.glob('*.models/Lasso_*.pkl'))
    for path in models:
        saved=_v9_read_pickle(path);gap=saved['audit'].get('lasso_relative_dual_gap')
        if gap is None or not np.isfinite(gap) or gap>cfg['lasso_objective_gap'] or not np.isfinite(saved['probs']).all():
            raise ValueError(f'Refusing uncertified legacy Lasso checkpoint: {path.name}')
    if 'Lasso' in cfg['base_models']:
        for mark in source.glob('*.fold*.complete'):
            folder=source/(mark.name.removesuffix('.complete')+'.models')
            if not all((folder/f'Lasso_{stage}.pkl').is_file() for stage in ('inner','outer')):
                raise ValueError('Legacy completed fold lacks verifiable Lasso certificates; no automatic reuse')
    files=[p for p in source.rglob('*') if p.is_file() and 'reports' not in p.relative_to(source).parts]
    target.mkdir(parents=True,exist_ok=True);copied=0
    for path in sorted(files,key=lambda p:p.name.endswith('.complete')):
        dest=target/path.relative_to(source)
        if dest.exists():continue
        dest.parent.mkdir(parents=True,exist_ok=True)
        _v9_atomic(dest,lambda temporary,p=path:shutil.copyfile(p,temporary));copied+=1
    manifest.write_text(json.dumps(dict(source_key=source.name,target_key=target.name,copied_files=copied,
        verified_lasso_checkpoints=len(models),acceptance_gap=cfg['lasso_objective_gap'],
        compatibility='Same weighted L1 objective and certificate; only solver execution changed. Legacy accepted predictions retained.',
        solver_revision='v10-working-set-resume-1'),indent=2))
    print(f'[resume] Preserved {len(models)} certified Lasso model checkpoints and compatible completed work ({copied} files copied); source untouched.',flush=True)

def run_unified_grid(m,ms,e,es,cfg,out):
    legacy=_lasso_legacy_fingerprint(m,ms,e,es,cfg)
    current=fitting_fingerprint(m,ms,e,es,cfg)
    root=Path(out)/'grid_checkpoints'
    _migrate_certified_lasso_checkpoints(root/legacy,root/current,cfg)
    return _L10_BEFORE_REPAIR['run_unified_grid'](m,ms,e,es,cfg,out)

print('[Lasso repair] Full-objective certification retained; zero-column/working-set acceleration and resumable iterates enabled. Runtime controls are in LASSO_RUNTIME.',flush=True)


## Sequence representation reference

The timing, normalization and convolution specifications below remain applicable. The active v12 experiment definition following this section governs model inputs and checkpoint reuse: no CNN receives ICD, and old model predictions are not imported.

## Prediction landmarks: a panel-triggered score, not a single-analyte score

The primary question is: **after the first qualifying recorded laboratory cluster becomes available, can routinely available information distinguish admissions that receive a BE diagnosis code?** This is an admission-time diagnostic-risk experiment. It is not established prediction of future onset, clinician suspicion or echocardiographic diagnosis.

A set is a **one-hour collection cluster containing at least two distinct core analytes**. The core trigger analytes are hemoglobin, WBC, platelets, sodium, potassium, creatinine, BUN and glucose. The cluster starts at a recorded core-analyte timestamp, not automatically at hospital admission. It is a pragmatic timestamp grouping, not a verified specimen accession, complete CBC/CMP, or one analyte. Eligibility depends on timing, analyte identity and availability, not on whether the results are abnormal.

1. **Initial score:** the first qualifying cluster beginning within 24 hours of admission. Predict at cluster end plus the assumed release lag. Use all qualifying current-admission values collected before that cluster end and available by scoring, plus selected historical features. A patient need not have prior history for the broad lab population.
2. **Update score:** a new qualifying cluster beginning at least six hours after the first cluster start and within 48 hours of admission. Use all available current-admission observations up to the new cutoff, not just the second cluster. Compare the two policies on their own eligible populations and on the shared repeat-tested admissions already reported by the grid. A second set is not required before issuing the initial score.
3. **Fixed admission-hour analysis:** a proposed secondary question, not an automatic replacement for the primary experiment. Restricting collection to `[0,1)` hours answers a different question and removes patients without a sufficiently observed early panel. With the reference four-hour release assumption, its score is at hour 5, **not hour 1**. The raw-data coverage audit below informs this proposal; this revision does not silently substitute this cohort or launch another full grid.

MIMIC uses collection/chart timestamps plus an assumed release lag; eICU additionally respects recorded revision availability. The retained 4/8/12-hour lag scenarios assess this assumption, not measured laboratory turnaround. Hospital admission time may follow emergency-department care, and an eICU record may begin after a hospital stay has already started. Absence of an early recorded panel therefore does not establish absence of early clinical testing. Do not backfill from later results to create an admission-hour score.

Historical labs remain optional: MIMIC history precedes the index admission, while separate eICU historical hospitalizations must have strictly earlier discharge years. Same-year separate eICU admissions are not added. Current index-admission labs are permitted in both cohorts. Historical summaries and valid within-stay trends retain subacute context without inventing a cross-admission eICU clock. ICD-history subgroup eligibility remains separate from the broad lab population.

### Clinical role and echocardiography

The intended research role is complementary risk recognition, not replacing echocardiography, ruling out BE, or delaying investigation until a second panel. ESC guidance places echocardiography early in suspected IE assessment; a patient may have an echo before this model's laboratory cutoff. Reliable suspicion, culture, antibiotic, echo-order and echo-result timestamps would be needed to evaluate a genuinely pre-workup score and its lead time. Discharge codes alone cannot establish that chronology. [2023 ESC endocarditis guideline](https://academic.oup.com/eurheartj/article/44/39/3948/7243107?login=false).

EndoPredict-Dx studied 805 adults with suspected left-sided IE using admission clinical/laboratory information. Its selected suspected-IE population differs substantially from this broad, low-prevalence, code-labeled admission cohort; its performance is not a benchmark to transplant here. [EndoPredict-Dx](https://pmc.ncbi.nlm.nih.gov/articles/PMC11593066/).

Before claiming added clinical value, evaluate calibration, alert burden and PPV at useful sensitivities, and determine whether flagged patients were already undergoing IE investigation. A higher retrospective AUROC alone cannot establish earlier diagnosis or benefit from an alert. Treatment-associated changes and test-ordering patterns can be predictive without identifying untreated BE biology.

## Sampling evidence and the CNN representation

The full cached adult broad-lab cohorts contain 48,132 MIMIC admissions at set 1 and 45,205 at set 2; corresponding eICU counts are 148,686 and 128,076. With three-hour bins, 78.7% of MIMIC and 86.1% of eICU first-set admissions have only one nonempty bin. No analyte repeats across bins in 86.4% and 91.2%, respectively. Thus, the first score is predominantly a panel-level experiment; fine temporal patterns cannot be learned for patients who have no repeat observations.

Across these full cohorts, binning retains approximately 98.8%/98.0% of MIMIC observations at sets 1/2 and 99.0%/97.7% of eICU observations. Counts refer to cleaned, available, revision-resolved observations, not all raw CSV rows. Median collection cutoffs are approximately 5.9/20.7 hours in MIMIC and 5.15/21.17 in eICU; the reference scoring times are four hours later. These are not median one-hour-admission predictions.

A separate label-blind sample of 2,000 currently eligible adult admissions per cohort examined the original lab timestamps and widths of 0.5, 1, 3 and 6 hours. MIMIC sampling excluded source patient fold 0; eICU sampling was descriptive, not performance-driven model selection. Both landmark observation totals were reconciled exactly to the saved summaries after applying the source pipeline's deduplication rules. This is a representation audit, not evidence of improved predictive performance.

In that sample, only **231/2,000 MIMIC admissions (11.55%)** and **588/2,000 eICU admissions (29.40%)** had at least two distinct core analytes collected in `[0,1)` hours and available by hour 5. These denominators are admissions already eligible for the first-available-panel analysis, not all hospital admissions. A strict admission-hour requirement would therefore discard approximately 88% and 71% of these sampled eligible admissions. This supports retaining the first qualifying recorded cluster as primary rather than imposing an immediate-admission panel. Earlier missing panels are not reconstructed using later measurements.

In the sampled MIMIC first-set data, 500 analyte/admission series support a raw trend spanning at least one hour. Three-hour binning retains trend eligibility in 85.6% of them; one-hour binning retains 100%. The corresponding observation retention is 98.52% versus 99.46%. In second-set MIMIC data, three-hour bins retain 99.52% of raw trend-eligible series. Median within-analyte repeat intervals in second-set records are about 10.18 hours in MIMIC and 11.95 hours in eICU. Early close repeats matter for a minority, while later monitoring is substantially sparser than hourly sampling.

**Design decision:** retain three-hour sequence bins as the computationally economical primary representation, but also provide the CNN with the requested **unbinned** current summaries and historical comparisons. Slopes use actual timestamps; means, medians, extrema, variability, counts, threshold directions and current-minus-history comparisons do not depend on tensor bins. This retains those summaries when several observations occupy one bin. It does not recover every within-bin waveform or exact ordering in the sequence branch. A one-hour-bin sensitivity is supported by setting `cnn_bin_hours=1.` and dilations `(1,2,4,8,16)`; it rebuilds tensors and refits rather than pretending the existing three-hour cache contains discarded observations. No extra finer-bin grid runs automatically. Neither resolution is claimed empirically optimal until compared using MIMIC training-side validation.

## CNN architecture and published precedents

These studies motivate design elements rather than supply a BE-specific validated architecture:

- **Catling & Wolff (2020):** hourly ICU inputs, causal temporal convolutions with increasing powers-of-two dilation, residual/skip structure and a demographic branch. Network depth covers the longitudinal lookback. We adopt full-window temporal access and static fusion, not their intervention/outcome targets or reported performance. [JAMIA paper](https://pmc.ncbi.nlm.nih.gov/articles/PMC7647248/).
- **Bednarski et al. (2022):** 24-hour MIMIC time series with value, observation and elapsed-time channels; exponential-dilation TCNs. Their largest tested configuration uses 12 layers, 200 filters and kernel size 5. That scale is not transferred to this rare-outcome cohort. [Scientific Reports paper](https://www.nature.com/articles/s41598-022-25472-z).
- **Rocheteau et al. (2021):** temporal-pointwise convolutions separate temporal processing from feature interaction and combine static data with hourly ICU series. Their temporal dilation increases linearly with layer number; forward-filled values have staleness information. Our depthwise/pointwise blocks operate on projected latent channels and use exponential dilation: this is an adaptation, **not an implementation of TPC**. [CHIL paper](https://emmarocheteau.netlify.app/publication/tpc/tpc.pdf).
- **Razavian et al. (2016):** 18 labs over 36 monthly time steps in 298,000 patients; multi-resolution CNNs use 64 filters, length-8-month kernels, pooling width/stride 3, two 100-unit dense layers and 0.5 dropout. Their multi-disease onset task has a three-month prediction gap. Monthly kernels and cohort-scale architecture are not appropriate defaults for one or two early hospital panels. [MLHC paper, supplement 8.1](https://proceedings.mlr.press/v56/Razavian16.pdf).
- **Che et al. (2018):** GRU-D is a recurrent model, not a CNN. It motivates distinguishing observed values, missingness and time since observation; its learned decay mechanism is not reproduced here. [Scientific Reports paper](https://www.nature.com/articles/s41598-018-24271-9).

### Implemented specification

The expanded sequence has 28 analytes, 18 three-hour slots over a 54-hour allocation, and 86 channels: normalized values, fresh-observation masks and time since last observation for each analyte, plus elapsed clock and valid-bin mask. The core-panel scenario uses 8 analytes and 26 channels. Only the prefix before the individual collection cutoff is valid; the allocated 54 hours never licenses future observations.

Sequence values use a fixed signed-log1p transform followed by **training-partition-only** median/IQR scaling (with the existing scale fallback and ±10 normalized clipping). Sampled MIMIC ALT, AST and bilirubin distributions are strongly right-skewed; the monotone transform reduces compression of high values by robust clipping. Clinical summaries retain original units before their own fold-local normalization. The log transform is a prespecified modeling choice, not a published BE-specific optimum; `cnn_value_transform='raw'` permits an explicit ablation.

Within the current admission only, the most recent normalized value is carried forward into empty valid bins. Before any observation, the value channel is the training median (zero after scaling). A carried value never becomes a new observation: its mask remains zero and its age increases from the actual timestamp. No backward fill, future interpolation or cross-admission carry is used. Unsupported/constant training channels remain at their training baseline externally. The observation-control models help assess how much signal reflects testing practices.

A 1×1 projection maps inputs to width 16. Four residual blocks each use a causal depthwise temporal convolution (kernel 3; dilations 1, 2, 4, 8), ReLU, pointwise channel mixing, per-time-point channel LayerNorm and dropout 0.1. Padding is masked after each block. The receptive field is `1 + 2*(1+2+4+8) = 31` bins, covering the entire 18-bin input. This permits interactions across the available observation window without widening the network substantially. Masked mean, masked maximum and last-valid representations are concatenated with a 16-unit tabular branch; a 16-unit hidden layer produces one logit. Parameter counts depend on selected tabular features and are exported per fit.

The static branch receives age/sex in both lab-only CNNs. The summary variant additionally receives the current and, when selected, historical lab-summary blocks. Neither CNN receives ICD features. Historical data remain summaries, not a fabricated continuous sequence.

AdamW uses learning rate 0.001 and weight decay 0.001, batch size 256, at most 40 epochs and patience 8 on inner-MIMIC AP. Gradient norm is clipped at 5; clipping frequency, epoch time, selected duration and model specification are logged. Fresh refitting uses all outer-training patients for the selected duration. Training loss follows the grid's sampling policy; no extra balanced loss is added after row resampling. Weighted/resampled scores are not assumed calibrated probabilities. Width and regularization are conservative research defaults; improved performance must be demonstrated, not presumed.

### Execution and interpretation

V12 writes a separate output tree and reuses only compatible raw feature caches. Its changed summaries and CNN comparisons require new model fits. Completed v12 inner/outer CNN stages are resumable; partial epochs are not. Prior source files and outputs remain untouched.

The same-run sampling report is generated before the grid. Revised predictions need execution: prior performance must not be relabeled as v12 performance. The design audit and smoke tests establish representation and execution behavior, not clinical validity, superiority, pre-echo lead time or prospective utility.


In [ ]:
"""Sampling-aware CNN. Raw feature calculations and non-CNN models are unchanged."""
if '_CNN_V10_BINDINGS' not in globals():
    _CNN_V10_BINDINGS={name:globals()[name] for name in (
        'SequenceTransform','HybridLabCNN','GridTransform','train_cnn',
        'fit_grid_partition','fitting_fingerprint','run_unified_grid','validate_config')}
CNN_V11_DEFAULTS=dict(cnn_dilations=(1,2,4,8),cnn_value_transform='log1p',
    cnn_carry_forward=True,cnn_add_current_summaries=True,cnn_grad_clip=5.)
for key,value in CNN_V11_DEFAULTS.items():CFG.setdefault(key,value)

class GridTransform(_CNN_V10_BINDINGS['GridTransform']):
    def fit(self,train,combo,cfg,seed,cnn=False):
        # The CNN receives the SAME requested tabular blocks as the other models,
        # plus the sequence branch. The flag supports a prespecified ablation.
        omit_current=cnn and not cfg['cnn_add_current_summaries']
        result=super().fit(train,combo,cfg,seed,cnn=omit_current)
        self.cnn=cnn
        return result

class SequenceTransform:
    def __init__(self,cfg):self.cfg=dict(cfg)
    def fit(self,seq):
        valid=seq['valid'].astype(bool)
        if not valid.any():raise ValueError('No valid sequence bins in training')
        x=seq['values'][valid]
        if self.cfg['cnn_value_transform']=='log1p':x=np.sign(x)*np.log1p(np.abs(x))
        self.norm=TrainOnlyTransform().fit(x)
        self.active=None;low=high=None
        # Do not materialize an entire extra transformed cohort to find constants.
        for start in range(0,len(valid),1024):
            sub=take_sequence(seq,slice(start,start+1024))
            a=self.transform(sub).transpose(0,2,1)[sub['valid'].astype(bool)]
            if not len(a):continue
            if low is None:self.baseline=a[0].copy();low=a.min(0);high=a.max(0)
            else:low=np.minimum(low,a.min(0));high=np.maximum(high,a.max(0))
        self.active=high>low
        return self
    def transform(self,seq):
        n,t,k=seq['values'].shape
        if k!=len(self.norm.median):raise ValueError('Sequence lab panel differs from fitted transform')
        out=np.empty((n,3*k+2,t),np.float32)
        for start in range(0,n,1024):
            end=min(start+1024,n)
            x=seq['values'][start:end];valid=seq['valid'][start:end].astype(bool)
            if np.any(np.diff(valid.astype(int),axis=1)>0):raise ValueError('Valid bins must form a prefix')
            observed=np.isfinite(x)&self.norm.supported[None,None,:]&valid[:,:,None]
            times=seq['hours'][start:end]
            if not np.isfinite(times[observed]).all():raise ValueError('Observed labs need actual timestamps')
            if self.cfg['cnn_value_transform']=='log1p':x=np.sign(x)*np.log1p(np.abs(x))
            z=(np.where(observed,x,self.norm.median)-self.norm.median)/self.norm.scale
            z=np.clip(z,-10,10).astype(np.float32)
            last_value=np.zeros((end-start,k),np.float32)
            last_time=np.full((end-start,k),np.nan)
            gap=np.ones_like(z)
            for b in range(t):
                obs=observed[:,b]
                last_value=np.where(obs,z[:,b],last_value)
                last_time=np.where(obs,times[:,b],last_time)
                if self.cfg['cnn_carry_forward']:z[:,b]=last_value
                clock=np.minimum((b+1)*self.cfg['cnn_bin_hours'],seq['cutoff'][start:end])[:,None]
                gap[:,b]=np.where(np.isfinite(last_time),np.maximum(clock-last_time,0)/self.cfg['cnn_window_hours'],1.)
            clock=np.minimum((np.arange(t)+1)*self.cfg['cnn_bin_hours'],seq['cutoff'][start:end,None])/self.cfg['cnn_window_hours']
            a=np.concatenate([z,observed.astype(np.float32),gap,clock[:,:,None],valid[:,:,None]],axis=2)
            if self.active is not None:a[:,:,~self.active]=self.baseline[~self.active]
            a*=valid[:,:,None]
            if not np.isfinite(a).all():raise ValueError('Nonfinite CNN input')
            out[start:end]=a.transpose(0,2,1)
        return out

class HybridLabCNN(nn.Module):
    def __init__(self,channels,tab_dim,cfg):
        super().__init__();w=cfg['width'];self.dilations=tuple(cfg['cnn_dilations'])
        self.proj=nn.Conv1d(channels,w,1)
        # Depthwise temporal + pointwise channel mixing limits CPU cost and parameters.
        self.temporal=nn.ModuleList([nn.Conv1d(w,w,3,padding=2*d,dilation=d,groups=w) for d in self.dilations])
        self.mix=nn.ModuleList([nn.Conv1d(w,w,1) for _ in self.dilations])
        self.norms=nn.ModuleList([nn.LayerNorm(w) for _ in self.dilations])
        self.drop=nn.Dropout1d(cfg['dropout'])
        self.tab=nn.Sequential(nn.Linear(tab_dim,w),nn.ReLU(),nn.Dropout(cfg['dropout']))
        self.out=nn.Sequential(nn.Linear(4*w,w),nn.ReLU(),nn.Dropout(cfg['dropout']),nn.Linear(w,1))
        self.receptive_field_bins=1+2*sum(self.dilations)
    def forward(self,s,x,mask):
        valid=mask[:,None,:];length=s.shape[2]
        h=torch.relu(self.proj(s*valid))*valid
        for conv,mix,norm in zip(self.temporal,self.mix,self.norms):
            # Trim right padding: no later-bin input enters an earlier-bin feature.
            z=mix(torch.relu(conv(h)[:,:,:length]))
            z=norm(z.transpose(1,2)).transpose(1,2)
            h=torch.relu(h+self.drop(z))*valid
        mean=h.sum(2)/valid.sum(2).clamp_min(1)
        maximum=h.masked_fill(~valid.bool(),torch.finfo(h.dtype).min).amax(2)
        maximum=torch.where(valid.sum(2)>0,maximum,torch.zeros_like(maximum))
        last=(mask.sum(1).long()-1).clamp_min(0)
        latest=h.gather(2,last[:,None,None].expand(-1,h.shape[1],1)).squeeze(2)
        return self.out(torch.cat([mean,maximum,latest,self.tab(x)],1)).squeeze(1)

def train_cnn(s,x,v,y,mode,cfg,seed,epochs,validation=None,row_indices=None):
    seed_everything(seed,cfg);model=HybridLabCNN(s.shape[1],x.shape[1],cfg)
    ix=np.arange(len(y)) if row_indices is None else np.asarray(row_indices,int)
    yr=np.asarray(y)[ix]
    weight=float((yr==0).sum()/max((yr==1).sum(),1)) if mode=='balanced' else 1.
    loss_fn=nn.BCEWithLogitsLoss(pos_weight=torch.tensor(weight,dtype=torch.float32))
    opt=torch.optim.AdamW(model.parameters(),lr=cfg['lr'],weight_decay=cfg['weight_decay'])
    bank=TensorDataset(torch.from_numpy(s),torch.from_numpy(x),torch.from_numpy(v),torch.tensor(np.asarray(y),dtype=torch.float32))
    loader=DataLoader(Subset(bank,ix.tolist()),batch_size=cfg['batch_size'],shuffle=True,
        generator=torch.Generator().manual_seed(seed))
    best=-np.inf;best_epoch=1;best_state=None;logs=[]
    for epoch in range(1,epochs+1):
        model.train();total=0.;clipped=0;started=time.perf_counter()
        for sb,xb,vb,yb in loader:
            opt.zero_grad();loss=loss_fn(model(sb,xb,vb),yb)
            if not torch.isfinite(loss):raise FloatingPointError('Nonfinite CNN loss; no checkpoint saved')
            loss.backward()
            norm=torch.nn.utils.clip_grad_norm_(model.parameters(),cfg['cnn_grad_clip'],error_if_nonfinite=True)
            clipped+=int(float(norm)>cfg['cnn_grad_clip']);opt.step();total+=loss.item()*len(yb)
        ap=np.nan
        if validation is not None:
            vs,vx,vv,vy=validation;ap=average_precision_score(vy,predict_cnn(model,vs,vx,vv))
            if ap>best+1e-6:best=ap;best_epoch=epoch;best_state=copy.deepcopy(model.state_dict())
        logs.append(dict(epoch=epoch,loss=total/len(ix),inner_AP=ap,
            clipped_batch_fraction=clipped/max(len(loader),1),epoch_seconds=time.perf_counter()-started))
        if epoch==1 or epoch%5==0 or epoch==epochs:
            print(f'[CNN epoch] {epoch}/{epochs}: loss={total/len(ix):.4f}, inner_AP={ap:.4f}, {time.perf_counter()-started:.1f}s',flush=True)
        if validation is not None and epoch-best_epoch>=cfg['patience']:break
    if validation is not None:model.load_state_dict(best_state)
    else:best_epoch=epochs
    return model,best_epoch,logs

def validate_config(cfg):
    _CNN_V10_BINDINGS['validate_config'](cfg)
    if cfg['cnn_value_transform'] not in ('raw','log1p'):raise ValueError('CNN value transform must be raw or log1p')
    if not cfg['cnn_dilations'] or any(not isinstance(d,int) or d<1 for d in cfg['cnn_dilations']):raise ValueError('CNN dilations must be positive integers')
    if 1+2*sum(cfg['cnn_dilations'])<int(np.ceil(cfg['cnn_window_hours']/cfg['cnn_bin_hours'])):
        raise ValueError('CNN receptive field does not cover the configured time window; increase dilations')
    if cfg['cnn_grad_clip']<=0 or not 0<=cfg['dropout']<1:raise ValueError('Invalid CNN optimization configuration')

def _cnn_partition_checkpoint(train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,ti,vi,ix,fx,folder):
    """Inner and outer CNN work saved separately; namespace fingerprints bind all inputs."""
    y=train.label.to_numpy(int);mode='balanced' if setting['mode']=='none' else 'unweighted'
    folder=None if folder is None else Path(folder)
    name='ICD_LabCNN_Fusion' if set(combo)&set(ICD_BLOCKS) else 'LabCNN'
    output={};records=[];logs=[];epochs=None
    for stage in ('inner','outer'):
        fit_idx=ti if stage=='inner' else np.arange(len(train))
        target_frame=train.iloc[vi] if stage=='inner' else target
        target_bank=take_sequence(seq,vi) if stage=='inner' else target_seq
        cohort='MIMIC' if stage=='inner' else target_cohort
        path=None if folder is None else folder/f'CNN_{stage}.pkl'
        if path is not None and path.is_file():
            saved=_v9_read_pickle(path)
            if saved['rows']!=len(target_frame) or not np.isfinite(saved['probs']).all():raise ValueError('Invalid CNN checkpoint')
            output[stage]=saved['probs'];records.append(saved['audit'])
            if stage=='inner':epochs=saved['epochs'];logs=saved['logs']
            print(f'[CNN] {stage}: resumed',flush=True);continue
        started=time.perf_counter();fit_frame=train.iloc[fit_idx]
        print(f'[CNN] {stage}: preparing {len(fit_frame):,} admissions, {cfg["cnn_bin_hours"]:g}-hour bins',flush=True)
        tab=GridTransform().fit(fit_frame,combo,cfg,seed,cnn=True)
        bank=take_sequence(seq,fit_idx);norm=SequenceTransform(cfg).fit(bank)
        sx=norm.transform(bank);tx=tab.transform(fit_frame,'MIMIC')
        sv=norm.transform(target_bank);tv=tab.transform(target_frame,cohort)
        validation=(sv,tv,target_bank['valid'],y[vi]) if stage=='inner' else None
        model,best,history=train_cnn(sx,tx,bank['valid'],y[fit_idx],mode,cfg,seed,
            cfg['max_epochs'] if stage=='inner' else epochs,validation,row_indices=ix if stage=='inner' else fx)
        probs=predict_cnn(model,sv,tv,target_bank['valid'])
        if stage=='inner':epochs=best;logs=history
        record=dict(model=name,stage=stage,seconds=time.perf_counter()-started,epochs=best,
            sequence_channels=sx.shape[1],sequence_bins=sx.shape[2],tabular_columns=tx.shape[1],
            parameters=sum(p.numel() for p in model.parameters()),receptive_field_bins=model.receptive_field_bins,
            summary_features_included=cfg['cnn_add_current_summaries'])
        if path is not None:
            _v9_pickle(path,dict(rows=len(target_frame),probs=probs,audit=record,epochs=best,logs=history,
                state_dict={k:v.detach().cpu().numpy() for k,v in model.state_dict().items()}))
        output[stage]=probs;records.append(record)
        print(f'[CNN] {stage}: saved, {best} epochs, {record["parameters"]:,} parameters, {record["seconds"]:.1f}s',flush=True)
    return name,output['inner'],output['outer'],logs,records

def _cnn_v10_fingerprint(m,ms,e,es,cfg):
    """Exact old code/config key, restored only while hashing. No model is run."""
    current={name:globals()[name] for name in _CNN_V10_BINDINGS}
    try:
        globals().update(_CNN_V10_BINDINGS)
        old_cfg={k:v for k,v in cfg.items() if k not in CNN_V11_DEFAULTS}
        return _CNN_V10_BINDINGS['fitting_fingerprint'](m,ms,e,es,old_cfg)
    finally:globals().update(current)

def fitting_fingerprint(m,ms,e,es,cfg):
    base=_CNN_V10_BINDINGS['fitting_fingerprint'](m,ms,e,es,cfg)
    code=stable_code_bytes(_cnn_partition_checkpoint.__code__)
    return hashlib.sha256(base.encode()+code+b'cnn-v11-sampling-aware-1').hexdigest()[:24]

def run_unified_grid(m,ms,e,es,cfg,out):
    validate_config(cfg)
    current=fitting_fingerprint(m,ms,e,es,cfg)
    old=_cnn_v10_fingerprint(m,ms,e,es,cfg)
    source=DATA_ROOT/'BE_integrated_grid_v10_outputs'/Path(out).name/'grid_checkpoints'/old
    target=Path(out)/'grid_checkpoints'/current;copied=0
    # Only unchanged tabular estimator predictions. Never copy completed folds,
    # metric reports, CNN models or CNN predictions into the revised experiment.
    if source.is_dir() and source.resolve()!=target.resolve():
        for file in source.glob('*.models/*.pkl'):
            if file.name not in {f'{name}_{stage}.pkl' for name in cfg['base_models'] for stage in ('inner','outer')}:continue
            dest=target/file.relative_to(source)
            if dest.exists():continue
            saved=_v9_read_pickle(file)
            if not np.isfinite(saved['probs']).all():raise ValueError('Invalid v10 tabular prediction cache')
            if file.name.startswith('Lasso_'):
                gap=saved['audit'].get('lasso_relative_dual_gap',np.inf)
                if not np.isfinite(gap) or gap>cfg['lasso_objective_gap']:raise ValueError('Uncertified v10 Lasso cache')
            _v9_pickle(dest,saved);copied+=1
        target.mkdir(parents=True,exist_ok=True)
        manifest=target/'cnn_v11_migration.json'
        if not manifest.exists():
            manifest.write_text(json.dumps(dict(source_key=old,target_key=current,imported_tabular_files=copied,
                cnn_reused=False,completed_folds_reused=False,source_untouched=True),indent=2))
    print(f'[CNN v11] Imported {copied} exact-key-compatible tabular checkpoints. CNN fits use their own new namespace.',flush=True)
    return _L10_BEFORE_REPAIR['run_unified_grid'](m,ms,e,es,cfg,out)

def cnn_sampling_diagnostics(m,ms,e,es,cfg,out):
    """Same-run representation audit, before training. No tuning using labels."""
    rows=[]
    for cohort,frame,seq in [('MIMIC',m,ms),('eICU',e,es)]:
        seq=panel_sequence(seq,cfg);labs=list(CORE) if cfg['lab_panel']=='core' else list(LABS)
        for landmark in (1,2):
            age=np.ones(len(frame),bool) if cfg['age_policy']=='all_ages' else frame.age.ge(18).to_numpy()
            indices=np.flatnonzero(age&frame.landmark.eq(landmark).to_numpy())
            nonempty=[];repeated=[];kept=0
            for start in range(0,len(indices),1024):
                obs=np.isfinite(seq['values'][indices[start:start+1024]])
                nonempty.extend(obs.any(2).sum(1).tolist());repeated.extend((obs.sum(1)>=2).sum(1).tolist());kept+=int(obs.sum())
            if not len(indices):continue
            counts=float(frame.iloc[indices][[f'current__{a}__count' for a in labs]].sum().sum())
            rows.append(dict(cohort=cohort,landmark=landmark,admissions=len(indices),bin_hours=cfg['cnn_bin_hours'],
                median_nonempty_bins=float(np.median(nonempty)),one_nonempty_bin_fraction=float(np.mean(np.asarray(nonempty)==1)),
                no_repeated_analyte_fraction=float(np.mean(np.asarray(repeated)==0)),raw_observations=counts,
                binned_observations=kept,observation_retention=kept/counts if counts else np.nan,
                median_collection_cutoff_hours=float(np.median(seq['cutoff'][indices])),
                median_score_hours=float(frame.iloc[indices].score_hour.median()),
                current_summaries_in_cnn=cfg['cnn_add_current_summaries']))
    report=pd.DataFrame(rows);report.to_csv(Path(out)/'cnn_sampling_diagnostics.csv',index=False)
    display(report)
    return report


def fit_grid_partition(train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,checkpoint_dir=None):
    """All selection/scaling nested. Same inner patient holdout for all model families."""
    seq=panel_sequence(seq,cfg);target_seq=panel_sequence(target_seq,cfg)
    ti,vi=inner_split(train,seed,cfg['inner_holdout']);y=train.label.to_numpy(int)
    inner=GridTransform().fit(train.iloc[ti],combo,cfg,seed)
    xt=inner.transform(train.iloc[ti],'MIMIC');xv=inner.transform(train.iloc[vi],'MIMIC')
    outer=GridTransform().fit(train,combo,cfg,seed)
    xf=outer.transform(train,'MIMIC');xe=outer.transform(target,target_cohort)
    ix=sample_indices(y[ti],setting,seed);fx=sample_indices(y,setting,seed)
    preds={};cal={};logs=[]
    fit_records=[]
    for name in cfg['base_models']:
        cal[name],inner_record=_fit_model_checkpoint(name,xt[ix],y[ti][ix],xv,setting,cfg,seed,'inner',checkpoint_dir)
        preds[name],outer_record=_fit_model_checkpoint(name,xf[fx],y[fx],xe,setting,cfg,seed,'outer',checkpoint_dir)
        fit_records.extend([inner_record,outer_record])
    if cfg['include_ensembles']:
        if not {'RandomForest','Lasso'}<=set(preds):raise ValueError('RF and Lasso are required for ensembles')
        ip=np.column_stack([cal['RandomForest'],cal['Lasso']]);ep=np.column_stack([preds['RandomForest'],preds['Lasso']])
        meta=LogisticRegression(max_iter=1000).fit(ip,y[vi])
        cal[ENSEMBLES[0]]=ip.mean(1);preds[ENSEMBLES[0]]=ep.mean(1)
        cal[ENSEMBLES[1]]=meta.predict_proba(ip)[:,1];preds[ENSEMBLES[1]]=meta.predict_proba(ep)[:,1]
    if 'lab_current' in combo and cfg['include_cnn']:
        name,inner_probs,outer_probs,logs,cnn_records=_cnn_partition_checkpoint(
            train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,ti,vi,ix,fx,checkpoint_dir)
        cal[name]=inner_probs;preds[name]=outer_probs;fit_records.extend(cnn_records)
    thresholds={name:youden_threshold(y[vi],p) for name,p in cal.items()}
    operating={model:{operating_key(t):operating_threshold(y[vi],p,t) for t in cfg['recall_targets']} for model,p in cal.items()}
    audit=dict(operating_thresholds_json=json.dumps(operating),train_admissions=len(train),train_patients=train.patient_id.nunique(),
        inner_validation_patients=train.iloc[vi].patient_id.nunique(),raw_predictors=outer.raw(train,'MIMIC').shape[1],
        natural_prevalence=float(y.mean()),fit_prevalence=float(y[fx].mean()),fit_rows=len(fx),
        duplicated_rows=len(fx)-len(np.unique(fx)),external_prevalence_used=setting['external_prevalence_used'],
        selected_ICD_features='|'.join(outer.icd_names),threshold_source='MIMIC inner patient holdout',model_timing_json=json.dumps(fit_records))
    return preds,thresholds,logs,audit


## Lab representation: question, evidence and active experiment

The intended use is to investigate whether available blood results, with recorded history, can raise suspicion for endocarditis and support consideration of echocardiography. This retrospective hospitalization-label experiment does **not** establish that the score precedes clinical suspicion, antibiotics, blood cultures or echo. It is not a validated echo referral rule. Those questions require reliable procedure/treatment/result-availability timing and prospective evaluation. First and second draw-cluster landmarks are different prediction times, not interchangeable estimates of the same task.

### What the actual sampling supports

A streaming audit of the existing adult raw-feature cache (54f4f143685242ca) included 48,132 MIMIC and 148,686 eICU first-landmark admissions, and 45,205 and 128,076 second-landmark admissions. At landmark 1, 97.91% of observed MIMIC analyte–admission pairs and 98.27% of eICU pairs had one timestamp. At landmark 2 these fractions were 29.11% and 32.97%. Among computable second-landmark slopes, 83.19% and 84.78%, respectively, used just two timestamps. These denominators are analyte–admission pairs, not patients. They describe the audited cache, not every possible sensitivity cohort. The same-run support report below recomputes these diagnostics for the configured cohort.

At the first landmark, only 36/382 MIMIC BE admissions and 11/439 eICU BE admissions had CRP; all were single observations. ESR was observed in 40/382 and 29/439, also without repeats. Biological plausibility of inflammatory markers does not create unavailable trajectories. Their ordering/missingness may reflect clinician suspicion and workup as well as disease.

The separate label-blind raw-timestamp sample (2,000 admissions per cohort; MIMIC sampling excluded one fixed study fold) found that three-hour last-value binning retained roughly 98–99% of observations overall. This aggregate hides analyte-specific losses: about 19.3% of first-landmark MIMIC lactate observations were discarded within bins; repeated lactate measurements had a median gap of about 1.58 hours. One-hour bins preserved substantially more eligible short trends in that sample. Raw summaries retain all qualifying observations and their actual times, independently of CNN bins.

This is evidence of sparse trajectories and within-bin information loss, **not** proof that the CNN learns only noise or a demonstrated Nyquist/aliasing threshold. Clinical sampling is irregular, clinician-directed and not a known band-limited signal. A convolution may exploit levels, cross-analyte relationships, elapsed time and ordering patterns rather than rich biological trajectories. Its incremental temporal value must be measured against simpler models on the same held-out patients. Lower training loss alone is insufficient.

### Model comparisons in the grid

1. `LabCNN_Sequence`: current-admission sequences plus age/sex, without ICD or summary inputs.
2. `LabCNN_Summary`: the same sequences and age/sex plus current and historical summaries and current-versus-history differences, without ICD.
3. Tabular models and RF/Lasso ensembles: requested ICD/lab/demographic combinations in the representative grid, with the full combinatorial profile available explicitly.

The two CNNs are evaluated on the all-labs population at both landmarks using unweighted natural-prevalence training. A targeted control-duplication prevalence-match comparison evaluates the summary-augmented CNN at both landmarks. The comparison tests adding the entire summary bundle, not current and historical summaries separately. No additional CNN is trained on the ICD-history subgroup or in policy sensitivities. Model audits list actual static source columns.

The architecture, train-only normalization, observed masks, elapsed-time channels, padding masks and current-admission forward fill remain as specified in the sequence-reference section. The three-hour representation is a computationally tractable reference, not a literature-proven optimum. A prespecified finer-resolution run can set `cnn_bin_hours=1.` and `cnn_dilations=(1,2,4,8,16)` before preparation; it needs newly built tensors and new CNN fits. Do not select bin width by eICU performance. Forward-filled values are not new observations. The hybrid summaries provide raw-time endpoint/slope information that binning can lose.

### Summary definitions and observational support

Level features retain first, last, mean, median, minimum and maximum over observations available at the prediction cutoff. The mean is observation-weighted, not time-weighted: frequent testing receives more weight. One observation makes these features identical; two points make several redundant. They are candidate representations, not six independent pieces of information. Their incremental contribution requires training-only ablation/selection rather than external-outcome-driven pruning.

Current SD and IQR, and historical SD/IQR where present, are missing with fewer than two observations. A singleton does not establish temporal stability. For repeated observations they remain descriptive dispersion, not measurement-error estimates or necessarily biological variation. Count and span remain explicit; no skewness/kurtosis is added to mostly one-/two-point series.

Endpoint change is `last-first` whenever two distinct timestamps are observed, even if the span is too short for a slope. The existing minimum-span rule still governs slope and numerical rise/fall. Raw rise/fall no longer requires a demographic reference interval. Separate indicators identify valid two-point slopes versus slopes supported by three or more observations. A two-point slope is a secant, not evidence for a trajectory shape. The per-day unit is a rate scaling, not a prediction of a 24-hour change.

Reference-relative direction and boundary-crossing features still require an applicable reference interval. Rising, approaching a boundary and moving toward a reference range are different quantities; none is labeled clinical improvement. No universal normal range or new demographic threshold is inferred from cohort outcomes. Stored raw caches remain intact: these row-local semantic rules are applied before each fold's train-only imputation/scaling and identically at inference.

Historical summaries remain distinct from current-admission dynamics. Pooled historical means/medians describe observed prior care, not a healthy personal baseline. Stay-level means reduce unequal weighting by number of tests but are not a chronological trend. Historical slopes are calculated only within a dated stay. For eICU, historical features use strictly earlier years; separate same-year hospitalizations are not ordered or joined into a fictional continuous sequence. Current-admission labs are permitted up to the landmark. Current-minus-history differences are not rates because the cross-stay interval is not uniformly known.

### Analyte-specific interpretation

- **WBC, neutrophil percentage, lymphocyte percentage, CRP and ESR:** preserve available levels and pre-landmark change, with count/span. Serial inflammatory-marker literature commonly studies several days of treatment response; it does not justify labeling a sparse admission measurement a response trajectory. Percent differentials are not absolute cell counts. [Serial CRP in sepsis](https://pmc.ncbi.nlm.nih.gov/articles/PMC3387609/) and [WBC trajectories in septic shock](https://pmc.ncbi.nlm.nih.gov/articles/PMC9334545/).
- **Lactate:** first/latest values, extrema and actual-time change are potentially complementary. A six-hour lactate-clearance study does not validate calling a variable-interval difference “six-hour clearance”; resuscitation and repeat-test selection confound interpretation. No clearance percentage with an arbitrary baseline/window is added. [Primary lactate comparison](https://pubmed.ncbi.nlm.nih.gov/23740148/).
- **Creatinine and BUN:** retain current levels and historical comparisons, clearly distinguishing prior-care values from admission values and inpatient nadirs. No new AKI label or automatically substituted “normal” baseline is created. Baseline choice changes the meaning of change. [Baseline renal-function comparison](https://pmc.ncbi.nlm.nih.gov/articles/PMC2929703/).
- **Hemoglobin, platelet, RDW and MCV:** retain levels, extrema and supported changes. Transfusion, bleeding, dilution and treatment can affect values. RDW already measures within-sample red-cell size dispersion; longitudinal SD of RDW is a different variable. Later hospital values cannot inform an earlier score. [RDW change in pneumonia](https://pmc.ncbi.nlm.nih.gov/articles/PMC5065335/).
- **Glucose, sodium, potassium, chloride, bicarbonate, calcium, magnesium and phosphate:** retain levels and supported dispersion/change, with observation intensity. Whole-stay or long-term variability associations do not establish stable early variability from two tests. Avoid adding CV indiscriminately or importing multi-year variability thresholds into an hourly task. [In-hospital glycemic variability](https://pmc.ncbi.nlm.nih.gov/articles/PMC3836112/).
- **Albumin, bilirubin, ALT, AST, alkaline phosphatase and LDH:** retain level/location, extrema and pre-cutoff change where observed. Albumin can reflect chronic illness, inflammation and fluid state; skewed enzymes and bilirubin warrant robust train-only scaling. Later recovery/prognostic patterns are not admission-time predictors. [Dynamic liver-model comparison](https://pmc.ncbi.nlm.nih.gov/articles/PMC11950807/).
- **INR and PTT:** reference-relative values and changes are descriptive; anticoagulant exposure and treatment targets can make “toward normal” an inappropriate clinical interpretation. Medication-adjusted causal interpretation is not available from these summaries alone.

These are representation/interpretation choices, not claims that each analyte is BE-specific or that a feature improves performance. Missingness and observation-only controls help assess dependence on the care process, but do not eliminate confounding.

### What the literature establishes—and does not

The evidence ledger separates clinical diagnostic/prognostic studies from temporal-method benchmarks, and abstract screening from complete main-methods review. Many papers evaluate associations or complete scores, not head-to-head summary types. Diseases, horizons, sampling frequencies and treatment stages differ; their AUROCs must not be ranked to choose a “best” BE feature.

A direct summary-statistic comparison found task-dependent value of central tendency versus extrema/dispersion in 24-hour ICU data, with nested selection. It supports testing representations, not assuming one combination wins in this sparse early-lab task. [Guo et al., 2020](https://pmc.ncbi.nlm.nih.gov/articles/PMC7059727/). Missingness-aware temporal models show why masks/gaps can matter, but do not establish CNN superiority for these cohorts. [GRU-D](https://www.nature.com/articles/s41598-018-24271-9).

The appropriate next performance analysis is the prespecified, same-patient comparison above, using MIMIC-held-out AP/AUROC, calibration and operating-point metrics, then external evaluation with all-row patient-cluster CIs. Prevalence-adapted runs remain explicitly separate. There is no demonstrated performance improvement from this implementation until it is run. Reusing eICU repeatedly for design decisions makes it an explored external cohort; a future untouched cohort is needed for a clean confirmatory claim.

### Reproducibility and execution

Raw feature caches can be reused if their timing/panel fingerprint matches. Exact-compatible V13 per-model predictions can be reused because V14 changes scheduling, not fitted-model definitions. Whole-fold V13 prediction bundles are not imported into the tabular grid. Inner/outer model checkpoints in this version remain resumable. No source CSV, existing notebook or prior output directory is deleted. The full results are saved; notebook display remains limited to the configured top rows, not the entire grid.


In [ ]:
"""Lab-only CNN comparisons and observation-supported summary semantics."""
if '_LAB_V11_BINDINGS' not in globals():
    _LAB_V11_BINDINGS={n:globals()[n] for n in (
        'GridTransform','fitting_fingerprint','wide_grid_results','validate_config')}
LAB_V12_DEFAULTS=dict(cnn_include_demographics=True,summary_semantics='observed_support_v12')
for key,value in LAB_V12_DEFAULTS.items():CFG.setdefault(key,value)

def supported_lab_summaries(frame):
    """Row-local corrections only; no labels, other patients or later results used.

    The stored raw caches remain unchanged. Counts refer to distinct timestamps,
    after the source builder resolves same-time observations.
    """
    updates={}
    for analyte in LABS:
        prefix=f'current__{analyte}__'
        if prefix+'count' not in frame:continue
        count=frame[prefix+'count'].to_numpy(float)
        first=frame[prefix+'first'].to_numpy(float);last=frame[prefix+'last'].to_numpy(float)
        span=frame[prefix+'span_hours'].to_numpy(float)
        pair=(count>=2)&np.isfinite(first)&np.isfinite(last)&(span>0)
        slope_ok=frame[prefix+'slope_valid'].eq(1).to_numpy()&pair
        for stat in ('sd','iqr'):
            updates[prefix+stat]=frame[prefix+stat].where(count>=2).to_numpy(float)
        # Endpoint difference is observed even when the interval is too short
        # for the prespecified slope calculation. It is not an hourly rate.
        updates[prefix+'change']=np.where(pair,last-first,np.nan)
        updates[prefix+'rising']=np.where(slope_ok,(last>first).astype(float),np.nan)
        updates[prefix+'falling']=np.where(slope_ok,(last<first).astype(float),np.nan)
        updates[prefix+'pair_observed']=pair.astype(float)
        updates[prefix+'slope_two_point']=(slope_ok&(count==2)).astype(float)
        updates[prefix+'slope_multi_point']=(slope_ok&(count>=3)).astype(float)
        for stat in ('sd','iqr'):
            name=f'history__{analyte}__{stat}';n=f'history__{analyte}__count'
            if name in frame and n in frame:
                updates[name]=frame[name].where(frame[n]>=2).to_numpy(float)
    if not updates:return frame
    # Shallow frame copy plus replacement arrays: do not modify shared raw data.
    out=frame.copy(deep=False)
    existing=[k for k in updates if k in out]
    out[existing]=pd.DataFrame({k:updates[k] for k in existing},index=out.index)
    new={k:v for k,v in updates.items() if k not in out}
    return pd.concat([out,pd.DataFrame(new,index=out.index)],axis=1) if new else out

class GridTransform(_LAB_V11_BINDINGS['GridTransform']):
    def fit(self,train,combo,cfg,seed,cnn=False):
        self.needs_lab_semantics=bool(set(combo)&set(('lab_current','lab_history')+OBS_BLOCKS))
        corrected=supported_lab_summaries(train) if self.needs_lab_semantics else train
        result=super().fit(corrected,combo,cfg,seed,cnn=cnn)
        # Raw directions no longer encode the presence of age/sex-specific
        # reference intervals. Preserve them when demographics are disabled.
        if not cfg['include_demographics'] and 'lab_current' in self.combo:
            self.lab_names += [n for n in corrected if n.startswith('current__')
                and n.split('__')[-1] in ('rising','falling') and n not in self.lab_names]
            self.norm=TrainOnlyTransform().fit(self.raw(corrected,'MIMIC'))
        return result
    def raw(self,frame,cohort):
        return super().raw(supported_lab_summaries(frame) if self.needs_lab_semantics else frame,cohort)

def cnn_variants(combo,cfg):
    """Do not refit the same sequence model for each ICD/history combination."""
    if not cfg['include_cnn'] or 'lab_current' not in combo or set(combo)&set(ICD_BLOCKS):return ()
    if not set(combo)<=set(('lab_current','lab_history')):return ()
    return ('LabCNN_Sequence','LabCNN_Summary') if 'lab_history' not in combo else ('LabCNN_Summary',)

def cnn_tabular_transform(train,combo,cfg,seed,variant):
    if set(combo)&set(ICD_BLOCKS):raise ValueError('CNN inputs must not contain ICD blocks')
    if variant not in ('LabCNN_Sequence','LabCNN_Summary'):raise ValueError(variant)
    setting=dict(cfg,include_demographics=cfg['cnn_include_demographics'],cnn_add_current_summaries=True)
    use=() if variant=='LabCNN_Sequence' else combo
    return GridTransform().fit(train,use,setting,seed,cnn=True)

def _lab_cnn_checkpoint(train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,ti,vi,ix,fx,folder,variant):
    y=train.label.to_numpy(int);mode='balanced' if setting['mode']=='none' else 'unweighted'
    folder=None if folder is None else Path(folder)/variant
    result={};records=[];logs=[];epochs=None
    for stage in ('inner','outer'):
        fit_idx=ti if stage=='inner' else np.arange(len(train))
        target_frame=train.iloc[vi] if stage=='inner' else target
        target_bank=take_sequence(seq,vi) if stage=='inner' else target_seq
        cohort='MIMIC' if stage=='inner' else target_cohort
        path=None if folder is None else folder/f'CNN_{stage}.pkl'
        if path is not None and path.is_file():
            saved=_v9_read_pickle(path)
            if saved['rows']!=len(target_frame) or saved['audit']['model']!=variant or not np.isfinite(saved['probs']).all():
                raise ValueError('Invalid CNN checkpoint')
            result[stage]=saved['probs'];records.append(saved['audit'])
            if stage=='inner':epochs=saved['epochs'];logs=saved['logs']
            print(f'[{variant}] {stage}: resumed',flush=True);continue
        started=time.perf_counter();fit_frame=train.iloc[fit_idx]
        tab=cnn_tabular_transform(fit_frame,combo,cfg,seed,variant)
        assert not tab.icd_presence and not tab.icd_names
        if variant=='LabCNN_Sequence':assert set(tab.lab_names)<=set(DEMOGRAPHICS)
        bank=take_sequence(seq,fit_idx);norm=SequenceTransform(cfg).fit(bank)
        sx=norm.transform(bank);tx=tab.transform(fit_frame,'MIMIC')
        sv=norm.transform(target_bank);tv=tab.transform(target_frame,cohort)
        validation=(sv,tv,target_bank['valid'],y[vi]) if stage=='inner' else None
        print(f'[{variant}] {stage}: {len(fit_frame):,} admissions, {tx.shape[1]} static columns',flush=True)
        model,best,history=train_cnn(sx,tx,bank['valid'],y[fit_idx],mode,cfg,seed,
            cfg['max_epochs'] if stage=='inner' else epochs,validation,row_indices=ix if stage=='inner' else fx)
        probs=predict_cnn(model,sv,tv,target_bank['valid'])
        history=[dict(r,model=variant) for r in history]
        if stage=='inner':epochs=best;logs=history
        record=dict(model=variant,stage=stage,seconds=time.perf_counter()-started,epochs=best,
            sequence_channels=sx.shape[1],sequence_bins=sx.shape[2],tabular_columns=tx.shape[1],
            parameters=sum(p.numel() for p in model.parameters()),receptive_field_bins=model.receptive_field_bins,
            summary_features_included=variant=='LabCNN_Summary',icd_features_included=False,
            demographics_included=cfg['cnn_include_demographics'],static_source_features=tab.lab_names)
        if path is not None:
            _v9_pickle(path,dict(rows=len(target_frame),probs=probs,audit=record,epochs=best,logs=history,
                state_dict={k:v.detach().cpu().numpy() for k,v in model.state_dict().items()}))
        result[stage]=probs;records.append(record)
        print(f'[{variant}] {stage}: saved after {record["seconds"]:.1f}s',flush=True)
    return result['inner'],result['outer'],logs,records

def grid_plan(cfg):
    ordinary=len(cfg['base_models'])+len(ENSEMBLES)*cfg['include_ensembles']
    sampling=len(sampling_plan(cfg));rows=0;cells=0;cnn=0
    for population in POPULATIONS:
        combos=population_combinations(population,cfg);cells+=2*sampling*len(combos)
        rows+=2*sampling*sum(ordinary+len(cnn_variants(c,cfg)) for c in combos)
        cnn+=2*sampling*sum(len(cnn_variants(c,cfg)) for c in combos)
    return dict(profile=cfg['grid_profile'],age_policy=cfg['age_policy'],populations=POPULATIONS,
        grid_cells=cells,expected_wide_result_rows=rows,cohort_metric_rows=rows*2,
        base_estimator_fits=cells*(cfg['outer_folds']+1)*2*len(cfg['base_models']),
        cnn_training_runs=cnn*(cfg['outer_folds']+1)*2,bootstrap_draws_per_row=cfg['n_boot'],
        note='Both CNNs use demographics, neither uses ICD. Sequence-only appears once per lab-current setting. Counts assume preflight passes.')

def validate_config(cfg):
    _LAB_V11_BINDINGS['validate_config'](cfg)
    if cfg['summary_semantics']!='observed_support_v12':raise ValueError('Unsupported summary semantics')
    if not isinstance(cfg['cnn_include_demographics'],bool):raise ValueError('CNN demographic flag must be boolean')

def fitting_fingerprint(m,ms,e,es,cfg):
    base=_LAB_V11_BINDINGS['fitting_fingerprint'](m,ms,e,es,cfg)
    funcs=[supported_lab_summaries,cnn_variants,cnn_tabular_transform,_lab_cnn_checkpoint,
           GridTransform.fit,GridTransform.raw,fit_grid_partition]
    # Captured parent classes remain executable dependencies of super() calls.
    # Hash their own implementations, not only the currently bound overrides.
    for cls in GridTransform.__mro__:
        funcs.extend(v for _,v in sorted(vars(cls).items()) if hasattr(v,'__code__'))
    return hashlib.sha256(base.encode()+b''.join(stable_code_bytes(f.__code__) for f in funcs)+b'lab-v12-2').hexdigest()[:24]

def run_unified_grid(m,ms,e,es,cfg,out):
    validate_config(cfg)
    # Revised lab semantics and model families require new prediction caches.
    # Reuse raw feature caches, never v10/v11 model predictions or completion flags.
    return _L10_BEFORE_REPAIR['run_unified_grid'](m,ms,e,es,cfg,out)

def wide_grid_results(metrics):
    w=_LAB_V11_BINDINGS['wide_grid_results'](metrics)
    w['CNN_input']=w.Model.map({'LabCNN_Sequence':'sequence',
        'LabCNN_Summary':'sequence + requested lab summaries'}).fillna('not CNN')
    return w

def summary_support_report(m,e,cfg,out):
    """Same-run, label-blind support by cohort/landmark/analyte; raw caches untouched."""
    rows=[]
    for cohort,frame in [('MIMIC',m),('eICU',e)]:
        eligible=frame if cfg['age_policy']=='all_ages' else frame[frame.age.ge(18)]
        for landmark,g in eligible.groupby('landmark'):
            for a in LABS:
                if cfg['lab_panel']=='core' and a not in CORE:continue
                p=f'current__{a}__';n=g[p+'count'];valid=g[p+'slope_valid'].eq(1)
                rows.append(dict(cohort=cohort,landmark=landmark,analyte=a,admissions=len(g),
                    observed=int(n.gt(0).sum()),singleton=int(n.eq(1).sum()),
                    valid_slope=int(valid.sum()),two_point_slope=int((valid&n.eq(2)).sum()),
                    three_plus=int(n.ge(3).sum()),history_observed=int(g[f'history__{a}__count'].gt(0).sum())))
    report=pd.DataFrame(rows);report.to_csv(Path(out)/'lab_summary_support.csv',index=False)
    pooled=report.groupby(['cohort','landmark'])[['observed','singleton','valid_slope','two_point_slope']].sum()
    pooled['singleton_fraction_of_observed_pairs']=pooled.singleton/pooled.observed
    pooled['two_point_fraction_of_slopes']=pooled.two_point_slope/pooled.valid_slope.replace(0,np.nan)
    print('Denominator: analyte–admission pairs, not patients. Full per-analyte audit saved to CSV.')
    display(pooled[['singleton_fraction_of_observed_pairs','two_point_fraction_of_slopes']].round(4))
    return report

def plot_results(pred,metrics,logs,cfg,out):
    """Prespecified lab comparisons, including sequence-only; no eICU selection."""
    out=Path(out);population=cfg.get('population_label','')
    for landmark in (1,2):
        fig,axes=plt.subplots(2,2,figsize=(14,10))
        for i,cohort in enumerate(['MIMIC_OOF','eICU_external']):
            d=pred[pred.landmark.eq(landmark)&pred.cohort.eq(cohort)&
                pred.scope.isin(['current_only','current_plus_history'])]
            for (model,scope,sampling),g in d.groupby(['model','scope','sampling']):
                if g.label.nunique()<2:continue
                fpr,tpr,_=roc_curve(g.label,g.probability);pr,re,_=precision_recall_curve(g.label,g.probability)
                label=f'{model} | {scope}'
                axes[i,0].plot(fpr,tpr,label=f'{label} ({roc_auc_score(g.label,g.probability):.3f})')
                axes[i,1].plot(re,pr,label=f'{label} ({average_precision_score(g.label,g.probability):.3f})')
            base=d.drop_duplicates('hospital_id').label.mean()
            axes[i,0].plot([0,1],[0,1],'k--',lw=1)
            axes[i,1].axhline(base,color='gray',ls='--',label=f'Prevalence {base:.3%}')
            for j,title in enumerate(['ROC','Precision–recall']):
                ax=axes[i,j];ax.set_title(f'{cohort} | set {landmark} | {title}');ax.set_xlim(0,1);ax.set_ylim(0,1)
                ax.set_xlabel('False positive rate' if j==0 else 'Recall');ax.set_ylabel('Recall' if j==0 else 'Precision')
                ax.legend(fontsize=6);ax.grid(alpha=.2)
        fig.suptitle(population);fig.tight_layout();fig.savefig(out/f'landmark_{landmark}_curves.png',dpi=180);plt.show()
    largest=metrics.groupby(['cohort','landmark']).size().max()
    fig,axes=plt.subplots(2,2,figsize=(17,max(10,largest*.58)))
    for i,cohort in enumerate(['MIMIC_OOF','eICU_external']):
        for j,landmark in enumerate([1,2]):
            d=metrics[metrics.cohort.eq(cohort)&metrics.landmark.eq(landmark)].sort_values(['sampling','model','scope'])
            ax=axes[i,j];x=np.arange(len(d))
            if d.empty:ax.axis('off');continue
            ax.barh(x,d.AP,height=.6,color=['tab:blue' if s=='current_plus_history' else 'tab:orange' if s=='current_only' else 'tab:gray' for s in d.scope])
            ax.hlines(x,d.AP_lo,d.AP_hi,color='black');ax.scatter(d.AP_lo,x,marker='|',c='black');ax.scatter(d.AP_hi,x,marker='|',c='black')
            ax.set_yticks(x,[f'{r.model} | {r.scope} | {r.sampling}' for r in d.itertuples()],fontsize=8)
            ax.invert_yaxis();ax.axvline(d.prevalence.iloc[0],color='gray',ls='--');ax.set_xlabel('Average precision (95% CI)')
            ax.set_title(f'{cohort} | set {landmark}');ax.grid(axis='x',alpha=.2)
    fig.suptitle(population);fig.tight_layout();fig.savefig(out/'scope_comparison_AP.png',dpi=180);plt.show()
    if not logs.empty:
        d=logs[logs.sampling.eq('none')&logs.fold.ge(0)]
        for model,gmodel in d.groupby('model'):
            fig,axes=plt.subplots(1,2,figsize=(12,4))
            for (landmark,fold,scope),g in gmodel.groupby(['landmark','fold','scope']):
                label=f'set {landmark}, fold {fold+1}, {scope}'
                axes[0].plot(g.epoch,g.loss,label=label);axes[1].plot(g.epoch,g.inner_AP,label=label)
            axes[0].set_title(f'{model}: inner-training loss');axes[1].set_title('Inner-validation AP')
            axes[0].set_ylabel('Weighted loss');axes[1].set_ylabel('Average precision')
            for ax in axes:ax.set_xlabel('Epoch');ax.grid(alpha=.2)
            axes[1].legend(fontsize=6);fig.suptitle(population);fig.tight_layout();fig.savefig(out/f'{model}_learning_curves.png',dpi=180);plt.show()


def fit_grid_partition(train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,checkpoint_dir=None):
    """All selection/scaling nested. Same inner patient holdout for all model families."""
    seq=panel_sequence(seq,cfg);target_seq=panel_sequence(target_seq,cfg)
    ti,vi=inner_split(train,seed,cfg['inner_holdout']);y=train.label.to_numpy(int)
    inner=GridTransform().fit(train.iloc[ti],combo,cfg,seed)
    xt=inner.transform(train.iloc[ti],'MIMIC');xv=inner.transform(train.iloc[vi],'MIMIC')
    outer=GridTransform().fit(train,combo,cfg,seed)
    xf=outer.transform(train,'MIMIC');xe=outer.transform(target,target_cohort)
    ix=sample_indices(y[ti],setting,seed);fx=sample_indices(y,setting,seed)
    preds={};cal={};logs=[]
    fit_records=[]
    for name in cfg['base_models']:
        cal[name],inner_record=_fit_model_checkpoint(name,xt[ix],y[ti][ix],xv,setting,cfg,seed,'inner',checkpoint_dir)
        preds[name],outer_record=_fit_model_checkpoint(name,xf[fx],y[fx],xe,setting,cfg,seed,'outer',checkpoint_dir)
        fit_records.extend([inner_record,outer_record])
    if cfg['include_ensembles']:
        if not {'RandomForest','Lasso'}<=set(preds):raise ValueError('RF and Lasso are required for ensembles')
        ip=np.column_stack([cal['RandomForest'],cal['Lasso']]);ep=np.column_stack([preds['RandomForest'],preds['Lasso']])
        meta=LogisticRegression(max_iter=1000).fit(ip,y[vi])
        cal[ENSEMBLES[0]]=ip.mean(1);preds[ENSEMBLES[0]]=ep.mean(1)
        cal[ENSEMBLES[1]]=meta.predict_proba(ip)[:,1];preds[ENSEMBLES[1]]=meta.predict_proba(ep)[:,1]
    for variant in cnn_variants(combo,cfg):
        ip,ep,history,records=_lab_cnn_checkpoint(
            train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,ti,vi,ix,fx,checkpoint_dir,variant)
        cal[variant]=ip;preds[variant]=ep;logs.extend(history);fit_records.extend(records)
    thresholds={name:youden_threshold(y[vi],p) for name,p in cal.items()}
    operating={model:{operating_key(t):operating_threshold(y[vi],p,t) for t in cfg['recall_targets']} for model,p in cal.items()}
    audit=dict(operating_thresholds_json=json.dumps(operating),train_admissions=len(train),train_patients=train.patient_id.nunique(),
        inner_validation_patients=train.iloc[vi].patient_id.nunique(),raw_predictors=outer.raw(train,'MIMIC').shape[1],
        natural_prevalence=float(y.mean()),fit_prevalence=float(y[fx].mean()),fit_rows=len(fx),
        duplicated_rows=len(fx)-len(np.unique(fx)),external_prevalence_used=setting['external_prevalence_used'],
        selected_ICD_features='|'.join(outer.icd_names),threshold_source='MIMIC inner patient holdout',model_timing_json=json.dumps(fit_records))
    return preds,thresholds,logs,audit


## Active design: focused V15 schedule

The V15 design reference at the top specifies the three main models, four ensemble anchors, optional L1 comparisons and full per-row metrics. The following V13/V14 compatibility definitions support accepted-checkpoint reuse; the V15 runtime cell installs the active schedule.


In [ ]:
"""Representative contrasts, fixed sensitivity anchors and exact-objective runtime work."""
from collections import OrderedDict
if '_V12_RUNTIME_BINDINGS' not in globals():
    _V12_RUNTIME_BINDINGS=dict(globals())
V13_KEYS=('sensitivity_profile','sensitivity_models','transform_cache_entries')
V13_TRANSFORMS=OrderedDict()

def primary_group_combinations():
    groups=[tuple(ICD_BLOCKS),('lab_current',),('lab_history',)]
    return [tuple(b for j in indices for b in groups[j])
        for size in (1,2,3) for indices in combinations(range(3),size)]

def grid_combinations(cfg):
    if cfg.get('sensitivity_profile')=='anchors_active':
        return [('lab_current',),('lab_current','lab_history'),tuple(ALL_GRID_BLOCKS)]
    if cfg['grid_profile']!='representative':return _V12_RUNTIME_BINDINGS['grid_combinations'](cfg)
    combos=primary_group_combinations()+[(b,) for b in ICD_BLOCKS]
    combos += [tuple(b for b in ALL_GRID_BLOCKS if b!=removed) for removed in ICD_BLOCKS]
    if cfg.get('include_observation_controls',True):combos += [('demographics',),('observation_current',),('observation_history',),OBS_BLOCKS]
    return list(dict.fromkeys(combos))

def sampling_for_combo(population,combo,cfg):
    if cfg.get('sensitivity_profile')=='anchors_active':
        wanted=[('lab_current',),('lab_current','lab_history')] if population=='all_labs' else [tuple(ALL_GRID_BLOCKS)]
        return [('none',None)] if tuple(combo) in wanted else []
    diagnostic=bool(set(combo)&set(OBS_BLOCKS)) or 'demographics' in combo
    ablation=cfg['grid_profile']=='representative' and tuple(combo) not in primary_group_combinations()
    return [('none',None)] if diagnostic or ablation else sampling_plan(cfg)

def sampling_plan(cfg):
    setting=dict(cfg)
    if setting['grid_profile']=='representative':setting['grid_profile']='compact'
    return _V12_RUNTIME_BINDINGS['sampling_plan'](setting)

def grid_plan(cfg):
    ordinary=len(cfg['base_models'])+len(ENSEMBLES)*cfg['include_ensembles'];cells=rows=cnn=0
    for pop in POPULATIONS:
        for combo in population_combinations(pop,cfg):
            n=2*len(sampling_for_combo(pop,combo,cfg));cells+=n
            rows+=n*(ordinary+len(cnn_variants(combo,cfg)));cnn+=n*len(cnn_variants(combo,cfg))
    return dict(profile=cfg['grid_profile'],age_policy=cfg['age_policy'],grid_cells=cells,expected_wide_result_rows=rows,
        base_estimator_fits=cells*(cfg['outer_folds']+1)*2*len(cfg['base_models']),
        cnn_training_runs=cnn*(cfg['outer_folds']+1)*2,bootstrap_draws_per_row=cfg['n_boot'],
        note='Five-fold evaluation retained; counts assume both populations/landmarks pass preflight. Sensitivities use fixed anchors.')

def validate_config(cfg):
    test=dict(cfg)
    if test['grid_profile']=='representative':test['grid_profile']='compact'
    _V12_RUNTIME_BINDINGS['validate_config'](test)
    if cfg.get('sensitivity_profile') not in ('anchors','anchors_active','full'):raise ValueError('Unknown sensitivity profile')
    if cfg['transform_cache_entries']<1:raise ValueError('Cache entries must be positive')

def cached_grid_transform(train,combo,cfg,seed):
    # Cache fitted preprocessing only, never dense patient matrices. Ordered IDs,
    # labels and settings bind the exact fold. Cache is cleared for each scenario.
    key=(tuple(combo),seed,frame_digest(train[['patient_id','hospital_id','label','landmark','score_hour']]),
         cfg['lab_panel'],cfg['include_demographics'],
         cfg['icd_min_patients'],cfg['icd_temporal_budget'],cfg['icd_ever_budget'],cfg['selection_trees'])
    if key in V13_TRANSFORMS:
        value=V13_TRANSFORMS.pop(key);V13_TRANSFORMS[key]=value;return value
    value=GridTransform().fit(train,combo,cfg,seed);V13_TRANSFORMS[key]=value
    while len(V13_TRANSFORMS)>cfg['transform_cache_entries']:V13_TRANSFORMS.popitem(last=False)
    return value

def fit_sampled_checkpoint(name,x,y,indices,target,setting,cfg,seed,stage,folder):
    # Only additive logistic losses have this exact row-multiplicity shortcut.
    # Trees/bootstrap and CNN minibatches retain the literal resampled indices.
    if name not in ('Lasso','SummaryLR'):
        return _V12_RUNTIME_BINDINGS['_fit_model_checkpoint'](name,x[indices],y[indices],target,setting,cfg,seed,stage,folder)
    path=None if folder is None else Path(folder)/f'{name}_{stage}.pkl'
    if path is not None and path.is_file():
        saved=_v9_read_pickle(path)
        if saved['rows']!=len(target) or not np.isfinite(saved['probs']).all():raise ValueError('Invalid model checkpoint')
        gap=saved['audit'].get('lasso_relative_dual_gap')
        if name=='Lasso' and (gap is None or not np.isfinite(gap) or gap>cfg['lasso_objective_gap']):raise ValueError('Uncertified Lasso')
        print(f'[model] {name} {stage}: resumed',flush=True);return saved['probs'],saved['audit']
    unique,multiplicity=np.unique(indices,return_counts=True)
    estimator=grid_estimator(name,y[indices],setting,cfg,seed)
    if name=='Lasso' and path is not None:estimator.checkpoint_path=path.with_name(path.stem+'.iterate.pkl')
    started=time.perf_counter()
    print(f'[model] {name} {stage}: {len(unique):,} unique rows, {len(indices):,} effective rows',flush=True)
    estimator.fit(x[unique],y[unique],sample_weight=multiplicity.astype(float))
    probs=estimator.predict_proba(target)[:,1]
    if not np.isfinite(probs).all():raise ValueError('Nonfinite predictions')
    audit=dict(model=name,stage=stage,seconds=time.perf_counter()-started,
        unique_rows=len(unique),effective_rows=len(indices),iterations=estimator.n_iter_.tolist(),
        lasso_relative_dual_gap=getattr(estimator,'relative_dual_gap_',None),solver_revision='v13-exact-multiplicity')
    if path is not None:_v9_pickle(path,dict(rows=len(target),probs=probs,audit=audit))
    print(f'[model] {name} {stage}: saved after {audit["seconds"]:.1f}s',flush=True)
    return probs,audit

def fitting_fingerprint(m,ms,e,es,cfg):
    base=_V12_RUNTIME_BINDINGS['fitting_fingerprint'](m,ms,e,es,cfg)
    funcs=[fit_sampled_checkpoint,cached_grid_transform,grid_combinations,
           primary_group_combinations,sampling_for_combo,sampling_plan,grid_estimator,_run_v13_grid_core]
    return hashlib.sha256(base.encode()+b''.join(stable_code_bytes(f.__code__) for f in funcs)+b'v13-runtime-1').hexdigest()[:24]

def _v12_source_fingerprint(m,ms,e,es,cfg):
    # Restore the exact earlier definitions only while hashing. No models execute.
    names=[n for n,v in _V12_RUNTIME_BINDINGS.items() if callable(v) and n in globals() and globals()[n] is not v]
    current={n:globals()[n] for n in names}
    oldcfg={k:v for k,v in cfg.items() if k not in V13_KEYS}
    oldcfg.update(grid_profile='compact',base_models=('RandomForest','Lasso','SummaryLR','XGBoost','DecisionTree'),
                  include_ensembles=True,include_cnn=True,include_observation_controls=True)
    try:
        globals().update({n:_V12_RUNTIME_BINDINGS[n] for n in names})
        return _V12_RUNTIME_BINDINGS['fitting_fingerprint'](m,ms,e,es,oldcfg)
    finally:globals().update(current)

def import_v12_models(m,ms,e,es,cfg,out):
    oldkey=_v12_source_fingerprint(m,ms,e,es,cfg)
    source=DATA_ROOT/'BE_integrated_grid_v12_outputs'/Path(out).name/'grid_checkpoints'/oldkey
    target=Path(out)/'grid_checkpoints'/fitting_fingerprint(m,ms,e,es,cfg);count=0
    if not source.is_dir():return 0
    allowed={f'{n}_{s}.pkl' for n in cfg['base_models'] for s in ('inner','outer')}
    for file in source.glob('*.models/**/*.pkl'):
        if file.name not in allowed and not (file.name in ('CNN_inner.pkl','CNN_outer.pkl') and file.parent.name in ('LabCNN_Sequence','LabCNN_Summary')):continue
        dest=target/file.relative_to(source)
        if dest.exists():continue
        saved=_v9_read_pickle(file)
        if not np.isfinite(saved['probs']).all():raise ValueError('Invalid v12 model cache')
        gap=saved['audit'].get('lasso_relative_dual_gap')
        if file.name.startswith('Lasso_') and (gap is None or not np.isfinite(gap) or gap>cfg['lasso_objective_gap']):raise ValueError('Uncertified v12 Lasso')
        _v9_pickle(dest,saved);count+=1
    print(f'[resume] Imported {count} exact-source-key v12 model predictions; source untouched.',flush=True)
    return count

def run_unified_grid(m,ms,e,es,cfg,out):
    validate_config(cfg);V13_TRANSFORMS.clear()
    import_v12_models(m,ms,e,es,cfg,out)
    return _run_v13_grid_core(m,ms,e,es,cfg,out)

def scenario_plan(cfg):
    rows=_V12_RUNTIME_BINDINGS['scenario_plan'](cfg);out=[]
    for name,setting in rows:
        if name!='primary' and cfg['sensitivity_profile']=='anchors':
            setting=dict(setting,sensitivity_profile='anchors_active',base_models=tuple(cfg['sensitivity_models']))
            if setting['include_ensembles'] and not {'RandomForest','Lasso'}<=set(setting['base_models']):raise ValueError('Sensitivity ensembles require RF/Lasso')
        out.append((name,setting))
    return out

def study_runtime_plan(cfg):
    plans=[dict(scenario=name,**grid_plan(setting)) for name,setting in scenario_plan(cfg)]
    for p in plans:print(p)
    print('TOTAL fits:',sum(p['base_estimator_fits'] for p in plans),'base;',sum(p['cnn_training_runs'] for p in plans),'CNN.')
    print('Counts are not a 24-hour guarantee. Check pilot timing before launching the complete study.')
    return plans

def compare_fixed_sensitivity_anchors(primary_wide,primary_checkpoint,other_checkpoint,cfg,out):
    if cfg['sensitivity_profile']!='anchors_active':
        return compare_scenario_overlap(primary_wide,primary_checkpoint,other_checkpoint,cfg,out)
    frames=[]
    for population,scopes in [('all_labs',['lab_current','lab_current+lab_history']),
                              ('icd_history',['+'.join(ALL_GRID_BLOCKS)])]:
        for scope in scopes:
            ref=primary_wide[primary_wide.population.eq(population)&primary_wide.Feature_combo.eq(scope)&primary_wide.Sampling.eq('none')]
            allowed=set(cfg['base_models'])|set(ENSEMBLES)|{'LabCNN_Sequence','LabCNN_Summary'}
            ref=ref[ref.Model.isin(allowed)]
            if not ref.empty:frames.append(compare_scenario_overlap(ref,primary_checkpoint,other_checkpoint,cfg,out))
    result=pd.concat(frames,ignore_index=True) if frames else pd.DataFrame()
    result.to_csv(Path(out)/'paired_fixed_anchor_comparison.csv',index=False)
    return result


def fit_grid_partition(train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,checkpoint_dir=None):
    """All selection/scaling nested. Same inner patient holdout for all model families."""
    seq=panel_sequence(seq,cfg);target_seq=panel_sequence(target_seq,cfg)
    ti,vi=inner_split(train,seed,cfg['inner_holdout']);y=train.label.to_numpy(int)
    inner=cached_grid_transform(train.iloc[ti],combo,cfg,seed)
    xt=inner.transform(train.iloc[ti],'MIMIC');xv=inner.transform(train.iloc[vi],'MIMIC')
    outer=cached_grid_transform(train,combo,cfg,seed)
    xf=outer.transform(train,'MIMIC');xe=outer.transform(target,target_cohort)
    ix=sample_indices(y[ti],setting,seed);fx=sample_indices(y,setting,seed)
    preds={};cal={};logs=[]
    fit_records=[]
    for name in cfg['base_models']:
        cal[name],inner_record=fit_sampled_checkpoint(name,xt,y[ti],ix,xv,setting,cfg,seed,'inner',checkpoint_dir)
        preds[name],outer_record=fit_sampled_checkpoint(name,xf,y,fx,xe,setting,cfg,seed,'outer',checkpoint_dir)
        fit_records.extend([inner_record,outer_record])
    if cfg['include_ensembles']:
        if not {'RandomForest','Lasso'}<=set(preds):raise ValueError('RF and Lasso are required for ensembles')
        ip=np.column_stack([cal['RandomForest'],cal['Lasso']]);ep=np.column_stack([preds['RandomForest'],preds['Lasso']])
        meta=LogisticRegression(max_iter=1000).fit(ip,y[vi])
        cal[ENSEMBLES[0]]=ip.mean(1);preds[ENSEMBLES[0]]=ep.mean(1)
        cal[ENSEMBLES[1]]=meta.predict_proba(ip)[:,1];preds[ENSEMBLES[1]]=meta.predict_proba(ep)[:,1]
    for variant in cnn_variants(combo,cfg):
        ip,ep,history,records=_lab_cnn_checkpoint(
            train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,ti,vi,ix,fx,checkpoint_dir,variant)
        cal[variant]=ip;preds[variant]=ep;logs.extend(history);fit_records.extend(records)
    thresholds={name:youden_threshold(y[vi],p) for name,p in cal.items()}
    operating={model:{operating_key(t):operating_threshold(y[vi],p,t) for t in cfg['recall_targets']} for model,p in cal.items()}
    audit=dict(operating_thresholds_json=json.dumps(operating),train_admissions=len(train),train_patients=train.patient_id.nunique(),
        inner_validation_patients=train.iloc[vi].patient_id.nunique(),raw_predictors=outer.raw(train,'MIMIC').shape[1],
        natural_prevalence=float(y.mean()),fit_prevalence=float(y[fx].mean()),fit_rows=len(fx),
        duplicated_rows=len(fx)-len(np.unique(fx)),external_prevalence_used=setting['external_prevalence_used'],
        selected_ICD_features='|'.join(outer.icd_names),threshold_source='MIMIC inner patient holdout',model_timing_json=json.dumps(fit_records))
    return preds,thresholds,logs,audit

def _run_v13_grid_core(m,ms,e,es,cfg,out):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    ICD_SELECTION_CACHE.clear()
    population_audit(m,e,cfg).to_csv(out/'population_eligibility.csv',index=False)
    checkpoint=out/'grid_checkpoints'/fitting_fingerprint(m,ms,e,es,cfg)
    checkpoint.mkdir(parents=True,exist_ok=True)
    tasks=[];status=[]
    # Validate all patient splits before fitting; a sparse subgroup never falls
    # back to the broader population or a globally fitted feature selection.
    for population in POPULATIONS:
        mi=population_indices(m,'MIMIC',population,cfg);ei=population_indices(e,'eICU',population,cfg)
        pm=m.iloc[mi].reset_index(drop=True);pe=e.iloc[ei].reset_index(drop=True)
        ps=take_sequence(ms,mi);pes=take_sequence(es,ei)
        try:fold_map=patient_folds(pm,cfg['outer_folds'],cfg['seed'])
        except ValueError as err:
            status.append(dict(population=population,landmark='both',status='skipped',reason=str(err)));continue
        for landmark in (1,2):
            mi=np.flatnonzero(pm.landmark.eq(landmark));ei=np.flatnonzero(pe.landmark.eq(landmark))
            mm=pm.iloc[mi].reset_index(drop=True);ee=pe.iloc[ei].reset_index(drop=True)
            ss=take_sequence(ps,mi);se=take_sequence(pes,ei);folds=mm.patient_id.map(fold_map).to_numpy()
            try:
                if mm.label.nunique()!=2 or ee.label.nunique()!=2:raise ValueError('Both cohorts need two outcome classes at this landmark')
                inner_split(mm,cfg['seed']+1000,cfg['inner_holdout'])
                for fold in range(cfg['outer_folds']):
                    tr=np.flatnonzero(folds!=fold);te=np.flatnonzero(folds==fold)
                    if not len(te) or mm.iloc[te].label.nunique()!=2:raise ValueError('Outer test fold empty or single-class; revise fold count explicitly')
                    inner_split(mm.iloc[tr],cfg['seed']+fold,cfg['inner_holdout'])
            except ValueError as err:
                status.append(dict(population=population,landmark=landmark,status='skipped',reason=str(err)));continue
            status.append(dict(population=population,landmark=landmark,status='ready',reason='',
                MIMIC_admissions=len(mm),MIMIC_cases=int(mm.label.sum()),eICU_admissions=len(ee),eICU_cases=int(ee.label.sum())))
            for combo in population_combinations(population,cfg):
                for mode,value in sampling_for_combo(population,combo,cfg):tasks.append((population,landmark,mm,ee,ss,se,folds,combo,mode,value))
    pd.DataFrame(status).to_csv(out/'population_run_status.csv',index=False)
    for s in status:print('[population]',s,flush=True)
    if not tasks:raise ValueError('No population/landmark passed preflight; inspect population_run_status.csv')
    completed=[];audits=[];curves=[]
    for number,(population,landmark,mm,ee,ss,se,folds,combo,mode,value) in enumerate(tasks,1):
        label='+'.join(combo);sl=mode if value is None else f'{mode} {value}'
        token=prediction_token(population,landmark,label,sl);tags=dict(population=population,landmark=landmark,scope=label,sampling=sl)
        pfile=checkpoint/f'{token}.csv.gz';afile=checkpoint/f'{token}.audit.csv';lfile=checkpoint/f'{token}.learning.csv'
        ready=checkpoint/f'{token}.complete'
        print(f'[fits] {number}/{len(tasks)} {population} set {landmark} {label} {sl}',flush=True)
        if not (ready.exists() and pfile.exists() and afile.exists()):
            cp=[];ca=[];cl=[]
            for fold in list(range(cfg['outer_folds']))+[-1]:
                part=checkpoint/f'{token}.fold{fold}.pkl';mark=checkpoint/f'{token}.fold{fold}.complete'
                # Only locally generated checkpoints in this exact fingerprint namespace.
                if part.exists() and mark.exists():q,a,ll=pd.read_pickle(part)
                else:
                    tr=np.arange(len(mm)) if fold==-1 else np.flatnonzero(folds!=fold)
                    te=np.flatnonzero(folds==fold) if fold!=-1 else np.arange(len(ee))
                    train=mm.iloc[tr];target=ee if fold==-1 else mm.iloc[te]
                    t_seq=se if fold==-1 else take_sequence(ss,te)
                    if fold!=-1:assert not set(train.patient_id)&set(target.patient_id)
                    setting=resolved_sampling(mode,value,train.label,ee.label)
                    pp,tt,ll,aa=fit_grid_partition(train,take_sequence(ss,tr),target,t_seq,
                        'eICU' if fold==-1 else 'MIMIC',combo,setting,cfg,cfg['seed']+(1000 if fold==-1 else fold),
                        checkpoint_dir=checkpoint/f'{token}.fold{fold}.models')
                    q=pd.concat([target[['patient_id','hospital_id','label']].copy().assign(**tags,
                        cohort='eICU_external' if fold==-1 else 'MIMIC_OOF',model=model,fold=fold,
                        probability=prob,threshold=tt[model]) for model,prob in pp.items()],ignore_index=True)
                    op=json.loads(aa['operating_thresholds_json'])
                    for target_recall in cfg['recall_targets']:
                        key=operating_key(target_recall)
                        q[key]=q.model.map({model:values[key] for model,values in op.items()})
                    if not np.isfinite(q.probability).all():raise ValueError('Nonfinite predictions')
                    a=dict(**tags,fold=fold,**aa);ll=[dict(**tags,fold=fold,**l) for l in ll]
                    pd.to_pickle((q,a,ll),part);mark.write_text('complete')
                cp.append(q);ca.append(a);cl.extend(ll)
            p=pd.concat(cp,ignore_index=True);a=pd.DataFrame(ca);ll=pd.DataFrame(cl)
            assert not p.duplicated(['cohort','hospital_id','model']).any()
            p.to_csv(pfile,index=False);a.to_csv(afile,index=False)
            if not ll.empty:ll.to_csv(lfile,index=False)
            ready.write_text('complete')
        audits.append(pd.read_csv(afile))
        if lfile.exists():curves.append(pd.read_csv(lfile))
        completed.append(dict(**tags,token=token,external_prevalence_used=mode=='prevalence_match' and value in ('eicu','midpoint')))
    # A separate phase: changing n_boot never retrains models. Every row is covered.
    reports=checkpoint/'reports'/report_fingerprint(cfg);reports.mkdir(parents=True,exist_ok=True)
    rows=[]
    for number,task in enumerate(completed,1):
        token=task['token'];file=reports/f'{token}.metrics.csv';mark=reports/f'{token}.complete'
        print(f'[all-row CIs] {number}/{len(completed)}',flush=True)
        if file.exists() and mark.exists():met=pd.read_csv(file,float_precision='round_trip')
        else:
            met=summarize_predictions(read_predictions(checkpoint/f'{token}.csv.gz'),cfg)
            met['external_prevalence_used']=task['external_prevalence_used']
            met.to_csv(file,index=False);mark.write_text('complete')
        rows.append(met)
    metrics=pd.concat(rows,ignore_index=True);audit=pd.concat(audits,ignore_index=True)
    learning=pd.concat(curves,ignore_index=True) if curves else pd.DataFrame()
    wide=wide_grid_results(metrics);wide.to_csv(out/'feature_sampling_grid.csv',index=False)
    metrics.to_csv(out/'unified_grid_metrics_long.csv',index=False);audit.to_csv(out/'unified_grid_fit_audit.csv',index=False)
    learning.to_csv(out/'unified_grid_learning_curves.csv',index=False)
    paired=[];tasks_frame=pd.DataFrame(completed)
    for _,g in tasks_frame.groupby(['population','scope','sampling'],sort=False):
        if set(g.landmark)!={1,2}:continue
        file=reports/f'{g.iloc[0].token}.paired.pkl';mark=file.with_suffix('.complete')
        if file.exists() and mark.exists():q=pd.read_pickle(file)
        else:
            p=pd.concat([read_predictions(checkpoint/f'{t}.csv.gz') for t in g.token],ignore_index=True)
            q=paired_landmarks(p,cfg);q.to_pickle(file);mark.write_text('complete')
        paired.append(q)
    paired=pd.concat(paired,ignore_index=True) if paired else pd.DataFrame()
    paired.to_csv(out/'paired_landmark_grid.csv',index=False)
    pd.DataFrame(completed).to_csv(checkpoint/'prediction_index.csv',index=False)
    return wide,metrics,paired,audit,learning,checkpoint

def run_v10_sensitivities(root,study_root,cfg,primary_wide,primary_checkpoint):
    """Load, rebuild/refit and release one scenario at a time. Never mutate CFG."""
    tables=[]
    for name,setting in scenario_plan(cfg)[1:]:
        out=Path(study_root)/name
        print(f'[sensitivity] {name}: fresh policy-specific feature/fit fingerprints',flush=True)
        m,ms,e,es,audit=prepare_v10_scenario(root,out,setting)
        result=run_unified_grid(m,ms,e,es,setting,out)
        wide,metrics,paired,fit_audit,learning,checkpoint=result
        finalist_diagnostics(wide,checkpoint,setting,out)
        compare_fixed_sensitivity_anchors(primary_wide,primary_checkpoint,checkpoint,setting,out)
        manifest,fingerprint=input_manifest(root,setting)
        (out/'run_manifest.json').write_text(json.dumps(dict(scenario=name,inputs=manifest,feature_fingerprint=fingerprint,
            patient_fold_digest=V10_FOLD_DIGEST,checkpoint_directory=str(checkpoint)),indent=2,default=str))
        tables.append(wide.assign(Scenario=name))
        del m,ms,e,es,audit,result,metrics,paired,fit_audit,learning
        ICD_RAW.clear();ICD_SELECTION_CACHE.clear();gc.collect()
    return pd.concat(tables,ignore_index=True) if tables else pd.DataFrame()


In [ ]:
"""V14 scheduling only: tabular grid plus prespecified, lab-only CNN comparisons.

CNN training remains unchanged; SummaryLR acceptance is checked below. Checkpoint
compatibility uses exact fingerprints, never output-folder names alone.
"""
import shutil


def v14_cnn_tasks():
    # Primary training uses natural prevalence without class weighting. This is
    # prespecified, not selected using MIMIC/eICU performance. Age/sex retained.
    return [(landmark, combo, variant, mode, value)
            for landmark in (1, 2)
            for combo, variant, mode, value in (
                (('lab_current',), 'LabCNN_Sequence', 'unweighted', None),
                (('lab_current', 'lab_history'), 'LabCNN_Summary', 'unweighted', None),
                (('lab_current', 'lab_history'), 'LabCNN_Summary', 'prevalence_match', 'eicu'))]


def v14_study_plan(cfg):
    plans = [dict(scenario=name, **grid_plan(dict(setting, include_cnn=False)))
             for name, setting in scenario_plan(cfg)]
    runs = len(v14_cnn_tasks()) * (cfg['outer_folds'] + 1) * 2 if RUN_CNN else 0
    print(f"[v14 plan] {sum(p['grid_cells'] for p in plans)} tabular configurations; "
          f"{sum(p['base_estimator_fits'] for p in plans)} base fits; "
          f"{runs} separate CNN fits (including inner selection and outer refit).")
    print('CNN: all_labs only, two landmarks, two unweighted comparisons; '
          'prevalence matching only for sequence + current/historical summaries. '
          'No CNN fits in tabular sensitivities. No 24-hour runtime guarantee.')
    return plans


def v14_source_checkpoint(m, ms, e, es, cfg, scenario):
    # Do not override any function used by the V13 fingerprint. Paths, the CNN
    # schedule and report switches live outside CFG and cannot alter old keys.
    source_cfg = dict(cfg, include_cnn=True)
    source_key = fitting_fingerprint(m, ms, e, es, source_cfg)
    return (DATA_ROOT / 'BE_integrated_grid_v13_outputs' / scenario /
            'grid_checkpoints' / source_key), source_key


def v14_copy_prediction(source, destination, *, model=None, rows=None, cfg=None):
    source, destination = Path(source), Path(destination)
    if destination.exists() or not source.is_file():
        return False
    saved = _v9_read_pickle(source)
    probs = np.asarray(saved['probs'])
    if probs.ndim != 1 or len(probs) != saved['rows'] or not np.isfinite(probs).all():
        raise ValueError(f'Invalid prediction checkpoint: {source.name}')
    if rows is not None and saved['rows'] != rows:
        raise ValueError('Checkpoint target row count mismatch')
    if model is not None and saved['audit'].get('model') != model:
        raise ValueError('CNN variant checkpoint mismatch')
    if source.name.startswith('Lasso_'):
        gap = saved['audit'].get('lasso_relative_dual_gap')
        if gap is None or not np.isfinite(gap) or gap > cfg['lasso_objective_gap']:
            raise ValueError('Cannot reuse uncertified Lasso fit')
    # Atomic destination write; never rename, rewrite or remove the old file.
    _v9_pickle(destination, saved)
    return True


def run_v14_tabular_grid(m, ms, e, es, cfg, out):
    setting = dict(cfg, include_cnn=False)
    validate_config(setting)
    V13_TRANSFORMS.clear()
    source, _ = v14_source_checkpoint(m, ms, e, es, setting, Path(out).name)
    target = Path(out) / 'grid_checkpoints' / fitting_fingerprint(m, ms, e, es, setting)
    copied = 0
    if source.is_dir():
        allowed = {f'{name}_{stage}.pkl' for name in setting['base_models']
                   for stage in ('inner', 'outer')}
        for file in source.glob('*.models/*.pkl'):
            if file.name in allowed:
                copied += v14_copy_prediction(file, target / file.relative_to(source), cfg=setting)
    print(f'[v14 resume] Reused {copied} exact-fingerprint tabular model checkpoints.')
    # Never import V13 whole-fold predictions: they also contain CNN rows.
    return _run_v13_grid_core(m, ms, e, es, setting, out)


def run_v14_cnn_comparisons(m, ms, e, es, cfg, out):
    setting_cfg = dict(cfg, include_cnn=True)
    validate_config(setting_cfg)
    source, source_key = v14_source_checkpoint(m, ms, e, es, setting_cfg, 'primary')
    recipe = dict(source_key=source_key, tasks=v14_cnn_tasks(), population='all_labs',
                  revision='separate-cnn-v14-1')
    digest = hashlib.sha256(json.dumps(recipe, sort_keys=True).encode() +
        b''.join(stable_code_bytes(f.__code__) for f in
                 (run_v14_cnn_comparisons, v14_cnn_tasks, v14_copy_prediction))).hexdigest()[:24]
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    checkpoint = out / 'cnn_checkpoints' / digest
    checkpoint.mkdir(parents=True, exist_ok=True)
    mi = population_indices(m, 'MIMIC', 'all_labs', setting_cfg)
    ei = population_indices(e, 'eICU', 'all_labs', setting_cfg)
    # Preflight only metadata. Copy the large feature/tensor bank for one
    # landmark at a time, not once for the full population and again per set.
    metadata = ['patient_id', 'hospital_id', 'label', 'landmark']
    pm = m[metadata].iloc[mi].reset_index(drop=True)
    pe = e[metadata].iloc[ei].reset_index(drop=True)
    fold_map = patient_folds(pm, cfg['outer_folds'], cfg['seed'])
    prepared = {}
    # Preflight all landmarks/folds before starting any CNN fit.
    for landmark in (1, 2):
        ii, jj = np.flatnonzero(pm.landmark.eq(landmark)), np.flatnonzero(pe.landmark.eq(landmark))
        mm, ee = pm.iloc[ii].reset_index(drop=True), pe.iloc[jj].reset_index(drop=True)
        folds = mm.patient_id.map(fold_map).to_numpy()
        if mm.label.nunique() != 2 or ee.label.nunique() != 2:
            raise ValueError(f'CNN set {landmark}: both cohorts require two outcome classes')
        inner_split(mm, cfg['seed'] + 1000, cfg['inner_holdout'])
        for fold in range(cfg['outer_folds']):
            train, test = mm.loc[folds != fold], mm.loc[folds == fold]
            if test.empty or test.label.nunique() != 2:
                raise ValueError(f'CNN set {landmark}, fold {fold}: invalid test cohort')
            if set(train.patient_id) & set(test.patient_id):
                raise ValueError('Patient overlap in CNN train/test partition')
            inner_split(train, cfg['seed'] + fold, cfg['inner_holdout'])
        prepared[landmark] = mi[ii], ei[jj], folds
    del pm, pe, mm, ee
    predictions, audit_rows, learning, metric_parts, index_rows = [], [], [], [], []
    reports = checkpoint / 'reports' / report_fingerprint(cfg)
    reports.mkdir(parents=True, exist_ok=True)
    tasks = v14_cnn_tasks()
    loaded_landmark = None
    for number, (landmark, combo, variant, mode, value) in enumerate(tasks, 1):
        if loaded_landmark != landmark:
            if loaded_landmark is not None:
                del mm, ee, ss, se, train, target, target_seq
                gc.collect()
            ii, jj, folds = prepared[landmark]
            mm, ee = m.iloc[ii].reset_index(drop=True), e.iloc[jj].reset_index(drop=True)
            ss = panel_sequence(take_sequence(ms, ii), cfg)
            se = panel_sequence(take_sequence(es, jj), cfg)
            loaded_landmark = landmark
        scope = '+'.join(combo); sampling = mode if value is None else f'{mode} {value}'
        token = prediction_token('all_labs', landmark, scope, sampling)
        tags = dict(population='all_labs', landmark=landmark, scope=scope, sampling=sampling)
        adapted = mode == 'prevalence_match'
        print(f'[CNN comparison] {number}/{len(tasks)} set {landmark}: {variant}, {sampling}', flush=True)
        chunks = []
        for fold in list(range(cfg['outer_folds'])) + [-1]:
            seed = cfg['seed'] + (1000 if fold == -1 else fold)
            tr = np.arange(len(mm)) if fold == -1 else np.flatnonzero(folds != fold)
            te = np.arange(len(ee)) if fold == -1 else np.flatnonzero(folds == fold)
            train, target = mm.iloc[tr], ee if fold == -1 else mm.iloc[te]
            target_seq = se if fold == -1 else take_sequence(ss, te)
            ti, vi = inner_split(train, seed, cfg['inner_holdout'])
            y = train.label.to_numpy(int)
            sampling_setting = resolved_sampling(mode, value, train.label, ee.label)
            ix = sample_indices(y[ti], sampling_setting, seed)
            fx = sample_indices(y, sampling_setting, seed)
            folder = checkpoint / f'{token}.fold{fold}.models'
            for stage, rows in (('inner', len(vi)), ('outer', len(target))):
                relative = Path(f'{token}.fold{fold}.models') / variant / f'CNN_{stage}.pkl'
                if v14_copy_prediction(source / relative, checkpoint / relative, model=variant, rows=rows):
                    print(f'[CNN resume] reused V13 {variant} fold {fold} {stage}', flush=True)
            ip, ep, logs, records = _lab_cnn_checkpoint(
                train, take_sequence(ss, tr), target, target_seq,
                'eICU' if fold == -1 else 'MIMIC', combo, sampling_setting, setting_cfg,
                seed, ti, vi, ix, fx, folder, variant)
            if len(ep) != len(target) or not np.isfinite(ep).all():
                raise ValueError('Invalid CNN prediction vector')
            q = target[['patient_id', 'hospital_id', 'label']].copy().assign(
                **tags, cohort='eICU_external' if fold == -1 else 'MIMIC_OOF',
                model=variant, fold=fold, probability=ep, threshold=youden_threshold(y[vi], ip))
            for recall in cfg['recall_targets']:
                q[operating_key(recall)] = operating_threshold(y[vi], ip, recall)
            chunks.append(q)
            for record in records:
                stage_y = y[ti] if record['stage'] == 'inner' else y
                stage_ix = ix if record['stage'] == 'inner' else fx
                audit_rows.append(dict(**tags, fold=fold, **record,
                    train_admissions=len(stage_y), natural_prevalence=float(stage_y.mean()),
                    fit_prevalence=float(stage_y[stage_ix].mean()), fit_rows=len(stage_ix),
                    duplicated_rows=len(stage_ix)-len(np.unique(stage_ix)),
                    external_prevalence_used=adapted))
            learning.extend(dict(**tags, fold=fold, **r) for r in logs)
        pred = pd.concat(chunks, ignore_index=True)
        if pred.duplicated(['cohort', 'hospital_id', 'model']).any():
            raise ValueError('Duplicate CNN out-of-fold or external predictions')
        pred.to_csv(checkpoint / f'{token}.csv.gz', index=False)
        predictions.append(pred)
        report = reports / f'{token}.metrics.csv'; mark = report.with_suffix('.complete')
        if report.is_file() and mark.is_file():
            met = pd.read_csv(report, float_precision='round_trip')
        else:
            met = summarize_predictions(pred, cfg)
            met['external_prevalence_used'] = adapted
            met.to_csv(report, index=False); mark.write_text('complete')
        metric_parts.append(met)
        index_rows.append(dict(**tags, token=token, external_prevalence_used=adapted))
        # Publish completed comparisons immediately; absence of a completion
        # manifest distinguishes these partial outputs from the finished study.
        wide_grid_results(pd.concat(metric_parts, ignore_index=True)).to_csv(out / 'cnn_comparison_results.csv', index=False)
    pred = pd.concat(predictions, ignore_index=True)
    metrics = pd.concat(metric_parts, ignore_index=True)
    wide = wide_grid_results(metrics)
    audit, logs = pd.DataFrame(audit_rows), pd.DataFrame(learning)
    paired_file = reports / 'paired_landmarks.pkl'
    paired_mark = reports / 'paired_landmarks.complete'
    if paired_file.is_file() and paired_mark.is_file():
        paired = pd.read_pickle(paired_file)
    else:
        paired = paired_landmarks(pred, cfg)
        _v9_pickle(paired_file, paired); paired_mark.write_text('complete')
    metrics.to_csv(out / 'cnn_metrics_long.csv', index=False)
    audit.to_csv(out / 'cnn_fit_audit.csv', index=False)
    logs.to_csv(out / 'cnn_learning_curves.csv', index=False)
    paired.to_csv(out / 'cnn_paired_landmarks.csv', index=False)
    pd.DataFrame(index_rows).to_csv(checkpoint / 'prediction_index.csv', index=False)
    manifest, feature_key = input_manifest(DATA_ROOT, cfg)
    (out / 'run_manifest.json').write_text(json.dumps(dict(
        recipe=recipe, inputs=manifest, feature_fingerprint=feature_key,
        patient_fold_digest=V10_FOLD_DIGEST, checkpoint_directory=str(checkpoint),
        status='complete', expected_training_runs=len(tasks)*(cfg['outer_folds']+1)*2), indent=2, default=str))
    return wide, metrics, paired, audit, logs, checkpoint, pred


def plot_v14_cnn(pred, learning, out):
    out = Path(out)
    for landmark in sorted(pred.landmark.unique()):
        fig, axes = plt.subplots(2, 2, figsize=(14, 10))
        for row, cohort in enumerate(('MIMIC_OOF', 'eICU_external')):
            d = pred[pred.landmark.eq(landmark) & pred.cohort.eq(cohort)]
            for (model, sampling), g in d.groupby(['model', 'sampling']):
                fpr, tpr, _ = roc_curve(g.label, g.probability)
                precision, recall, _ = precision_recall_curve(g.label, g.probability)
                label = f'{model} | {sampling}'
                axes[row, 0].plot(fpr, tpr, label=f'{label}: {roc_auc_score(g.label,g.probability):.3f}')
                axes[row, 1].plot(recall, precision, label=f'{label}: AP {average_precision_score(g.label,g.probability):.3f}')
            axes[row, 0].plot([0, 1], [0, 1], '--', color='gray')
            axes[row, 1].axhline(d.drop_duplicates('hospital_id').label.mean(), ls='--', color='gray', label='Prevalence')
            for col, (xlabel, ylabel) in enumerate((('False positive rate', 'Recall'), ('Recall', 'Precision'))):
                axes[row, col].set(xlabel=xlabel, ylabel=ylabel, title=f'{cohort} | lab set {landmark}', xlim=(0, 1), ylim=(0, 1))
                axes[row, col].legend(fontsize=7); axes[row, col].grid(alpha=.2)
        fig.tight_layout(); fig.savefig(out / f'cnn_set{landmark}_roc_pr.png', dpi=180); plt.show()
    if not learning.empty:
        for (model, sampling), group in learning[learning.fold.ge(0)].groupby(['model', 'sampling']):
            fig, axes = plt.subplots(1, 2, figsize=(13, 4))
            for (landmark, fold), g in group.groupby(['landmark', 'fold']):
                label = f'set {landmark}, fold {fold+1}'
                axes[0].plot(g.epoch, g.loss, label=label)
                axes[1].plot(g.epoch, g.inner_AP, label=label)
            axes[0].set(xlabel='Epoch', ylabel='Training loss')
            axes[1].set(xlabel='Epoch', ylabel='Inner-validation AP')
            axes[1].legend(fontsize=7); fig.suptitle(f'{model} | {sampling}')
            fig.tight_layout(); fig.savefig(out / f'{model}_{sampling.replace(" ","_")}_learning.png', dpi=180); plt.show()


def run_v14_sensitivities(root, study_root, cfg, primary_wide, primary_checkpoint):
    tables = []
    for name, setting in scenario_plan(dict(cfg, include_cnn=False))[1:]:
        out = Path(study_root) / name
        print(f'[tabular sensitivity] {name}: fixed anchors; no CNN fits', flush=True)
        m, ms, e, es, audit = prepare_v10_scenario(root, out, setting)
        result = run_v14_tabular_grid(m, ms, e, es, setting, out)
        wide, metrics, paired, fit_audit, learning, checkpoint = result
        finalist_diagnostics(wide, checkpoint, setting, out)
        compare_fixed_sensitivity_anchors(primary_wide, primary_checkpoint, checkpoint, setting, out)
        manifest, fingerprint = input_manifest(root, setting)
        (out / 'run_manifest.json').write_text(json.dumps(dict(scenario=name, inputs=manifest,
            feature_fingerprint=fingerprint, patient_fold_digest=V10_FOLD_DIGEST,
            checkpoint_directory=str(checkpoint), cnn_included=False), indent=2, default=str))
        tables.append(wide.assign(Scenario=name, Experiment_family='tabular'))
        del m, ms, e, es, audit, result, metrics, paired, fit_audit, learning
        ICD_RAW.clear(); ICD_SELECTION_CACHE.clear(); gc.collect()
    return pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()


def display_v14_results(grid):
    shown = grid[grid.Model.ne('DecisionTree')].sort_values('eICU_Recall', ascending=False).head(50)
    columns = [c for c in ('Scenario', 'Experiment_family', 'population', 'Experiment', 'Feature_combo',
        'Sampling', 'Model', 'MIMIC_PR_AUC', 'MIMIC_AUROC', 'eICU_PR_AUC', 'eICU_PR_AUC_lo',
        'eICU_PR_AUC_hi', 'eICU_AUROC', 'eICU_Recall', 'eICU_Specificity', 'eICU_PPV',
        'external_prevalence_used') if c in shown]
    with pd.option_context('display.max_rows', 50, 'display.max_columns', None, 'display.max_colwidth', None):
        display(shown[columns].round(4))


"""V14 acceptance checks and reporting; unchanged CNN training and data policies."""
import warnings
import uuid
from datetime import datetime, timezone
from sklearn.exceptions import ConvergenceWarning

# Retain the exact earlier code for fingerprint-based migration, not execution.
# Guard also permits re-executing this cell in the same kernel.
if '_V14_FIX_ORIGINALS' not in globals():
    _V14_FIX_ORIGINALS = {n: v for n, v in globals().copy().items() if callable(v)}


def v14_legacy_fingerprint(m, ms, e, es, cfg):
    changed = {n: globals()[n] for n, v in _V14_FIX_ORIGINALS.items()
               if callable(v) and n in globals() and globals()[n] is not v}
    try:
        globals().update({n: _V14_FIX_ORIGINALS[n] for n in changed})
        return _V14_FIX_ORIGINALS['fitting_fingerprint'](m, ms, e, es, cfg)
    finally:
        globals().update(changed)


def v14_source_checkpoint(m, ms, e, es, cfg, scenario):
    key = v14_legacy_fingerprint(m, ms, e, es, dict(cfg, include_cnn=True))
    return DATA_ROOT / 'BE_integrated_grid_v13_outputs' / scenario / 'grid_checkpoints' / key, key


def v14_summary_certified(saved):
    audit = saved.get('audit', {})
    iterations = np.asarray(audit.get('iterations', []), dtype=float)
    return (audit.get('summary_converged') is True and
            audit.get('summary_convergence_warning') is False and
            iterations.size > 0 and np.isfinite(iterations).all() and
            (iterations < audit.get('summary_max_iter', 0)).all())


def fit_sampled_checkpoint(name, x, y, indices, target, setting, cfg, seed, stage, folder):
    """Only converged SummaryLR fits may enter the accepted checkpoint namespace."""
    if name != 'SummaryLR':
        return _V14_FIX_ORIGINALS['fit_sampled_checkpoint'](
            name, x, y, indices, target, setting, cfg, seed, stage, folder)
    path = None if folder is None else Path(folder) / f'{name}_{stage}.pkl'
    if path is not None and path.is_file():
        saved = _v9_read_pickle(path)
        probs = np.asarray(saved['probs'])
        if (saved['rows'] != len(target) or probs.shape != (len(target),) or
                not np.isfinite(probs).all() or not v14_summary_certified(saved)):
            raise ValueError('Uncertified SummaryLR checkpoint; no predictions were accepted.')
        print(f'[model] SummaryLR {stage}: resumed converged fit', flush=True)
        return probs, saved['audit']
    unique, multiplicity = np.unique(indices, return_counts=True)
    estimator = grid_estimator(name, y[indices], setting, cfg, seed)
    started = time.perf_counter()
    print(f'[model] SummaryLR {stage}: {len(unique):,} unique rows, {len(indices):,} effective rows', flush=True)
    # Explicitly override any notebook-wide suppression for convergence warnings.
    with warnings.catch_warnings(record=True) as caught:
        warnings.simplefilter('always', ConvergenceWarning)
        estimator.fit(x[unique], y[unique], sample_weight=multiplicity.astype(float))
    convergence_warning = any(issubclass(w.category, ConvergenceWarning) for w in caught)
    for warning in caught:
        if not issubclass(warning.category, ConvergenceWarning):
            warnings.warn_explicit(warning.message, warning.category, warning.filename, warning.lineno)
    iterations = np.asarray(estimator.n_iter_)
    converged = (not convergence_warning and iterations.size > 0 and
                 np.isfinite(iterations).all() and (iterations < estimator.max_iter).all())
    if not converged:
        raise RuntimeError(f'SummaryLR {stage} did not converge (iterations={iterations.tolist()}, '
                           f'limit={estimator.max_iter}); no accepted checkpoint saved. '
                           'Completed other-model checkpoints remain reusable.')
    probs = estimator.predict_proba(target)[:, 1]
    if not np.isfinite(probs).all():
        raise ValueError('Nonfinite SummaryLR predictions')
    audit = dict(model=name, stage=stage, seconds=time.perf_counter()-started,
                 unique_rows=len(unique), effective_rows=len(indices), iterations=iterations.tolist(),
                 lasso_relative_dual_gap=None, solver_revision='v14-convergence-checked',
                 summary_converged=True, summary_convergence_warning=False,
                 summary_max_iter=int(estimator.max_iter), summary_solver=estimator.solver)
    if path is not None:
        _v9_pickle(path, dict(rows=len(target), probs=probs, audit=audit))
    print(f'[model] SummaryLR {stage}: converged and saved after {audit["seconds"]:.1f}s', flush=True)
    return probs, audit


def v14_copy_prediction(source, destination, *, model=None, rows=None, cfg=None):
    source, destination = Path(source), Path(destination)
    if destination.exists() or not source.is_file():
        return False
    if source.name.startswith('SummaryLR_') and not v14_summary_certified(_v9_read_pickle(source)):
        # Old caches did not record optimizer warnings. Iteration count alone is
        # insufficient to rule out abnormal optimizer termination. Refit only LR.
        return False
    return _V14_FIX_ORIGINALS['v14_copy_prediction'](
        source, destination, model=model, rows=rows, cfg=cfg)


def run_v14_tabular_grid(m, ms, e, es, cfg, out):
    setting = dict(cfg, include_cnn=False)
    validate_config(setting)
    V13_TRANSFORMS.clear()
    source, _ = v14_source_checkpoint(m, ms, e, es, setting, Path(out).name)
    old_v14 = Path(out) / 'grid_checkpoints' / v14_legacy_fingerprint(m, ms, e, es, setting)
    target = Path(out) / 'grid_checkpoints' / fitting_fingerprint(m, ms, e, es, setting)
    if target == old_v14:
        raise RuntimeError('Convergence-checked fits must have a new acceptance namespace')
    allowed = {f'{name}_{stage}.pkl' for name in setting['base_models'] for stage in ('inner', 'outer')}
    copied = 0; uncertified = set()
    for origin in (old_v14, source):
        for file in origin.glob('*.models/*.pkl'):
            if file.name not in allowed:
                continue
            if file.name.startswith('SummaryLR_') and not v14_summary_certified(_v9_read_pickle(file)):
                uncertified.add(str(file.relative_to(origin)))
                continue
            copied += v14_copy_prediction(file, target / file.relative_to(origin), cfg=setting)
    print(f'[v14 resume] Reused {copied} exact-fingerprint model checkpoints; '
          f'{len(uncertified)} legacy SummaryLR stages require convergence-checked refits.', flush=True)
    # Never migrate whole-fold/config/report caches: these could bypass the new
    # acceptance check. Other estimators retain their saved predictions.
    return _run_v13_grid_core(m, ms, e, es, setting, out)


def v14_atomic_csv(frame, path):
    path = Path(path)
    compression = 'gzip' if path.name.endswith('.gz') else None
    _v9_atomic(path, lambda temporary: frame.to_csv(temporary, index=False, compression=compression))


def v14_update_status(out, **updates):
    path = Path(out) / 'run_manifest.json'
    state = json.loads(path.read_text()) if path.exists() else {}
    state.update(updates, updated_at=datetime.now(timezone.utc).isoformat())
    _v9_atomic(path, lambda p: p.write_text(json.dumps(state, indent=2, default=str), encoding='utf-8'))


def v14_cnn_diagnostics(pred, metrics, cfg, out):
    """All prespecified CNN comparisons; descriptive calibration, no recalibration."""
    digest = hashlib.sha256((frame_digest(pred) + frame_digest(metrics) + report_fingerprint(cfg)).encode()
                            + stable_code_bytes(v14_cnn_diagnostics.__code__)
                            + stable_code_bytes(calibration_parameters.__code__)).hexdigest()[:24]
    folder = Path(out) / 'diagnostics' / digest
    folder.mkdir(parents=True, exist_ok=True)
    marker = folder / 'complete.json'
    if marker.exists():
        return folder
    keys = ['population', 'landmark', 'scope', 'model', 'sampling', 'cohort']
    calibration, operating, bin_rows = [], [], []
    for key, group in pred.groupby(keys, sort=False):
        tags = dict(zip(keys, key))
        tags['external_prevalence_used'] = str(tags['sampling']).startswith('prevalence_match')
        calibration.append(dict(**tags, **calibration_parameters(group.label, group.probability),
                                admissions=len(group), patients=group.patient_id.nunique(), cases=int(group.label.sum())))
        cuts = np.unique(np.quantile(group.probability, np.linspace(0, 1, 11)))
        bins = (pd.cut(group.probability, cuts, include_lowest=True, duplicates='drop')
                if len(cuts) > 1 else pd.Series(0, index=group.index))
        grouped = group.groupby(bins, observed=True).agg(predicted=('probability', 'mean'),
                    observed=('label', 'mean'), admissions=('label', 'size'))
        bin_rows.extend(dict(**tags, **row) for row in grouped.to_dict('records'))
        for target in cfg['recall_targets']:
            col = operating_key(target)
            if col not in group or not np.isfinite(group[col]).all():
                raise ValueError('Missing/invalid MIMIC-derived CNN operating thresholds')
            q = group.copy(); q['threshold'] = q[col]
            met = metric_values(q.label, q.probability, q.probability >= q.threshold)
            boot = patient_bootstrap(q, cfg['n_boot'], cfg['seed'])
            row = dict(**tags, target_inner_recall=target, threshold_source='MIMIC inner patient holdout',
                       admissions=len(q), patients=q.patient_id.nunique(), cases=int(q.label.sum()),
                       bootstrap_requested=cfg['n_boot'], bootstrap_valid=len(boot), **met)
            for name in met:
                row[name+'_lo'] = boot[name].quantile(.025) if name in boot else np.nan
                row[name+'_hi'] = boot[name].quantile(.975) if name in boot else np.nan
            operating.append(row)
    cal, op, bins = pd.DataFrame(calibration), pd.DataFrame(operating), pd.DataFrame(bin_rows)
    for name, frame in [('calibration', cal), ('operating_points', op), ('calibration_bins', bins)]:
        v14_atomic_csv(frame, folder / f'{name}.csv')
    for landmark in sorted(pred.landmark.unique()):
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))
        for ax, cohort in zip(axes, ('MIMIC_OOF', 'eICU_external')):
            for (model, sampling), g in bins[bins.landmark.eq(landmark) & bins.cohort.eq(cohort)].groupby(['model', 'sampling']):
                ax.plot(g.predicted, g.observed, 'o-', label=f'{model} | {sampling}')
            ax.plot([0, 1], [0, 1], '--', color='gray')
            ax.set(xlabel='Mean predicted probability', ylabel='Observed BE fraction', title=f'{cohort} | set {landmark}')
            ax.legend(fontsize=7); ax.grid(alpha=.2)
        fig.tight_layout(); fig.savefig(folder / f'cnn_set{landmark}_calibration.png', dpi=180); plt.close(fig)
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))
    for ax, cohort in zip(axes, ('MIMIC_OOF', 'eICU_external')):
        g = metrics[metrics.cohort.eq(cohort)].sort_values(['landmark', 'model', 'sampling'])
        positions = np.arange(len(g))
        # Interval endpoints need not bracket the point estimate in a percentile CI.
        ax.hlines(positions, g.AP_lo, g.AP_hi, color='steelblue')
        ax.scatter(g.AP, positions, color='steelblue')
        ax.set_yticks(positions, [f'set {r.landmark} | {r.model} | {r.sampling}' for r in g.itertuples()], fontsize=7)
        ax.set(xlabel='Average precision (95% patient-bootstrap CI)', title=cohort)
        ax.grid(axis='x', alpha=.2)
    fig.tight_layout(); fig.savefig(folder / 'cnn_ap_95ci.png', dpi=180); plt.close(fig)
    _v9_atomic(marker, lambda p: p.write_text(json.dumps(dict(status='complete',
        comparisons=len(metrics), operating_rows=len(op), fingerprint=digest)), encoding='utf-8'))
    return folder


def run_v14_cnn_comparisons(m, ms, e, es, cfg, out):
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    manifest_path = out / 'run_manifest.json'
    if manifest_path.exists():
        archive = out / 'manifest_history' / f'{uuid.uuid4().hex}.json'
        _v9_atomic(archive, lambda p: p.write_bytes(manifest_path.read_bytes()))
    # Replace rather than merge: no old complete flag, key or results pointer survives.
    state = dict(status='running', run_id=uuid.uuid4().hex,
                 expected_comparisons=len(v14_cnn_tasks()), completed_comparisons=0,
                 expected_training_runs=len(v14_cnn_tasks())*(cfg['outer_folds']+1)*2,
                 note='Root CSVs are current only when their fingerprint is listed in this manifest.')
    _v9_atomic(manifest_path, lambda p: p.write_text(json.dumps(state, indent=2), encoding='utf-8'))
    try:
        result = _v14_cnn_core_checked(m, ms, e, es, cfg, out)
        wide, metrics, paired, audit, logs, checkpoint, pred = result
        folder = v14_cnn_diagnostics(pred, metrics, cfg, out)
        v14_update_status(out, status='complete', completed_comparisons=len(v14_cnn_tasks()),
                          diagnostics_directory=str(folder), results_file=str(out / 'cnn_comparison_results.csv'))
        return result
    except BaseException as error:
        v14_update_status(out, status='interrupted' if isinstance(error, (KeyboardInterrupt, SystemExit)) else 'failed',
                          error_type=type(error).__name__, error=str(error))
        raise


def _v14_cnn_core_checked(m, ms, e, es, cfg, out):
    setting_cfg = dict(cfg, include_cnn=True)
    validate_config(setting_cfg)
    source, source_key = v14_source_checkpoint(m, ms, e, es, setting_cfg, 'primary')
    recipe = dict(source_key=source_key, tasks=v14_cnn_tasks(), population='all_labs',
                  revision='separate-cnn-v14-1')
    digest = hashlib.sha256(json.dumps(recipe, sort_keys=True).encode() +
        b''.join(stable_code_bytes(f.__code__) for f in
                 (_V14_FIX_ORIGINALS['run_v14_cnn_comparisons'], _V14_FIX_ORIGINALS['v14_cnn_tasks'], _V14_FIX_ORIGINALS['v14_copy_prediction']))).hexdigest()[:24]
    out = Path(out); out.mkdir(parents=True, exist_ok=True)
    checkpoint = out / 'cnn_checkpoints' / digest
    checkpoint.mkdir(parents=True, exist_ok=True)
    v14_update_status(out, fingerprint=digest, report_fingerprint=report_fingerprint(cfg), recipe=recipe, checkpoint_directory=str(checkpoint))
    mi = population_indices(m, 'MIMIC', 'all_labs', setting_cfg)
    ei = population_indices(e, 'eICU', 'all_labs', setting_cfg)
    # Preflight only metadata. Copy the large feature/tensor bank for one
    # landmark at a time, not once for the full population and again per set.
    metadata = ['patient_id', 'hospital_id', 'label', 'landmark']
    pm = m[metadata].iloc[mi].reset_index(drop=True)
    pe = e[metadata].iloc[ei].reset_index(drop=True)
    fold_map = patient_folds(pm, cfg['outer_folds'], cfg['seed'])
    prepared = {}
    # Preflight all landmarks/folds before starting any CNN fit.
    for landmark in (1, 2):
        ii, jj = np.flatnonzero(pm.landmark.eq(landmark)), np.flatnonzero(pe.landmark.eq(landmark))
        mm, ee = pm.iloc[ii].reset_index(drop=True), pe.iloc[jj].reset_index(drop=True)
        folds = mm.patient_id.map(fold_map).to_numpy()
        if mm.label.nunique() != 2 or ee.label.nunique() != 2:
            raise ValueError(f'CNN set {landmark}: both cohorts require two outcome classes')
        inner_split(mm, cfg['seed'] + 1000, cfg['inner_holdout'])
        for fold in range(cfg['outer_folds']):
            train, test = mm.loc[folds != fold], mm.loc[folds == fold]
            if test.empty or test.label.nunique() != 2:
                raise ValueError(f'CNN set {landmark}, fold {fold}: invalid test cohort')
            if set(train.patient_id) & set(test.patient_id):
                raise ValueError('Patient overlap in CNN train/test partition')
            inner_split(train, cfg['seed'] + fold, cfg['inner_holdout'])
        prepared[landmark] = mi[ii], ei[jj], folds
    del pm, pe, mm, ee
    predictions, audit_rows, learning, metric_parts, index_rows = [], [], [], [], []
    reports = checkpoint / 'reports' / report_fingerprint(cfg)
    reports.mkdir(parents=True, exist_ok=True)
    tasks = v14_cnn_tasks()
    loaded_landmark = None
    for number, (landmark, combo, variant, mode, value) in enumerate(tasks, 1):
        if loaded_landmark != landmark:
            if loaded_landmark is not None:
                del mm, ee, ss, se, train, target, target_seq
                gc.collect()
            ii, jj, folds = prepared[landmark]
            mm, ee = m.iloc[ii].reset_index(drop=True), e.iloc[jj].reset_index(drop=True)
            ss = panel_sequence(take_sequence(ms, ii), cfg)
            se = panel_sequence(take_sequence(es, jj), cfg)
            loaded_landmark = landmark
        scope = '+'.join(combo); sampling = mode if value is None else f'{mode} {value}'
        token = prediction_token('all_labs', landmark, scope, sampling)
        tags = dict(population='all_labs', landmark=landmark, scope=scope, sampling=sampling)
        adapted = mode == 'prevalence_match'
        print(f'[CNN comparison] {number}/{len(tasks)} set {landmark}: {variant}, {sampling}', flush=True)
        chunks = []
        for fold in list(range(cfg['outer_folds'])) + [-1]:
            seed = cfg['seed'] + (1000 if fold == -1 else fold)
            tr = np.arange(len(mm)) if fold == -1 else np.flatnonzero(folds != fold)
            te = np.arange(len(ee)) if fold == -1 else np.flatnonzero(folds == fold)
            train, target = mm.iloc[tr], ee if fold == -1 else mm.iloc[te]
            target_seq = se if fold == -1 else take_sequence(ss, te)
            ti, vi = inner_split(train, seed, cfg['inner_holdout'])
            y = train.label.to_numpy(int)
            sampling_setting = resolved_sampling(mode, value, train.label, ee.label)
            ix = sample_indices(y[ti], sampling_setting, seed)
            fx = sample_indices(y, sampling_setting, seed)
            folder = checkpoint / f'{token}.fold{fold}.models'
            for stage, rows in (('inner', len(vi)), ('outer', len(target))):
                relative = Path(f'{token}.fold{fold}.models') / variant / f'CNN_{stage}.pkl'
                if v14_copy_prediction(source / relative, checkpoint / relative, model=variant, rows=rows):
                    print(f'[CNN resume] reused V13 {variant} fold {fold} {stage}', flush=True)
            ip, ep, logs, records = _lab_cnn_checkpoint(
                train, take_sequence(ss, tr), target, target_seq,
                'eICU' if fold == -1 else 'MIMIC', combo, sampling_setting, setting_cfg,
                seed, ti, vi, ix, fx, folder, variant)
            if len(ep) != len(target) or not np.isfinite(ep).all():
                raise ValueError('Invalid CNN prediction vector')
            q = target[['patient_id', 'hospital_id', 'label']].copy().assign(
                **tags, cohort='eICU_external' if fold == -1 else 'MIMIC_OOF',
                model=variant, fold=fold, probability=ep, threshold=youden_threshold(y[vi], ip))
            for recall in cfg['recall_targets']:
                q[operating_key(recall)] = operating_threshold(y[vi], ip, recall)
            chunks.append(q)
            for record in records:
                stage_y = y[ti] if record['stage'] == 'inner' else y
                stage_ix = ix if record['stage'] == 'inner' else fx
                audit_rows.append(dict(**tags, fold=fold, **record,
                    train_admissions=len(stage_y), natural_prevalence=float(stage_y.mean()),
                    fit_prevalence=float(stage_y[stage_ix].mean()), fit_rows=len(stage_ix),
                    duplicated_rows=len(stage_ix)-len(np.unique(stage_ix)),
                    external_prevalence_used=adapted))
            learning.extend(dict(**tags, fold=fold, **r) for r in logs)
        pred = pd.concat(chunks, ignore_index=True)
        if pred.duplicated(['cohort', 'hospital_id', 'model']).any():
            raise ValueError('Duplicate CNN out-of-fold or external predictions')
        v14_atomic_csv(pred, checkpoint / f'{token}.csv.gz')
        predictions.append(pred)
        report = reports / f'{token}.metrics.csv'; mark = report.with_suffix('.complete')
        if report.is_file() and mark.is_file():
            met = pd.read_csv(report, float_precision='round_trip')
        else:
            met = summarize_predictions(pred, cfg)
            met['external_prevalence_used'] = adapted
            v14_atomic_csv(met, report); mark.write_text('complete')
        metric_parts.append(met)
        index_rows.append(dict(**tags, token=token, external_prevalence_used=adapted))
        # Publish atomic partial results with explicit run status and progress.
        v14_atomic_csv(wide_grid_results(pd.concat(metric_parts, ignore_index=True)), out / 'cnn_comparison_results.csv')
        v14_update_status(out, completed_comparisons=number, partial_results_file=str(out / 'cnn_comparison_results.csv'))
    pred = pd.concat(predictions, ignore_index=True)
    metrics = pd.concat(metric_parts, ignore_index=True)
    wide = wide_grid_results(metrics)
    audit, logs = pd.DataFrame(audit_rows), pd.DataFrame(learning)
    paired_file = reports / 'paired_landmarks.pkl'
    paired_mark = reports / 'paired_landmarks.complete'
    if paired_file.is_file() and paired_mark.is_file():
        paired = pd.read_pickle(paired_file)
    else:
        paired = paired_landmarks(pred, cfg)
        _v9_pickle(paired_file, paired); paired_mark.write_text('complete')
    v14_atomic_csv(metrics, out / 'cnn_metrics_long.csv')
    v14_atomic_csv(audit, out / 'cnn_fit_audit.csv')
    v14_atomic_csv(logs, out / 'cnn_learning_curves.csv')
    v14_atomic_csv(paired, out / 'cnn_paired_landmarks.csv')
    v14_atomic_csv(pd.DataFrame(index_rows), checkpoint / 'prediction_index.csv')
    manifest, feature_key = input_manifest(DATA_ROOT, cfg)
    v14_update_status(out, status='reporting', inputs=manifest, feature_fingerprint=feature_key,
                      patient_fold_digest=V10_FOLD_DIGEST, checkpoint_directory=str(checkpoint))
    return wide, metrics, paired, audit, logs, checkpoint, pred


"""Runtime-only continuation of the existing certified Lasso solver.

Load in an idle V14 kernel with %run -i, then rerun the grid cell. No training
starts on import. Saved predictions, iterates, objective and tolerance are unchanged.
"""
from types import MethodType
import tempfile

if not all(n in globals() for n in ('grid_estimator', 'fitting_fingerprint',
                                   'ResumableWeightedL1Logistic', '_v9_read_pickle')):
    raise RuntimeError('Run the V14 definition cells first; existing feature variables need not be rebuilt.')
if '_V14_AUTOCONTINUE_BINDINGS' not in globals():
    _V14_AUTOCONTINUE_BINDINGS = dict(grid_estimator=grid_estimator,
        fitting_fingerprint=fitting_fingerprint, solver_step=ResumableWeightedL1Logistic.fit)


def _v14_lasso_progress(path):
    if not Path(path).is_file():
        return 0, None, None
    state = _v9_read_pickle(path)
    trace = state['trace']
    if not trace:
        return 0, None, None
    last = trace[-1]
    gap, objective = float(last['relative_dual_gap']), float(last['objective'])
    if not np.isfinite([gap, objective]).all():
        raise RuntimeError('Nonfinite Lasso progress; automatic continuation stopped.')
    return len(trace), gap, objective


def v14_auto_continue_lasso(self, X, y, sample_weight=None):
    """Repeat only resumable budget exits, never arbitrary optimizer exceptions."""
    stall_limit = int(LASSO_RUNTIME.get('max_stalled_chunks', 3))
    if stall_limit < 1:
        raise ValueError('max_stalled_chunks must be positive')
    original_path = getattr(self, 'checkpoint_path', None)
    temporary = None
    if original_path is None:
        temporary = tempfile.TemporaryDirectory(prefix='be-lasso-continuation-')
        self.checkpoint_path = Path(temporary.name) / 'iterate.pkl'
    stalled_chunks = 0
    try:
        while True:
            before = _v14_lasso_progress(self.checkpoint_path)
            try:
                # The original step function still validates the full matrix,
                # objective, solver signature and global convergence certificate.
                return _V14_AUTOCONTINUE_BINDINGS['solver_step'](self, X, y, sample_weight=sample_weight)
            except (TimeoutError, RuntimeError) as error:
                message = str(error)
                time_boundary = (type(error) is TimeoutError and
                    message.startswith('Lasso reached its ') and
                    'per-invocation budget. Accepted iterates and completed models are saved;' in message)
                iteration_boundary = (type(error) is RuntimeError and
                    message.startswith('Lasso gap ') and ' exceeds ' in message and
                    message.endswith('accepted iterates saved. Rerun to continue; no uncertified predictions saved.'))
                if not (time_boundary or iteration_boundary):
                    raise
                if not Path(self.checkpoint_path).is_file():
                    raise RuntimeError('Lasso budget exit did not save resumable state; refusing a blind retry.') from error
                after = _v14_lasso_progress(self.checkpoint_path)
                progressed = after[0] > before[0] and (before[1] is None or
                    after[1] < before[1] - 1e-12 * max(abs(before[1]), 1e-12) or
                    after[2] < before[2] - 1e-12 * max(abs(before[2]), 1.))
                stalled_chunks = 0 if progressed else stalled_chunks + 1
                if stalled_chunks >= stall_limit:
                    raise RuntimeError(f'Lasso made no material optimization progress across {stall_limit} '
                        'consecutive work chunks. State is preserved, but no unconverged predictions '
                        'were accepted; inspect the solver before retrying.') from error
                gap_text = 'not yet available' if after[1] is None else f'{after[1]:.6g}'
                print(f'[Lasso] checkpoint boundary: {after[0]} accepted steps, gap {gap_text}; '
                      'continuing automatically with the same tolerance.', flush=True)
    finally:
        if temporary is not None:
            self.checkpoint_path = original_path
            temporary.cleanup()


def grid_estimator(name, y, setting, cfg, seed):
    estimator = _V14_AUTOCONTINUE_BINDINGS['grid_estimator'](name, y, setting, cfg, seed)
    if name == 'Lasso':
        estimator.fit = MethodType(v14_auto_continue_lasso, estimator)
    return estimator


def fitting_fingerprint(m, ms, e, es, cfg):
    # Only orchestration changed: each chunk uses precisely the earlier solver
    # and certificate, equivalent to pressing Run again. Explicit compatibility
    # keeps all existing whole-fold, per-model and partial-iterate checkpoints.
    current_factory = globals()['grid_estimator']
    try:
        globals()['grid_estimator'] = _V14_AUTOCONTINUE_BINDINGS['grid_estimator']
        return _V14_AUTOCONTINUE_BINDINGS['fitting_fingerprint'](m, ms, e, es, cfg)
    finally:
        globals()['grid_estimator'] = current_factory


print('[Lasso] Automatic checkpoint continuation enabled; convergence tolerance unchanged. '
      'Interrupt remains available; no model training was started by this patch.', flush=True)


"""Runtime-only continuation of the existing certified Lasso solver.

Load in an idle V14 kernel with %run -i, then rerun the grid cell. No training
starts on import. Saved predictions, iterates, objective and tolerance are unchanged.
"""
from types import MethodType
import tempfile

if not all(n in globals() for n in ('grid_estimator', 'fitting_fingerprint',
                                   'ResumableWeightedL1Logistic', '_v9_read_pickle')):
    raise RuntimeError('Run the V14 definition cells first; existing feature variables need not be rebuilt.')
if '_V14_AUTOCONTINUE_BINDINGS' not in globals():
    _V14_AUTOCONTINUE_BINDINGS = dict(grid_estimator=grid_estimator,
        fitting_fingerprint=fitting_fingerprint, solver_step=ResumableWeightedL1Logistic.fit)


def _v14_lasso_progress(path):
    if not Path(path).is_file():
        return 0, None, None
    state = _v9_read_pickle(path)
    trace = state['trace']
    if not trace:
        return 0, None, None
    last = trace[-1]
    gap, objective = float(last['relative_dual_gap']), float(last['objective'])
    if not np.isfinite([gap, objective]).all():
        raise RuntimeError('Nonfinite Lasso progress; automatic continuation stopped.')
    return len(trace), gap, objective


def v14_auto_continue_lasso(self, X, y, sample_weight=None):
    """Repeat only resumable budget exits, never arbitrary optimizer exceptions."""
    stall_limit = int(LASSO_RUNTIME.get('max_stalled_chunks', 3))
    if stall_limit < 1:
        raise ValueError('max_stalled_chunks must be positive')
    original_path = getattr(self, 'checkpoint_path', None)
    temporary = None
    if original_path is None:
        temporary = tempfile.TemporaryDirectory(prefix='be-lasso-continuation-')
        self.checkpoint_path = Path(temporary.name) / 'iterate.pkl'
    stalled_chunks = 0
    try:
        while True:
            before = _v14_lasso_progress(self.checkpoint_path)
            try:
                # The original step function still validates the full matrix,
                # objective, solver signature and global convergence certificate.
                return _V14_AUTOCONTINUE_BINDINGS['solver_step'](self, X, y, sample_weight=sample_weight)
            except (TimeoutError, RuntimeError) as error:
                message = str(error)
                time_boundary = (type(error) is TimeoutError and
                    message.startswith('Lasso reached its ') and
                    'per-invocation budget. Accepted iterates and completed models are saved;' in message)
                iteration_boundary = (type(error) is RuntimeError and
                    message.startswith('Lasso gap ') and ' exceeds ' in message and
                    message.endswith('accepted iterates saved. Rerun to continue; no uncertified predictions saved.'))
                if not (time_boundary or iteration_boundary):
                    raise
                if not Path(self.checkpoint_path).is_file():
                    raise RuntimeError('Lasso budget exit did not save resumable state; refusing a blind retry.') from error
                after = _v14_lasso_progress(self.checkpoint_path)
                progressed = after[0] > before[0] and (before[1] is None or
                    after[1] < before[1] - 1e-12 * max(abs(before[1]), 1e-12) or
                    after[2] < before[2] - 1e-12 * max(abs(before[2]), 1.))
                stalled_chunks = 0 if progressed else stalled_chunks + 1
                if stalled_chunks >= stall_limit:
                    raise RuntimeError(f'Lasso made no material optimization progress across {stall_limit} '
                        'consecutive work chunks. State is preserved, but no unconverged predictions '
                        'were accepted; inspect the solver before retrying.') from error
                gap_text = 'not yet available' if after[1] is None else f'{after[1]:.6g}'
                print(f'[Lasso] checkpoint boundary: {after[0]} accepted steps, gap {gap_text}; '
                      'continuing automatically with the same tolerance.', flush=True)
    finally:
        if temporary is not None:
            self.checkpoint_path = original_path
            temporary.cleanup()


def grid_estimator(name, y, setting, cfg, seed):
    estimator = _V14_AUTOCONTINUE_BINDINGS['grid_estimator'](name, y, setting, cfg, seed)
    if name == 'Lasso':
        estimator.fit = MethodType(v14_auto_continue_lasso, estimator)
    return estimator


def fitting_fingerprint(m, ms, e, es, cfg):
    # Only orchestration changed: each chunk uses precisely the earlier solver
    # and certificate, equivalent to pressing Run again. Explicit compatibility
    # keeps all existing whole-fold, per-model and partial-iterate checkpoints.
    current_factory = globals()['grid_estimator']
    try:
        globals()['grid_estimator'] = _V14_AUTOCONTINUE_BINDINGS['grid_estimator']
        return _V14_AUTOCONTINUE_BINDINGS['fitting_fingerprint'](m, ms, e, es, cfg)
    finally:
        globals()['grid_estimator'] = current_factory


print('[Lasso] Automatic checkpoint continuation enabled; convergence tolerance unchanged. '
      'Interrupt remains available; no model training was started by this patch.', flush=True)


"""Failure-isolated tabular grid. No failed/partial CV score is accepted."""
import traceback

if '_V14_UNATTENDED' not in globals():
    _V14_UNATTENDED = {n:v for n,v in globals().copy().items() if callable(v)}
    _V14_UNATTENDED['solver_step'] = _V14_AUTOCONTINUE_BINDINGS['solver_step']
GRID_RUNTIME = globals().get('GRID_RUNTIME', dict(retry_failed=False))
LASSO_RUNTIME.setdefault('max_fit_seconds', 3600.)


def v14_bounded_lasso_step(self, X, y, sample_weight=None):
    path = getattr(self, 'checkpoint_path', None)
    spent = float(_v9_read_pickle(path)['seconds']) if path is not None and Path(path).exists() else 0.
    limit = float(LASSO_RUNTIME['max_fit_seconds'])
    if not np.isfinite(limit) or limit <= 0 or not np.isfinite(spent) or spent < 0:
        raise ValueError('Invalid cumulative Lasso runtime budget/state')
    if spent >= limit:
        raise RuntimeError(f'Lasso cumulative compute limit ({limit:g}s) reached without certification. '
                           'Iterates retained; this fit is incomplete, not an accepted result.')
    previous = self.max_seconds
    try:
        self.max_seconds = min(previous, limit-spent)
        return _V14_UNATTENDED['solver_step'](self, X, y, sample_weight=sample_weight)
    finally:
        self.max_seconds = previous


_V14_AUTOCONTINUE_BINDINGS['solver_step'] = v14_bounded_lasso_step


def fitting_fingerprint(m, ms, e, es, cfg):
    # Failure bookkeeping and compute budgets do not change accepted fits.
    # Restore exact earlier callable bindings only during compatibility hashing.
    changed = {n:globals()[n] for n,v in _V14_UNATTENDED.items()
               if n in globals() and callable(v) and globals()[n] is not v}
    try:
        globals().update({n:_V14_UNATTENDED[n] for n in changed})
        return _V14_UNATTENDED['fitting_fingerprint'](m,ms,e,es,cfg)
    finally:
        globals().update(changed)


def run_v14_tabular_grid(m,ms,e,es,cfg,out):
    v14_runtime_preflight()
    v14_update_status(out,status='running',incomplete_rows=None,error_type=None,error=None)
    try:
        return _V14_UNATTENDED['run_v14_tabular_grid'](m,ms,e,es,cfg,out)
    except BaseException as error:
        v14_update_status(out,status='interrupted' if isinstance(error,(KeyboardInterrupt,SystemExit)) else 'failed',
                          error_type=type(error).__name__,error=str(error))
        raise


def v14_failure(folder, model, stage, error):
    record = dict(model=model, stage=stage, fit_status='failed',
                  error_type=type(error).__name__, error=str(error),
                  traceback=traceback.format_exc(), recorded_at=datetime.now(timezone.utc).isoformat())
    if folder is not None:
        path=Path(folder)/f'{model}_{stage}.failure.json'
        _v9_atomic(path,lambda p:p.write_text(json.dumps(record,indent=2),encoding='utf-8'))
    print(f'[fit failed] {model} {stage}: {error}. Continuing other fits.',flush=True)
    return record


def v14_attempt_model(name,x,y,indices,target,setting,cfg,seed,stage,folder):
    failure=None if folder is None else Path(folder)/f'{name}_{stage}.failure.json'
    accepted=None if folder is None else Path(folder)/f'{name}_{stage}.pkl'
    if (failure is not None and failure.exists() and not GRID_RUNTIME['retry_failed']
            and not accepted.exists()):
        record=json.loads(failure.read_text())
        return np.full(len(target),np.nan),record
    try:
        probs,audit=fit_sampled_checkpoint(name,x,y,indices,target,setting,cfg,seed,stage,folder)
        probs=np.asarray(probs)
        if probs.shape!=(len(target),) or not np.isfinite(probs).all() or np.any((probs<0)|(probs>1)):
            raise ValueError('Invalid model probability vector')
        return probs,dict(audit,fit_status='complete')
    except (RuntimeError,ValueError,FloatingPointError,np.linalg.LinAlgError) as error:
        # I/O failures, memory exhaustion and manual interrupts propagate.
        return np.full(len(target),np.nan),v14_failure(folder,name,stage,error)


def v14_valid_predictions(pred,cfg):
    if pred.empty or not np.isfinite(pred.probability).all() or pred.threshold.isna().any():
        return False
    if ((pred.probability<0)|(pred.probability>1)).any():return False
    for cohort,g in pred.groupby('cohort'):
        expected=set(range(cfg['outer_folds'])) if cohort=='MIMIC_OOF' else {-1}
        if set(g.fold)!=expected:return False
        if g.duplicated(['hospital_id','model']).any():return False
    return True


def summarize_predictions(pred,cfg):
    keys=['population','cohort','landmark','scope','model','sampling'];rows=[]
    names=('AUROC','AP','Brier','Recall','Specificity','PPV','Alert_rate')
    for key,g in pred.groupby(keys,sort=False):
        if v14_valid_predictions(g,cfg):
            row=_V14_UNATTENDED['summarize_predictions'](g,cfg).iloc[0].to_dict()
            row.update(status='complete',failed_folds='[]')
        else:
            failed=[str(f) for f,q in g.groupby('fold')
                    if not np.isfinite(q.probability).all() or q.threshold.isna().any()]
            expected=set(range(cfg['outer_folds'])) if key[1]=='MIMIC_OOF' else {-1}
            failed += ['missing:'+str(f) for f in sorted(expected-set(g.fold))]
            row=dict(zip(keys,key),status='incomplete',failed_folds=json.dumps(failed),
                admissions=len(g),patients=g.patient_id.nunique(),cases=int(g.label.sum()),
                prevalence=float(g.label.mean()),bootstrap_requested=cfg['n_boot'],bootstrap_valid=0)
            for name in names:
                for suffix in ('','_lo','_hi'):row[name+suffix]=np.nan
        rows.append(row)
    return pd.DataFrame(rows)


def wide_grid_results(metrics):
    result=_V14_UNATTENDED['wide_grid_results'](metrics)
    for cohort in ('MIMIC','eICU'):
        if cohort+'_status' not in result:result[cohort+'_status']='complete'
    result['Status']=np.where(result.MIMIC_status.eq('complete')&result.eICU_status.eq('complete'),
                              'complete','incomplete')
    return result


def v14_complete_grid(grid):
    return grid[grid.Status.eq('complete')].copy() if 'Status' in grid else grid


def v14_prediction_digest(path):
    digest=hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda:stream.read(1024*1024),b''):digest.update(chunk)
    return digest.hexdigest()[:24]


def v14_complete_predictions(pred,cfg):
    keys=['population','landmark','scope','model','sampling']
    groups=[g for _,g in pred.groupby(keys,sort=False) if v14_valid_predictions(g,cfg)]
    return pd.concat(groups,ignore_index=True) if groups else pred.iloc[:0].copy()


def paired_landmarks(pred,cfg):
    keys=['population','cohort','scope','model','sampling'];parts=[]
    for _,g in pred.groupby(keys,sort=False):
        if set(g.landmark)=={1,2} and all(v14_valid_predictions(q,cfg) for _,q in g.groupby('landmark')):
            parts.append(g)
    if not parts:return pd.DataFrame()
    return _V14_UNATTENDED['paired_landmarks'](pd.concat(parts,ignore_index=True),cfg)


def display_v14_results(grid):
    valid=v14_complete_grid(grid)
    print(f'[results] {len(valid)} complete rows; {len(grid)-len(valid)} incomplete rows excluded from rankings. '
          'See grid_fit_status.csv and grid_failures.csv for details.')
    return _V14_UNATTENDED['display_v14_results'](valid)


def finalist_diagnostics(wide,checkpoint,cfg,out):
    valid=v14_complete_grid(wide)
    if valid.empty:
        print('[diagnostics] No complete model/configuration; finalist diagnostics skipped.')
        return pd.DataFrame(),pd.DataFrame()
    return _V14_UNATTENDED['finalist_diagnostics'](valid,checkpoint,cfg,out)


def integrated_plots(grid,paired,learning,out):
    valid=v14_complete_grid(grid)
    if valid.empty:
        print('[plots] No complete rows to plot.');return
    return _V14_UNATTENDED['integrated_plots'](valid,paired,learning,out)


def plot_results(pred,metrics,learning,cfg,out):
    pred=v14_complete_predictions(pred,cfg)
    metrics=metrics[metrics.status.eq('complete')] if 'status' in metrics else metrics
    if pred.empty or metrics.empty:
        print('[plots] No complete lab model predictions to plot.');return
    return _V14_UNATTENDED['plot_results'](pred,metrics,learning,cfg,out)


def v14_grid_failure_report(wide,audit,out):
    out=Path(out);rows=[]
    for r in audit.to_dict('records'):
        tags={k:r[k] for k in ('population','landmark','scope','sampling','fold') if k in r}
        for stage in json.loads(r.get('model_timing_json','[]')):
            if stage.get('fit_status') in ('failed','dependency_failed'):
                rows.append(dict(**tags,**stage))
    failures=pd.DataFrame(rows,columns=list(dict.fromkeys(
        ['population','landmark','scope','sampling','fold','model','stage','fit_status','error_type','error']+
        [k for r in rows for k in r])))
    v14_atomic_csv(failures,out/'grid_failures.csv')
    cols=[c for c in ['population','landmark','Feature_combo','Sampling','Model','Status',
          'MIMIC_status','eICU_status','MIMIC_failed_folds','eICU_failed_folds'] if c in wide]
    v14_atomic_csv(wide[cols],out/'grid_fit_status.csv')
    print(f'[grid finished] {len(v14_complete_grid(wide))} complete rows; '
          f'{int(wide.Status.ne("complete").sum())} incomplete. Failures explicitly logged.',flush=True)


print('[grid] Failure isolation enabled. Failed fits are logged, never scored as partial CV; '
      'other models continue. Lasso cumulative compute cap:',LASSO_RUNTIME['max_fit_seconds'],'seconds.',flush=True)

def fit_grid_partition(train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,checkpoint_dir=None):
    """All selection/scaling nested. Same inner patient holdout for all model families."""
    seq=panel_sequence(seq,cfg);target_seq=panel_sequence(target_seq,cfg)
    ti,vi=inner_split(train,seed,cfg['inner_holdout']);y=train.label.to_numpy(int)
    inner=cached_grid_transform(train.iloc[ti],combo,cfg,seed)
    xt=inner.transform(train.iloc[ti],'MIMIC');xv=inner.transform(train.iloc[vi],'MIMIC')
    outer=cached_grid_transform(train,combo,cfg,seed)
    xf=outer.transform(train,'MIMIC');xe=outer.transform(target,target_cohort)
    ix=sample_indices(y[ti],setting,seed);fx=sample_indices(y,setting,seed)
    preds={};cal={};logs=[]
    fit_records=[]
    for name in cfg['base_models']:
        cal[name],inner_record=v14_attempt_model(name,xt,y[ti],ix,xv,setting,cfg,seed,'inner',checkpoint_dir)
        if not np.isfinite(cal[name]).all():
            preds[name]=np.full(len(target),np.nan)
            fit_records.extend([inner_record,dict(model=name,stage='outer',fit_status='dependency_failed',error='Inner fit failed; outer scoring not attempted')])
            continue
        preds[name],outer_record=v14_attempt_model(name,xf,y,fx,xe,setting,cfg,seed,'outer',checkpoint_dir)
        fit_records.extend([inner_record,outer_record])
    if cfg['include_ensembles']:
        available=all(n in preds and np.isfinite(preds[n]).all() and np.isfinite(cal[n]).all()
                      for n in ('RandomForest','Lasso'))
        if available:
            ip=np.column_stack([cal['RandomForest'],cal['Lasso']]);ep=np.column_stack([preds['RandomForest'],preds['Lasso']])
            cal[ENSEMBLES[0]]=ip.mean(1);preds[ENSEMBLES[0]]=ep.mean(1)
            try:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always',ConvergenceWarning)
                    meta=LogisticRegression(max_iter=1000).fit(ip,y[vi])
                if any(issubclass(w.category,ConvergenceWarning) for w in caught):
                    raise RuntimeError('Stack meta-learner did not converge')
                cal[ENSEMBLES[1]]=meta.predict_proba(ip)[:,1];preds[ENSEMBLES[1]]=meta.predict_proba(ep)[:,1]
                if not np.isfinite(preds[ENSEMBLES[1]]).all():raise ValueError('Invalid Stack probabilities')
            except (RuntimeError,ValueError,FloatingPointError,np.linalg.LinAlgError) as error:
                fit_records.append(v14_failure(checkpoint_dir,ENSEMBLES[1],'meta',error))
                cal[ENSEMBLES[1]]=np.full(len(vi),np.nan);preds[ENSEMBLES[1]]=np.full(len(target),np.nan)
        else:
            for name in ENSEMBLES:
                cal[name]=np.full(len(vi),np.nan);preds[name]=np.full(len(target),np.nan)
                fit_records.append(dict(model=name,stage='ensemble',fit_status='dependency_failed',
                                        error='Required RF/Lasso fit unavailable'))
    for variant in cnn_variants(combo,cfg):
        ip,ep,history,records=_lab_cnn_checkpoint(
            train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,ti,vi,ix,fx,checkpoint_dir,variant)
        cal[variant]=ip;preds[variant]=ep;logs.extend(history);fit_records.extend(records)
    thresholds={name:(youden_threshold(y[vi],p) if np.isfinite(p).all() else np.nan) for name,p in cal.items()}
    operating={model:{operating_key(t):(operating_threshold(y[vi],p,t) if np.isfinite(p).all() else np.nan) for t in cfg['recall_targets']} for model,p in cal.items()}
    audit=dict(operating_thresholds_json=json.dumps(operating),train_admissions=len(train),train_patients=train.patient_id.nunique(),
        inner_validation_patients=train.iloc[vi].patient_id.nunique(),raw_predictors=outer.raw(train,'MIMIC').shape[1],
        natural_prevalence=float(y.mean()),fit_prevalence=float(y[fx].mean()),fit_rows=len(fx),
        duplicated_rows=len(fx)-len(np.unique(fx)),external_prevalence_used=setting['external_prevalence_used'],
        selected_ICD_features='|'.join(outer.icd_names),threshold_source='MIMIC inner patient holdout',model_timing_json=json.dumps(fit_records))
    return preds,thresholds,logs,audit

def _run_v13_grid_core(m,ms,e,es,cfg,out):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    ICD_SELECTION_CACHE.clear()
    population_audit(m,e,cfg).to_csv(out/'population_eligibility.csv',index=False)
    checkpoint=out/'grid_checkpoints'/fitting_fingerprint(m,ms,e,es,cfg)
    checkpoint.mkdir(parents=True,exist_ok=True)
    tasks=[];status=[]
    # Validate all patient splits before fitting; a sparse subgroup never falls
    # back to the broader population or a globally fitted feature selection.
    for population in POPULATIONS:
        mi=population_indices(m,'MIMIC',population,cfg);ei=population_indices(e,'eICU',population,cfg)
        pm=m.iloc[mi].reset_index(drop=True);pe=e.iloc[ei].reset_index(drop=True)
        ps=take_sequence(ms,mi);pes=take_sequence(es,ei)
        try:fold_map=patient_folds(pm,cfg['outer_folds'],cfg['seed'])
        except ValueError as err:
            status.append(dict(population=population,landmark='both',status='skipped',reason=str(err)));continue
        for landmark in (1,2):
            mi=np.flatnonzero(pm.landmark.eq(landmark));ei=np.flatnonzero(pe.landmark.eq(landmark))
            mm=pm.iloc[mi].reset_index(drop=True);ee=pe.iloc[ei].reset_index(drop=True)
            ss=take_sequence(ps,mi);se=take_sequence(pes,ei);folds=mm.patient_id.map(fold_map).to_numpy()
            try:
                if mm.label.nunique()!=2 or ee.label.nunique()!=2:raise ValueError('Both cohorts need two outcome classes at this landmark')
                inner_split(mm,cfg['seed']+1000,cfg['inner_holdout'])
                for fold in range(cfg['outer_folds']):
                    tr=np.flatnonzero(folds!=fold);te=np.flatnonzero(folds==fold)
                    if not len(te) or mm.iloc[te].label.nunique()!=2:raise ValueError('Outer test fold empty or single-class; revise fold count explicitly')
                    inner_split(mm.iloc[tr],cfg['seed']+fold,cfg['inner_holdout'])
            except ValueError as err:
                status.append(dict(population=population,landmark=landmark,status='skipped',reason=str(err)));continue
            status.append(dict(population=population,landmark=landmark,status='ready',reason='',
                MIMIC_admissions=len(mm),MIMIC_cases=int(mm.label.sum()),eICU_admissions=len(ee),eICU_cases=int(ee.label.sum())))
            for combo in population_combinations(population,cfg):
                for mode,value in sampling_for_combo(population,combo,cfg):tasks.append((population,landmark,mm,ee,ss,se,folds,combo,mode,value))
    pd.DataFrame(status).to_csv(out/'population_run_status.csv',index=False)
    for s in status:print('[population]',s,flush=True)
    if not tasks:raise ValueError('No population/landmark passed preflight; inspect population_run_status.csv')
    completed=[];audits=[];curves=[]
    for number,(population,landmark,mm,ee,ss,se,folds,combo,mode,value) in enumerate(tasks,1):
        label='+'.join(combo);sl=mode if value is None else f'{mode} {value}'
        token=prediction_token(population,landmark,label,sl);tags=dict(population=population,landmark=landmark,scope=label,sampling=sl)
        pfile=checkpoint/f'{token}.csv.gz';afile=checkpoint/f'{token}.audit.csv';lfile=checkpoint/f'{token}.learning.csv'
        ready=checkpoint/f'{token}.complete'
        attempted=checkpoint/f'{token}.attempted.json'
        print(f'[fits] {number}/{len(tasks)} {population} set {landmark} {label} {sl}',flush=True)
        if not ((ready.exists() or (attempted.exists() and not GRID_RUNTIME['retry_failed'])) and pfile.exists() and afile.exists()):
            cp=[];ca=[];cl=[]
            for fold in list(range(cfg['outer_folds']))+[-1]:
                part=checkpoint/f'{token}.fold{fold}.pkl';mark=checkpoint/f'{token}.fold{fold}.complete'
                # Only locally generated checkpoints in this exact fingerprint namespace.
                attempt=checkpoint/f'{token}.fold{fold}.attempted.json'
                if part.exists() and (mark.exists() or (attempt.exists() and not GRID_RUNTIME['retry_failed'])):q,a,ll=pd.read_pickle(part)
                else:
                    tr=np.arange(len(mm)) if fold==-1 else np.flatnonzero(folds!=fold)
                    te=np.flatnonzero(folds==fold) if fold!=-1 else np.arange(len(ee))
                    train=mm.iloc[tr];target=ee if fold==-1 else mm.iloc[te]
                    t_seq=se if fold==-1 else take_sequence(ss,te)
                    if fold!=-1:assert not set(train.patient_id)&set(target.patient_id)
                    setting=resolved_sampling(mode,value,train.label,ee.label)
                    pp,tt,ll,aa=fit_grid_partition(train,take_sequence(ss,tr),target,t_seq,
                        'eICU' if fold==-1 else 'MIMIC',combo,setting,cfg,cfg['seed']+(1000 if fold==-1 else fold),
                        checkpoint_dir=checkpoint/f'{token}.fold{fold}.models')
                    q=pd.concat([target[['patient_id','hospital_id','label']].copy().assign(**tags,
                        cohort='eICU_external' if fold==-1 else 'MIMIC_OOF',model=model,fold=fold,
                        probability=prob,threshold=tt[model]) for model,prob in pp.items()],ignore_index=True)
                    op=json.loads(aa['operating_thresholds_json'])
                    for target_recall in cfg['recall_targets']:
                        key=operating_key(target_recall)
                        q[key]=q.model.map({model:values[key] for model,values in op.items()})
                    for _,g in q.groupby('model'):
                        if not (np.isfinite(g.probability).all() or g.probability.isna().all()):
                            raise ValueError('Unexpected partially invalid probability vector')
                    q['fit_status']=np.where(q.probability.notna(),'complete','failed')
                    a=dict(**tags,fold=fold,**aa);ll=[dict(**tags,fold=fold,**l) for l in ll]
                    _v9_pickle(part,(q,a,ll))
                    if np.isfinite(q.probability).all():mark.write_text('complete')
                    else:_v9_atomic(attempt,lambda p:p.write_text(json.dumps(dict(status='incomplete',failed_models=q.loc[q.probability.isna(),'model'].unique().tolist())),encoding='utf-8'))
                cp.append(q);ca.append(a);cl.extend(ll)
            p=pd.concat(cp,ignore_index=True);a=pd.DataFrame(ca);ll=pd.DataFrame(cl)
            assert not p.duplicated(['cohort','hospital_id','model']).any()
            v14_atomic_csv(p,pfile);v14_atomic_csv(a,afile)
            if not ll.empty:ll.to_csv(lfile,index=False)
            if np.isfinite(p.probability).all():ready.write_text('complete')
            else:_v9_atomic(attempted,lambda f:f.write_text(json.dumps(dict(status='incomplete',failed_models=p.loc[p.probability.isna(),'model'].unique().tolist())),encoding='utf-8'))
        audits.append(pd.read_csv(afile))
        if lfile.exists():curves.append(pd.read_csv(lfile))
        completed.append(dict(**tags,token=token,external_prevalence_used=mode=='prevalence_match' and value in ('eicu','midpoint')))
    # A separate phase: changing n_boot never retrains models. Every row is covered.
    reports=checkpoint/'reports'/report_fingerprint(cfg);reports.mkdir(parents=True,exist_ok=True)
    rows=[]
    for number,task in enumerate(completed,1):
        token=task['token'];source_key=v14_prediction_digest(checkpoint/f'{token}.csv.gz')
        file=reports/f'{token}.{source_key}.metrics.csv';mark=file.with_suffix('.complete')
        print(f'[all-row CIs] {number}/{len(completed)}',flush=True)
        if file.exists() and mark.exists():met=pd.read_csv(file,float_precision='round_trip')
        else:
            met=summarize_predictions(read_predictions(checkpoint/f'{token}.csv.gz'),cfg)
            met['external_prevalence_used']=task['external_prevalence_used']
            met.to_csv(file,index=False);mark.write_text('complete')
        rows.append(met)
    metrics=pd.concat(rows,ignore_index=True);audit=pd.concat(audits,ignore_index=True)
    learning=pd.concat(curves,ignore_index=True) if curves else pd.DataFrame()
    wide=wide_grid_results(metrics);wide.to_csv(out/'feature_sampling_grid.csv',index=False)
    metrics.to_csv(out/'unified_grid_metrics_long.csv',index=False);audit.to_csv(out/'unified_grid_fit_audit.csv',index=False)
    learning.to_csv(out/'unified_grid_learning_curves.csv',index=False)
    paired=[];tasks_frame=pd.DataFrame(completed)
    for _,g in tasks_frame.groupby(['population','scope','sampling'],sort=False):
        if set(g.landmark)!={1,2}:continue
        source_key=hashlib.sha256(''.join(v14_prediction_digest(checkpoint/f'{t}.csv.gz') for t in g.token).encode()).hexdigest()[:24]
        file=reports/f'{g.iloc[0].token}.{source_key}.paired.pkl';mark=file.with_suffix('.complete')
        if file.exists() and mark.exists():q=pd.read_pickle(file)
        else:
            p=pd.concat([read_predictions(checkpoint/f'{t}.csv.gz') for t in g.token],ignore_index=True)
            q=paired_landmarks(p,cfg);q.to_pickle(file);mark.write_text('complete')
        paired.append(q)
    paired=pd.concat(paired,ignore_index=True) if paired else pd.DataFrame()
    paired.to_csv(out/'paired_landmark_grid.csv',index=False)
    pd.DataFrame(completed).to_csv(checkpoint/'prediction_index.csv',index=False)
    v14_grid_failure_report(wide,audit,out)
    v14_update_status(out,status='complete_with_failures' if wide.Status.ne('complete').any() else 'complete',
        incomplete_rows=int(wide.Status.ne('complete').sum()),checkpoint_directory=str(checkpoint))
    return wide,metrics,paired,audit,learning,checkpoint

def compare_scenario_overlap(primary_wide,primary_checkpoint,other_checkpoint,cfg,out):
    """Paired held-out policy comparison on common patients/admissions, no refit."""
    out=Path(out)/'paired_policy_comparison';out.mkdir(exist_ok=True)
    digest=hashlib.sha256((str(primary_checkpoint)+str(other_checkpoint)+frame_digest(primary_wide)+report_fingerprint(cfg)).encode()+
        stable_code_bytes(compare_scenario_overlap.__code__)).hexdigest()[:16]
    path=out/f'{digest}.csv';mark=out/f'{digest}.complete'
    if path.exists() and mark.exists():return pd.read_csv(path)
    primary_wide=v14_complete_grid(primary_wide)
    winners=primary_wide[primary_wide.Input_kind.eq('values_and_observation_patterns')].sort_values(
        ['MIMIC_PR_AUC','Feature_combo','Sampling'],ascending=[False,True,True]).drop_duplicates(
        ['population','landmark','Model','external_prevalence_used'])
    records=[]
    for r in winners.itertuples():
        token=prediction_token(r.population,r.landmark,r.Feature_combo,r.Sampling)
        first=Path(primary_checkpoint)/f'{token}.csv.gz';second=Path(other_checkpoint)/f'{token}.csv.gz'
        if not first.exists() or not second.exists():
            records.append(dict(population=r.population,landmark=r.landmark,model=r.Model,status='configuration_unavailable'));continue
        a=read_predictions(first);b=read_predictions(second)
        a=a[a.model.eq(r.Model)];b=b[b.model.eq(r.Model)]
        if not v14_valid_predictions(a,cfg) or not v14_valid_predictions(b,cfg):
            records.append(dict(population=r.population,landmark=r.landmark,model=r.Model,status='incomplete_fit'));continue
        p=a.merge(b,on=['cohort','patient_id','hospital_id','model'],suffixes=('_primary','_sensitivity'),validate='one_to_one')
        assert p.label_primary.eq(p.label_sensitivity).all()
        internal=p.cohort.eq('MIMIC_OOF')
        assert p.loc[internal,'fold_primary'].eq(p.loc[internal,'fold_sensitivity']).all(), 'Cross-policy fold mismatch'
        for cohort,g in p.groupby('cohort'):
            tags=dict(population=r.population,landmark=r.landmark,model=r.Model,scope=r.Feature_combo,sampling=r.Sampling,
                external_prevalence_used=bool(r.external_prevalence_used),cohort=cohort,
                common_admissions=len(g),common_patients=g.patient_id.nunique(),common_cases=int(g.label_primary.sum()))
            if g.label_primary.nunique()<2:records.append(dict(**tags,status='single_class_overlap'));continue
            f=WeightedPredictionMetrics(g.label_primary,g.probability_primary,g.probability_primary>=g.threshold_primary)
            s=WeightedPredictionMetrics(g.label_sensitivity,g.probability_sensitivity,g.probability_sensitivity>=g.threshold_sensitivity)
            x=f(np.ones(len(g)));z=s(np.ones(len(g)));delta=[]
            for weights in cluster_draws(g.patient_id,cfg['n_boot'],cfg['seed']):
                xx=f(weights);zz=s(weights)
                if xx is not None:delta.append({k:zz[k]-xx[k] for k in ('AP','AUROC','Brier')})
            boot=pd.DataFrame(delta);row=dict(**tags,status='ok',bootstrap_valid=len(boot))
            for metric in ('AP','AUROC','Brier'):
                row[metric+'_primary']=x[metric];row[metric+'_sensitivity']=z[metric];row[metric+'_delta']=z[metric]-x[metric]
                row[metric+'_delta_lo']=boot[metric].quantile(.025) if metric in boot else np.nan
                row[metric+'_delta_hi']=boot[metric].quantile(.975) if metric in boot else np.nan
            records.append(row)
    frame=pd.DataFrame(records);frame.to_csv(path,index=False);mark.write_text('complete')
    return frame

def run_v14_sensitivities(root, study_root, cfg, primary_wide, primary_checkpoint):
    tables = []
    for name, setting in scenario_plan(dict(cfg, include_cnn=False))[1:]:
        out = Path(study_root) / name
        print(f'[tabular sensitivity] {name}: fixed anchors; no CNN fits', flush=True)
        m, ms, e, es, audit = prepare_v10_scenario(root, out, setting)
        result = run_v14_tabular_grid(m, ms, e, es, setting, out)
        wide, metrics, paired, fit_audit, learning, checkpoint = result
        finalist_diagnostics(wide, checkpoint, setting, out)
        compare_fixed_sensitivity_anchors(primary_wide, primary_checkpoint, checkpoint, setting, out)
        manifest, fingerprint = input_manifest(root, setting)
        (out / 'run_manifest.json').write_text(json.dumps(dict(scenario=name, inputs=manifest,
            feature_fingerprint=fingerprint, patient_fold_digest=V10_FOLD_DIGEST,
            checkpoint_directory=str(checkpoint), cnn_included=False,
            status='complete_with_failures' if wide.Status.ne('complete').any() else 'complete',
            incomplete_rows=int(wide.Status.ne('complete').sum())), indent=2, default=str))
        tables.append(wide.assign(Scenario=name, Experiment_family='tabular'))
        del m, ms, e, es, audit, result, metrics, paired, fit_audit, learning
        ICD_RAW.clear(); ICD_SELECTION_CACHE.clear(); gc.collect()
    return pd.concat(tables, ignore_index=True) if tables else pd.DataFrame()


# Check the active kernel bindings, not merely the file version on disk.
def v14_runtime_preflight():
    replaced=[name for name,expected in _V14_ACTIVE_HANDLERS.items()
              if globals().get(name) is not expected]
    if replaced:
        raise RuntimeError('V14 runtime definitions were overwritten: '+', '.join(replaced)+
            '. Load v14_unattended.py in this kernel before starting the grid; '
            'existing feature variables and checkpoints can be retained.')
    print('[v14 runtime] Automatic Lasso continuation and per-model failure isolation ACTIVE.',flush=True)


_V14_ACTIVE_HANDLERS={name:globals()[name] for name in
    ('grid_estimator','fit_grid_partition','_run_v13_grid_core','v14_attempt_model',
     'v14_auto_continue_lasso','fitting_fingerprint','run_v14_tabular_grid')}
v14_runtime_preflight()


In [ ]:
"""Focused model comparison; unchanged patient splits and per-row operating points."""
if '_V15_PARENT' not in globals():
    _V15_PARENT={n:v for n,v in globals().copy().items() if callable(v)}
    _V15_PARENT_CFG=dict(CFG)

# Keep the cache name SummaryLR: it is L2 logistic regression on the selected
# feature combination, not a restriction to laboratory-summary predictors.
V15_MAIN_MODELS=globals().get('V15_MAIN_MODELS',('RandomForest','SummaryLR','XGBoost'))
CFG.update(base_models=tuple(V15_MAIN_MODELS),include_ensembles=False,
           sensitivity_models=tuple(V15_MAIN_MODELS),
           v15_l1_anchors=V15_L1_ANCHORS,v15_ensemble_anchors=True)
V15_ENSEMBLES=('SoftVote(RF+L2)','Stack(RF+L2)')
V15_METRICS=('AUROC','AP','Brier','Recall','Specificity','PPV','Alert_rate',
             'F1','PR_AUC_lift','AUROC_lift')


def v15_old_config(cfg):
    old={k:v for k,v in cfg.items() if not k.startswith('v15_') and k!='_v15_population'}
    old['base_models']=tuple(_V15_PARENT_CFG['sensitivity_models'] if
        old.get('sensitivity_profile')=='anchors_active' else _V15_PARENT_CFG['base_models'])
    old['sensitivity_models']=_V15_PARENT_CFG['sensitivity_models']
    old['include_ensembles']=_V15_PARENT_CFG['include_ensembles']
    return old


def v15_is_anchor(population,combo,mode,cfg):
    if mode!='none' or cfg.get('sensitivity_profile')=='anchors_active':return False
    return ((population=='all_labs' and tuple(combo)==('lab_current','lab_history')) or
            (population=='icd_history' and tuple(combo)==tuple(ALL_GRID_BLOCKS)))


def metric_values(y,p,decision):
    met=_V15_PARENT['metric_values'](y,p,decision)
    y=np.asarray(y,int);decision=np.asarray(decision,bool)
    tp=float(np.sum((y==1)&decision));den=float(np.sum(y==1)+decision.sum())
    prevalence=float(y.mean())
    met.update(F1=2*tp/den if den else np.nan,
               PR_AUC_lift=met['AP']/prevalence if prevalence>0 else np.nan,
               AUROC_lift=met['AUROC']/0.5)
    return met


class WeightedPredictionMetrics(_V15_PARENT['WeightedPredictionMetrics']):
    def __call__(self,weights):
        met=super().__call__(weights)
        if met is None:return None
        w=np.asarray(weights,float);pos=float(w@self.y);alerts=float(w@self.decision)
        tp=float(w@(self.y*self.decision));prevalence=pos/float(w.sum())
        met.update(F1=2*tp/(pos+alerts),PR_AUC_lift=met['AP']/prevalence,
                   AUROC_lift=met['AUROC']/0.5)
        return met


def summarize_predictions(pred,cfg):
    # Every complete row gets the same patient-cluster bootstrap and all metrics.
    # An incomplete fold invalidates the entire model/configuration internal row.
    keys=['population','cohort','landmark','scope','model','sampling'];rows=[]
    for key,g in pred.groupby(keys,sort=False):
        row=dict(zip(keys,key),admissions=len(g),patients=g.patient_id.nunique(),
                 cases=int(g.label.sum()),prevalence=float(g.label.mean()),
                 bootstrap_requested=cfg['n_boot'])
        if v14_valid_predictions(g,cfg):
            met=metric_values(g.label,g.probability,g.probability>=g.threshold)
            boot=patient_bootstrap(g,cfg['n_boot'],cfg['seed'])
            row.update(met,status='complete',failed_folds='[]',bootstrap_valid=len(boot))
            for name in V15_METRICS:
                row[name+'_lo']=boot[name].quantile(.025) if name in boot else np.nan
                row[name+'_hi']=boot[name].quantile(.975) if name in boot else np.nan
        else:
            failed=[str(f) for f,q in g.groupby('fold') if
                    not np.isfinite(q.probability).all() or q.threshold.isna().any()]
            expected=set(range(cfg['outer_folds'])) if key[1]=='MIMIC_OOF' else {-1}
            failed+=['missing:'+str(f) for f in sorted(expected-set(g.fold))]
            row.update(status='incomplete',failed_folds=json.dumps(failed),bootstrap_valid=0)
            row.update({name+suffix:np.nan for name in V15_METRICS for suffix in ('','_lo','_hi')})
        rows.append(row)
    return pd.DataFrame(rows)


def wide_grid_results(metrics):
    wide=_V15_PARENT['wide_grid_results'](metrics)
    # Existing PR_AUC column is average precision, not trapezoidal PR integration.
    for prefix in ('MIMIC_','eICU_'):
        for suffix in ('','_lo','_hi'):wide[prefix+'AP'+suffix]=wide[prefix+'PR_AUC'+suffix]
    wide['Model_description']=wide.Model.replace({'SummaryLR':'L2 logistic regression'})
    wide['Model_schedule']=np.where(wide.Model.isin(V15_ENSEMBLES+('Lasso',)),
                                   'prespecified_primary_anchors','all_configurations')
    return wide


def display_v14_results(grid):
    valid=v14_complete_grid(grid)
    shown=valid.sort_values('eICU_Recall',ascending=False,kind='stable').head(50)
    identity=['Scenario','Experiment_family','population','Experiment','Feature_combo','Sampling',
              'Model','Model_description','Model_schedule','Status','external_prevalence_used']
    columns=[c for c in identity if c in shown]
    for cohort in ('MIMIC_','eICU_'):
        columns += [cohort+'prevalence'] if cohort+'prevalence' in shown else []
        for metric in ('Recall','Specificity','PPV','F1','AUROC','AP','PR_AUC','PR_AUC_lift','AUROC_lift'):
            columns += [cohort+metric+s for s in ('','_lo','_hi') if cohort+metric+s in shown]
    print(f'[results] Showing {len(shown)} of {len(valid)} complete rows, sorted by eICU recall. '
          f'{len(grid)-len(valid)} incomplete rows excluded; all rows retained in CSV.')
    with pd.option_context('display.max_rows',50,'display.max_columns',None,
                           'display.max_colwidth',None,'display.width',None):
        display(shown[columns].round(5))


def report_fingerprint(cfg):
    funcs=(metric_values,WeightedPredictionMetrics.__call__,summarize_predictions,wide_grid_results)
    return hashlib.sha256((_V15_PARENT['report_fingerprint'](cfg)+'v15-full-metrics-1').encode()+
        b''.join(stable_code_bytes(f.__code__) for f in funcs)).hexdigest()[:24]


def fitting_fingerprint(m,ms,e,es,cfg):
    # Old keys are used ONLY to locate individually compatible model checkpoints.
    # V15 whole-fold outputs get their own namespace and cannot reuse old ensembles.
    old=_V15_PARENT['fitting_fingerprint'](m,ms,e,es,v15_old_config(cfg))
    if cfg.get('include_cnn'):return old
    recipe=dict(base_models=cfg['base_models'],l1_anchors=cfg.get('v15_l1_anchors',False),
                ensemble_anchors=cfg.get('v15_ensemble_anchors',True),members=V15_ENSEMBLES)
    return hashlib.sha256((old+json.dumps(recipe,sort_keys=True)).encode()+
        stable_code_bytes(fit_grid_partition.__code__)+stable_code_bytes(v15_is_anchor.__code__)+
        b'v15-focused-models-1').hexdigest()[:24]


def v14_source_checkpoint(m,ms,e,es,cfg,scenario):
    return _V15_PARENT['v14_source_checkpoint'](m,ms,e,es,v15_old_config(cfg),scenario)


def grid_plan(cfg):
    cells=rows=fits=anchors=0
    stages=2*(cfg['outer_folds']+1)
    for pop in POPULATIONS:
        for combo in population_combinations(pop,cfg):
            for mode,value in sampling_for_combo(pop,combo,cfg):
                cells+=2;rows+=2*len(cfg['base_models']);fits+=2*len(cfg['base_models'])*stages
                if v15_is_anchor(pop,combo,mode,cfg):
                    anchors+=2
                    if cfg.get('v15_ensemble_anchors',True):rows+=4
                    if cfg.get('v15_l1_anchors',False):rows+=2;fits+=2*stages
    return dict(profile=cfg['grid_profile'],age_policy=cfg['age_policy'],grid_cells=cells,
                expected_wide_result_rows=rows,base_estimator_fits=fits,cnn_training_runs=0,
                bootstrap_draws_per_row=cfg['n_boot'],ensemble_anchor_configurations=anchors,
                note='All rows retain inner threshold fits, five-fold evaluation, external refit and CIs.')


def v15_migrate_models(origin,target,cfg):
    allowed=set(cfg['base_models'])|({'Lasso'} if cfg.get('v15_l1_anchors',False) else set())
    copied=0
    for name in sorted(allowed):
        for stage in ('inner','outer'):
            for file in Path(origin).glob(f'*.models/{name}_{stage}.pkl'):
                copied+=v14_copy_prediction(file,Path(target)/file.relative_to(origin),model=name,cfg=cfg)
    return copied


def run_v14_tabular_grid(m,ms,e,es,cfg,out):
    # Deliberately do not invoke V14's runner: it owns a different model schedule.
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    setting=dict(cfg,include_cnn=False)
    validate_config(setting);V13_TRANSFORMS.clear()
    v14_update_status(out,status='running',revision='v15',error=None,error_type=None)
    try:
        old_key=_V15_PARENT['fitting_fingerprint'](m,ms,e,es,v15_old_config(setting))
        target=out/'grid_checkpoints'/fitting_fingerprint(m,ms,e,es,setting)
        origin=DATA_ROOT/'BE_integrated_grid_v14_outputs'/out.name/'grid_checkpoints'/old_key
        copied=v15_migrate_models(origin,target,setting)
        print(f'[v15 resume] Reused {copied} compatible individual model stages. '
              'No old whole-fold results, ensembles or failure markers imported.',flush=True)
        return _run_v13_grid_core(m,ms,e,es,setting,out)
    except BaseException as error:
        v14_update_status(out,status='interrupted' if isinstance(error,(KeyboardInterrupt,SystemExit)) else 'failed',
                          error_type=type(error).__name__,error=str(error))
        raise


print('[v15] RF + L2 logistic + XGBoost on every configuration; inner threshold fitting retained. '
      'RF+L2 ensembles on four primary anchors; optional L1 anchors:',V15_L1_ANCHORS,flush=True)

def fit_grid_partition(train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,checkpoint_dir=None):
    """All selection/scaling nested. Same inner patient holdout for all model families."""
    seq=panel_sequence(seq,cfg);target_seq=panel_sequence(target_seq,cfg)
    ti,vi=inner_split(train,seed,cfg['inner_holdout']);y=train.label.to_numpy(int)
    inner=cached_grid_transform(train.iloc[ti],combo,cfg,seed)
    xt=inner.transform(train.iloc[ti],'MIMIC');xv=inner.transform(train.iloc[vi],'MIMIC')
    outer=cached_grid_transform(train,combo,cfg,seed)
    xf=outer.transform(train,'MIMIC');xe=outer.transform(target,target_cohort)
    ix=sample_indices(y[ti],setting,seed);fx=sample_indices(y,setting,seed)
    preds={};cal={};logs=[]
    fit_records=[]
    anchor=v15_is_anchor(cfg.get('_v15_population'),combo,setting['mode'],cfg)
    models=tuple(cfg['base_models'])+(('Lasso',) if anchor and cfg.get('v15_l1_anchors',False) else ())
    for name in models:
        cal[name],inner_record=v14_attempt_model(name,xt,y[ti],ix,xv,setting,cfg,seed,'inner',checkpoint_dir)
        if not np.isfinite(cal[name]).all():
            preds[name]=np.full(len(target),np.nan)
            fit_records.extend([inner_record,dict(model=name,stage='outer',fit_status='dependency_failed',error='Inner fit failed; outer scoring not attempted')])
            continue
        preds[name],outer_record=v14_attempt_model(name,xf,y,fx,xe,setting,cfg,seed,'outer',checkpoint_dir)
        fit_records.extend([inner_record,outer_record])
    if anchor and cfg.get('v15_ensemble_anchors',True):
        available=all(n in preds and np.isfinite(preds[n]).all() and np.isfinite(cal[n]).all()
                      for n in ('RandomForest','SummaryLR'))
        if available:
            ip=np.column_stack([cal['RandomForest'],cal['SummaryLR']]);ep=np.column_stack([preds['RandomForest'],preds['SummaryLR']])
            cal[V15_ENSEMBLES[0]]=ip.mean(1);preds[V15_ENSEMBLES[0]]=ep.mean(1)
            try:
                with warnings.catch_warnings(record=True) as caught:
                    warnings.simplefilter('always',ConvergenceWarning)
                    meta=LogisticRegression(max_iter=1000).fit(ip,y[vi])
                if any(issubclass(w.category,ConvergenceWarning) for w in caught):
                    raise RuntimeError('Stack meta-learner did not converge')
                cal[V15_ENSEMBLES[1]]=meta.predict_proba(ip)[:,1];preds[V15_ENSEMBLES[1]]=meta.predict_proba(ep)[:,1]
                if not np.isfinite(preds[V15_ENSEMBLES[1]]).all():raise ValueError('Invalid Stack probabilities')
            except (RuntimeError,ValueError,FloatingPointError,np.linalg.LinAlgError) as error:
                fit_records.append(v14_failure(checkpoint_dir,V15_ENSEMBLES[1],'meta',error))
                cal[V15_ENSEMBLES[1]]=np.full(len(vi),np.nan);preds[V15_ENSEMBLES[1]]=np.full(len(target),np.nan)
        else:
            for name in V15_ENSEMBLES:
                cal[name]=np.full(len(vi),np.nan);preds[name]=np.full(len(target),np.nan)
                fit_records.append(dict(model=name,stage='ensemble',fit_status='dependency_failed',
                                        error='Required RF/L2 fit unavailable'))
    for variant in cnn_variants(combo,cfg):
        ip,ep,history,records=_lab_cnn_checkpoint(
            train,seq,target,target_seq,target_cohort,combo,setting,cfg,seed,ti,vi,ix,fx,checkpoint_dir,variant)
        cal[variant]=ip;preds[variant]=ep;logs.extend(history);fit_records.extend(records)
    thresholds={name:(youden_threshold(y[vi],p) if np.isfinite(p).all() else np.nan) for name,p in cal.items()}
    operating={model:{operating_key(t):(operating_threshold(y[vi],p,t) if np.isfinite(p).all() else np.nan) for t in cfg['recall_targets']} for model,p in cal.items()}
    audit=dict(operating_thresholds_json=json.dumps(operating),train_admissions=len(train),train_patients=train.patient_id.nunique(),
        inner_validation_patients=train.iloc[vi].patient_id.nunique(),raw_predictors=outer.raw(train,'MIMIC').shape[1],
        natural_prevalence=float(y.mean()),fit_prevalence=float(y[fx].mean()),fit_rows=len(fx),
        duplicated_rows=len(fx)-len(np.unique(fx)),external_prevalence_used=setting['external_prevalence_used'],
        selected_ICD_features='|'.join(outer.icd_names),threshold_source='MIMIC inner patient holdout',model_timing_json=json.dumps(fit_records))
    return preds,thresholds,logs,audit

def _run_v13_grid_core(m,ms,e,es,cfg,out):
    out=Path(out);out.mkdir(parents=True,exist_ok=True)
    ICD_SELECTION_CACHE.clear()
    population_audit(m,e,cfg).to_csv(out/'population_eligibility.csv',index=False)
    checkpoint=out/'grid_checkpoints'/fitting_fingerprint(m,ms,e,es,cfg)
    checkpoint.mkdir(parents=True,exist_ok=True)
    tasks=[];status=[]
    # Validate all patient splits before fitting; a sparse subgroup never falls
    # back to the broader population or a globally fitted feature selection.
    for population in POPULATIONS:
        mi=population_indices(m,'MIMIC',population,cfg);ei=population_indices(e,'eICU',population,cfg)
        pm=m.iloc[mi].reset_index(drop=True);pe=e.iloc[ei].reset_index(drop=True)
        ps=take_sequence(ms,mi);pes=take_sequence(es,ei)
        try:fold_map=patient_folds(pm,cfg['outer_folds'],cfg['seed'])
        except ValueError as err:
            status.append(dict(population=population,landmark='both',status='skipped',reason=str(err)));continue
        for landmark in (1,2):
            mi=np.flatnonzero(pm.landmark.eq(landmark));ei=np.flatnonzero(pe.landmark.eq(landmark))
            mm=pm.iloc[mi].reset_index(drop=True);ee=pe.iloc[ei].reset_index(drop=True)
            ss=take_sequence(ps,mi);se=take_sequence(pes,ei);folds=mm.patient_id.map(fold_map).to_numpy()
            try:
                if mm.label.nunique()!=2 or ee.label.nunique()!=2:raise ValueError('Both cohorts need two outcome classes at this landmark')
                inner_split(mm,cfg['seed']+1000,cfg['inner_holdout'])
                for fold in range(cfg['outer_folds']):
                    tr=np.flatnonzero(folds!=fold);te=np.flatnonzero(folds==fold)
                    if not len(te) or mm.iloc[te].label.nunique()!=2:raise ValueError('Outer test fold empty or single-class; revise fold count explicitly')
                    inner_split(mm.iloc[tr],cfg['seed']+fold,cfg['inner_holdout'])
            except ValueError as err:
                status.append(dict(population=population,landmark=landmark,status='skipped',reason=str(err)));continue
            status.append(dict(population=population,landmark=landmark,status='ready',reason='',
                MIMIC_admissions=len(mm),MIMIC_cases=int(mm.label.sum()),eICU_admissions=len(ee),eICU_cases=int(ee.label.sum())))
            for combo in population_combinations(population,cfg):
                for mode,value in sampling_for_combo(population,combo,cfg):tasks.append((population,landmark,mm,ee,ss,se,folds,combo,mode,value))
    pd.DataFrame(status).to_csv(out/'population_run_status.csv',index=False)
    for s in status:print('[population]',s,flush=True)
    if not tasks:raise ValueError('No population/landmark passed preflight; inspect population_run_status.csv')
    completed=[];audits=[];curves=[]
    for number,(population,landmark,mm,ee,ss,se,folds,combo,mode,value) in enumerate(tasks,1):
        label='+'.join(combo);sl=mode if value is None else f'{mode} {value}'
        token=prediction_token(population,landmark,label,sl);tags=dict(population=population,landmark=landmark,scope=label,sampling=sl)
        pfile=checkpoint/f'{token}.csv.gz';afile=checkpoint/f'{token}.audit.csv';lfile=checkpoint/f'{token}.learning.csv'
        ready=checkpoint/f'{token}.complete'
        attempted=checkpoint/f'{token}.attempted.json'
        print(f'[fits] {number}/{len(tasks)} {population} set {landmark} {label} {sl}',flush=True)
        if not ((ready.exists() or (attempted.exists() and not GRID_RUNTIME['retry_failed'])) and pfile.exists() and afile.exists()):
            cp=[];ca=[];cl=[]
            for fold in list(range(cfg['outer_folds']))+[-1]:
                part=checkpoint/f'{token}.fold{fold}.pkl';mark=checkpoint/f'{token}.fold{fold}.complete'
                # Only locally generated checkpoints in this exact fingerprint namespace.
                attempt=checkpoint/f'{token}.fold{fold}.attempted.json'
                if part.exists() and (mark.exists() or (attempt.exists() and not GRID_RUNTIME['retry_failed'])):q,a,ll=pd.read_pickle(part)
                else:
                    tr=np.arange(len(mm)) if fold==-1 else np.flatnonzero(folds!=fold)
                    te=np.flatnonzero(folds==fold) if fold!=-1 else np.arange(len(ee))
                    train=mm.iloc[tr];target=ee if fold==-1 else mm.iloc[te]
                    t_seq=se if fold==-1 else take_sequence(ss,te)
                    if fold!=-1:assert not set(train.patient_id)&set(target.patient_id)
                    setting=resolved_sampling(mode,value,train.label,ee.label)
                    pp,tt,ll,aa=fit_grid_partition(train,take_sequence(ss,tr),target,t_seq,
                        'eICU' if fold==-1 else 'MIMIC',combo,setting,dict(cfg,_v15_population=population),cfg['seed']+(1000 if fold==-1 else fold),
                        checkpoint_dir=checkpoint/f'{token}.fold{fold}.models')
                    q=pd.concat([target[['patient_id','hospital_id','label']].copy().assign(**tags,
                        cohort='eICU_external' if fold==-1 else 'MIMIC_OOF',model=model,fold=fold,
                        probability=prob,threshold=tt[model]) for model,prob in pp.items()],ignore_index=True)
                    op=json.loads(aa['operating_thresholds_json'])
                    for target_recall in cfg['recall_targets']:
                        key=operating_key(target_recall)
                        q[key]=q.model.map({model:values[key] for model,values in op.items()})
                    for _,g in q.groupby('model'):
                        if not (np.isfinite(g.probability).all() or g.probability.isna().all()):
                            raise ValueError('Unexpected partially invalid probability vector')
                    q['fit_status']=np.where(q.probability.notna(),'complete','failed')
                    a=dict(**tags,fold=fold,**aa);ll=[dict(**tags,fold=fold,**l) for l in ll]
                    _v9_pickle(part,(q,a,ll))
                    if np.isfinite(q.probability).all():mark.write_text('complete')
                    else:_v9_atomic(attempt,lambda p:p.write_text(json.dumps(dict(status='incomplete',failed_models=q.loc[q.probability.isna(),'model'].unique().tolist())),encoding='utf-8'))
                cp.append(q);ca.append(a);cl.extend(ll)
            p=pd.concat(cp,ignore_index=True);a=pd.DataFrame(ca);ll=pd.DataFrame(cl)
            assert not p.duplicated(['cohort','hospital_id','model']).any()
            v14_atomic_csv(p,pfile);v14_atomic_csv(a,afile)
            if not ll.empty:ll.to_csv(lfile,index=False)
            if np.isfinite(p.probability).all():ready.write_text('complete')
            else:_v9_atomic(attempted,lambda f:f.write_text(json.dumps(dict(status='incomplete',failed_models=p.loc[p.probability.isna(),'model'].unique().tolist())),encoding='utf-8'))
        audits.append(pd.read_csv(afile))
        if lfile.exists():curves.append(pd.read_csv(lfile))
        completed.append(dict(**tags,token=token,external_prevalence_used=mode=='prevalence_match' and value in ('eicu','midpoint')))
    # A separate phase: changing n_boot never retrains models. Every row is covered.
    reports=checkpoint/'reports'/report_fingerprint(cfg);reports.mkdir(parents=True,exist_ok=True)
    rows=[]
    for number,task in enumerate(completed,1):
        token=task['token'];source_key=v14_prediction_digest(checkpoint/f'{token}.csv.gz')
        file=reports/f'{token}.{source_key}.metrics.csv';mark=file.with_suffix('.complete')
        print(f'[all-row CIs] {number}/{len(completed)}',flush=True)
        if file.exists() and mark.exists():met=pd.read_csv(file,float_precision='round_trip')
        else:
            met=summarize_predictions(read_predictions(checkpoint/f'{token}.csv.gz'),cfg)
            met['external_prevalence_used']=task['external_prevalence_used']
            met.to_csv(file,index=False);mark.write_text('complete')
        rows.append(met)
    metrics=pd.concat(rows,ignore_index=True);audit=pd.concat(audits,ignore_index=True)
    learning=pd.concat(curves,ignore_index=True) if curves else pd.DataFrame()
    wide=wide_grid_results(metrics);wide.to_csv(out/'feature_sampling_grid.csv',index=False)
    metrics.to_csv(out/'unified_grid_metrics_long.csv',index=False);audit.to_csv(out/'unified_grid_fit_audit.csv',index=False)
    learning.to_csv(out/'unified_grid_learning_curves.csv',index=False)
    paired=[];tasks_frame=pd.DataFrame(completed)
    for _,g in tasks_frame.groupby(['population','scope','sampling'],sort=False):
        if set(g.landmark)!={1,2}:continue
        source_key=hashlib.sha256(''.join(v14_prediction_digest(checkpoint/f'{t}.csv.gz') for t in g.token).encode()).hexdigest()[:24]
        file=reports/f'{g.iloc[0].token}.{source_key}.paired.pkl';mark=file.with_suffix('.complete')
        if file.exists() and mark.exists():q=pd.read_pickle(file)
        else:
            p=pd.concat([read_predictions(checkpoint/f'{t}.csv.gz') for t in g.token],ignore_index=True)
            q=paired_landmarks(p,cfg);q.to_pickle(file);mark.write_text('complete')
        paired.append(q)
    paired=pd.concat(paired,ignore_index=True) if paired else pd.DataFrame()
    paired.to_csv(out/'paired_landmark_grid.csv',index=False)
    pd.DataFrame(completed).to_csv(checkpoint/'prediction_index.csv',index=False)
    v14_grid_failure_report(wide,audit,out)
    v14_update_status(out,status='complete_with_failures' if wide.Status.ne('complete').any() else 'complete',
        incomplete_rows=int(wide.Status.ne('complete').sum()),checkpoint_directory=str(checkpoint))
    return wide,metrics,paired,audit,learning,checkpoint


In [ ]:
v14_plans=v14_study_plan(CFG)
display(pd.DataFrame(v14_plans))


In [ ]:
initialize_study_folds(DATA_ROOT,CFG,OUTPUT_ROOT)
mimic_features,mimic_sequences,eicu_features,eicu_sequences,data_audit=prepare_v10_scenario(
    DATA_ROOT,OUTPUT_DIR,CFG)
if RUN_CRP_AUDIT:
    crp_unit_review=audit_crp_source_units(DATA_ROOT,OUTPUT_ROOT/'source_unit_review',CFG)
    display(crp_unit_review.groupby('decision').lab_rows.sum().rename('lab_rows').to_frame())

cnn_sampling_report=cnn_sampling_diagnostics(mimic_features,mimic_sequences,eicu_features,eicu_sequences,CFG,OUTPUT_DIR)

lab_support_report=summary_support_report(mimic_features,eicu_features,CFG,OUTPUT_DIR)


## Execute the tabular grid

No CNN is fitted by this cell. Existing exact-compatible base-model checkpoints are
reused. The top 50 rows sorted by eICU recall are displayed when the primary grid
finishes, before CNN training or sensitivity execution. Ranking external results is
descriptive, not an independent model-selection validation.
Checkpoints contain patient-level research data and must remain access-controlled.


In [ ]:
if RUN_GRID:
    feature_sampling_grid,grid_metrics_long,paired_grid,grid_fit_audit,grid_learning_curves,grid_checkpoint_dir=run_v14_tabular_grid(
        mimic_features,mimic_sequences,eicu_features,eicu_sequences,CFG,OUTPUT_DIR)
    manifest,fingerprint=input_manifest(DATA_ROOT,CFG)
    (OUTPUT_DIR/'run_manifest.json').write_text(json.dumps(dict(scenario='primary',
        inputs=manifest,lab_feature_fingerprint=fingerprint,patient_fold_digest=V10_FOLD_DIGEST,
        grid_plan=grid_plan(CFG),checkpoint_directory=str(grid_checkpoint_dir),
        status='complete_with_failures' if feature_sampling_grid.Status.ne('complete').any() else 'complete',
        incomplete_rows=int(feature_sampling_grid.Status.ne('complete').sum())),indent=2,default=str))
    calibration_summary,operating_points=finalist_diagnostics(feature_sampling_grid,grid_checkpoint_dir,CFG,OUTPUT_DIR)
    print(f'Primary finished: {len(feature_sampling_grid)} rows saved; {int(feature_sampling_grid.Status.ne("complete").sum())} incomplete. Rankings exclude incomplete fits.')
else:
    print('Features and diagnostics prepared. Set RUN_GRID=True to fit the grid.')

if RUN_GRID:
    display_v14_results(feature_sampling_grid.assign(Scenario='primary',Experiment_family='tabular'))


## Primary visual diagnostics and paired landmarks

Existing scatter, heatmap and sampling plots describe measured-value feature combinations.
Diagnostic controls remain in the exported grid. Paired landmark estimates compare shared
admissions, with patient-cluster uncertainty. First- and second-set training cohorts can differ,
so these estimates compare complete policies, not only the effect of extra measurements.


In [ ]:
if RUN_GRID:
    integrated_plots(feature_sampling_grid[feature_sampling_grid.Input_kind.eq('values_and_observation_patterns')],paired_grid,grid_learning_curves,OUTPUT_DIR)
    print('Paired first/second-set comparisons saved to paired_landmark_grid.csv. Use these for the time-point comparison.')


## Tabular lab-only diagnostics

These figures reuse weighted-natural (`none`) tabular predictions. Separate CNN
figures below use their explicitly labelled unweighted/prevalence-matched settings.
No fits are performed by either plotting cell.


In [ ]:
if RUN_GRID:
    lab_scopes={'lab_current':'current_only','lab_history':'history_only','lab_current+lab_history':'current_plus_history'}
    index=pd.read_csv(grid_checkpoint_dir/'prediction_index.csv')
    for population in index.population.unique():
        chosen=index[index.population.eq(population)&index.scope.isin(lab_scopes)&index.sampling.eq('none')]
        if chosen.empty:continue
        parts=[]
        for token in chosen.token:
            p=read_predictions(grid_checkpoint_dir/f'{token}.csv.gz')
            p=p[p.model.isin(['RandomForest','SummaryLR','LabCNN_Sequence','LabCNN_Summary'])].copy()
            p['scope']=p.scope.map(lab_scopes);parts.append(p)
        lab_predictions=pd.concat(parts,ignore_index=True)
        lab_metrics=grid_metrics_long[grid_metrics_long.population.eq(population)&grid_metrics_long.scope.isin(lab_scopes)&grid_metrics_long.sampling.eq('none')&grid_metrics_long.model.isin(['RandomForest','SummaryLR','LabCNN_Sequence','LabCNN_Summary'])].copy()
        lab_metrics['scope']=lab_metrics.scope.map(lab_scopes)
        lab_learning=grid_learning_curves.copy()
        if not lab_learning.empty:
            lab_learning=lab_learning[lab_learning.population.eq(population)&lab_learning.scope.isin(lab_scopes)].copy()
            lab_learning['scope']=lab_learning.scope.map(lab_scopes)
        destination=OUTPUT_DIR/population;destination.mkdir(parents=True,exist_ok=True)
        if set(lab_predictions.landmark)=={1,2}:
            plot_results(lab_predictions,lab_metrics,lab_learning,dict(CFG,sampling_modes=('none',),population_label=population),destination)
        else:print(population,': paired-landmark plots unavailable; inspect population_run_status.csv')

## Separate CNN comparisons

Two prespecified lab-only models at both landmarks; prevalence matching is a targeted third comparison. Raw features and patient-fold assignments are shared with the tabular study. Historical labs remain optional.


In [ ]:
if RUN_CNN:
    cnn_comparison_grid,cnn_metrics_long,cnn_paired,cnn_fit_audit,cnn_learning_curves,cnn_checkpoint_dir,cnn_predictions=run_v14_cnn_comparisons(
        mimic_features,mimic_sequences,eicu_features,eicu_sequences,CFG,OUTPUT_ROOT/'cnn_primary')
    display_v14_results(cnn_comparison_grid.assign(Scenario='primary',Experiment_family='CNN'))
    print('CNN diagnostics:', json.loads((OUTPUT_ROOT/'cnn_primary'/'run_manifest.json').read_text())['diagnostics_directory'])
    primary_tables=[cnn_comparison_grid.assign(Scenario='primary',Experiment_family='CNN')]
    if RUN_GRID:
        primary_tables.insert(0,feature_sampling_grid.assign(Scenario='primary',Experiment_family='tabular'))
    pd.concat(primary_tables,ignore_index=True).to_csv(OUTPUT_ROOT/'primary_combined_grid.csv',index=False)
else:
    print('Separate CNN experiment disabled; no CNN fits scheduled.')


In [ ]:
if RUN_CNN:
    plot_v14_cnn(cnn_predictions,cnn_learning_curves,OUTPUT_ROOT/'cnn_primary')

if RUN_CNN:
    _cnn_diag = v14_cnn_diagnostics(cnn_predictions, cnn_metrics_long, CFG, OUTPUT_ROOT/'cnn_primary')
    print('CNN descriptive calibration (no recalibration)')
    display(pd.read_csv(_cnn_diag/'calibration.csv').round(4))
    _cnn_op = pd.read_csv(_cnn_diag/'operating_points.csv')
    print('CNN operating points: thresholds chosen on MIMIC inner holdouts, then frozen')
    display(_cnn_op[['landmark','model','sampling','cohort','target_inner_recall','Recall','Recall_lo','Recall_hi','Specificity','PPV','Alert_rate']].round(4))
    from IPython.display import Image as _CNNImage
    for _cnn_plot in sorted(_cnn_diag.glob('*.png')):
        display(_CNNImage(filename=str(_cnn_plot)))


## Tabular sensitivity execution and combined reporting

The fixed-anchor release-lag, all-ages and core-panel sensitivities refit tabular
models only. They do not establish robustness of the CNN to those policies.
Primary CNN outputs are retained in the combined study table as a separate
Experiment_family; all rows retain their population, scope and sampling labels.


In [ ]:
if RUN_GRID:
    for variable in ('mimic_features','mimic_sequences','eicu_features','eicu_sequences','cnn_predictions'):
        globals().pop(variable,None)
    ICD_RAW.clear();ICD_SELECTION_CACHE.clear();gc.collect()
    sensitivity_grid=run_v14_sensitivities(DATA_ROOT,OUTPUT_ROOT,CFG,feature_sampling_grid,grid_checkpoint_dir)
    primary_tables=[feature_sampling_grid.assign(Scenario='primary',Experiment_family='tabular')]
    if RUN_CNN:
        primary_tables.append(cnn_comparison_grid.assign(Scenario='primary',Experiment_family='CNN'))
    study_grid=pd.concat(primary_tables+[sensitivity_grid],ignore_index=True)
    study_grid.to_csv(OUTPUT_ROOT/'all_scenario_grid.csv',index=False)
    display_v14_results(study_grid[study_grid.Scenario.eq(DISPLAY_SCENARIO)])
    print('Study finished. All rows and explicit failure statuses saved; only complete rows are ranked.')


## Design sources and interpretation

Dataset timing/definitions: [MIMIC LABEVENTS](https://mimic.mit.edu/docs/iii/tables/labevents.html), [MIMIC item dictionary](https://physionet.org/content/mimiciii-demo/1.4/D_LABITEMS.csv), [eICU lab offsets](https://eicu.mit.edu/eicutables/lab/), [eICU hospital identifiers and dates](https://eicu.mit.edu/eicutables/patient/).

Clinical time-series research motivates explicit observation masks and elapsed-time information: [Che et al., GRU-D](https://www.nature.com/articles/s41598-018-24271-9). This notebook adapts those input concepts to a small CNN; it is not an implementation or validation of GRU-D. [Harutyunyan et al.](https://arxiv.org/abs/1703.07771) provides a clinical time-series benchmarking precedent, not proof that a CNN should outperform summary models for BE. [Tomašev et al.](https://www.nature.com/articles/s41586-019-1390-1) illustrates temporally anchored clinical prediction with longitudinal records; its disease/task differs.

Published current-admission BE work includes [SABIER](https://academic.oup.com/jid/article/230/3/606/7616139) and [MEFIER](https://academic.oup.com/ofid/article/12/6/ofaf287/8128787). Their eligibility, outcome and timing choices are not assumed equivalent to this experiment. No cross-paper metric comparison is made here.

Reference intervals: [Mayo CBC](https://www.mayocliniclabs.com/test-catalog/overview/9109), [Mayo CMP](https://www.mayocliniclabs.com/api/sitecore/TestCatalog/DownloadTestCatalog?testId=113631), [Mayo CRP](https://www.mayocliniclabs.com/api/sitecore/TestCatalog/DownloadTestCatalog?testId=800164), [Bronson ESR](https://bronsonlab.testcatalog.org/show/ESR-or-LAB322), [Mayo magnesium](https://www.mayocliniclabs.com/api/sitecore/TestCatalog/DownloadTestCatalog?testId=8448), [Mayo phosphate](https://www.mayocliniclabs.com/api/sitecore/TestCatalog/DownloadTestCatalog?testId=8408), [Central Peninsula LDH](https://cphlab.testcatalog.org/show/LDH), [Mayo PTT](https://www.mayocliniclabs.com/api/sitecore/TestCatalog/DownloadTestCatalog?testId=40935), [Marshfield INR](https://marshfieldlabs.testcatalog.org/show/PT), [Mayo lactate](https://prd1.mayocliniclabs.com/test-catalog/overview/601685). Exact feature units, bounds and source links are exported with the run.

Before presentation: inspect age composition, case counts, missingness, unit exclusions and scoring times; examine how conclusions change under plausible release-lag and eligibility sensitivities. Code-based BE labels, incomplete historical coverage, treatment effects and approximate result availability remain substantive limitations. New-cohort results must not be described as a like-for-like increase over the prior historical-only task.

## Evidence ledger
This is a targeted evidence map, not a systematic review or meta-analysis. A screened publication is not necessarily an independent cohort. Depth labels distinguish complete main-methods review from abstract-only/access-limited review. No cross-disease AUROC ranking establishes a universally best summary.

- [Utility of serum C-reactive protein in assessing the outcome of infective endocarditis](https://pubmed.ncbi.nlm.nih.gov/15855194/) (2005); Abstract only.. Use serial findings for landmark prognosis, not admission diagnosis or proof of an optimal summary.

- [Prognostic value of serial C-reactive protein measurements in left-sided native valve endocarditis](https://pubmed.ncbi.nlm.nih.gov/18268172/) (2008); Abstract only.. Separate surgery, complications and mortality targets; do not infer that percentage decline outperforms absolute CRP.

- [Baseline C-reactive protein levels and prognosis in patients with infective endocarditis: A prospective cohort study](https://pmc.ncbi.nlm.nih.gov/articles/PMC6309136/) (2018); Full methods and results inspected; PMC XML.. Do not encode as a single admission CRP or resolve the aggregation discrepancy silently.

- [The impact of admission neutrophil-to-platelet ratio on in-hospital and long-term mortality in patients with infective endocarditis](https://pubmed.ncbi.nlm.nih.gov/27987356/) (2017); Abstract only.. Candidate baseline ratio; association does not establish superiority over its component counts.

- [The VIRSTA score, a prediction score to estimate risk of infective endocarditis and determine priority for echocardiography in patients with Staphylococcus aureus bacteremia](https://pubmed.ncbi.nlm.nih.gov/26916042/) (2016); Abstract only.. Relevant diagnostic baseline; prevent 48-hour information entering an admission-only model.

- [Who needs a blood culture? A prospectively derived and validated prediction rule](https://pubmed.ncbi.nlm.nih.gov/18486413/) (2008); Abstract only.. Use as a clinically interpretable bacteremia benchmark; preserve culture-order selection and distinguish diagnosis from mortality.

- [Relation between lymphopenia and bacteraemia in UK adults with medical emergencies](https://pubmed.ncbi.nlm.nih.gov/15333656/) (2004); Abstract only; full-text XML retrieval access-limited by server error.. Retain differential counts alongside WBC; this compares analytes, not time summaries.

- [Lymphocytopenia and neutrophil-lymphocyte count ratio predict bacteremia better than conventional infection markers in an emergency care unit](https://pmc.ncbi.nlm.nih.gov/articles/PMC3219299/) (2010); Full methods and abstract inspected; PMC XML.. Compare NLR with component counts; recalibrate in the intended population.

- [C-reactive protein and albumin kinetics after antibiotic therapy in community-acquired bloodstream infection](https://pubmed.ncbi.nlm.nih.gov/32251802/) (2020); Abstract only.. Candidate day-4 CRP ratio plus baseline albumin; not an admission feature.

- [Prediction Rules for Ruling Out Endocarditis in Patients With Staphylococcus aureus Bacteremia](https://pmc.ncbi.nlm.nih.gov/articles/PMC9049276/) (2022); Full methods, score table and abstract inspected; PMC XML.. Evaluate sensitivity, calibration and testing burden at the correct prediction time.

- [C-reactive protein, an early marker of community-acquired sepsis resolution: a multi-center prospective observational study](https://pmc.ncbi.nlm.nih.gov/articles/PMC3387609/) (2011); Full methods and results inspected; PMC XML.. Test baseline plus trajectory at landmarks; association differences do not prove predictive-summary superiority.

- [The time course of blood C-reactive protein concentrations in relation to the response to initial antimicrobial therapy in patients with sepsis](https://pubmed.ncbi.nlm.nih.gov/18463788/) (2008); Abstract only.. CRP change partly reflects treatment adequacy; retain antibiotic/source-control timing.

- [White blood cell count trajectory and mortality in septic shock: a historical cohort study](https://pmc.ncbi.nlm.nih.gov/articles/PMC9334545/) (2022); Full methods and results inspected; PMC XML.. Do not assign seven-day trajectory classes at admission; compare simpler landmark features.

- [Thrombocytopenia in adult patients with sepsis: incidence, risk factors, and its association with clinical outcome](https://pmc.ncbi.nlm.nih.gov/articles/PMC4373028/) (2013); Full methods and abstract inspected; PMC XML.. Daily minimum is documented usage, not evidence it outperforms first/mean/last platelets.

- [Early lactate clearance is associated with improved outcome in severe sepsis and septic shock](https://pubmed.ncbi.nlm.nih.gov/15286537/) (2004); Abstract only.. Treat clearance as a post-treatment change feature; do not transfer the 10% threshold as universally optimal.

- [Serum lactate as a predictor of mortality in emergency department patients with infection](https://pubmed.ncbi.nlm.nih.gov/15855951/) (2005); Abstract only.. Include baseline lactate but avoid treating a low value as sufficient exclusion of deterioration.

- [Serum lactate as a predictor of mortality in patients with infection](https://pubmed.ncbi.nlm.nih.gov/17431582/) (2007); Abstract only.. Preserve baseline lactate and selection context; no evidence here for averaging or clearance superiority.

- [Whole blood lactate kinetics in patients undergoing quantitative resuscitation for severe sepsis and septic shock](https://pubmed.ncbi.nlm.nih.gov/23740148/) (2013); Abstract only; PMC/XML full-text access attempts unsuccessful.. Compare normalization, change and baseline locally; exact clearance-rate formula remains unverified.

- [Lactate clearance for death prediction in severe sepsis or septic shock patients during the first 24 hours in Intensive Care Unit: an observational study](https://pmc.ncbi.nlm.nih.gov/articles/PMC3614435/) (2013); Full methods and abstract inspected; PMC XML.. Align clocks and available information before comparing admission values with clearance.

- [Serum lactate is associated with mortality in severe sepsis independent of organ failure and shock](https://pubmed.ncbi.nlm.nih.gov/19325467/) (2009); Abstract only.. Baseline lactate may contribute beyond overt shock; assess incremental performance directly rather than infer it from significance.

- [C-reactive protein is an independent predictor of severity in community-acquired pneumonia](https://pubmed.ncbi.nlm.nih.gov/18328306/) (2008); Abstract only.. Use separate admission and day-4 models; differing associations do not demonstrate superior aggregation.

- [Usefulness of consecutive C-reactive protein measurements in follow-up of severe community-acquired pneumonia](https://pubmed.ncbi.nlm.nih.gov/18508833/) (2008); Abstract only.. Retain steroid exposure and antibiotic adequacy; avoid equating biomarker decline with untreated disease severity.

- [Usefulness of C-reactive protein in monitoring the severe community-acquired pneumonia clinical course](https://pmc.ncbi.nlm.nih.gov/articles/PMC2206486/) (2007); Full methods and abstract inspected; PMC XML.. Distinguish relative change from ratio-to-baseline; reproduce only using observations available at prediction time.

- [Markers of treatment failure in hospitalised community acquired pneumonia](https://pubmed.ncbi.nlm.nih.gov/18245147/) (2008); Abstract only.. Do not use day-3 measurements to predict failures already occurring by 72 hours.

- [C-reactive protein. A clinical marker in community-acquired pneumonia](https://pubmed.ncbi.nlm.nih.gov/7587431/) (1995); Abstract only.. Record pre-admission antibiotic exposure; admission is not necessarily a pretreatment baseline.

- [Serum Creatinine Trajectories for Community- versus Hospital-Acquired Acute Kidney Injury](https://pmc.ncbi.nlm.nih.gov/articles/PMC5490993/) (2016); full_methods. Retain extrema timestamps and direction; restrict every feature to the prediction cutoff.

- [Acute kidney injury subphenotypes based on creatinine trajectory identifies patients at increased risk of death](https://pmc.ncbi.nlm.nih.gov/articles/PMC5112626/) (2016); full_methods. Evaluate recovery alongside severity at an explicit 72-hour landmark.

- [Commonly used surrogates for baseline renal function affect the classification and prognosis of acute kidney injury](https://pmc.ncbi.nlm.nih.gov/articles/PMC2929703/) (2010); access_limited: complete PMC/PubMed abstracts; full-method retrieval failed.. Preserve baseline source and age; sensitivity-test baseline definitions.

- [The comparison of the commonly used surrogates for baseline renal function in acute kidney injury diagnosis and staging](https://pubmed.ncbi.nlm.nih.gov/26748909/) (2016); abstract_only. Do not pool measured and imputed baseline deltas without provenance.

- [Clinically meaningful eGFR slope as a surrogate endpoint differs across CKD stages and slope evaluation periods: the CKD-JAC study](https://pmc.ncbi.nlm.nih.gov/articles/PMC11822292/) (2025); full_methods. Treat observation duration and baseline disease severity as determinants of slope reliability.

- [Variability in eGFR and the Risk of Adverse Kidney Outcomes and All-Cause Mortality: A Retrospective Observational Study Based on National General Practice Dataset](https://pmc.ncbi.nlm.nih.gov/articles/PMC12935336/) (2026); full_methods. Test variability alongside level/slope; separate composite components.

- [Recovery of kidney function after acute kidney disease—a multi-cohort analysis](https://pubmed.ncbi.nlm.nih.gov/37573145/) (2024); abstract_only. Distinguish lookback length, transient normalization and sustained recovery.

- [Comparison of Static and Dynamic Baseline Creatinine Surrogates for Defining Acute Kidney Injury](https://pubmed.ncbi.nlm.nih.gov/34419950/) (2021); abstract_only. Candidate direct comparison supporting a causal-time rolling baseline; verify full implementation before reuse.

- [The Lille model: a new tool for therapeutic strategy in patients with severe alcoholic hepatitis treated with steroids](https://pubmed.ncbi.nlm.nih.gov/17518367/) (2007); abstract_only. Combine baseline state with response change; encode treatment-relative timing.

- [Prospective derivation and validation of early dynamic model for predicting outcome in patients with acute liver failure](https://pubmed.ncbi.nlm.nih.gov/22337947/) (2012); abstract_only. Persistence/crossing are candidate summaries, not evidence that generic slopes dominate.

- [A dynamic prediction model for prognosis of acute-on-chronic liver failure based on the trend of clinical indicators](https://www.nature.com/articles/s41598-021-81431-0) (2021); full_methods. Compare summaries at matched landmarks and information availability.

- [Blood lactate as an early predictor of outcome in paracetamol-induced acute liver failure: a cohort study](https://www.sciencedirect.com/science/article/pii/S0140673602077437) (2002); abstract_only: complete publisher summary.. Record intervention-relative timestamps; distinguish repeated levels from computed change.

- [Trajectory of Serum Bilirubin Predicts Spontaneous Recovery in a Real-World Cohort of Patients With Alcoholic Hepatitis](https://pmc.ncbi.nlm.nih.gov/articles/PMC9188123/) (2022); access_limited: abstract and indexed methods/results passages.. Retain continuous ratios; resolve source inconsistencies before implementing categorical thresholds.

- [Comparison Between Dynamic Models for Predicting Response to Corticosteroids in Alcohol-Associated Hepatitis: A Global Cohort Study](https://pmc.ncbi.nlm.nih.gov/articles/PMC11950807/) (2025); full_methods. Report uncertainty and matched timing; no established universal day-seven superiority.

- [Variability of blood glucose concentration and short-term mortality in critically ill patients](https://pubmed.ncbi.nlm.nih.gov/16871057/) (2006); abstract_only. Candidate mean-plus-variability representation; test transportability to sparse labs.

- [Increased Glycemic Variability Is Independently Associated With Length of Stay and Mortality in Noncritically Ill Hospitalized Patients](https://pmc.ncbi.nlm.nih.gov/articles/PMC3836112/) (2013); full_methods. Compare reading-weighted versus day-weighted means; retain hypoglycemia separately.

- [Diabetic status and the relation of the three domains of glycemic control to mortality in critically ill patients: an international multicenter cohort study](https://pubmed.ncbi.nlm.nih.gov/23452622/) (2013); abstract_only. Evaluate disease-context interactions rather than assuming identical variability effects.

- [Effect of in-hospital glycemic variability on mortality in patients with diabetes](https://pubmed.ncbi.nlm.nih.gov/32646753/) (2021); abstract_only. Test level and variability jointly; do not import thresholds uncritically.

- [Visit-to-Visit Glycemic Variability and Risks of Cardiovascular Events and All-Cause Mortality: The ALLHAT Study](https://pubmed.ncbi.nlm.nih.gov/30659073/) (2019); abstract_only; full-text retrieval unsuccessful.. Separate long-term variability from short-term swings; retain count and time span.

- [Impact of visit-to-visit glycemic variability on the risks of macrovascular and microvascular events and all-cause mortality in type 2 diabetes: the ADVANCE trial](https://pubmed.ncbi.nlm.nih.gov/24812434/) (2014); abstract_only. Use nonoverlapping exposure/outcome windows; distinguish analyte-specific chronic variability.

- [Trajectories of Serum Sodium on In-Hospital and 1-Year Survival among Hospitalized Patients](https://pmc.ncbi.nlm.nih.gov/articles/PMC7269204/) (2020); access_limited: complete abstract and indexed methods/results passages.. Test observation-count sensitivity and fixed-window trajectories; avoid whole-stay admission predictors.

- [Sodium Fluctuations and Mortality in a General Hospitalized Population](https://pubmed.ncbi.nlm.nih.gov/31330509/) (2019); abstract_only. Keep baseline excursion distinct from range and SD; test at fixed landmarks.

- [Serum potassium variability as a predictor of clinical outcomes in patients with cardiorenal disease or diabetes: a retrospective UK database study](https://pubmed.ncbi.nlm.nih.gov/35371436/) (2022); abstract_only. Retain negative findings; do not equate variability with universal mortality risk.

- [Dysnatremia is an Independent Indicator of Mortality in Hospitalized Patients](https://pmc.ncbi.nlm.nih.gov/articles/PMC5447666/) (2017); full_methods. Use only past persistence at a defined landmark; document reference ranges.

- [The relationship between serum potassium, potassium variability and in-hospital mortality in critically ill patients and a before-after analysis on the impact of computer-assisted potassium control](https://pmc.ncbi.nlm.nih.gov/articles/PMC4312438/) (2015); full_methods. Separate early/late windows; check specimen artifacts before constructing extrema.

- [Red cell distribution width as a novel prognostic marker in heart failure: data from the CHARM Program and the Duke Databank](https://pubmed.ncbi.nlm.nih.gov/17601544/) (2007); abstract_only. Keep RDW as a candidate level feature; effect scaling by SD is not longitudinal variability.

- [Progressive rise in red cell distribution width is associated with disease progression in ambulatory patients with chronic heart failure](https://pmc.ncbi.nlm.nih.gov/articles/PMC3272251/) (2012); full_methods. Use explicitly timed differences; report repeat-measurement eligibility and baseline-dependent effects.

- [Relation between changes in red cell distribution width and clinical outcomes in acute decompensated heart failure](https://pubmed.ncbi.nlm.nih.gov/22560496/) (2013); abstract_only. Threshold transitions are candidates only after their observation window closes.

- [Timing of hemoconcentration during treatment of acute decompensated heart failure and subsequent survival: importance of sustained decongestion](https://pmc.ncbi.nlm.nih.gov/articles/PMC3892152/) (2013); full_methods. Peak timing normalized by completed stay is unavailable for admission prediction.

- [Prognostic impact of hemoglobin drop during hospital stay in patients with acute coronary syndromes](https://pubmed.ncbi.nlm.nih.gov/19634496/) (2009); abstract_only. Retain baseline and observed minimum separately; truncate nadir calculation at prediction time.

- [Relation of Baseline Hemoglobin Levels and Adverse Events in Patients With Acute Coronary Syndromes (from the Acute Catheterization and Urgent Intervention Triage strategY and Harmonizing Outcomes with RevasculariZatiON and Stents in Acute Myocardial Infarction Trials)](https://pubmed.ncbi.nlm.nih.gov/28388994/) (2017); abstract_only. Evaluate nonlinear Hb effects without assuming monotonic risk.

- [Temporal dynamics of peripheral neutrophil and lymphocytes following acute ischemic stroke](https://pmc.ncbi.nlm.nih.gov/articles/PMC7898194/) (2019); full_methods. Anchor samples to disease onset; later measurements answer a later prediction question.

- [Development and validation of a predictive model for chemotherapy-associated thrombosis](https://pubmed.ncbi.nlm.nih.gov/18216292/) (2008); abstract_only. Use as a threshold-feature precedent, not proof cutoffs outperform continuous labs.

- [A clinical prediction model for cancer-associated venous thromboembolism: a development and validation study in two independent prospective cohorts](https://pmc.ncbi.nlm.nih.gov/articles/PMC7338218/) (2018); full_methods. Log2 transformation is reproducible; retain assay metadata and compare representations within one cohort.

- [Risk assessment of recurrence in patients with unprovoked deep vein thrombosis or pulmonary embolism: the Vienna prediction model](https://pubmed.ncbi.nlm.nih.gov/20351233/) (2010); abstract_only. Keep post-treatment measurement context; do not count the update as independent replication.

- [D-dimer levels over time and the risk of recurrent venous thromboembolism: an update of the Vienna prediction model](https://pmc.ncbi.nlm.nih.gov/articles/PMC3959721/) (2014); full_methods. Dynamic landmark modeling uses updated measurements; descriptive random slopes are not necessarily predictive features.

- [Usefulness of repeated D-dimer testing after stopping anticoagulation for a first episode of unprovoked venous thromboembolism: the PROLONG II prospective study](https://pubmed.ncbi.nlm.nih.gov/19965693/) (2010); abstract_only. Define usable persistence features using past measurements only.

- [Hematological indices for differential diagnosis of Beta thalassemia trait and iron deficiency anemia](https://pubmed.ncbi.nlm.nih.gov/24818016/) (2014); abstract_only. Genuine head-to-head transformation evidence, but diagnostic and population-specific; not evidence for temporal-summary superiority.

- [A new index to discriminate between iron deficiency anemia and thalassemia trait](https://pmc.ncbi.nlm.nih.gov/articles/PMC4997903/) (2016); full_methods. Reproduce units and reference diagnosis; distinguish algebraic composites from serial summaries.

- [Diagnostic performance of hematological discrimination indices to discriminate between βeta thalassemia trait and iron deficiency anemia and using cluster analysis: Introducing two new indices tested in Iranian population](https://pmc.ncbi.nlm.nih.gov/articles/PMC6901548/) (2019); full_methods. Compare formulas on held-out patients; avoid transporting cohort-optimized thresholds untested.

- [SMART-COP: a tool for predicting the need for intensive respiratory or vasopressor support in community-acquired pneumonia](https://pubmed.ncbi.nlm.nih.gov/18558884/) (2008); abstract_only. Thresholds can support a validated combined score without proving thresholds outperform continuous values.

- [Neutrophil-to-lymphocyte ratio, blood eosinophils and COPD exacerbations: a cohort study](https://pmc.ncbi.nlm.nih.gov/articles/PMC8711083/) (2021); full_methods. Log transformation was used for reliability analysis, not demonstrated predictive improvement; retain ratio components.

- [The clinical significance of changes in red blood cell distribution width in patients with community-acquired pneumonia](https://pmc.ncbi.nlm.nih.gov/articles/PMC5065335/) (2016); full_methods. Positive change means declining RDW here. Use a day4 landmark; significance does not establish validated incremental prediction.

- [Association of Red Blood Cell Distribution Width With Mortality Risk in Hospitalized Adults With SARS-CoV-2 Infection](https://pmc.ncbi.nlm.nih.gov/articles/PMC7512057/) (2020); full_methods. RDW delta is percentage points. Group trajectory interpolation is descriptive, not a validated individual mean feature.

- [An interpretable mortality prediction model for COVID-19 patients](https://www.nature.com/articles/s42256-020-0180-7) (2020); access_limited. Do not implement an assumed latest/mean/maximum aggregation from this abstract.

- [Red cell distribution width as a prognostic marker in patients with community-acquired pneumonia](https://pubmed.ncbi.nlm.nih.gov/22867824/) (2013); abstract_only. Test incremental value against the clinical baseline; do not equate quartile associations with optimal discretization.

- [Time-to-Death Longitudinal Characterization of Clinical Variables and Longitudinal Prediction of Mortality in COVID-19 Patients: A Two-Center Study](https://www.frontiersin.org/journals/medicine/articles/10.3389/fmed.2021.661940/full) (2021); full_methods. SD/mean was a group-comparison statistic, not an established predictive feature; use prospective landmarks.

- [Longitudinal Profile of Laboratory Parameters and Their Application in the Prediction for Fatal Outcome Among Patients Infected With SARS-CoV-2: A Retrospective Cohort Study](https://pubmed.ncbi.nlm.nih.gov/33048116/) (2021); abstract_only. Specify phase anchors and available-at-prediction measurements before reproducing features.

- [Prognosing the risk of COVID-19 death through a machine learning-based routine blood panel: A retrospective study in Brazil](https://pmc.ncbi.nlm.nih.gov/articles/PMC9327247/) (2022); full_methods. Anchor to the actual prediction timestamp, not an assumed admission window; feature importance is not transformation superiority.

- [Machine learning based early warning system enables accurate mortality risk prediction for COVID-19](https://www.nature.com/articles/s41467-020-18684-2) (2020); full_methods. Useful explicit first-value baseline; transformation use alone does not prove its benefit.

- [Multitask learning and benchmarking with clinical time series data](https://pmc.ncbi.nlm.nih.gov/articles/PMC6572845/) (2019); Methods-detail/full text: preprocessing, tasks, baselines and evaluation.. Reproduce matched-window summary and sequence baselines; no isolated proof that any summary defeats first values.

- [An evaluation of time series summary statistics as features for clinical prediction tasks](https://pmc.ncbi.nlm.nih.gov/articles/PMC7059727/) (2020); Methods-detail/full text: selection, statistics, RF and genetic search.. Retain first-value baseline; this study cannot support a blanket first-value-never claim.

- [Learning Predictive and Interpretable Timeseries Summaries from ICU Data](https://pmc.ncbi.nlm.nih.gov/articles/PMC8861716/) (2021); Methods-detail/indexed primary full-text excerpts: cohort, functions, experiments, Table 3; direct access limited.. Test learned or prespecified lookbacks only where repeat observations exist.

- [Multi-task Prediction of Disease Onsets from Longitudinal Laboratory Tests](https://proceedings.mlr.press/v56/Razavian16.html) (2016); Methods-detail/full PMLR paper: model definitions, experiments and appendix.. CNN superiority cannot be assumed for sparse, small-sample landmarks.

- [Recurrent Neural Networks for Multivariate Time Series with Missing Values](https://pmc.ncbi.nlm.nih.gov/articles/PMC5904216/) (2018); Methods-detail/full text: architecture, datasets and ablations.. Include count, observed flag and time-since-measurement baselines before complex sequences.

- [Temporal convolutional networks allow early prediction of events in critical care](https://pmc.ncbi.nlm.nih.gov/articles/PMC7647248/) (2020); Methods-detail/indexed primary full-text excerpts: lab table, missingness, comparators and evaluation; supplement uninspected.. History benefits are task-specific, not evidence against first-panel prediction.

- [Temporal convolutional networks and data rebalancing for clinical length of stay and mortality prediction](https://www.nature.com/articles/s41598-022-25472-z) (2022); Methods-detail/full text: extraction, models, balancing and validation.. Evaluate calibration and thresholds; no universal TCN winner.

- [Temporal Pointwise Convolutional Networks for Length of Stay Prediction in the Intensive Care Unit](https://dl.acm.org/doi/10.1145/3450439.3451860) (2021); Methods-detail/full publisher-formatted author reprint, including ablation tables.. Time and cross-analyte interactions are separable hypotheses, requiring adequate repeated data.

- [Biases in electronic health record data due to processes within the healthcare system: retrospective observational study](https://www.bmj.com/content/361/bmj.k1479) (2018); Access-limited: complete abstract and indexed cohort-method excerpts, not complete analytic methods.. Repeat testing may encode care intensity rather than biological variability.

- [Democratizing EHR analyses with FIDDLE: a flexible data-driven preprocessing pipeline for structured clinical data](https://pmc.ncbi.nlm.nih.gov/articles/PMC7727385/) (2020); Methods-detail/full primary XML: variable frequency, encoding, experiments and validation.. Do not attribute vital-sign min/max/mean processing to every sparse lab.

- [Early Prediction of Sepsis From Clinical Data: The PhysioNet/Computing in Cardiology Challenge 2019](https://pmc.ncbi.nlm.nih.gov/articles/PMC6964870/) (2020); Methods-detail/full primary XML: variables, cohort, scoring and testing.. Evaluate transfer and alarm/utility metrics, not only discrimination.

- [Incorporating informatively collected laboratory data from EHR in clinical prediction models](https://link.springer.com/article/10.1186/s12911-024-02612-1) (2024); Methods-detail/full text: missingness representations, model comparisons and results.. Treat missingness handling as a separately validated modeling choice.

- [HiRID-ICU-Benchmark — A Comprehensive Machine Learning Benchmark on High-resolution ICU Data](https://datasets-benchmarks-proceedings.neurips.cc/paper_files/paper/2021/file/5878a7ab84fb43402106c575658472fa-Paper-round1.pdf) (2021); Methods-detail/full paper: preprocessing, features, splits and model comparisons.. Include nonlinear tabular models with summaries as serious comparators.

- [Yet Another ICU Benchmark: A Flexible Multi-Center Framework for Clinical ML](https://proceedings.iclr.cc/paper_files/paper/2024/file/c26a89073d972f2e6643617b0f3a9e8a-Paper-Conference.pdf) (2024); Methods-detail/full paper: experimental setup, preprocessing, input table and external-validation design.. Lock cohort, label, horizon and preprocessing before comparing architectures.

- [Set Functions for Time Series](https://proceedings.mlr.press/v119/horn20a.html) (2020); Methods-detail/full PMLR paper: representation, datasets and online evaluation.. Audit interpolation and sequence normalization for future-information leakage.

- [Benchmarking machine learning models on multi-centre eICU critical care dataset](https://journals.plos.org/plosone/article?id=10.1371/journal.pone.0235424) (2020); Methods-detail/full publisher text: cohort, preprocessing, tasks and comparisons.. Separate feature-set effects from temporal architecture effects.

- [Learning to Detect Sepsis with a Multitask Gaussian Process RNN Classifier](https://proceedings.mlr.press/v70/futoma17a.html) (2017); Methods-detail/full PMLR paper: GP/RNN construction, lab inventory and experiments.. Imputation uncertainty matters, but complex interpolation needs repeat-information justification.

- [Machine learning for early detection of sepsis: an internal and temporal validation study](https://pmc.ncbi.nlm.nih.gov/articles/PMC7382639/) (2020); Abstract-only: complete PubMed/Europe PMC abstract; full methods inaccessible.. Require later-period validation; do not infer exact aggregators or summary superiority.

- [Directly Modeling Missing Data in Sequences with RNNs: Improved Classification of Clinical Time Series](https://proceedings.mlr.press/v56/Lipton16.html) (2016); Methods-detail/full PMLR paper: preprocessing, summary baselines and missingness ablations.. Ablate observation-process features separately from measured values.

- [MIMIC-Extract: A Data Extraction, Preprocessing, and Representation Pipeline for MIMIC-III](https://doi.org/10.1145/3368555.3384469) (2020); Methods-detail/full publisher-formatted author reprint: pipeline and experiments.. Keep semantic harmonization distinct from temporal summarization.

- [Scalable and accurate deep learning with electronic health records](https://www.nature.com/articles/s41746-018-0029-1) (2018); Methods-detail/full publisher text: representation, outcomes, timing and model development.. Do not cite whole-EHR gains as evidence against simple lab summaries.

- [Early prediction of circulatory failure in the intensive care unit using machine learning](https://www.nature.com/articles/s41591-020-0789-4) (2020); Methods-detail/indexed primary full-text excerpts; direct publisher access limited.. Test simple imputation and compact feature baselines; distinguish label construction from predictor processing.

- [Dynamic survival prediction in intensive care units from heterogeneous time series without the need for variable selection or curation](https://www.nature.com/articles/s41598-020-79142-z) (2020); Methods-detail/full publisher text: quantization, architecture and ablations.. Robust encoding and temporal-history effects should be tested separately.

- [Feature Robustness in Non-stationary Health Records: Caveats to Deployable Model Performance in Common Clinical Machine Learning Tasks](https://proceedings.mlr.press/v106/nestor19a.html) (2019); Methods-detail/full PMLR paper: feature mapping, cohorts and temporal designs.. Audit analyte identity, units and source changes before attributing drift to physiology.

- [Early Recognition of Sepsis with Gaussian Process Temporal Convolutional Networks and Dynamic Time Warping](https://proceedings.mlr.press/v106/moor19a.html) (2019); Methods-detail/full PMLR paper: cohort, modeling and lab/imputation appendices.. Use matched preprocessing and realistic event prevalence when comparing temporal models.
